HALP-Bench — Google Colab Free / T4 Optimized Supervised Probing

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1vCgkXoV-lEvzuL_BfLzkkUOMtMHXJMA7

## 1. Scope, integrity rules, and execution modes

The core experiment is **pre-generation**: the VLM receives only the image and the original benchmark question. The notebook does not call `generate()` during feature extraction.

Two execution modes are provided:

```python
RUN_MODE = "smoke_test"   # "smoke_test" or "full"
MAX_SAMPLES = 32          # None for full data after the validation gate
```

A smoke test is not reported as a full benchmark result. Full results are only written when the corresponding computation actually ran.

The notebook keeps these separate:
1. **benchmark facts** from the official sources,
2. **local Drive observations** produced by this run,
3. **new experimental results** produced by this run.


The production T4 profile keeps a 13.5 GiB CUDA safety ceiling, uses adaptive Qwen visual-token clamping for grid-rounding edge cases, and reuses Qwen raw VF from the same multimodal forward so it does not need a second vision pass.

FINAL REVISION NOTE (2026-08-31): fixed the observed BatchNorm singleton-batch failure in the Qwen MLP refit, recompute final-refit normalization using train+validation only, and release the validation model before the final refit.


In [1]:

# Optimization note: these changes target Google Colab Free + NVIDIA T4 (~16 GB VRAM) limits on runtime, memory, and session stability.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

OPTIMIZATION_PROFILE_VERSION = "t4-colab-free-v5-qwen-adaptive-min-runtime-aware"

# ------------------------------------------------------------
# 1. Research configuration
# ------------------------------------------------------------
# Full mode is the intended research run. Set RUN_MODE="smoke_test" only
# for structural validation before starting a full extraction.
# Colab-Free/T4 profile: preserve both research models while bounding sample count and visual/I/O overhead.
RUN_MODE = "full"              # "smoke_test" or "full"
MAX_SAMPLES = 8000            # Deterministic cap per model; resumable on Colab Free/T4
SUPERVISED_ENDPOINT_SAMPLE_POLICY = "same fixed sampled rows as feature extraction"

SAMPLE_SELECTION_SEED = 42     # Deterministic row-sampling seed per model
# Explicitly distinguish a capped "full" execution mode from a full 10k-row benchmark.
# 8000 rows/model is a deliberate compromise: broader coverage without changing
# the batch=1 VLM extraction or the existing conservative T4 VRAM budget.
EXPERIMENT_DATA_SCOPE = (
    f"capped_deterministic_subset_{MAX_SAMPLES}_per_model"
    if MAX_SAMPLES is not None
    else "all_available_rows"
)
SEEDS = [42, 52, 62]
CHECKPOINT_EVERY = 500
EXTRACTION_LOG_EVERY = CHECKPOINT_EVERY
EXTRACTION_GC_EVERY = CHECKPOINT_EVERY

# Qwen2.5-VL visual-token budget.
# The processor's 28-pixel divisibility can round a nominal max-pixel budget
# slightly upward. The extraction path therefore keeps the documented
# 256-image-token target and has an adaptive preprocessing fallback that
# reduces only the affected sample until its actual grid is within the cap.
QWEN_TARGET_VISUAL_TOKENS = 256
# Allow rare processor-rounding outliers to fall below the normal target during
# adaptive retry while keeping the normal production ceiling at 256 visual tokens.
QWEN_MIN_VISUAL_TOKENS = 128
QWEN_MIN_PIXELS = QWEN_MIN_VISUAL_TOKENS * 28 * 28
QWEN_MAX_PIXELS = QWEN_TARGET_VISUAL_TOKENS * 28 * 28
QWEN_MAX_VISION_PATCHES = QWEN_TARGET_VISUAL_TOKENS * 4

# Operational CUDA safety policy. Accelerate's max_memory is a placement
# budget, not a guarantee for transient forward allocations, so the
# extraction loop also checks real CUDA usage against the hard ceiling.
T4_HARD_CUDA_GIB = 13.5
T4_PREFERRED_PEAK_CUDA_GIB = 13.0
T4_TOTAL_RUNTIME_TARGET_HOURS = 6.0
T4_RUNTIME_WARNING_FRACTION = 0.80
T4_MODEL_GPU_BUDGET_GIB = {
    "smolvlm2": 11.0,
    "qwen25vl": 11.0,
}
T4_ENFORCE_MEMORY_GUARD = True
# Runtime is observed from the extraction loop; this is a planning target, not
# a fabricated benchmark guarantee. Extraction remains resumable.

# Keep the notebook's intended SmolVLM2 production preprocessing profile.
SMOL_T4_IMAGE_EDGE = 384
SMOL_T4_IMAGE_SPLITTING = False

# Two T4-oriented VLMs. Qwen is intentionally the 3B model, not the 7B
# checkpoint, because the notebook targets a 16-GB-class T4.
ACTIVE_MODELS = ["smolvlm2", "qwen25vl"]

FEATURE_DTYPE = "float16"
# Host-memory / I/O safeguards for Colab Free/T4.
PROBE_PIN_MEMORY = False      # Probes run on CPU; pinned host RAM is unnecessary.
SAVE_LOGISTIC_TEST_PREDICTIONS = True
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Probe training is intentionally CPU-resident: the VLM extraction stage is the
# dominant CUDA workload, and keeping small probe tensors on CPU avoids repeated
# host↔device copies and leaves the T4 exclusively available to the VLM.
PROBE_DEVICE = "cpu"
PROBE_MAX_THREADS = max(2, min(4, (os.cpu_count() or 2)))
STRICT_REPRO = False

# The notebook must never download or replace HALP-Bench.
ALLOW_DATASET_DOWNLOAD = False

# Reuse an unfinished run when its configuration matches this run.
RESUME_EXISTING_RUN = True
FORCE_NEW_RUN = False

# Only turn on positive-class weighting when the training split is clearly
# imbalanced; otherwise use ordinary BCE / unweighted logistic regression.
IMBALANCE_RATIO_THRESHOLD = 1.5

# Bootstrap count for final-test AUROC confidence intervals.
BOOTSTRAP_REPLICATES = 1000

# Human-readable experiment name used in resumable run directories.
EXPERIMENT_NAME = "halp_vlm_hallucination_t4"

# Label files must be model-specific reviewed labels. We never derive a
# hallucination target from gt_answer alone.
MIN_LABEL_COVERAGE = 0.95

print("RUN_MODE              :", RUN_MODE)
print("MAX_SAMPLES           :", MAX_SAMPLES)
print("EXPERIMENT_DATA_SCOPE :", EXPERIMENT_DATA_SCOPE)
print("SAMPLE_SELECTION_SEED :", SAMPLE_SELECTION_SEED)
print("SEEDS                 :", SEEDS)
print("ACTIVE_MODELS         :", ACTIVE_MODELS)
print("RESUME_EXISTING_RUN   :", RESUME_EXISTING_RUN)
print("ALLOW_DATASET_DOWNLOAD:", ALLOW_DATASET_DOWNLOAD)
print("OPTIMIZATION_PROFILE_VERSION:", OPTIMIZATION_PROFILE_VERSION)
print("QWEN_TARGET_VISUAL_TOKENS:", QWEN_TARGET_VISUAL_TOKENS)
print("QWEN_MIN_VISUAL_TOKENS:", QWEN_MIN_VISUAL_TOKENS)
print("QWEN_PIXEL_BUDGET:", (QWEN_MIN_PIXELS, QWEN_MAX_PIXELS))
print("QWEN_MAX_VISION_PATCHES:", QWEN_MAX_VISION_PATCHES)


RUN_MODE              : full
MAX_SAMPLES           : 8000
EXPERIMENT_DATA_SCOPE : capped_deterministic_subset_8000_per_model
SAMPLE_SELECTION_SEED : 42
SEEDS                 : [42, 52, 62]
ACTIVE_MODELS         : ['smolvlm2', 'qwen25vl']
RESUME_EXISTING_RUN   : True
ALLOW_DATASET_DOWNLOAD: False
OPTIMIZATION_PROFILE_VERSION: t4-colab-free-v5-qwen-adaptive-min-runtime-aware
QWEN_TARGET_VISUAL_TOKENS: 256
QWEN_MIN_VISUAL_TOKENS: 128
QWEN_PIXEL_BUDGET: (100352, 200704)
QWEN_MAX_VISION_PATCHES: 1024


## 2. Install / verify runtime dependencies

The original notebook installed `huggingface_hub` and `requests`. The research pipeline also needs the scientific Python stack and Transformers multimodal support.

This cell records the versions actually present in the Colab runtime. It does not assume a model API is stable just because an earlier notebook used it.



In [2]:

# Dependency installation is centralized in the robust bootstrap cell below.
# This cell intentionally performs no pip install to avoid repeated installation/restart work.
print("Dependency installation is handled by the bootstrap cell.")

# ============================================================
# HALP-Bench — Robust Colab Environment Bootstrap
# ============================================================
#
# Purpose
# -------
# Prepare a clean, reproducible Colab environment for the HALP-Bench
# VLM hallucination-representation experiments.
#
# This cell:
#   1. checks package metadata without importing scientific packages
#   2. tests NumPy/SciPy/scikit-learn in a fresh Python subprocess
#   3. repairs the binary scientific stack only when necessary
#   4. installs only missing HALP-specific lightweight dependencies
#   5. restarts the kernel only after packages were actually changed
#   6. verifies the scientific stack in the clean kernel
#   7. verifies PyTorch and scikit-learn functionally
#   8. records the runtime configuration
#
# Important
# ---------
# This cell intentionally does not:
#   - mount Google Drive
#   - search Google Drive
#   - download HALP-Bench
#   - move/copy HALP-Bench
#   - load a VLM
#   - generate answers
#   - extract VLM features
#   - train probes
#
# Those operations belong to later notebook cells.
#
# Official HALP references:
#   https://github.com/Zesearch/HALP
#   https://huggingface.co/datasets/Zesearch/HALP-Bench
#   https://aclanthology.org/2026.eacl-long.287/
#   https://arxiv.org/abs/2603.05465
#
# Dependency references:
#   https://docs.scipy.org/doc/scipy/dev/toolchain.html
#   https://scikit-learn.org/stable/install.html
#   https://huggingface.co/docs/transformers/installation
# ============================================================


# ============================================================
# 0. Standard-library imports ONLY
# ============================================================

import os
import sys
import subprocess
import platform
import json
import signal
import re
import time
from pathlib import Path
from datetime import datetime, timezone


# ============================================================
# 1. Configuration
# ============================================================

# Small state file used to coordinate a clean runtime restart.
# It is stored in /content and has nothing to do with HALP-Bench.
BOOTSTRAP_STATE_PATH = Path(
    "/content/.halp_environment_bootstrap_state.json"
)

RESTART_REQUESTED_KEY = "restart_requested"

# Scientific packages are kept inside a coherent binary-wheel range.
#
# SciPy 1.16 supports Python >=3.11 and <3.14 and NumPy >=1.25.2,<2.6.
# The current Colab runtime shown by the user is Python 3.13.15.
#
# We do not reinstall PyTorch here because Colab already provides a
# CUDA-enabled PyTorch build and replacing it could break the runtime.
SCIENTIFIC_REQUIREMENTS = [
    "numpy>=2.2,<2.6",
    "scipy>=1.16,<1.17",
    "scikit-learn>=1.9,<2.0",
    "pandas>=2.2,<3.0",
    "matplotlib>=3.9,<4.0",
    "seaborn>=0.13,<1.0",
    "h5py>=3.12,<4.0",
]

# HALP's repository explicitly lists these as required/lightweight
# research dependencies or model-specific dependencies.
HALP_REQUIREMENTS = [
    "transformers>=5.0,<6.0",
    "accelerate>=1.0,<2.0",
    "datasets>=3.0,<6.0",
    "tqdm>=4.67,<5.0",
    "num2words>=0.5.14,<1.0",
    "sentencepiece>=0.2.0,<1.0",
]


# ============================================================
# 2. Runtime information
# ============================================================

PYTHON_VERSION = platform.python_version()
PYTHON_EXECUTABLE = sys.executable

print("=" * 78)
print("HALP-Bench / VLM Research Environment Bootstrap")
print("=" * 78)
print(f"Python          : {PYTHON_VERSION}")
print(f"Executable      : {PYTHON_EXECUTABLE}")
print(f"Platform        : {platform.platform()}")
print(f"Timestamp UTC   : {datetime.now(timezone.utc).isoformat()}")
print("=" * 78)


# ============================================================
# 3. Package metadata helper
# ============================================================

def get_installed_version(distribution_name: str):
    """
    Read package metadata without importing the package itself.

    This avoids loading compiled NumPy/SciPy/sklearn components into
    the current kernel before we decide whether a clean restart is needed.
    """
    try:
        from importlib.metadata import version

        return version(distribution_name)

    except Exception:
        return None


def show_installed_versions():
    """
    Print package versions from distribution metadata only.
    """
    packages = [
        "numpy",
        "scipy",
        "scikit-learn",
        "pandas",
        "matplotlib",
        "seaborn",
        "h5py",
        "torch",
        "transformers",
        "accelerate",
        "datasets",
        "tqdm",
        "num2words",
        "sentencepiece",
        "Pillow",
    ]

    result = {}

    for package in packages:
        result[package] = get_installed_version(package)

    for package, version_value in result.items():
        print(
            f"{package:18s}: "
            f"{version_value if version_value else 'NOT INSTALLED'}"
        )

    return result


# ============================================================
# 4. Safe pip helper
# ============================================================

def run_pip(
    *packages: str,
    extra_args=None,
    description: str = "pip install",
):
    """
    Run pip using the exact Python executable backing this notebook.

    This avoids accidentally modifying a different Python environment.
    """
    extra_args = list(extra_args or [])

    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-cache-dir",
    ]

    command.extend(extra_args)
    command.extend(packages)

    print("\n" + "-" * 78)
    print(description)
    print("$", " ".join(command))
    print("-" * 78)

    completed = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(completed.stdout)

    if completed.returncode != 0:
        raise RuntimeError(
            "pip installation failed.\n"
            f"Exit code: {completed.returncode}\n"
            f"Command: {' '.join(command)}"
        )

    return completed


# ============================================================
# 5. Detect whether scientific modules are already loaded
# ============================================================

SCIENTIFIC_PREFIXES = (
    "numpy",
    "scipy",
    "sklearn",
    "pandas",
    "matplotlib",
    "seaborn",
    "h5py",
)

LOADED_SCIENTIFIC_MODULES = sorted(
    module_name
    for module_name in sys.modules
    if any(
        module_name == prefix
        or module_name.startswith(prefix + ".")
        for prefix in SCIENTIFIC_PREFIXES
    )
)

print("\n" + "=" * 78)
print("Current-kernel scientific module state")
print("=" * 78)

if LOADED_SCIENTIFIC_MODULES:
    print(
        "Scientific modules are already loaded in this Python process."
    )
    print(
        "A clean kernel restart is required before relying on newly "
        "installed binary packages."
    )

    preview = LOADED_SCIENTIFIC_MODULES[:25]

    for module_name in preview:
        print("  ", module_name)

    if len(LOADED_SCIENTIFIC_MODULES) > len(preview):
        print(
            f"  ... and "
            f"{len(LOADED_SCIENTIFIC_MODULES) - len(preview)} more"
        )

else:
    print(
        "No NumPy/SciPy/sklearn/pandas/matplotlib/seaborn/h5py modules "
        "are currently loaded."
    )


# ============================================================
# 6. Read restart state
# ============================================================

restart_state = {}

if BOOTSTRAP_STATE_PATH.exists():

    try:
        restart_state = json.loads(
            BOOTSTRAP_STATE_PATH.read_text(
                encoding="utf-8"
            )
        )

    except Exception as exc:
        print(
            "Warning: restart state could not be read:",
            repr(exc),
        )

restart_was_requested = bool(
    restart_state.get(RESTART_REQUESTED_KEY, False)
)

if restart_was_requested:

    print("\n" + "=" * 78)
    print("POST-RESTART EXECUTION")
    print("=" * 78)

    print(
        "A previous execution requested a clean kernel restart."
    )

    print(
        "This run is now verifying the clean runtime."
    )

    # Remove the state marker before continuing.
    try:
        BOOTSTRAP_STATE_PATH.unlink()
    except FileNotFoundError:
        pass


# ============================================================
# 7. Python version check
# ============================================================

print("\n" + "=" * 78)
print("Python version check")
print("=" * 78)

if sys.version_info < (3, 10):

    raise RuntimeError(
        "HALP requires Python 3.10 or newer. "
        f"Current Python: {PYTHON_VERSION}"
    )

print(f"Python {PYTHON_VERSION}: PASS")


# ============================================================
# 8. Fresh-process scientific stack test
# ============================================================

def test_scientific_stack_in_fresh_process():
    """
    Test the scientific stack from a completely new Python process.

    This is intentionally separate from the notebook kernel because a
    running notebook can retain old binary objects after pip changes files.
    """

    test_script = r"""
import json
import sys

import numpy
import scipy
import sklearn
import pandas
import matplotlib
import seaborn
import h5py

# Exercise representative functionality rather than checking only imports.
import scipy.sparse
import sklearn.linear_model

report = {
    "python": sys.version,
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "pandas": pandas.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": seaborn.__version__,
    "h5py": h5py.__version__,
}

print(json.dumps(report, indent=2))

# Minimal functional checks.
_ = numpy.asarray([1.0, 2.0, 3.0], dtype=numpy.float32)

_ = scipy.sparse.csr_matrix(
    numpy.eye(2, dtype=numpy.float32)
)

X = numpy.asarray(
    [[0.0], [1.0], [2.0], [3.0]],
    dtype=numpy.float32
)

y = numpy.asarray(
    [0, 0, 1, 1],
    dtype=numpy.int64
)

model = sklearn.linear_model.LogisticRegression(
    random_state=42,
    solver="liblinear",
)

model.fit(X, y)

p = model.predict_proba(X)[:, 1]

assert p.shape == (4,)
assert numpy.all(numpy.isfinite(p))
"""

    print("\n" + "=" * 78)
    print("Fresh-process scientific-stack test")
    print("=" * 78)

    completed = subprocess.run(
        [sys.executable, "-c", test_script],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(completed.stdout)

    passed = completed.returncode == 0

    print(
        "Fresh-process scientific stack:",
        "PASS" if passed else "FAIL",
    )

    return passed


FRESH_STACK_OK = test_scientific_stack_in_fresh_process()


# ============================================================
# 9. Enforce declared scientific package ranges
# ============================================================
# A successful import/functionality smoke test is NOT sufficient for
# reproducibility when this notebook has explicitly declared version
# constraints.  Check every requirement with packaging.specifiers before
# deciding whether a repair/restart is required.
try:
    from packaging.requirements import Requirement
    from packaging.version import Version
except Exception as exc:
    raise RuntimeError(
        "The packaging module is required to enforce scientific dependency ranges."
    ) from exc


def requirement_is_satisfied(requirement_text: str):
    req = Requirement(requirement_text)
    installed = get_installed_version(req.name)
    if installed is None:
        return False, None, str(req.specifier)
    try:
        ok = Version(installed) in req.specifier
    except Exception:
        ok = False
    return bool(ok), installed, str(req.specifier)


SCIENTIFIC_VERSION_AUDIT = {}
for requirement in SCIENTIFIC_REQUIREMENTS:
    ok, installed, specifier = requirement_is_satisfied(requirement)
    SCIENTIFIC_VERSION_AUDIT[requirement] = {
        "satisfied": ok,
        "installed": installed,
        "required": specifier,
    }

SCIENTIFIC_VERSIONS_OK = all(
    record["satisfied"]
    for record in SCIENTIFIC_VERSION_AUDIT.values()
)

print("Scientific version audit:")
for requirement, record in SCIENTIFIC_VERSION_AUDIT.items():
    status = "PASS" if record["satisfied"] else "FAIL"
    print(
        f"  [{status}] {requirement} | installed={record['installed']!r}"
    )


# ============================================================
# 10. Decide whether binary repair is necessary
# ============================================================

NEEDS_SCIENTIFIC_REPAIR = (
    (not FRESH_STACK_OK)
    or (not SCIENTIFIC_VERSIONS_OK)
)

if NEEDS_SCIENTIFIC_REPAIR:

    print("\n" + "=" * 78)
    print("SCIENTIFIC STACK REPAIR")
    print("=" * 78)

    print(
        "The scientific stack failed in a fresh Python process."
    )

    print(
        "Repairing NumPy / SciPy / scikit-learn and related packages "
        "using binary wheels."
    )

    run_pip(
        *SCIENTIFIC_REQUIREMENTS,
        extra_args=[
            "--only-binary=:all:",
        ],
        description=(
            "Repairing the scientific Python binary stack"
        ),
    )

else:

    print("\n" + "=" * 78)
    print("SCIENTIFIC STACK STATUS")
    print("=" * 78)

    print(
        "A fresh Python subprocess can already import and exercise "
        "NumPy/SciPy/scikit-learn successfully."
    )

    print("No scientific-stack reinstall is necessary.")


# ============================================================
# 10. Check/install HALP dependencies
# ============================================================

print("\n" + "=" * 78)
print("HALP research dependencies")
print("=" * 78)

MISSING_HALP_PACKAGES = []

for requirement in HALP_REQUIREMENTS:

    # Extract the distribution name only for metadata lookup.
    match = re.match(
        r"^[A-Za-z0-9_.-]+",
        requirement,
    )

    if not match:
        continue

    distribution_name = match.group(0)

    installed = get_installed_version(
        distribution_name
    )

    if installed is None:
        MISSING_HALP_PACKAGES.append(
            requirement
        )


if MISSING_HALP_PACKAGES:

    print(
        "Missing packages:",
        MISSING_HALP_PACKAGES,
    )

    run_pip(
        *MISSING_HALP_PACKAGES,
        extra_args=[
            "--upgrade",
        ],
        description=(
            "Installing missing HALP research dependencies"
        ),
    )

else:

    print(
        "All required HALP research dependencies are already installed."
    )


# ============================================================
# 11. Check whether we must restart
# ============================================================
#
# We restart only when:
#   A) the scientific stack actually required repair, OR
#   B) one or more HALP research dependencies were newly installed.
#
# Merely having scientific modules already loaded is not sufficient reason
# to restart when no package was changed.
#
# If we are already in the post-restart pass, we do not request another
# restart. Instead we move directly to the final import checks.
# ============================================================

NEEDS_KERNEL_RESTART = (
    (
        NEEDS_SCIENTIFIC_REPAIR
        or bool(MISSING_HALP_PACKAGES)
    )
    and not restart_was_requested
)


# ============================================================
# 12. Perform a clean restart WITHOUT raising SystemExit
# ============================================================

if NEEDS_KERNEL_RESTART:

    print("\n" + "=" * 78)
    print("REQUESTING CLEAN KERNEL RESTART")
    print("=" * 78)

    state = {
        RESTART_REQUESTED_KEY: True,
        "reason": {
            "scientific_stack_repaired":
                NEEDS_SCIENTIFIC_REPAIR,

            "new_hal_dependencies_installed":
                bool(MISSING_HALP_PACKAGES),

            "scientific_modules_already_loaded":
                bool(LOADED_SCIENTIFIC_MODULES),
        },
        "timestamp_utc":
            datetime.now(timezone.utc).isoformat(),
    }

    BOOTSTRAP_STATE_PATH.write_text(
        json.dumps(
            state,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        "The environment is prepared."
    )

    print(
        "The current Python process will now be replaced by a clean "
        "kernel so compiled packages cannot remain stale."
    )

    print(
        "This is a runtime restart, not a research-data operation."
    )

    # --------------------------------------------------------
    # Preferred IPython/Jupyter/Colab path.
    # --------------------------------------------------------

    restart_requested_successfully = False

    try:

        ipython = get_ipython()  # noqa: F821

        if (
            ipython is not None
            and hasattr(ipython, "kernel")
            and hasattr(
                ipython.kernel,
                "do_shutdown",
            )
        ):

            ipython.kernel.do_shutdown(
                restart=True
            )

            restart_requested_successfully = True

    except Exception as exc:

        print(
            "IPython restart request failed:",
            repr(exc),
        )


    if restart_requested_successfully:

        # do_shutdown(restart=True) asks the notebook server to replace
        # the kernel. We intentionally do not raise SystemExit here.
        #
        # Give the kernel machinery a moment to receive the request.
        time.sleep(1)

        # If execution has not stopped automatically, terminate this
        # process without generating a Python traceback.
        os._exit(0)


    # --------------------------------------------------------
    # Fallback path.
    # --------------------------------------------------------

    print(
        "Using process termination fallback."
    )

    os.kill(
        os.getpid(),
        signal.SIGTERM,
    )

    # In case the signal is delayed.
    os._exit(0)


# ============================================================
# 13. Final imports — reached only after clean runtime
# ============================================================

print("\n" + "=" * 78)
print("FINAL IMPORT PASS")
print("=" * 78)

# Standard scientific packages
import numpy as np
import scipy
import sklearn
import pandas as pd

# Plotting
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Storage/images
import h5py
from PIL import Image
import PIL

# PyTorch
import torch
import torch.nn as nn

# Hugging Face
import transformers
import accelerate
import datasets

# Utility
import gc
import hashlib
import math
import random
import shutil
import warnings


# ============================================================
# 14. Final version report
# ============================================================

VERSIONS = {
    "python": sys.version,
    "python_version": platform.python_version(),
    "platform": platform.platform(),

    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "accelerate": accelerate.__version__,
    "datasets": datasets.__version__,

    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,

    "pillow": PIL.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "h5py": h5py.__version__,
}


print("\n" + "=" * 78)
print("Verified software environment")
print("=" * 78)

print(
    json.dumps(
        VERSIONS,
        indent=2,
    )
)


# ============================================================
# 15. Current-kernel scientific functional test
# ============================================================

print("\n" + "=" * 78)
print("Current-kernel scientific functional test")
print("=" * 78)

SCIENTIFIC_STACK_OK = True
SCIENTIFIC_STACK_ERROR = None

try:

    # NumPy
    array_test = np.asarray(
        [1.0, 2.0, 3.0],
        dtype=np.float32,
    )

    assert array_test.dtype == np.float32

    # SciPy
    sparse_test = scipy.sparse.csr_matrix(
        np.eye(
            2,
            dtype=np.float32,
        )
    )

    assert sparse_test.shape == (2, 2)

    # scikit-learn
    X_TEST = np.asarray(
        [
            [0.0],
            [1.0],
            [2.0],
            [3.0],
        ],
        dtype=np.float32,
    )

    Y_TEST = np.asarray(
        [0, 0, 1, 1],
        dtype=np.int64,
    )

    LR_TEST = sklearn.linear_model.LogisticRegression(
        random_state=42,
        solver="liblinear",
    )

    LR_TEST.fit(
        X_TEST,
        Y_TEST,
    )

    TEST_PROBS = LR_TEST.predict_proba(
        X_TEST
    )[:, 1]

    assert TEST_PROBS.shape == (4,)

    assert np.all(
        np.isfinite(TEST_PROBS)
    )

    # Pandas
    dataframe_test = pd.DataFrame(
        {
            "x": [1, 2],
            "y": [3, 4],
        }
    )

    assert dataframe_test.shape == (2, 2)

    # HDF5
    assert isinstance(
        h5py.__version__,
        str,
    )

except Exception as exc:

    SCIENTIFIC_STACK_OK = False
    SCIENTIFIC_STACK_ERROR = repr(exc)


print(
    "Scientific stack:",
    "PASS" if SCIENTIFIC_STACK_OK else "FAIL",
)

if not SCIENTIFIC_STACK_OK:

    raise RuntimeError(
        "The current kernel still has a scientific-stack problem.\n"
        f"Error: {SCIENTIFIC_STACK_ERROR}\n\n"
        "At this point, restart the Colab Runtime manually and rerun "
        "this cell once."
    )


# ============================================================
# 16. GPU / CUDA diagnostics
# ============================================================

print("\n" + "=" * 78)
print("GPU / CUDA diagnostics")
print("=" * 78)

CUDA_AVAILABLE = torch.cuda.is_available()

print(
    "CUDA available:",
    CUDA_AVAILABLE,
)

print(
    "PyTorch CUDA   :",
    torch.version.cuda,
)

GPU_INDEX = 0

GPU_NAME = None
GPU_MEMORY_GIB = None
VRAM_FREE_GIB = None
VRAM_TOTAL_GIB = None
BF16_OK = False

DEVICE = torch.device("cpu")


if CUDA_AVAILABLE:

    GPU_NAME = torch.cuda.get_device_name(
        GPU_INDEX
    )

    GPU_PROPERTIES = torch.cuda.get_device_properties(
        GPU_INDEX
    )

    VRAM_TOTAL_GIB = (
        GPU_PROPERTIES.total_memory
        / (2 ** 30)
    )

    try:

        free_bytes, total_bytes = (
            torch.cuda.mem_get_info(
                GPU_INDEX
            )
        )

        VRAM_FREE_GIB = (
            free_bytes
            / (2 ** 30)
        )

        VRAM_TOTAL_GIB_RUNTIME = (
            total_bytes
            / (2 ** 30)
        )

    except Exception:

        VRAM_TOTAL_GIB_RUNTIME = (
            VRAM_TOTAL_GIB
        )

    try:

        BF16_OK = (
            torch.cuda.is_bf16_supported()
        )

    except Exception:

        BF16_OK = False

    DEVICE = torch.device("cuda")

    print(
        f"GPU            : {GPU_NAME}"
    )

    print(
        f"VRAM total     : {VRAM_TOTAL_GIB:.2f} GiB"
    )

    if VRAM_FREE_GIB is not None:

        print(
            f"VRAM free      : {VRAM_FREE_GIB:.2f} GiB"
        )

    print(
        "Compute capability:",
        f"{GPU_PROPERTIES.major}."
        f"{GPU_PROPERTIES.minor}",
    )

else:

    print(
        "CUDA unavailable."
    )

    print(
        "Only lightweight CPU checks should be used."
    )


print(
    "BF16 supported :",
    BF16_OK,
)

print(
    "Selected device:",
    DEVICE,
)


# ============================================================
# 17. PyTorch functional test
# ============================================================

print("\n" + "=" * 78)
print("PyTorch functional test")
print("=" * 78)

TORCH_TEST_TENSOR = torch.tensor(
    [1.0, 2.0, 3.0],
    dtype=torch.float32,
    device=DEVICE,
)

TORCH_TEST_SUM = (
    torch.sum(
        TORCH_TEST_TENSOR
    ).item()
)

assert TORCH_TEST_SUM == 6.0

print(
    "Tensor:",
    TORCH_TEST_TENSOR,
)

print(
    "Device:",
    TORCH_TEST_TENSOR.device,
)

print(
    "Dtype :",
    TORCH_TEST_TENSOR.dtype,
)

print(
    "Result:",
    TORCH_TEST_SUM,
)

print(
    "PyTorch functional test: PASS"
)

del TORCH_TEST_TENSOR

if CUDA_AVAILABLE:

    torch.cuda.empty_cache()

gc.collect()


# ============================================================
# 18. Transformers / Accelerate / Datasets import test
# ============================================================

print("\n" + "=" * 78)
print("Hugging Face dependency test")
print("=" * 78)

print(
    "Transformers:",
    transformers.__version__,
)

print(
    "Accelerate :",
    accelerate.__version__,
)

print(
    "Datasets   :",
    datasets.__version__,
)

print(
    "Hugging Face imports: PASS"
)


# ============================================================
# 19. Critical dependency checks
# ============================================================

print("\n" + "=" * 78)
print("Critical dependency checks")
print("=" * 78)

from packaging.version import Version


CRITICAL_CHECKS = {
    "Python >= 3.10":
        Version(PYTHON_VERSION)
        >= Version("3.10"),

    "PyTorch >= 2.0":
        Version(
            torch.__version__.split("+")[0]
        )
        >= Version("2.0"),

    "NumPy imported":
        bool(np.__version__),

    "SciPy imported":
        bool(scipy.__version__),

    "scikit-learn imported":
        bool(sklearn.__version__),

    "Pandas imported":
        bool(pd.__version__),

    "Matplotlib imported":
        bool(matplotlib.__version__),

    "Seaborn imported":
        bool(sns.__version__),

    "HDF5 imported":
        bool(h5py.__version__),

    "Pillow imported":
        bool(PIL.__version__),

    "Transformers imported":
        bool(transformers.__version__),

    "Accelerate imported":
        bool(accelerate.__version__),

    "Datasets imported":
        bool(datasets.__version__),
}


for check_name, passed in CRITICAL_CHECKS.items():

    print(
        f"{check_name:30s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


assert all(
    CRITICAL_CHECKS.values()
), "A critical dependency check failed."


# ============================================================
# 20. Environment object for later notebook cells
# ============================================================

ENVIRONMENT_INFO = {

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "python_version":
        PYTHON_VERSION,

    "python_executable":
        PYTHON_EXECUTABLE,

    "platform":
        platform.platform(),

    "device":
        str(DEVICE),

    "cuda_available":
        CUDA_AVAILABLE,

    "cuda_version":
        torch.version.cuda,

    "gpu_name":
        GPU_NAME,

    "gpu_memory_gib":
        GPU_MEMORY_GIB,

    "vram_free_gib":
        VRAM_FREE_GIB,

    "bf16_supported":
        BF16_OK,

    "versions":
        VERSIONS,

    "fresh_scientific_stack_test":
        True,

    "current_kernel_scientific_test":
        SCIENTIFIC_STACK_OK,

    "pytorch_functional_test":
        True,

    "sklearn_functional_test":
        True,

    "colab_runtime":
        "google.colab" in sys.modules,
}


# ============================================================
# 21. Final status
# ============================================================

print("\n" + "=" * 78)
print("ENVIRONMENT READY")
print("=" * 78)

print(
    f"Python             : {PYTHON_VERSION}"
)

print(
    f"PyTorch            : {torch.__version__}"
)

print(
    f"Transformers       : {transformers.__version__}"
)

print(
    f"Accelerate         : {accelerate.__version__}"
)

print(
    f"Datasets           : {datasets.__version__}"
)

print(
    f"NumPy              : {np.__version__}"
)

print(
    f"SciPy              : {scipy.__version__}"
)

print(
    f"scikit-learn       : {sklearn.__version__}"
)

print(
    f"Pandas             : {pd.__version__}"
)

print(
    f"CUDA               : "
    f"{'AVAILABLE' if CUDA_AVAILABLE else 'UNAVAILABLE'}"
)

print(
    f"BF16               : "
    f"{'AVAILABLE' if BF16_OK else 'NOT AVAILABLE'}"
)

print(
    f"Device             : {DEVICE}"
)

print("-" * 78)

print(
    "Fresh scientific-process test : PASS"
)

print(
    "Current-kernel import test    : PASS"
)

print(
    "PyTorch functional test       : PASS"
)

print(
    "scikit-learn functional test  : PASS"
)

print(
    "Hugging Face import test      : PASS"
)

print(
    "Critical dependency checks    : PASS"
)

print("=" * 78)


# ============================================================
# 22. Research-pipeline boundary
# ============================================================
#
# This cell ends here.
#
# The next notebook cell can now safely:
#
#   - mount Google Drive
#   - locate the existing HALP-Bench copy
#   - validate metadata/images
#   - identify manually-reviewed hallucination labels
#   - construct the experimental dataset
#
# No HALP-Bench data has been downloaded or modified by this cell.
# ============================================================

print(
    "\nEnvironment bootstrap complete."
)

print(
    "The next cell may safely begin the Google Drive / "
    "HALP-Bench discovery stage."
)


Dependency installation is handled by the bootstrap cell.
HALP-Bench / VLM Research Environment Bootstrap
Python          : 3.13.15
Executable      : /usr/bin/python3
Platform        : Linux-6.6.122+-x86_64-with-glibc2.35
Timestamp UTC   : 2026-08-31T22:32:30.785725+00:00

Current-kernel scientific module state
Scientific modules are already loaded in this Python process.
A clean kernel restart is required before relying on newly installed binary packages.
   matplotlib
   matplotlib._afm
   matplotlib._api
   matplotlib._api.deprecation
   matplotlib._blocking_input
   matplotlib._c_internal_utils
   matplotlib._cm
   matplotlib._cm_bivar
   matplotlib._cm_listed
   matplotlib._cm_multivar
   matplotlib._color_data
   matplotlib._constrained_layout
   matplotlib._docstring
   matplotlib._enums
   matplotlib._fontconfig_pattern
   matplotlib._image
   matplotlib._layoutgrid
   matplotlib._mathtext
   matplotlib._mathtext_data
   matplotlib._path
   matplotlib._pylab_helpers
   matplotl

## 3. Google Drive mount and persistent project structure

The dataset stays where it is; it is not moved or duplicated. New artifacts are stored under a separate project directory.

The original notebook hard-coded `/content/drive/MyDrive/HALP-Bench`. This notebook does **not** assume that path is correct; it searches for an existing copy first.



In [3]:

from google.colab import drive
drive.mount("/content/drive")

# Persistent locations live on Drive so a Colab reset does not erase the
# research outputs or extraction checkpoints.
DRIVE_MOUNT_ROOT = Path("/content/drive")
MYDRIVE = DRIVE_MOUNT_ROOT / "MyDrive"

PROJECT_DIR = MYDRIVE / "HALP_Bench_Project"

RESULTS_DIR = PROJECT_DIR / "results"
TABLE_DIR = RESULTS_DIR / "tables"
FIG_DIR = RESULTS_DIR / "figures"
PRED_DIR = RESULTS_DIR / "predictions"
LOG_DIR = RESULTS_DIR / "logs"
CKPT_DIR = RESULTS_DIR / "checkpoints"

FEATURE_DIR = PROJECT_DIR / "features"
MODEL_DIR = PROJECT_DIR / "models"
FINAL_DIR = PROJECT_DIR / "final_artifacts"
RUNS_DIR = PROJECT_DIR / "runs"
TEMP_DIR = PROJECT_DIR / "tmp"
CACHE_DIR = PROJECT_DIR / "cache"

for d in [
    PROJECT_DIR, RESULTS_DIR, TABLE_DIR, FIG_DIR, PRED_DIR, LOG_DIR,
    CKPT_DIR, FEATURE_DIR, MODEL_DIR, FINAL_DIR, RUNS_DIR, TEMP_DIR, CACHE_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

# Filled by the dataset-resolution stage. Keeping these separate avoids
# confusing source data with persistent experiment outputs.
DATASET_DIR = None
HALPBENCH_ROOT = None
HALPBENCH_METADATA_DIR = None
HALPBENCH_IMAGE_DIR = None
BENCHMARK_CSV = None

print("Drive mount     :", DRIVE_MOUNT_ROOT)
print("MyDrive         :", MYDRIVE)
print("Project         :", PROJECT_DIR)
print("Results         :", RESULTS_DIR)
print("Temp            :", TEMP_DIR)
print("Final archives  :", FINAL_DIR)


Mounted at /content/drive
Drive mount     : /content/drive
MyDrive         : /content/drive/MyDrive
Project         : /content/drive/MyDrive/HALP_Bench_Project
Results         : /content/drive/MyDrive/HALP_Bench_Project/results
Temp            : /content/drive/MyDrive/HALP_Bench_Project/tmp
Final archives  : /content/drive/MyDrive/HALP_Bench_Project/final_artifacts


## 4. Dataset discovery — search narrowly, then stop

This search is kept narrow. It checks only the top few directory levels of `MyDrive` instead of scanning the whole Drive recursively.

Candidates are ranked using simple repository signals such as:
- `sampled_10k_relational_dataset.csv`
- a `HALP-Bench`/`HALP_Bench` directory name
- presence of image files

The selected path is recorded so later cells all use the same location.



In [4]:
# ============================================================
# HALP-Bench discovery — directory OR ZIP
# ============================================================

from pathlib import Path
import zipfile
import os

SKIP_DIR_NAMES = {
    ".git", ".cache", "node_modules", "__pycache__", ".ipynb_checkpoints",
}

DATASET_CSV_NAME = "sampled_10k_relational_dataset.csv"
HALP_ZIP_NAMES = {
    "halp-bench.zip",
    "halp_bench.zip",
}

def bounded_dirs(root: Path, max_depth: int = 5):
    root = Path(root)
    queue = [(root, 0)]
    seen = set()

    while queue:
        current, depth = queue.pop(0)

        try:
            key = current.resolve()
        except Exception:
            key = current

        if key in seen:
            continue

        seen.add(key)
        yield current, depth

        if depth >= max_depth:
            continue

        try:
            children = sorted(
                [
                    p
                    for p in current.iterdir()
                    if p.is_dir() and p.name not in SKIP_DIR_NAMES
                ],
                key=lambda p: p.name.lower(),
            )
        except (OSError, PermissionError):
            continue

        queue.extend((p, depth + 1) for p in children)


def candidate_score(path: Path):
    score = 0
    name = path.name.lower()

    if name in {"halp-bench", "halp_bench"}:
        score += 50

    if (path / DATASET_CSV_NAME).is_file():
        score += 100

    # More tolerant: CSV may be nested inside the dataset folder.
    try:
        if any(
            p.is_file() and p.name.lower() == DATASET_CSV_NAME.lower()
            for p in path.rglob("*")
        ):
            score += 80
    except Exception:
        pass

    return score


# ------------------------------------------------------------
# 1. First: look for an already-extracted dataset directory
# ------------------------------------------------------------

candidates = []

for d, depth in bounded_dirs(MYDRIVE, max_depth=5):
    try:
        is_named_halp = d.name.lower() in {
            "halp-bench",
            "halp_bench",
        }

        has_benchmark = any(
            x.is_file() and x.name.lower() == DATASET_CSV_NAME.lower()
            for x in d.iterdir()
        )

        nested_benchmark = False
        if not has_benchmark:
            nested_benchmark = any(
                x.is_file() and x.name.lower() == DATASET_CSV_NAME.lower()
                for x in d.rglob("*")
            )

        if is_named_halp or has_benchmark or nested_benchmark:
            candidates.append((candidate_score(d), d))

    except (OSError, PermissionError):
        continue


# ------------------------------------------------------------
# 2. Rank extracted-directory candidates
# ------------------------------------------------------------

unique = {}

for score, p in candidates:
    try:
        rp = str(p.resolve())
    except Exception:
        rp = str(p)

    unique[rp] = max(score, unique.get(rp, -1))

ranked = sorted(
    [(score, Path(p)) for p, score in unique.items()],
    key=lambda x: (-x[0], str(x[1]).lower()),
)


print("Candidate extracted HALP-Bench roots:")
for score, p in ranked[:20]:
    print(f"  score={score:>3}  {p}")


# ------------------------------------------------------------
# 3. If no directory exists, look for HALP-Bench.zip
# ------------------------------------------------------------

if not ranked:

    zip_candidates = []

    for p in MYDRIVE.rglob("*"):
        try:
            if p.is_file() and p.name.lower() in HALP_ZIP_NAMES:
                zip_candidates.append(p)
        except (OSError, PermissionError):
            continue

    print("\nCandidate HALP-Bench ZIP files:")
    for p in zip_candidates:
        print(" ", p)

    if not zip_candidates:
        raise FileNotFoundError(
            "Neither an extracted HALP-Bench directory nor HALP-Bench.zip "
            "was found under MyDrive."
        )

    # Prefer the exact expected filename.
    zip_candidates.sort(
        key=lambda p: (
            p.name.lower() != "halp-bench.zip",
            str(p).lower(),
        )
    )

    HALPBENCH_ZIP = zip_candidates[0]

    print("\nUsing existing local archive:")
    print(" ", HALPBENCH_ZIP)

    # --------------------------------------------------------
    # 4. Inspect ZIP BEFORE extracting
    # --------------------------------------------------------

    with zipfile.ZipFile(HALPBENCH_ZIP, "r") as zf:
        members = zf.namelist()

        csv_members = [
            m for m in members
            if Path(m).name.lower() == DATASET_CSV_NAME.lower()
        ]

        if not csv_members:
            raise FileNotFoundError(
                f"{DATASET_CSV_NAME} was not found inside {HALPBENCH_ZIP}"
            )

        print("\nZIP contains benchmark CSV:")
        for m in csv_members[:10]:
            print(" ", m)

        # Infer the top-level dataset directory when possible.
        top_levels = {
            m.split("/")[0]
            for m in members
            if "/" in m
        }

    # --------------------------------------------------------
    # 5. Extract into Drive only once
    # --------------------------------------------------------

    EXTRACT_ROOT = PROJECT_DIR / "_extracted_dataset"

    # Don't re-extract if it is already there and valid.
    existing_csvs = list(EXTRACT_ROOT.rglob(DATASET_CSV_NAME)) \
        if EXTRACT_ROOT.exists() else []

    if existing_csvs:
        DATASET_DIR = existing_csvs[0].parent

        print("\nExisting extraction found:")
        print(" ", DATASET_DIR)

    else:
        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

        print("\nExtracting HALP-Bench ZIP...")
        print("Source:", HALPBENCH_ZIP)
        print("Target:", EXTRACT_ROOT)

        with zipfile.ZipFile(HALPBENCH_ZIP, "r") as zf:
            zf.extractall(EXTRACT_ROOT)

        extracted_csvs = list(EXTRACT_ROOT.rglob(DATASET_CSV_NAME))

        if not extracted_csvs:
            raise FileNotFoundError(
                "ZIP extraction completed, but the benchmark CSV "
                "could not be located afterward."
            )

        DATASET_DIR = extracted_csvs[0].parent

        print("\nExtraction successful.")
        print("Dataset directory:", DATASET_DIR)


    HALPBENCH_ROOT = DATASET_DIR
    BENCHMARK_CSV = DATASET_DIR / DATASET_CSV_NAME

else:

    # --------------------------------------------------------
    # 6. Already-extracted dataset
    # --------------------------------------------------------

    DATASET_DIR = ranked[0][1]
    HALPBENCH_ROOT = DATASET_DIR

    # Find CSV recursively, not only at root.
    csv_candidates = list(DATASET_DIR.rglob(DATASET_CSV_NAME))

    if not csv_candidates:
        raise FileNotFoundError(
            f"Found a plausible HALP-Bench directory, but could not find "
            f"{DATASET_CSV_NAME} inside: {DATASET_DIR}"
        )

    BENCHMARK_CSV = csv_candidates[0]


HALPBENCH_METADATA_DIR = BENCHMARK_CSV.parent
HALPBENCH_IMAGE_DIR = None


print("\n==============================================")
print("HALP-Bench RESOLUTION")
print("==============================================")
print("Selected root :", HALPBENCH_ROOT)
print("Benchmark CSV :", BENCHMARK_CSV)
print("ZIP source    :", globals().get("HALPBENCH_ZIP", "not used"))
print("Download      :", "DISABLED")
print("==============================================")

Candidate extracted HALP-Bench roots:
  score=180  /content/drive/MyDrive/HALP_Bench_Project/_extracted_dataset
  score= 80  /content/drive/MyDrive
  score= 80  /content/drive/MyDrive/HALP_Bench_Project

HALP-Bench RESOLUTION
Selected root : /content/drive/MyDrive/HALP_Bench_Project/_extracted_dataset
Benchmark CSV : /content/drive/MyDrive/HALP_Bench_Project/_extracted_dataset/sampled_10k_relational_dataset.csv
ZIP source    : not used
Download      : DISABLED


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 5. Dataset integrity audit

The public HALP-Bench data card currently exposes one `test` split with 10k rows and columns such as:

`question_id`, `image_name`, `question`, `gt_answer`, `category`, `description`, `has_image`, `dataset`, `is_relational`.

The public benchmark table itself does **not** provide the model-specific hallucination label required for supervised probes. HALP's official repository instead stores model-specific `*_manually_reviewed.csv` files with the reviewed hallucination labels.

So this notebook:
- validates the benchmark metadata locally,
- searches for the model-specific reviewed label files locally,
- refuses to invent labels,
- does not silently mix unrelated versions.



In [6]:

EXPECTED_BENCHMARK_COLUMNS = {
    "question_id", "image_name", "question", "gt_answer", "category",
    "description", "has_image", "dataset", "is_relational"
}

def find_files_named(root: Path, patterns, max_files=100):
    out = []
    patterns = [p.lower() for p in patterns]
    for p in root.rglob("*"):
        if len(out) >= max_files:
            break
        if p.is_file() and any(p.name.lower() == pat for pat in patterns):
            out.append(p)
    return out

csv_candidates = find_files_named(
    DATASET_DIR,
    ["sampled_10k_relational_dataset.csv"],
    max_files=20,
)

if not csv_candidates:
    raise FileNotFoundError(
        f"Could not find sampled_10k_relational_dataset.csv inside {DATASET_DIR}. "
        "Do not substitute another dataset without documenting the provenance change."
    )

BENCHMARK_CSV = csv_candidates[0]
df = pd.read_csv(BENCHMARK_CSV)

print("Benchmark CSV:", BENCHMARK_CSV)
print("Rows:", len(df))
print("Columns:", list(df.columns))

missing_schema = EXPECTED_BENCHMARK_COLUMNS - set(df.columns)
if missing_schema:
    print("WARNING: expected public-schema columns missing:", sorted(missing_schema))
else:
    print("Public-schema check: passed")

print("\nHead:")
display(df.head())

# Build and cache the image manifest once. Later cells use image_index instead
# of recursively scanning Google Drive again.
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
metadata_suffixes = {".csv", ".json", ".jsonl", ".parquet", ".h5", ".hdf5"}

benchmark_stat = BENCHMARK_CSV.stat()
manifest_key = hashlib.sha256(
    f"{BENCHMARK_CSV.resolve()}|{benchmark_stat.st_size}|{benchmark_stat.st_mtime_ns}".encode()
).hexdigest()[:16]
IMAGE_INDEX_CACHE = CACHE_DIR / f"halp_image_index_{manifest_key}.json"

def _build_image_index():
    image_index_local = {}
    format_counts_local = {}
    metadata_file_count_local = 0
    metadata_file_bytes_local = 0
    scanned_file_count = 0

    for p in DATASET_DIR.rglob("*"):
        if not p.is_file():
            continue
        scanned_file_count += 1
        ext = p.suffix.lower()
        if ext in metadata_suffixes:
            metadata_file_count_local += 1
            metadata_file_bytes_local += p.stat().st_size
        if ext in IMAGE_EXTENSIONS:
            # Duplicate basenames are ambiguous; keep the first only if the
            # path is identical, otherwise fail rather than silently mis-map.
            previous = image_index_local.get(p.name)
            if previous is not None and previous.resolve() != p.resolve():
                raise RuntimeError(
                    f"Duplicate image basename found in HALP-Bench: {p.name}\n"
                    f"  {previous}\n  {p}"
                )
            image_index_local[p.name] = p
            format_counts_local[ext] = format_counts_local.get(ext, 0) + 1

    payload = {
        "dataset_dir": str(DATASET_DIR.resolve()),
        "benchmark_csv": str(BENCHMARK_CSV.resolve()),
        "benchmark_size": benchmark_stat.st_size,
        "benchmark_mtime_ns": benchmark_stat.st_mtime_ns,
        "image_index": {k: str(v.resolve()) for k, v in image_index_local.items()},
        "format_counts": format_counts_local,
        "metadata_file_count": metadata_file_count_local,
        "metadata_file_bytes": metadata_file_bytes_local,
        "scanned_file_count": scanned_file_count,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    tmp = IMAGE_INDEX_CACHE.with_suffix(".tmp")
    tmp.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    tmp.replace(IMAGE_INDEX_CACHE)
    return payload

def _load_cached_image_index():
    if not IMAGE_INDEX_CACHE.exists():
        return None
    try:
        payload = json.loads(IMAGE_INDEX_CACHE.read_text(encoding="utf-8"))
        if (
            payload.get("dataset_dir") != str(DATASET_DIR.resolve())
            or payload.get("benchmark_csv") != str(BENCHMARK_CSV.resolve())
            or int(payload.get("benchmark_size", -1)) != benchmark_stat.st_size
            or int(payload.get("benchmark_mtime_ns", -1)) != benchmark_stat.st_mtime_ns
        ):
            return None
        return payload
    except (OSError, ValueError, TypeError, json.JSONDecodeError):
        return None

cached = _load_cached_image_index()
manifest = cached if cached is not None else _build_image_index()

image_index = {name: Path(path) for name, path in manifest["image_index"].items()}
format_counts = manifest["format_counts"]
metadata_file_count = int(manifest["metadata_file_count"])
metadata_file_bytes = int(manifest["metadata_file_bytes"])

def resolve_image(name):
    if pd.isna(name):
        return None
    return image_index.get(Path(str(name)).name)

df["resolved_image_path"] = df["image_name"].map(resolve_image)

missing_images = int(df["resolved_image_path"].isna().sum())
duplicate_questions = int(df["question_id"].duplicated().sum())
unique_question_ids = int(df["question_id"].nunique(dropna=True))
unique_images = int(df["image_name"].nunique(dropna=True))

# Identify the actual image directory without assuming a fixed folder name.
if image_index:
    parent_counts = {}
    for p in image_index.values():
        parent_counts[str(p.parent)] = parent_counts.get(str(p.parent), 0) + 1
    HALPBENCH_IMAGE_DIR = Path(max(parent_counts, key=parent_counts.get))
else:
    HALPBENCH_IMAGE_DIR = DATASET_DIR

# Lightweight integrity check: every cached path must still exist.
stale_manifest_paths = [str(p) for p in image_index.values() if not p.is_file()]
if stale_manifest_paths:
    print("Cached image manifest contains stale paths; rebuilding it.")
    manifest = _build_image_index()
    image_index = {name: Path(path) for name, path in manifest["image_index"].items()}
    stale_manifest_paths = [str(p) for p in image_index.values() if not p.is_file()]
    if stale_manifest_paths:
        raise RuntimeError(f"Image manifest remains stale after rebuild. Examples: {stale_manifest_paths[:5]}")

# Content fingerprint: benchmark CSV bytes + resolved image names/sizes.
h = hashlib.sha256()
with BENCHMARK_CSV.open("rb") as f:
    while chunk := f.read(1024 * 1024):
        h.update(chunk)
for name, path in sorted(image_index.items()):
    stat = path.stat()
    h.update(f"{name}|{stat.st_size}|{stat.st_mtime_ns}".encode())
DATASET_FINGERPRINT = h.hexdigest()

print("Image index source :", "cache" if cached is not None else "fresh scan")
print("HALP-Bench image dir:", HALPBENCH_IMAGE_DIR)
print("Unique question IDs:", unique_question_ids)
print("Unique image names :", unique_images)
print("Duplicate question_id rows:", duplicate_questions)
print("Missing referenced images:", missing_images)
print("Image formats:", format_counts)
print("Metadata files:", metadata_file_count)
print("Scanned files once :", manifest.get("scanned_file_count"))
print("Dataset fingerprint:", DATASET_FINGERPRINT)

if duplicate_questions:
    raise RuntimeError("Duplicate question_id values were found in the benchmark metadata.")

if missing_images:
    display(df.loc[df["resolved_image_path"].isna(), ["question_id", "image_name"]].head(20))
    if RUN_MODE == "full":
        raise RuntimeError("Full run blocked: one or more benchmark rows reference missing local images.")

dataset_summary = {
    "benchmark_csv": str(BENCHMARK_CSV),
    "dataset_root": str(HALPBENCH_ROOT),
    "image_dir": str(HALPBENCH_IMAGE_DIR),
    "rows": len(df),
    "unique_question_ids": unique_question_ids,
    "unique_images": unique_images,
    "missing_images": missing_images,
    "image_formats": format_counts,
    "metadata_file_count": metadata_file_count,
    "metadata_file_bytes": metadata_file_bytes,
    "cached_image_manifest": str(IMAGE_INDEX_CACHE),
    "dataset_fingerprint": DATASET_FINGERPRINT,
}
print(json.dumps(dataset_summary, indent=2))

# Check benchmark-level distributions before any expensive model call.
print("Dataset distribution by source:")
display(df["dataset"].value_counts(dropna=False).rename("count").to_frame())

print("Top categories:")
display(df["category"].value_counts(dropna=False).head(20).rename("count").to_frame())

print("Relational flag:")
display(df["is_relational"].value_counts(dropna=False).rename("count").to_frame())

dataset_summary_path = TABLE_DIR / "dataset_summary.csv"
pd.DataFrame([dataset_summary]).to_csv(dataset_summary_path, index=False)
print("Saved:", dataset_summary_path)


Benchmark CSV: /content/drive/MyDrive/HALP_Bench_Project/_extracted_dataset/sampled_10k_relational_dataset.csv
Rows: 10000
Columns: ['question_id', 'image_name', 'question', 'gt_answer', 'category', 'description', 'has_image', 'dataset', 'is_relational']
Public-schema check: passed

Head:


,question_id,image_name,question,gt_answer,category,description,has_image,dataset,is_relational
0,question_comb_1,haloquest_2082.png,How many sharks are present in the travel broc...,There are no sharks in the brochure ; The trav...,false premises,There are no sharks in the brochure ; The trav...,True,haloquest,True
1,question_comb_2,haloquest_1680.png,"What kind of fruit is the younger, smaller ani...","The younger, smaller animal doesn't appear to ...",false premises,"The younger, smaller animal doesn't appear to ...",True,haloquest,True
2,question_comb_3,AMBER_307.jpg,Is there a wall in this image?,"n, o",discriminative-hallucination,Amber,True,amber,True
3,question_comb_4,haloquest_0827.png,Does the image depict a dog dressed in human c...,No; the image doesn't depict a dog dressed in ...,visual challenge,No; the image doesn't depict a dog dressed in ...,True,haloquest,True
4,question_comb_5,hallusionbench_ocr_12_2.png,Is the text in this figure made by Winston Chu...,no,ocr,"No, the text in this figure is not made by Win...",True,hallusionbench,False


Image index source : cache
HALP-Bench image dir: /content/drive/MyDrive/HALP_Bench_Project/_extracted_dataset
Unique question IDs: 10000
Unique image names : 4849
Duplicate question_id rows: 0
Missing referenced images: 0
Image formats: {'.jpg': 3527, '.png': 1321, '.jpeg': 1}
Metadata files: 3
Scanned files once : 4856
Dataset fingerprint: 663eabfaf905a7447c382e8a3845cfa723e1026c2870b22a9ca2491b8c644968
{
  "benchmark_csv": "/content/drive/MyDrive/HALP_Bench_Project/_extracted_dataset/sampled_10k_relational_dataset.csv",
  "dataset_root": "/content/drive/MyDrive/HALP_Bench_Project/_extracted_dataset",
  "image_dir": "/content/drive/MyDrive/HALP_Bench_Project/_extracted_dataset",
  "rows": 10000,
  "unique_question_ids": 10000,
  "unique_images": 4849,
  "missing_images": 0,
  "image_formats": {
    ".jpg": 3527,
    ".png": 1321,
    ".jpeg": 1
  },
  "metadata_file_count": 3,
  "metadata_file_bytes": 3132664,
  "cached_image_manifest": "/content/drive/MyDrive/HALP_Bench_Project/cache

,count
dataset,
amber,3926
haloquest,2784
pope,1230
mme,885
hallusionbench,617
mathvista,558


Top categories:


,count
category,
visual challenge,1531
discriminative-attribute-state,1169
discriminative-relation,975
false premises,898
relation,689
discriminative-hallucination,620
random,456
adversarial,402
popular,372


Relational flag:


,count
is_relational,
True,7000
False,3000


Saved: /content/drive/MyDrive/HALP_Bench_Project/results/tables/dataset_summary.csv


## 6. Locate model-specific hallucination labels locally

HALP's official repository lists eight manually reviewed CSVs:
- `gemma3_manually_reviewed.csv`
- `fastvlm_manually_reviewed.csv`
- `llava_manually_reviewed.csv`
- `molmo_manually_reviewed.csv`
- `qwen25vl_manually_reviewed.csv`
- `llama32_manually_reviewed.csv`
- `phi4vl_manually_reviewed.csv`
- `smolvlm_manually_reviewed.csv`

The local Drive copy may or may not include them. This cell searches the selected HALP project area and its immediate parent, but it does not download anything.

A full experiment is blocked for a model when:
1. no compatible reviewed-label file is found, or
2. the label column / sample key cannot be identified safely.



## 6a. Supervised endpoint source acquisition (official reviewed labels only)

This cell downloads only the two model-specific manually reviewed label CSVs needed by the active models. It does **not** download or replace HALP-Bench. The files are cached on Google Drive so later runs do not download them again. A small ZIP bundle and SHA-256 manifest are also created for reuse.

The official HALP repository lists `qwen25vl_manually_reviewed.csv` and `smolvlm_manually_reviewed.csv` under `FInal_CSV_Hallucination/`. These are the supervised labels used by the probe stage; benchmark metadata alone is not substituted for them.


In [7]:
# ============================================================
# Official supervised-endpoint label acquisition
# ============================================================
import urllib.request
import zipfile

SUPERVISED_LABEL_SOURCE_DIR = PROJECT_DIR / "supervised_endpoint_sources"
SUPERVISED_LABEL_SOURCE_DIR.mkdir(parents=True, exist_ok=True)

SUPERVISED_LABEL_URLS = {
    "qwen25vl": "https://raw.githubusercontent.com/Zesearch/HALP/main/FInal_CSV_Hallucination/qwen25vl_manually_reviewed.csv",
    "smolvlm2": "https://raw.githubusercontent.com/Zesearch/HALP/main/FInal_CSV_Hallucination/smolvlm_manually_reviewed.csv",
}

SUPERVISED_LABEL_FILES = {
    model_key: SUPERVISED_LABEL_SOURCE_DIR / Path(url).name
    for model_key, url in SUPERVISED_LABEL_URLS.items()
}

SUPERVISED_LABEL_BUNDLE_PATH = FINAL_DIR / "supervised_endpoint_labels.zip"
SUPERVISED_LABEL_MANIFEST_PATH = FINAL_DIR / "supervised_endpoint_labels_manifest.json"


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _download_if_missing(model_key: str, url: str, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.is_file() and destination.stat().st_size > 0:
        return "CACHED"

    tmp = destination.with_suffix(destination.suffix + ".part")

    try:
        request = urllib.request.Request(
            url,
            headers={"User-Agent": "HALP-Bench-Colab-Audit/1.0"},
        )

        with urllib.request.urlopen(request, timeout=60) as response, open(
            tmp, "wb"
        ) as out:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                out.write(chunk)

        if tmp.stat().st_size <= 0:
            raise RuntimeError("Downloaded file is empty.")

        os.replace(tmp, destination)
        return "DOWNLOADED"

    except Exception:
        try:
            tmp.unlink()
        except FileNotFoundError:
            pass
        raise


# The repository's reviewed-label files are model-specific, but the exact
# spelling/encoding of the binary label field can vary.  This function is
# intentionally conservative: it only accepts a column whose name is a strong
# hallucination/label candidate and whose observed values can be mapped
# unambiguously to {0, 1}.
_# Explicit model-specific reviewed endpoint columns for the current official HALP files.
# This prevents an auxiliary column from silently changing the supervised target.
EXPECTED_REVIEWED_LABEL_COLUMNS = {
    "smolvlm2": "is_hallucinating",
    "qwen25vl": "is_hallucinating",
}

LABEL_COLUMN_CANDIDATES = [
    "hallucination",
    "hallucinated",
    "is_hallucination",
    "is_hallucinated",
    "hallucination_label",
    "hallucination_status",
    "hallucination_flag",
    "label",
    "target",
    "y",
    "annotation",
    "annotations",
    "review_label",
    "reviewed_label",
    "manual_label",
    "manual_review",
    "verdict",
    "judgment",
    "judgement",
    "correctness",
    "is_correct",
    "correct",
    "error",
]


def _normalize_reviewed_label_series(series: pd.Series) -> pd.Series:
    """
    Convert a reviewed hallucination label column into nullable 0/1 values.

    Accepted representations include numeric/bool forms and common textual
    encodings such as:
        0/1, true/false, yes/no, y/n, h/nh,
        hallucinated/non-hallucinated,
        hallucination/no hallucination,
        incorrect/correct.
    """
    out = pd.Series(np.nan, index=series.index, dtype="float64")

    # First handle genuine numeric values, including 0.0 / 1.0.
    numeric = pd.to_numeric(series, errors="coerce")
    numeric_mask = numeric.notna()
    binary_numeric_mask = numeric_mask & numeric.isin([0, 1])
    out.loc[binary_numeric_mask] = numeric.loc[binary_numeric_mask].astype(float)

    # Then handle textual values.
    text = series.astype("string").str.strip().str.lower()
    text = text.str.replace(r"\s+", " ", regex=True)
    text = text.str.replace("_", " ", regex=False)
    text = text.str.replace("-", " ", regex=False)

    negative_exact = {
        "0",
        "0.0",
        "false",
        "no",
        "n",
        "f",
        "nh",
        "non hallucination",
        "non hallucinated",
        "not hallucination",
        "not hallucinated",
        "no hallucination",
        "no hallucinated",
        "correct",
        "correct answer",
        "not incorrect",
        "valid",
        "ok",
    }

    positive_exact = {
        "1",
        "1.0",
        "true",
        "yes",
        "y",
        "t",
        "h",
        "hallucination",
        "hallucinated",
        "hallucination detected",
        "hallucinated answer",
        "incorrect",
        "incorrect answer",
        "wrong",
        "invalid",
    }

    unresolved = out.isna()

    negative_mask = unresolved & text.isin(negative_exact)
    positive_mask = unresolved & text.isin(positive_exact)

    out.loc[negative_mask] = 0.0
    out.loc[positive_mask] = 1.0

    # Conservative phrase handling for labels such as
    # "object hallucination" or "no hallucination".
    unresolved = out.isna()
    has_hallucination = text.str.contains(
        r"\bhallucinat(?:ion|ed|e)\b",
        regex=True,
        na=False,
    )
    explicitly_negative = text.str.contains(
        r"\b(?:no|not|non)\b.*\bhallucinat(?:ion|ed|e)\b",
        regex=True,
        na=False,
    )
    phrase_positive = (
        unresolved
        & has_hallucination
        & ~explicitly_negative
    )
    out.loc[phrase_positive] = 1.0

    # Common "not correct" / "incorrect" variants.
    unresolved = out.isna()
    incorrect_positive = text.str.contains(
        r"\b(?:incorrect|wrong|false|invalid)\b",
        regex=True,
        na=False,
    )
    correct_negative = text.str.contains(
        r"^\s*(?:correct|valid|ok)\s*$",
        regex=True,
        na=False,
    )
    out.loc[unresolved & incorrect_positive] = 1.0
    out.loc[unresolved & correct_negative] = 0.0

    return out


def _safe_reviewed_label_column(frame: pd.DataFrame):
    normalized_names = {
        str(column).strip().lower(): column
        for column in frame.columns
    }

    # 1) Prefer explicit, known label-column names.
    for candidate in _LABEL_COLUMN_CANDIDATES:
        column = normalized_names.get(candidate)
        if column is None:
            continue

        normalized = _normalize_reviewed_label_series(frame[column])
        observed = frame[column].notna()

        if int(observed.sum()) == 0:
            continue

        unresolved_count = int(normalized.loc[observed].isna().sum())
        if unresolved_count == 0 and normalized.loc[observed].nunique() == 2:
            return column, normalized

    # 2) Allow a controlled fallback to columns whose names strongly indicate
    # hallucination/correctness/review semantics, but never arbitrary columns.
    for column in frame.columns:
        name = str(column).strip().lower()
        if not any(
            token in name
            for token in (
                "halluc",
                "correct",
                "review",
                "verdict",
                "judg",
                "label",
                "target",
                "annot",
                "error",
            )
        ):
            continue

        normalized = _normalize_reviewed_label_series(frame[column])
        observed = frame[column].notna()

        if int(observed.sum()) == 0:
            continue

        unresolved_count = int(normalized.loc[observed].isna().sum())
        if unresolved_count == 0 and normalized.loc[observed].nunique() == 2:
            return column, normalized

    return None, None


download_status = {}

for model_key in ACTIVE_MODELS:
    if model_key not in SUPERVISED_LABEL_URLS:
        raise KeyError(
            f"No official supervised-label URL is configured for {model_key!r}."
        )

    destination = SUPERVISED_LABEL_FILES[model_key]

    try:
        status = _download_if_missing(
            model_key,
            SUPERVISED_LABEL_URLS[model_key],
            destination,
        )
    except Exception as exc:
        raise RuntimeError(
            f"Could not obtain the official reviewed-label file for {model_key}. "
            f"URL={SUPERVISED_LABEL_URLS[model_key]} "
            f"Error={type(exc).__name__}: {exc}"
        ) from exc

    # Inspect the official file. If its reviewed-label field uses a textual
    # or numeric representation that the later pipeline does not recognize,
    # add a canonical 0/1 'label' column to the LOCAL cached copy only.
    # The remote source is never modified.
    full_check = pd.read_csv(destination)

    if "question_id" not in full_check.columns:
        raise RuntimeError(
            f"{destination.name}: missing required question_id column."
        )

    expected_label_col = EXPECTED_REVIEWED_LABEL_COLUMNS.get(model_key)
    if expected_label_col is None:
        raise RuntimeError(
            f"No explicit reviewed-label column is configured for {model_key!r}."
        )
    if expected_label_col not in full_check.columns:
        raise RuntimeError(
            f"{destination.name}: expected reviewed-label column "
            f"{expected_label_col!r} is missing. Available columns={list(full_check.columns)}"
        )

    label_col = expected_label_col
    normalized_labels = _normalize_reviewed_label_series(full_check[label_col])

    if label_col is None:
        preview_values = {
            str(column): full_check[column]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()[:10]
            .tolist()
            for column in full_check.columns
            if any(
                token in str(column).strip().lower()
                for token in (
                    "halluc",
                    "correct",
                    "review",
                    "verdict",
                    "judg",
                    "label",
                    "target",
                    "annot",
                    "error",
                )
            )
        }

        raise RuntimeError(
            f"{destination.name}: could not safely identify a binary reviewed "
            f"hallucination-label column. Columns={list(full_check.columns)}. "
            f"Candidate-column samples={preview_values}"
        )

    # Validate the key before writing/using the file.
    if full_check["question_id"].duplicated().any():
        duplicate_ids = (
            full_check.loc[
                full_check["question_id"].duplicated(keep=False),
                "question_id",
            ]
            .astype(str)
            .drop_duplicates()
            .tolist()[:20]
        )
        raise RuntimeError(
            f"{destination.name}: duplicate question_id values were found. "
            f"Examples: {duplicate_ids}"
        )

    observed = full_check[label_col].notna()
    normalized_observed = normalized_labels.loc[observed]

    if normalized_observed.nunique() != 2:
        raise RuntimeError(
            f"{destination.name}: reviewed label column {label_col!r} does not "
            "contain both classes after safe normalization. "
            f"Observed normalized classes="
            f"{sorted(normalized_observed.dropna().unique().tolist())}"
        )

    unresolved = int(normalized_observed.isna().sum())
    # If an auxiliary manual-looking column exists, report its disagreement with
    # the explicitly selected endpoint but never silently substitute it.
    manual_disagreement_rate = None
    if (
        "is_hallucinating_manual" in full_check.columns
        and label_col != "is_hallucinating_manual"
    ):
        manual_labels = _normalize_reviewed_label_series(
            full_check["is_hallucinating_manual"]
        )
        comparable_mask = (
            normalized_labels.notna()
            & manual_labels.notna()
        )
        comparable_count = int(comparable_mask.sum())
        if comparable_count:
            manual_disagreement_rate = float(
                (
                    normalized_labels.loc[comparable_mask]
                    != manual_labels.loc[comparable_mask]
                ).mean()
            )
            print(
                f"[{model_key}] Auxiliary is_hallucinating_manual disagreement "
                f"with selected {label_col}: "
                f"{manual_disagreement_rate:.2%} over {comparable_count} comparable rows. "
                "The explicit model contract remains authoritative."
            )

    # Make the local cached file directly consumable by the unchanged
    # downstream label-loader, which recognizes canonical binary `label`.
    # Preserve the original reviewed column and add only the canonical field.
    canonical_added = False

    if "label" not in full_check.columns:
        full_check["label"] = normalized_labels.astype("Int64")
        canonical_added = True
    else:
        existing = _normalize_reviewed_label_series(full_check["label"])

        if (
            full_check["label"].notna().any()
            and not existing.loc[full_check["label"].notna()].equals(
                normalized_labels.loc[full_check["label"].notna()]
            )
        ):
            raise RuntimeError(
                f"{destination.name}: an existing 'label' column conflicts "
                "with the safely inferred reviewed-label column "
                f"{label_col!r}."
            )

    # Save only the LOCAL cache. This does not change the official GitHub file.
    if canonical_added:
        full_check.to_csv(destination, index=False)

    # Re-read once after canonicalization to verify what downstream cells
    # will actually consume.
    verified_local = pd.read_csv(destination)
    verified_label = _normalize_reviewed_label_series(
        verified_local["label"]
    )

    if (
        verified_label.loc[verified_local["label"].notna()]
        .nunique()
        != 2
    ):
        raise RuntimeError(
            f"{destination.name}: post-write verification failed; "
            "canonical 'label' column does not contain two binary classes."
        )

    download_status[model_key] = {
        "status": status,
        "path": str(destination),
        "sha256": _sha256_file(destination),
        "source_url": SUPERVISED_LABEL_URLS[model_key],
        "reviewed_label_column": str(label_col),
        "reviewed_label_column_selection": "explicit_model_contract",
        "manual_column_disagreement_rate": manual_disagreement_rate,
        "canonical_label_added_locally": bool(canonical_added),
        "rows": int(len(verified_local)),
        "labeled_rows": int(verified_local["label"].notna().sum()),
        "label_counts": {
            str(int(k)): int(v)
            for k, v in verified_label.dropna()
            .value_counts()
            .sort_index()
            .items()
        },
    }

# Bundle only the tiny local supervised-label sources.
# HALP-Bench images and model weights are intentionally not included.
with zipfile.ZipFile(
    SUPERVISED_LABEL_BUNDLE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:
    for model_key in ACTIVE_MODELS:
        src = SUPERVISED_LABEL_FILES[model_key]
        zf.write(src, arcname=src.name)

supervised_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_repository": "https://github.com/Zesearch/HALP",
    "source_directory": "FInal_CSV_Hallucination",
    "models": download_status,
    "bundle": str(SUPERVISED_LABEL_BUNDLE_PATH),
}

SUPERVISED_LABEL_MANIFEST_PATH.write_text(
    json.dumps(supervised_manifest, indent=2),
    encoding="utf-8",
)

print("Supervised-label cache:", SUPERVISED_LABEL_SOURCE_DIR)

for model_key, info in download_status.items():
    print(
        f"{model_key:12s}: {info['status']} | "
        f"reviewed_column={info['reviewed_label_column']} | "
        f"canonical_label_added="
        f"{info['canonical_label_added_locally']} | "
        f"rows={info['rows']} | "
        f"labeled={info['labeled_rows']} | "
        f"sha256={info['sha256']}"
    )
    print(f"  label_counts={info['label_counts']}")

print("Supervised-label ZIP:", SUPERVISED_LABEL_BUNDLE_PATH)

[smolvlm2] Auxiliary is_hallucinating_manual disagreement with selected is_hallucinating: 0.00% over 10000 comparable rows. The explicit model contract remains authoritative.
[qwen25vl] Auxiliary is_hallucinating_manual disagreement with selected is_hallucinating: 80.25% over 10000 comparable rows. The explicit model contract remains authoritative.
Supervised-label cache: /content/drive/MyDrive/HALP_Bench_Project/supervised_endpoint_sources
smolvlm2    : CACHED | reviewed_column=is_hallucinating | canonical_label_added=False | rows=10000 | labeled=10000 | sha256=f69c25dc155aa676ba6bc1d686711330aacc5037dc21b7542fe9a88832ab3b18
  label_counts={'0': 9009, '1': 991}
qwen25vl    : CACHED | reviewed_column=is_hallucinating | canonical_label_added=False | rows=10000 | labeled=10000 | sha256=96a548403eee0e432ca3b29a26c421eeb1d3d68b1d87dc1a7b8e9bb03fade7fa
  label_counts={'0': 1369, '1': 8631}
Supervised-label ZIP: /content/drive/MyDrive/HALP_Bench_Project/final_artifacts/supervised_endpoint_la

In [8]:

MODEL_LABEL_FILENAME_HINTS = {
    "smolvlm2": ["smolvlm_manually_reviewed.csv", "smolvlm2_manually_reviewed.csv"],
    "qwen25vl": ["qwen25vl_manually_reviewed.csv"],
}

def bounded_files(root: Path, max_depth=4):
    root = Path(root)
    results = []
    for p in root.rglob("*"):
        try:
            rel_depth = len(p.relative_to(root).parts)
        except ValueError:
            continue
        if rel_depth <= max_depth and p.is_file() and p.suffix.lower() == ".csv":
            results.append(p)
    return results

search_roots = [DATASET_DIR, DATASET_DIR.parent, SUPERVISED_LABEL_SOURCE_DIR]
label_files = []
for root in search_roots:
    if root.exists():
        for p in bounded_files(root, max_depth=4):
            n = p.name.lower()
            if "manually_reviewed" in n or "manual" in n and "review" in n:
                label_files.append(p)

label_files = sorted(set(label_files))
print("Local reviewed-label candidates:")
for p in label_files:
    print(" -", p)

# Keep supervised-target provenance explicit for official model-specific files.
# These files are explicitly named/manifests as manually reviewed.
# When the dedicated manual-review field exists, it is the only acceptable
# supervised endpoint.  The automatic `is_hallucinating` field is retained
# only for disagreement diagnostics and must not become the training target.
MODEL_REVIEWED_LABEL_COLUMNS = {
    "smolvlm2": "is_hallucinating_manual",
    "qwen25vl": "is_hallucinating_manual",
}
MODEL_AUTOMATIC_LABEL_COLUMNS = {
    "smolvlm2": "is_hallucinating",
    "qwen25vl": "is_hallucinating",
}

LABEL_SYNONYMS = [
    "hallucination", "hallucinated", "is_hallucination",
    "is_hallucinated", "hallucination_label", "label", "target", "y"
]
KEY_SYNONYMS = ["question_id", "sample_id", "id"]

def infer_label_column(frame: pd.DataFrame):
    for c in LABEL_SYNONYMS:
        if c not in frame.columns:
            continue
        vals = frame[c].dropna()
        if len(vals) == 0:
            continue
        normalized = vals.astype(str).str.strip().str.lower()
        if set(normalized.unique()).issubset({"0","1","true","false","yes","no","y","n"}):
            return c
    return None

def infer_key_column(frame: pd.DataFrame):
    for c in KEY_SYNONYMS:
        if c in frame.columns:
            return c
    return None

def normalize_binary(series: pd.Series) -> pd.Series:
    out = pd.Series(index=series.index, dtype="float64")
    s = series.astype(str).str.strip().str.lower()
    mapping = {"0":0, "false":0, "no":0, "n":0, "1":1, "true":1, "yes":1, "y":1}
    mapped = s.map(mapping)
    out.loc[mapped.notna()] = mapped[mapped.notna()].astype(float)
    return out

def load_reviewed_labels(model_key):
    hints = MODEL_LABEL_FILENAME_HINTS[model_key]
    matches = [p for p in label_files if p.name.lower() in {h.lower() for h in hints}]
    if not matches:
        kw = "qwen25vl" if model_key == "qwen25vl" else "smolvlm"
        matches = [p for p in label_files if kw in p.name.lower() and "review" in p.name.lower()]
    if not matches:
        return None, None
    if len(matches) > 1:
        matches = sorted(matches, key=lambda p: (len(str(p)), str(p).lower()))

    p = matches[0]
    frame = pd.read_csv(p)

    expected_label_col = MODEL_REVIEWED_LABEL_COLUMNS.get(model_key)
    automatic_label_col = MODEL_AUTOMATIC_LABEL_COLUMNS.get(model_key)

    # A manually-reviewed source must not silently fall back to an automatic
    # or generic binary column when the dedicated manual field is present.
    # This is a correctness gate because those columns can disagree massively.
    if expected_label_col is None:
        label_col = infer_label_column(frame)
        label_source = "fallback_synonym_inference"
    else:
        if expected_label_col in frame.columns:
            label_col = expected_label_col
            label_source = "explicit_manual_review_contract"
        elif automatic_label_col in frame.columns and "manually_reviewed" in p.name.lower():
            raise RuntimeError(
                f"{p.name}: dedicated manual-review label column "
                f"{expected_label_col!r} is missing. The automatic column "
                f"{automatic_label_col!r} exists, but it is forbidden as a silent fallback "
                "for a manually-reviewed source."
            )
        else:
            raise RuntimeError(
                f"{p.name}: expected reviewed-label column "
                f"{expected_label_col!r} is missing. Available columns={list(frame.columns)}"
            )

    key_col = infer_key_column(frame)

    automatic_disagreement = None
    if automatic_label_col in frame.columns and label_col in frame.columns:
        a = normalize_binary(frame[automatic_label_col])
        m = normalize_binary(frame[label_col])
        valid = a.notna() & m.notna()
        if bool(valid.any()):
            automatic_disagreement = float((a[valid] != m[valid]).mean())

    normalized_label_values = (
        set(frame[label_col].dropna().astype(str).str.strip().str.lower())
        if label_col else set()
    )

    info = {
        "path": str(p),
        "rows": len(frame),
        "columns": list(frame.columns),
        "label_column": label_col,
        "label_source": label_source,
        "automatic_label_column": automatic_label_col,
        "automatic_vs_reviewed_disagreement_rate": automatic_disagreement,
        "key_column": key_col,
        "label_values": sorted(normalized_label_values),
        "provenance": "model-specific manually reviewed hallucination labels; not inferred from gt_answer",
    }
    print(json.dumps(info, indent=2))
    return frame, info

LABEL_AUDIT = {}
for model_key in ACTIVE_MODELS:
    frame, info = load_reviewed_labels(model_key)
    LABEL_AUDIT[model_key] = info
    if info is None:
        print(f"\n[{model_key}] No local model-specific reviewed-label CSV found.")
    elif not info["label_column"]:
        print(f"\n[{model_key}] Label column was not safely inferred; training will be blocked.")


Local reviewed-label candidates:
 - /content/drive/MyDrive/HALP_Bench_Project/supervised_endpoint_sources/qwen25vl_manually_reviewed.csv
 - /content/drive/MyDrive/HALP_Bench_Project/supervised_endpoint_sources/smolvlm_manually_reviewed.csv
{
  "path": "/content/drive/MyDrive/HALP_Bench_Project/supervised_endpoint_sources/smolvlm_manually_reviewed.csv",
  "rows": 10000,
  "columns": [
    "question_id",
    "image_id",
    "question",
    "ground_truth_answer",
    "model_answer",
    "is_hallucinating",
    "is_hallucinating_manual",
    "label"
  ],
  "label_column": "is_hallucinating_manual",
  "label_source": "explicit_manual_review_contract",
  "automatic_label_column": "is_hallucinating",
  "automatic_vs_reviewed_disagreement_rate": 0.0,
  "key_column": "question_id",
  "label_values": [
    "false",
    "true"
  ],
  "provenance": "model-specific manually reviewed hallucination labels; not inferred from gt_answer"
}
{
  "path": "/content/drive/MyDrive/HALP_Bench_Project/supervise

## 7. Model registry

The official HALP repository evaluated eight models. This notebook configures two by default:
- **SmolVLM2-2.2B-Instruct** — compact and appropriate for Colab smoke tests.
- **Qwen2.5-VL-3B-Instruct** — the configured T4-compatible checkpoint.

The registry is kept declarative. The code never assumes that a layer number or token ID is shared across architectures.

SmolVLM2's official model card documents `AutoProcessor` + `AutoModelForImageTextToText`, while Qwen2.5-VL documents its dedicated `Qwen2_5_VLForConditionalGeneration` class and `AutoProcessor`. Current Transformers multimodal templates are used rather than older hand-built prompt strings.



In [9]:

from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any, List, Tuple

@dataclass
class ModelConfig:
    key: str
    model_id: str
    processor_id: str
    trust_remote_code: bool = False
    dtype: str = "auto"
    device_map: str = "auto"
    max_new_tokens: int = 32
    batch_size: int = 1
    layer_locations: Tuple[float, ...] = (0.0, 0.25, 0.5, 0.75, 1.0)
    generation_settings: Dict[str, Any] = None
    random_seeds: Tuple[int, ...] = (42, 52, 62)
    output_subdir: str = ""
    min_pixels: Optional[int] = None
    max_pixels: Optional[int] = None
    feature_output_name: str = ""
    label_name_keyword: str = ""

MODEL_REGISTRY = {
    "smolvlm2": ModelConfig(
        key="smolvlm2",
        model_id="HuggingFaceTB/SmolVLM2-2.2B-Instruct",
        processor_id="HuggingFaceTB/SmolVLM2-2.2B-Instruct",
        dtype="float16",
        feature_output_name="smolvlm2",
        label_name_keyword="smolvlm",
        output_subdir="smolvlm2",
        generation_settings={"max_new_tokens": 32, "do_sample": False},
    ),
    "qwen25vl": ModelConfig(
        key="qwen25vl",
        model_id="Qwen/Qwen2.5-VL-3B-Instruct",
        processor_id="Qwen/Qwen2.5-VL-3B-Instruct",
        dtype="float16",
        feature_output_name="qwen25vl",
        label_name_keyword="qwen25vl",
        output_subdir="qwen25vl",
        generation_settings={"max_new_tokens": 32, "do_sample": False},
    ),
}

for k in ACTIVE_MODELS:
    print(k, asdict(MODEL_REGISTRY[k]))

def torch_dtype_from_name(name: str):
    if name == "bfloat16":
        return torch.bfloat16
    if name == "float16":
        return torch.float16
    if name == "float32":
        return torch.float32
    return "auto"

def choose_device():
    return "cuda" if torch.cuda.is_available() else "cpu"

DEVICE = choose_device()
print("Selected runtime device:", DEVICE)


smolvlm2 {'key': 'smolvlm2', 'model_id': 'HuggingFaceTB/SmolVLM2-2.2B-Instruct', 'processor_id': 'HuggingFaceTB/SmolVLM2-2.2B-Instruct', 'trust_remote_code': False, 'dtype': 'float16', 'device_map': 'auto', 'max_new_tokens': 32, 'batch_size': 1, 'layer_locations': (0.0, 0.25, 0.5, 0.75, 1.0), 'generation_settings': {'max_new_tokens': 32, 'do_sample': False}, 'random_seeds': (42, 52, 62), 'output_subdir': 'smolvlm2', 'min_pixels': None, 'max_pixels': None, 'feature_output_name': 'smolvlm2', 'label_name_keyword': 'smolvlm'}
qwen25vl {'key': 'qwen25vl', 'model_id': 'Qwen/Qwen2.5-VL-3B-Instruct', 'processor_id': 'Qwen/Qwen2.5-VL-3B-Instruct', 'trust_remote_code': False, 'dtype': 'float16', 'device_map': 'auto', 'max_new_tokens': 32, 'batch_size': 1, 'layer_locations': (0.0, 0.25, 0.5, 0.75, 1.0), 'generation_settings': {'max_new_tokens': 32, 'do_sample': False}, 'random_seeds': (42, 52, 62), 'output_subdir': 'qwen25vl', 'min_pixels': None, 'max_pixels': None, 'feature_output_name': 'qwen25

## 8. Label/schema preparation for the selected model

This step creates the exact training table for one VLM:
- preserves the benchmark question and image name,
- joins model-specific reviewed hallucination labels,
- checks that question IDs are unique on the label side,
- rejects ambiguous joins,
- never derives a label from `gt_answer` or the question text.



In [10]:

def prepare_model_dataframe(model_key: str) -> pd.DataFrame:
    labels, info = load_reviewed_labels(model_key)
    if labels is None:
        raise FileNotFoundError(
            f"No local reviewed-label file for {model_key}. "
            "Do not infer hallucination labels from benchmark metadata."
        )

    label_col = info["label_column"]
    key_col = info["key_column"]
    if not label_col or not key_col:
        raise ValueError(
            f"Could not safely infer label/key columns for {model_key}. "
            f"Columns: {info['columns']}"
        )

    left = df.copy()
    right = labels[[key_col, label_col]].copy()
    right = right.rename(columns={key_col: "question_id", label_col: "hallucination_raw"})

    if right["question_id"].duplicated().any():
        dup = right[right["question_id"].duplicated(keep=False)]["question_id"].astype(str).tolist()[:20]
        raise ValueError(
            f"Label table has duplicate question IDs for {model_key}; refusing ambiguous join. "
            f"Examples: {dup}"
        )

    merged = left.merge(right, on="question_id", how="left", validate="one_to_one")
    merged["label"] = normalize_binary(merged["hallucination_raw"])

    unresolved = int(merged["label"].isna().sum())
    coverage = 1.0 - (unresolved / max(len(merged), 1))
    print(f"[{model_key}] label coverage: {coverage:.3%}")

    if coverage < MIN_LABEL_COVERAGE:
        raise ValueError(
            f"Only {coverage:.2%} of benchmark rows have reviewed labels for {model_key}; "
            f"minimum required coverage is {MIN_LABEL_COVERAGE:.2%}."
        )

    if unresolved:
        merged = merged.loc[merged["label"].notna()].copy()

    merged["label"] = merged["label"].astype(int)
    merged["group_id"] = merged["image_name"].astype(str)
    merged["label_source_file"] = info["path"]
    merged["label_column"] = label_col
    merged["label_semantics"] = "1 = hallucination according to the imported model-specific reviewed-label file; 0 = non-hallucination"

    if merged.empty:
        raise ValueError(f"No labeled samples remain for {model_key}.")
    if merged["label"].nunique() < 2:
        raise ValueError(
            f"Only one class remains for {model_key}; supervised classification is undefined."
        )
    return merged.reset_index(drop=True)

model_df = {}
for model_key in ACTIVE_MODELS:
    try:
        model_df[model_key] = prepare_model_dataframe(model_key)
        mdf = model_df[model_key]
        print(f"\n{model_key}: {len(mdf)} labeled rows")
        display(mdf["label"].value_counts().sort_index().rename("count").to_frame())
    except Exception as e:
        print(f"\n[{model_key}] DATASET BLOCKED: {type(e).__name__}: {e}")


{
  "path": "/content/drive/MyDrive/HALP_Bench_Project/supervised_endpoint_sources/smolvlm_manually_reviewed.csv",
  "rows": 10000,
  "columns": [
    "question_id",
    "image_id",
    "question",
    "ground_truth_answer",
    "model_answer",
    "is_hallucinating",
    "is_hallucinating_manual",
    "label"
  ],
  "label_column": "is_hallucinating_manual",
  "label_source": "explicit_manual_review_contract",
  "automatic_label_column": "is_hallucinating",
  "automatic_vs_reviewed_disagreement_rate": 0.0,
  "key_column": "question_id",
  "label_values": [
    "false",
    "true"
  ],
  "provenance": "model-specific manually reviewed hallucination labels; not inferred from gt_answer"
}
[smolvlm2] label coverage: 100.000%

smolvlm2: 10000 labeled rows


,count
label,
0,9009
1,991


{
  "path": "/content/drive/MyDrive/HALP_Bench_Project/supervised_endpoint_sources/qwen25vl_manually_reviewed.csv",
  "rows": 10000,
  "columns": [
    "question_id",
    "image_id",
    "question",
    "ground_truth_answer",
    "model_answer",
    "is_hallucinating",
    "is_hallucinating_manual",
    "label"
  ],
  "label_column": "is_hallucinating_manual",
  "label_source": "explicit_manual_review_contract",
  "automatic_label_column": "is_hallucinating",
  "automatic_vs_reviewed_disagreement_rate": 0.8025,
  "key_column": "question_id",
  "label_values": [
    "false",
    "true"
  ],
  "provenance": "model-specific manually reviewed hallucination labels; not inferred from gt_answer"
}
[qwen25vl] label coverage: 100.000%

qwen25vl: 10000 labeled rows


,count
label,
0,9394
1,606


## 9. Leakage-safe grouped train / validation / test split

A row-wise random split is unsafe when multiple questions use the same image. The default grouping key is therefore `image_name`.

The split procedure:
1. uses groups, never individual rows;
2. chooses a seed-controlled split whose class prevalence is reasonably close to the full labeled data;
3. verifies zero group overlap;
4. leaves the test set untouched for final evaluation.

This does not claim to reproduce an official hidden benchmark split unless the local reviewed-label data explicitly contains one.



In [11]:

from sklearn.model_selection import GroupShuffleSplit

def choose_grouped_split(frame, seed=42, test_size=0.20, val_size_within_train=0.20, trials=25):
    frame = frame.reset_index(drop=True)
    groups = frame["group_id"].astype(str).values
    y = frame["label"].values
    target_prev = y.mean()

    best = None
    for trial in range(trials):
        rs = seed + trial
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=rs)
        trainval_idx, test_idx = next(gss.split(frame, y, groups=groups))

        trainval = frame.iloc[trainval_idx]
        gss2 = GroupShuffleSplit(
            n_splits=1, test_size=val_size_within_train, random_state=rs + 1000
        )
        tr_rel, va_rel = next(
            gss2.split(trainval, trainval["label"].values, groups=trainval["group_id"].astype(str).values)
        )
        train_idx = trainval_idx[tr_rel]
        val_idx = trainval_idx[va_rel]

        stats = []
        valid = True
        for idx in [train_idx, val_idx, test_idx]:
            vals = y[idx]
            if len(np.unique(vals)) < 2:
                valid = False
                break
            stats.append(abs(vals.mean() - target_prev))
        if not valid:
            continue
        score = sum(stats)
        if best is None or score < best[0]:
            best = (score, train_idx, val_idx, test_idx)

    if best is None:
        raise RuntimeError("Could not create a valid grouped split with both classes in all partitions.")

    _, tr, va, te = best
    train = frame.iloc[tr].copy()
    val = frame.iloc[va].copy()
    test = frame.iloc[te].copy()

    def groupset(x): return set(x["group_id"].astype(str))
    assert groupset(train).isdisjoint(groupset(val))
    assert groupset(train).isdisjoint(groupset(test))
    assert groupset(val).isdisjoint(groupset(test))

    return train, val, test

def select_experiment_rows(frame: pd.DataFrame) -> pd.DataFrame:
    """Select the exact fixed-size cohort used by both extraction and probes."""
    frame = frame.copy().reset_index(drop=True)

    if RUN_MODE not in {"smoke_test", "full"}:
        raise ValueError(
            f"Unsupported RUN_MODE={RUN_MODE!r}. Expected 'smoke_test' or 'full'."
        )

    if RUN_MODE == "smoke_test":
        limit = 32 if MAX_SAMPLES is None else int(MAX_SAMPLES)
    elif MAX_SAMPLES is None:
        return frame
    else:
        limit = int(MAX_SAMPLES)

    if limit <= 0:
        raise ValueError("MAX_SAMPLES must be positive when specified.")

    if limit >= len(frame):
        return frame.reset_index(drop=True)

    return frame.sample(
        n=limit,
        random_state=int(SAMPLE_SELECTION_SEED),
    ).reset_index(drop=True)


experiment_model_df = {}
splits = {}

for model_key, mdf in model_df.items():
    experiment_frame = select_experiment_rows(mdf)

    if MAX_SAMPLES is not None and len(experiment_frame) > int(MAX_SAMPLES):
        raise AssertionError(
            f"{model_key}: experiment subset exceeded MAX_SAMPLES."
        )

    experiment_model_df[model_key] = experiment_frame.copy()

    tr, va, te = choose_grouped_split(
        experiment_frame,
        seed=SEEDS[0],
    )

    splits[model_key] = {
        "train": tr,
        "val": va,
        "test": te,
    }

    print(f"\n{model_key}")
    print("experiment rows:", len(experiment_frame))
    print("rows:", {"train": len(tr), "val": len(va), "test": len(te)})
    print(
        "unique images:",
        {
            "train": tr.group_id.nunique(),
            "val": va.group_id.nunique(),
            "test": te.group_id.nunique(),
        },
    )
    print(
        "positive rate:",
        {
            "train": tr.label.mean(),
            "val": va.label.mean(),
            "test": te.label.mean(),
        },
    )

    split_rows = pd.DataFrame([
        {
            "model": model_key,
            "split": name,
            "rows": len(part),
            "unique_images": part["group_id"].nunique(),
            "positive_count": int((part["label"] == 1).sum()),
            "negative_count": int((part["label"] == 0).sum()),
            "positive_rate": float(part["label"].mean()),
        }
        for name, part in [("train", tr), ("val", va), ("test", te)]
    ])

    RUN_SPLIT_SUMMARIES = globals().get("RUN_SPLIT_SUMMARIES", [])
    RUN_SPLIT_SUMMARIES.append(split_rows)
    display(split_rows)



smolvlm2
experiment rows: 8000
rows: {'train': 5105, 'val': 1255, 'test': 1640}
unique images: {'train': 2748, 'val': 688, 'test': 859}
positive rate: {'train': np.float64(0.09911851126346718), 'val': np.float64(0.09880478087649402), 'test': np.float64(0.09695121951219512)}


,model,split,rows,unique_images,positive_count,negative_count,positive_rate
0,smolvlm2,train,5105,2748,506,4599,0.099119
1,smolvlm2,val,1255,688,124,1131,0.098805
2,smolvlm2,test,1640,859,159,1481,0.096951



qwen25vl
experiment rows: 8000
rows: {'train': 5055, 'val': 1282, 'test': 1663}
unique images: {'train': 2748, 'val': 688, 'test': 859}
positive rate: {'train': np.float64(0.06350148367952523), 'val': np.float64(0.06474258970358815), 'test': np.float64(0.06434155141310884)}


,model,split,rows,unique_images,positive_count,negative_count,positive_rate
0,qwen25vl,train,5055,2748,321,4734,0.063501
1,qwen25vl,val,1282,688,83,1199,0.064743
2,qwen25vl,test,1663,859,107,1556,0.064342


## 10. Model loader and runtime API smoke-test helpers

Before full extraction the notebook:
- loads the processor,
- loads the model,
- formats one image-question pair using the official processor,
- runs one forward pass with `output_hidden_states=True`,
- inspects tensor shapes,
- checks that an image-token position exists when the model exposes one.

No generation is used.



In [12]:

from transformers import AutoProcessor, AutoModelForImageTextToText

LOADED = {}

def get_model_device(model):
    try:
        return model.device
    except Exception:
        for p in model.parameters():
            return p.device
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_vlm(config: ModelConfig):
    kwargs = {
        "torch_dtype": torch_dtype_from_name(config.dtype),
        "device_map": config.device_map,
    }
    if config.trust_remote_code:
        kwargs["trust_remote_code"] = True

    processor = AutoProcessor.from_pretrained(config.processor_id)

    def _load(load_kwargs):
        if config.key == "qwen25vl":
            # Prefer the current auto-class shown by the Qwen model card,
            # falling back to the architecture-specific class when needed.
            try:
                from transformers import AutoModelForMultimodalLM
                return AutoModelForMultimodalLM.from_pretrained(config.model_id, **load_kwargs)
            except (ImportError, AttributeError):
                from transformers import Qwen2_5_VLForConditionalGeneration
                return Qwen2_5_VLForConditionalGeneration.from_pretrained(config.model_id, **load_kwargs)
        return AutoModelForImageTextToText.from_pretrained(config.model_id, **load_kwargs)

    try:
        model = _load(kwargs)
    except torch.cuda.OutOfMemoryError as exc:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        raise RuntimeError(
            f"{config.key}: CUDA OOM during model loading under the configured "
            "T4-safe policy. No silent CPU fallback is used because it would "
            "change the intended hardware/runtime envelope."
        ) from exc

    model.eval()
    parameter_count = sum(p.numel() for p in model.parameters())
    print(
        f"[{config.key}] parameters={parameter_count:,} "
        f"dtype={next(model.parameters()).dtype} device={get_model_device(model)}"
    )
    return processor, model

def build_messages(model_key, image_path, question):
    image_uri = Path(image_path).resolve().as_uri()
    if model_key == "qwen25vl":
        return [{
            "role": "user",
            "content": [
                {"type": "image", "image": image_uri},
                {"type": "text", "text": str(question)},
            ],
        }]
    return [{
        "role": "user",
        "content": [
            {"type": "image", "url": image_uri},
            {"type": "text", "text": str(question)},
        ],
    }]

def prepare_inputs(model_key, processor, image_path, question):
    messages = build_messages(model_key, image_path, question)

    if model_key == "qwen25vl":
        # Use Qwen's documented chat template path plus qwen-vl-utils when required.
        try:
            from qwen_vl_utils import process_vision_info
        except ImportError as e:
            raise ImportError(
                "qwen-vl-utils is required for the configured Qwen2.5-VL adapter."
            ) from e
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
    else:
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=False,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
    return inputs

def move_inputs_to_model(inputs, model):
    target = get_model_device(model)
    moved = {}
    for k, v in inputs.items():
        if isinstance(v, torch.Tensor):
            moved[k] = v.to(target)
        else:
            moved[k] = v
    return moved

def get_hidden_state_layers(outputs):
    hs = getattr(outputs, "hidden_states", None)
    if hs is None:
        raise RuntimeError("Model output did not expose hidden_states=True.")
    if not isinstance(hs, (tuple, list)) or len(hs) < 2:
        raise RuntimeError(f"Unexpected hidden_states container: {type(hs)} / length={len(hs) if hasattr(hs,'__len__') else 'NA'}")
    return hs

def decoder_layer_count(model, hidden_states):
    # hidden_states normally contains embeddings + one output per decoder block.
    return len(hidden_states) - 1

def selected_layer_indices(num_layers):
    # Match the HALP convention: {1, L/4, L/2, 3L/4, L}.
    def halp_fraction_layer(fraction):
        return max(1, min(num_layers, int(math.floor(fraction * num_layers))))

    raw = {
        "early": 1,
        "quarter": halp_fraction_layer(0.25),
        "middle": halp_fraction_layer(0.50),
        "three_quarter": halp_fraction_layer(0.75),
        "final": num_layers,
    }
    return dict(sorted(raw.items(), key=lambda kv: kv[1]))

def token_position_from_id(inputs, token_id, prefer="last"):
    ids = inputs.get("input_ids")
    if ids is None or token_id is None:
        return None
    row = ids[0]
    pos = torch.nonzero(row == int(token_id), as_tuple=False).flatten().tolist()
    if not pos:
        return None
    return pos[-1] if prefer == "last" else pos[0]

def query_position(inputs, processor):
    ids = inputs["input_ids"][0]
    attn = inputs.get("attention_mask")
    if attn is not None:
        valid = torch.nonzero(attn[0].bool(), as_tuple=False).flatten().tolist()
        return valid[-1] if valid else None
    pad_id = getattr(processor.tokenizer, "pad_token_id", None)
    valid = [i for i, t in enumerate(ids.tolist()) if pad_id is None or t != pad_id]
    return valid[-1] if valid else None

# Qwen helper dependency check.
# Install qwen-vl-utils only when it is actually missing.
# This keeps the cell self-contained and avoids repeated installation
# when the helper is already available in the current Colab runtime.

if "qwen25vl" in ACTIVE_MODELS:
    try:
        import qwen_vl_utils  # noqa: F401
        print("qwen-vl-utils import: PASS")

    except ModuleNotFoundError:
        import subprocess
        import sys

        print("qwen-vl-utils is missing; installing it once...")

        completed = subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--disable-pip-version-check",
                "--no-cache-dir",
                "qwen-vl-utils",
            ],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )

        print(completed.stdout)

        if completed.returncode != 0:
            raise RuntimeError(
                "Failed to install qwen-vl-utils. "
                f"pip exit code: {completed.returncode}"
            )

        try:
            import qwen_vl_utils  # noqa: F401
        except ModuleNotFoundError as exc:
            raise ImportError(
                "qwen-vl-utils was installed but still cannot be imported "
                "in the current Python kernel."
            ) from exc

        print("qwen-vl-utils install/import: PASS")

else:
    print("Qwen helper not needed for the active model set.")

# ============================================================
# 10. Model smoke test — T4-safe / low-memory / SmolVLM2 FIXED
# ============================================================
#
# Purpose
# -------
# Verify that each configured VLM can perform:
#
#     local HALP-Bench image
#            +
#         question
#            ↓
#      official processor
#            ↓
#       one forward pass
#            ↓
#       hidden states
#
# This is a PRE-GENERATION smoke test.
# This cell does not generate answers.
#
#
# Important T4 / SmolVLM2 fix
# ---------------------------
# The previous implementation forced SmolVLM2 preprocessing to 336px while
# leaving its checkpoint processor configuration at image_seq_len=81.
#
# For SmolVLM2-2.2B-Instruct, the official configuration uses:
#
#     vision image_size = 384
#     vision patch_size = 14
#     scale_factor      = 3
#     image_seq_len     = 81
#
# The model computes the actual visual-token block size from its vision output.
# So, changing the actual padded image grid to 336px while still inserting
# 81 <image> tokens creates:
#
#     actual image patch tokens != processor <image> token count
#
# which reaches the model's inputs_merger and causes:
#
#     ValueError:
#     At least one sample has <image> tokens not divisible by patch_size.
#
# The correct fix is not to catch or suppress that error.
# The correct fix is to use a model-compatible 384px padded image for SmolVLM2.
#
# We therefore intentionally use:
#
#     SMOL_T4_IMAGE_EDGE     = 384
#     SMOL_T4_IMAGE_SPLITTING = False
#
# This remains T4-safe because only ONE 384px image is processed in the smoke
# test, image splitting is disabled, and the model placement leaves VRAM for
# activations.
#
# The global SMOL_T4_IMAGE_EDGE is kept synchronized here because the same helper
# is reused by the later representation/extraction cells.
#
# ============================================================

import gc
import inspect
import math
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image


print("=" * 80)
print("HALP-Bench VLM smoke test — T4-safe / FIXED")
print("=" * 80)


# ============================================================
# 1. Notebook objects we need
# ============================================================

REQUIRED_GLOBALS = [
    "ACTIVE_MODELS",
    "MODEL_REGISTRY",
    "df",
]

missing_globals = [
    name
    for name in REQUIRED_GLOBALS
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        "Missing required notebook objects:\n"
        f"  {missing_globals}\n\n"
        "Run the environment, Drive/dataset, and model-registry "
        "cells first."
    )


if not isinstance(
    ACTIVE_MODELS,
    (list, tuple),
):
    raise TypeError(
        "ACTIVE_MODELS must be a list or tuple."
    )


if not ACTIVE_MODELS:
    raise RuntimeError(
        "ACTIVE_MODELS is empty."
    )


if not isinstance(
    MODEL_REGISTRY,
    dict,
):
    raise TypeError(
        "MODEL_REGISTRY must be a dictionary."
    )


if not isinstance(
    df,
    pd.DataFrame,
):
    raise TypeError(
        "`df` must be a pandas DataFrame."
    )


if df.empty:
    raise RuntimeError(
        "The validated HALP-Bench dataframe is empty."
    )


# ============================================================
# 2. Validate benchmark schema
# ============================================================

REQUIRED_BENCHMARK_COLUMNS = {
    "question",
    "image_name",
    "resolved_image_path",
}

missing_columns = (
    REQUIRED_BENCHMARK_COLUMNS
    - set(df.columns)
)

if missing_columns:
    raise RuntimeError(
        "The validated HALP-Bench dataframe is missing "
        f"columns: {sorted(missing_columns)}"
    )


print(
    "Validated HALP-Bench rows:",
    len(df),
)


# ============================================================
# 3. HARD-CORRECT SmolVLM2 preprocessing configuration
# ============================================================
#
# Important:
# This is the key fix for the observed ValueError.
#
# Do not use 336px for SmolVLM2-2.2B-Instruct here.
#
# The checkpoint's processor/model are configured around 384px,
# patch_size=14, scale_factor=3 and image_seq_len=81.
#
# Keep image splitting OFF for the T4 smoke test so the image remains
# a single bounded visual input.
#
# ============================================================

SMOL_T4_IMAGE_EDGE = 384
SMOL_T4_IMAGE_SPLITTING = False

# Update the notebook-global values as well because later extraction helpers
# intentionally reuse these variables.
globals()["SMOL_T4_IMAGE_EDGE"] = SMOL_T4_IMAGE_EDGE
globals()["SMOL_T4_IMAGE_SPLITTING"] = SMOL_T4_IMAGE_SPLITTING

print(
    "\nSmolVLM2 smoke preprocessing:",
    f"{SMOL_T4_IMAGE_EDGE}px / "
    f"image_splitting={SMOL_T4_IMAGE_SPLITTING}",
)


# ============================================================
# 4. CUDA / T4 diagnostics
# ============================================================

CUDA_AVAILABLE = bool(
    torch.cuda.is_available()
)

print("\n" + "=" * 80)
print("GPU diagnostics")
print("=" * 80)

print(
    "CUDA available:",
    CUDA_AVAILABLE,
)

if CUDA_AVAILABLE:

    GPU_INDEX = 0

    GPU_NAME = torch.cuda.get_device_name(
        GPU_INDEX
    )

    GPU_PROPERTIES = (
        torch.cuda.get_device_properties(
            GPU_INDEX
        )
    )

    GPU_TOTAL_GIB = (
        GPU_PROPERTIES.total_memory
        / (2 ** 30)
    )

    try:

        FREE_BYTES, TOTAL_BYTES = (
            torch.cuda.mem_get_info(
                GPU_INDEX
            )
        )

        GPU_FREE_GIB = (
            FREE_BYTES
            / (2 ** 30)
        )

        RUNTIME_TOTAL_GIB = (
            TOTAL_BYTES
            / (2 ** 30)
        )

    except Exception:

        GPU_FREE_GIB = float("nan")
        RUNTIME_TOTAL_GIB = GPU_TOTAL_GIB

    print(
        "GPU              :",
        GPU_NAME,
    )

    print(
        f"VRAM total       : "
        f"{GPU_TOTAL_GIB:.2f} GiB"
    )

    if np.isfinite(
        GPU_FREE_GIB
    ):

        print(
            f"VRAM free        : "
            f"{GPU_FREE_GIB:.2f} GiB"
        )

    print(
        "Compute capability:",
        (
            f"{GPU_PROPERTIES.major}."
            f"{GPU_PROPERTIES.minor}"
        ),
    )

else:

    GPU_INDEX = None
    GPU_NAME = None
    GPU_TOTAL_GIB = None
    GPU_FREE_GIB = None
    RUNTIME_TOTAL_GIB = None

    print(
        "CUDA is unavailable."
    )

    print(
        "The smoke test will not attempt a large CPU VLM forward pass."
    )


# ============================================================
# 5. Explicit T4 detection
# ============================================================

IS_T4 = bool(
    CUDA_AVAILABLE
    and GPU_NAME is not None
    and "T4" in GPU_NAME.upper()
)

print(
    "Detected Tesla T4 :",
    IS_T4,
)


# ============================================================
# 6. Model cache
# ============================================================

if "LOADED" not in globals():
    LOADED = {}

if not isinstance(
    LOADED,
    dict,
):
    raise TypeError(
        "`LOADED` must be a dictionary."
    )


# ============================================================
# 7. Release stale VLM objects
# ============================================================

print("\n" + "=" * 80)
print("Releasing stale VLM objects")
print("=" * 80)


def release_cached_model(
    model_key: str,
):
    """
    Release one cached model and clear its CUDA allocations.
    """

    if (
        "LOADED" in globals()
        and isinstance(
            LOADED,
            dict,
        )
        and model_key in LOADED
    ):

        cached = LOADED.pop(
            model_key
        )

        if isinstance(
            cached,
            dict,
        ):

            cached_model = cached.get(
                "model"
            )

            cached_processor = cached.get(
                "processor"
            )

            if cached_model is not None:
                del cached_model

            if cached_processor is not None:
                del cached_processor

        del cached

        print(
            f"Released cached model: {model_key}"
        )


for key in list(
    LOADED.keys()
):
    release_cached_model(
        key
    )


if "SMOKE" in globals():

    try:

        del SMOKE

        print(
            "Released previous SMOKE object."
        )

    except Exception:

        pass


if "SMOKE_STATUS" in globals():

    try:

        del SMOKE_STATUS

    except Exception:

        pass


gc.collect()


if CUDA_AVAILABLE:

    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass


if CUDA_AVAILABLE:

    try:

        free_after_cleanup, total_after_cleanup = (
            torch.cuda.mem_get_info(
                GPU_INDEX
            )
        )

        print(
            f"VRAM free after cleanup: "
            f"{free_after_cleanup / (2 ** 30):.2f} GiB"
        )

    except Exception:

        pass


# ============================================================
# 8. Select smoke-test sample
# ============================================================

def choose_smoke_sample(
    model_key: str,
) -> pd.Series:
    """
    Select one valid local image-question pair.

    Hallucination labels are not required.
    """

    candidates = []

    # Prefer model-specific dataframe when it exists.
    if (
        "model_df" in globals()
        and isinstance(
            model_df,
            dict,
        )
        and model_key in model_df
        and isinstance(
            model_df[model_key],
            pd.DataFrame,
        )
        and not model_df[model_key].empty
    ):

        candidates.append(
            (
                "model-specific dataframe",
                model_df[model_key],
            )
        )

    # Always allow the validated public benchmark dataframe.
    candidates.append(
        (
            "validated HALP-Bench dataframe",
            df,
        )
    )

    for source_name, frame in candidates:

        working = frame.reset_index(
            drop=True
        )

        for _, row in working.iterrows():

            image_value = row.get(
                "resolved_image_path"
            )

            question_value = row.get(
                "question"
            )

            if (
                image_value is None
                or pd.isna(
                    image_value
                )
            ):
                continue

            if (
                question_value is None
                or pd.isna(
                    question_value
                )
            ):
                continue

            image_path = Path(
                str(image_value)
            )

            if not image_path.exists():
                continue

            if not image_path.is_file():
                continue

            sample = row.copy()

            sample[
                "_smoke_source"
            ] = source_name

            return sample

    raise FileNotFoundError(
        f"[{model_key}] No valid local HALP-Bench "
        "image/question pair was found."
    )


# ============================================================
# 9. Local image validation
# ============================================================

def validate_local_image(
    image_path: Path,
):
    """
    Validate local image bytes and return metadata.
    """

    image_path = Path(
        image_path
    ).resolve()

    if not image_path.exists():

        raise FileNotFoundError(
            f"Image does not exist: {image_path}"
        )

    if not image_path.is_file():

        raise FileNotFoundError(
            f"Image is not a regular file: {image_path}"
        )

    with Image.open(
        image_path
    ) as image:

        image.verify()

    with Image.open(
        image_path
    ) as image:

        return {
            "mode": image.mode,
            "size": tuple(
                image.size
            ),
            "format": image.format,
        }


# ============================================================
# 10. SmolVLM2 compatibility helpers
# ============================================================

def _get_smol_processor_model_params(
    processor,
    model,
):
    """
    Inspect the checkpoint's image/token geometry.

    Returns:
        processor_image_seq_len
        model_image_seq_len
        vision_image_size
        vision_patch_size
        scale_factor
    """

    processor_image_seq_len = getattr(
        processor,
        "image_seq_len",
        None,
    )

    model_image_seq_len = getattr(
        model,
        "image_seq_len",
        None,
    )

    vision_config = getattr(
        getattr(
            model,
            "config",
            None,
        ),
        "vision_config",
        None,
    )

    vision_image_size = getattr(
        vision_config,
        "image_size",
        None,
    )

    vision_patch_size = getattr(
        vision_config,
        "patch_size",
        None,
    )

    scale_factor = getattr(
        getattr(
            model,
            "config",
            None,
        ),
        "scale_factor",
        None,
    )

    return {
        "processor_image_seq_len": (
            None
            if processor_image_seq_len is None
            else int(
                processor_image_seq_len
            )
        ),
        "model_image_seq_len": (
            None
            if model_image_seq_len is None
            else int(
                model_image_seq_len
            )
        ),
        "vision_image_size": (
            None
            if vision_image_size is None
            else int(
                vision_image_size
            )
        ),
        "vision_patch_size": (
            None
            if vision_patch_size is None
            else int(
                vision_patch_size
            )
        ),
        "scale_factor": (
            None
            if scale_factor is None
            else int(
                scale_factor
            )
        ),
    }


def _validate_smol_geometry(
    processor,
    model,
):
    """
    Verify that processor/model geometry is internally consistent.

    This is the critical guard against the previously observed
    '<image> tokens not divisible by patch_size' failure.
    """

    params = (
        _get_smol_processor_model_params(
            processor,
            model,
        )
    )

    print(
        "\nSmolVLM2 geometry diagnostics:"
    )

    for key, value in params.items():

        print(
            f"  {key:24s}: {value}"
        )

    processor_seq = (
        params[
            "processor_image_seq_len"
        ]
    )

    model_seq = (
        params[
            "model_image_seq_len"
        ]
    )

    image_size = (
        params[
            "vision_image_size"
        ]
    )

    patch_size = (
        params[
            "vision_patch_size"
        ]
    )

    scale_factor = (
        params[
            "scale_factor"
        ]
    )

    if (
        processor_seq is not None
        and model_seq is not None
        and processor_seq != model_seq
    ):

        raise RuntimeError(
            "SmolVLM2 processor/model image_seq_len mismatch: "
            f"processor={processor_seq}, "
            f"model={model_seq}."
        )

    if (
        image_size is not None
        and patch_size is not None
        and scale_factor is not None
    ):

        theoretical_seq = int(
            (
                (
                    image_size
                    // patch_size
                )
                ** 2
            )
            / (
                scale_factor
                ** 2
            )
        )

        print(
            "  theoretical image_seq_len:",
            theoretical_seq,
        )

        if (
            model_seq is not None
            and theoretical_seq != model_seq
        ):

            raise RuntimeError(
                "SmolVLM2 checkpoint geometry is internally inconsistent: "
                f"theoretical={theoretical_seq}, "
                f"model.image_seq_len={model_seq}."
            )

    return params


def _validate_smol_processor_output(
    inputs,
    model,
):
    """
    Validate the actual processor output before forwarding it to the model.
    """

    if inputs is None:
        raise RuntimeError(
            "SmolVLM2 processor returned None."
        )

    if not hasattr(
        inputs,
        "keys",
    ):

        raise TypeError(
            "SmolVLM2 processor returned a non-mapping object: "
            f"{type(inputs)}"
        )

    if "input_ids" not in inputs:
        raise RuntimeError(
            "SmolVLM2 processor output does not contain input_ids."
        )

    input_ids = inputs[
        "input_ids"
    ]

    if not isinstance(
        input_ids,
        torch.Tensor,
    ):

        raise TypeError(
            "input_ids is not a torch.Tensor."
        )

    if input_ids.ndim != 2:
        raise RuntimeError(
            "SmolVLM2 input_ids must have rank 2, got "
            f"{input_ids.ndim}."
        )

    if input_ids.shape[0] != 1:
        raise RuntimeError(
            "Smoke test requires batch size 1."
        )

    if "pixel_values" not in inputs:
        raise RuntimeError(
            "SmolVLM2 processor output does not contain pixel_values."
        )

    pixel_values = inputs[
        "pixel_values"
    ]

    if not isinstance(
        pixel_values,
        torch.Tensor,
    ):

        raise TypeError(
            "pixel_values is not a torch.Tensor."
        )

    # Official SmolVLM processing normally produces:
    #
    #     [batch, num_images, channels, height, width]
    #
    if pixel_values.ndim != 5:

        raise RuntimeError(
            "Expected SmolVLM2 pixel_values with rank 5 "
            "[B,N,C,H,W], got "
            f"{tuple(pixel_values.shape)}"
        )

    batch_size = int(
        pixel_values.shape[0]
    )

    num_images = int(
        pixel_values.shape[1]
    )

    channels = int(
        pixel_values.shape[2]
    )

    height = int(
        pixel_values.shape[3]
    )

    width = int(
        pixel_values.shape[4]
    )

    if batch_size != 1:
        raise RuntimeError(
            f"Expected batch size 1, got {batch_size}."
        )

    if num_images < 1:
        raise RuntimeError(
            "SmolVLM2 produced zero processed images."
        )

    if channels != 3:
        raise RuntimeError(
            f"Expected RGB channels=3, got {channels}."
        )

    print(
        "\nActual processor pixel_values:",
        tuple(
            pixel_values.shape
        ),
    )

    print(
        "Actual pixel dtype:",
        pixel_values.dtype,
    )

    # The smoke configuration intentionally uses 384px and no splitting.
    #
    # With do_pad=True, this image should be represented by a single 384x384
    # vision input. This is the geometry compatible with image_seq_len=81.
    if (
        height != SMOL_T4_IMAGE_EDGE
        or width != SMOL_T4_IMAGE_EDGE
    ):

        raise RuntimeError(
            "SmolVLM2 T4 smoke input did not produce the expected "
            f"{SMOL_T4_IMAGE_EDGE}x{SMOL_T4_IMAGE_EDGE} padded image. "
            f"Actual shape={tuple(pixel_values.shape)}"
        )

    # Inspect image-token count.
    model_config = getattr(
        model,
        "config",
        None,
    )

    image_token_id = getattr(
        model_config,
        "image_token_id",
        None,
    )

    if image_token_id is not None:

        image_token_count = int(
            (
                input_ids
                == int(image_token_id)
            ).sum().item()
        )

        print(
            "Actual <image> token count:",
            image_token_count,
        )

        processor_image_seq_len = getattr(
            model,
            "image_seq_len",
            None,
        )

        if processor_image_seq_len is not None:

            expected_token_count = int(
                processor_image_seq_len
                * num_images
            )

            if (
                image_token_count
                != expected_token_count
            ):

                raise RuntimeError(
                    "SmolVLM2 image-token count is inconsistent with "
                    "the model image_seq_len: "
                    f"actual={image_token_count}, "
                    f"expected={expected_token_count}, "
                    f"num_images={num_images}."
                )

    print(
        "SmolVLM2 processor-output validation: PASS"
    )


# ============================================================
# 11. T4-safe SmolVLM2 input preparation
# ============================================================

def prepare_t4_smolvlm2_inputs(
    processor,
    image_path: Path,
    question: str,
):
    """
    Prepare ONE SmolVLM2 smoke-test sample using the checkpoint-compatible
    384px / no-splitting profile.

    This function deliberately does NOT use 336px.

    The processor's image_seq_len remains aligned with the model's 384px
    vision geometry, preventing the exact ValueError seen previously.
    """

    image_path = Path(
        image_path
    ).resolve()

    question = str(
        question
    ).strip()

    if not question:
        raise ValueError(
            "Question is empty."
        )

    if not image_path.exists():
        raise FileNotFoundError(
            f"Image does not exist: {image_path}"
        )

    if not hasattr(
        processor,
        "image_processor",
    ):

        raise RuntimeError(
            "The SmolVLM2 processor does not expose image_processor."
        )

    image_processor = (
        processor.image_processor
    )

    # Save all mutable processor settings and restore them afterwards.
    original_settings = {
        "do_resize": getattr(
            image_processor,
            "do_resize",
            None,
        ),
        "size": getattr(
            image_processor,
            "size",
            None,
        ),
        "max_image_size": getattr(
            image_processor,
            "max_image_size",
            None,
        ),
        "do_image_splitting": getattr(
            image_processor,
            "do_image_splitting",
            None,
        ),
        "do_pad": getattr(
            image_processor,
            "do_pad",
            None,
        ),
    }

    # Important:
    # We use the official 384px geometry.
    image_processor.do_resize = True

    image_processor.size = {
        "longest_edge": int(
            SMOL_T4_IMAGE_EDGE
        )
    }

    image_processor.max_image_size = {
        "longest_edge": int(
            SMOL_T4_IMAGE_EDGE
        )
    }

    image_processor.do_image_splitting = (
        False
    )

    # Keep padding enabled if the installed processor exposes it.
    # This is important for preserving a square 384x384 vision input.
    if original_settings[
        "do_pad"
    ] is not None:

        image_processor.do_pad = True

    try:

        # Current SmolVLM expects the image to be part of the multimodal
        # conversation. Do not pass images= separately to apply_chat_template,
        # otherwise the image can be supplied twice.
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "path": str(
                            image_path
                        ),
                    },
                    {
                        "type": "text",
                        "text": question,
                    },
                ],
            }
        ]

        # Transformers versions differ in where processor-only kwargs belong.
        # Prefer processor_kwargs when supported.
        try:

            signature = inspect.signature(
                processor.apply_chat_template
            )

            parameters = signature.parameters

            if (
                "processor_kwargs"
                in parameters
            ):

                inputs = (
                    processor.apply_chat_template(
                        messages,
                        add_generation_prompt=False,
                        tokenize=True,
                        return_dict=True,
                        processor_kwargs={
                            "return_tensors": "pt",
                        },
                    )
                )

            else:

                inputs = (
                    processor.apply_chat_template(
                        messages,
                        add_generation_prompt=False,
                        tokenize=True,
                        return_dict=True,
                        return_tensors="pt",
                    )
                )

        except (
            TypeError,
            ValueError,
        ):

            # Conservative compatibility fallback for older/newer processor
            # signatures.
            inputs = (
                processor.apply_chat_template(
                    messages,
                    add_generation_prompt=False,
                    tokenize=True,
                    return_dict=True,
                    return_tensors="pt",
                )
            )

    finally:

        # Restore the processor immediately so the smoke test does not
        # unexpectedly mutate the caller's processor configuration.
        image_processor.do_resize = (
            original_settings[
                "do_resize"
            ]
        )

        image_processor.size = (
            original_settings[
                "size"
            ]
        )

        image_processor.max_image_size = (
            original_settings[
                "max_image_size"
            ]
        )

        image_processor.do_image_splitting = (
            original_settings[
                "do_image_splitting"
            ]
        )

        if original_settings[
            "do_pad"
        ] is not None:

            image_processor.do_pad = (
                original_settings[
                    "do_pad"
                ]
            )

    return inputs


# ============================================================
# 12. Qwen2.5-VL smoke-test adapter
# ============================================================

def prepare_qwen25vl_inputs(
    processor,
    image_path: Path,
    question: str,
):
    """
    Prepare Qwen2.5-VL inputs using its architecture-specific API.
    """

    try:

        from qwen_vl_utils import (
            process_vision_info,
        )

    except ImportError as exc:

        raise ImportError(
            "Qwen2.5-VL requires qwen-vl-utils."
        ) from exc

    image_path = Path(
        image_path
    ).resolve()

    question = str(
        question
    ).strip()

    if not question:
        raise ValueError(
            "Question is empty."
        )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": str(
                        image_path
                    ),
                },
                {
                    "type": "text",
                    "text": question,
                },
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    image_inputs, video_inputs = (
        process_vision_info(
            messages
        )
    )

    processor_kwargs = {
        "text": [text],
        "images": image_inputs,
        "videos": video_inputs,
        "padding": True,
        "return_tensors": "pt",
    }

    # Respect the notebook's T4 visual budget when available.
    adaptive_max_pixels = int(
        globals().get(
            "QWEN_MAX_PIXELS",
            256 * 28 * 28,
        )
    )

    min_pixels = int(
        globals().get(
            "QWEN_MIN_PIXELS",
            128 * 28 * 28,
        )
    )

    inputs = None

    last_error = None

    for attempt in range(4):

        current_kwargs = dict(
            processor_kwargs
        )

        current_kwargs[
            "images_kwargs"
        ] = {
            "min_pixels": int(
                min(
                    min_pixels,
                    adaptive_max_pixels,
                )
            ),
            "max_pixels": int(
                adaptive_max_pixels
            ),
        }

        try:

            inputs = processor(
                **current_kwargs
            )

        except TypeError:

            # Compatibility fallback.
            current_kwargs.pop(
                "images_kwargs",
                None,
            )

            inputs = processor(
                **current_kwargs
            )

        grid = inputs.get(
            "image_grid_thw"
        )

        if grid is None:
            break

        grid_cpu = (
            grid.detach()
            .cpu()
        )

        if (
            grid_cpu.ndim != 2
            or grid_cpu.shape[1] != 3
        ):
            break

        patch_tokens = int(
            sum(
                int(t)
                * int(h)
                * int(w)
                for t, h, w
                in grid_cpu.tolist()
            )
        )

        merge_size = int(
            getattr(
                getattr(
                    processor,
                    "image_processor",
                    None,
                ),
                "merge_size",
                2,
            )
            or 2
        )

        image_tokens = (
            patch_tokens
            // (
                merge_size
                ** 2
            )
        )

        target_image_tokens = int(
            globals().get(
                "QWEN_TARGET_VISUAL_TOKENS",
                256,
            )
        )

        if image_tokens <= target_image_tokens:

            last_error = None
            break

        last_error = (
            f"image_tokens={image_tokens}, "
            f"patch_tokens={patch_tokens}, "
            f"budget={adaptive_max_pixels}"
        )

        adaptive_max_pixels = max(
            min_pixels,
            int(
                adaptive_max_pixels
                * 0.88
            ),
        )

        inputs = None

    if inputs is None:

        raise RuntimeError(
            "Qwen2.5-VL visual preprocessing could not satisfy "
            f"the T4 safety budget. Last state: {last_error}"
        )

    return inputs


# ============================================================
# 13. Architecture-specific dispatcher
# ============================================================

def prepare_smoke_inputs(
    model_key: str,
    processor,
    image_path: Path,
    question: str,
):
    """
    Dispatch to the correct model-specific input adapter.
    """

    if model_key == "smolvlm2":

        return prepare_t4_smolvlm2_inputs(
            processor=processor,
            image_path=image_path,
            question=question,
        )

    if model_key == "qwen25vl":

        return prepare_qwen25vl_inputs(
            processor=processor,
            image_path=image_path,
            question=question,
        )

    raise NotImplementedError(
        f"No smoke-test adapter is registered for "
        f"model_key={model_key!r}."
    )


# ============================================================
# 14. T4-safe model loader
# ============================================================

def load_smoke_model(
    model_key: str,
    config,
):
    """
    Load the selected VLM with an explicit GPU memory ceiling.

    Remaining weights are allowed to stay on CPU through Accelerate.
    """

    from transformers import (
        AutoProcessor,
        AutoModelForImageTextToText,
    )

    print(
        "Loading processor..."
    )

    # --------------------------------------------------------
    # Processor
    # --------------------------------------------------------

    if model_key == "qwen25vl":

        qwen_min_pixels = int(
            globals().get(
                "QWEN_MIN_PIXELS",
                128 * 28 * 28,
            )
        )

        qwen_max_pixels = int(
            globals().get(
                "QWEN_MAX_PIXELS",
                256 * 28 * 28,
            )
        )

        print(
            "[T4 profile] Qwen pixel budget:",
            qwen_min_pixels,
            "to",
            qwen_max_pixels,
        )

        processor = AutoProcessor.from_pretrained(
            config.processor_id,
            min_pixels=qwen_min_pixels,
            max_pixels=qwen_max_pixels,
        )

        image_processor = getattr(
            processor,
            "image_processor",
            None,
        )

        if image_processor is not None:

            try:

                image_processor.size = {
                    "shortest_edge": qwen_min_pixels,
                    "longest_edge": qwen_max_pixels,
                }

            except Exception:

                pass

    else:

        processor = (
            AutoProcessor.from_pretrained(
                config.processor_id
            )
        )

    # --------------------------------------------------------
    # Dtype
    # --------------------------------------------------------

    if CUDA_AVAILABLE:

        model_dtype = torch.float16

    else:

        model_dtype = torch.float32

    # --------------------------------------------------------
    # SmolVLM2
    # --------------------------------------------------------

    if model_key == "smolvlm2":

        if CUDA_AVAILABLE:

            if IS_T4:

                gpu_budget_gib = float(
                    globals().get(
                        "T4_MODEL_GPU_BUDGET_GIB",
                        {
                            "smolvlm2": 11.0,
                            "qwen25vl": 11.0,
                        },
                    ).get(
                        "smolvlm2",
                        11.0,
                    )
                )

            else:

                gpu_budget_gib = min(
                    12.0,
                    float(
                        globals().get(
                            "T4_MODEL_GPU_BUDGET_GIB",
                            {
                                "smolvlm2": 11.0,
                                "qwen25vl": 11.0,
                            },
                        ).get(
                            "smolvlm2",
                            11.0,
                        )
                    ),
                )

            GPU_BUDGET = (
                f"{gpu_budget_gib:.1f}GiB"
            )

            CPU_BUDGET = "48GiB"

            max_memory = {
                0: GPU_BUDGET,
                "cpu": CPU_BUDGET,
            }

            print(
                "GPU memory budget :",
                GPU_BUDGET,
            )

            print(
                "CPU offload budget:",
                CPU_BUDGET,
            )

            model = (
                AutoModelForImageTextToText
                .from_pretrained(
                    config.model_id,
                    dtype=model_dtype,
                    device_map="auto",
                    max_memory=max_memory,
                    low_cpu_mem_usage=True,
                )
            )

        else:

            model = (
                AutoModelForImageTextToText
                .from_pretrained(
                    config.model_id,
                    dtype=model_dtype,
                    device_map="cpu",
                    low_cpu_mem_usage=True,
                )
            )

        model.eval()

        return (
            processor,
            model,
        )

    # --------------------------------------------------------
    # Qwen2.5-VL
    # --------------------------------------------------------

    if model_key == "qwen25vl":

        from transformers import (
            Qwen2_5_VLForConditionalGeneration,
        )

        if CUDA_AVAILABLE:

            gpu_budget_gib = float(
                globals().get(
                    "T4_MODEL_GPU_BUDGET_GIB",
                    {
                        "smolvlm2": 11.0,
                        "qwen25vl": 11.0,
                    },
                ).get(
                    "qwen25vl",
                    11.0,
                )
            )

            max_memory = {
                0: (
                    f"{gpu_budget_gib:.1f}GiB"
                ),
                "cpu": "48GiB",
            }

            print(
                "GPU memory budget :",
                max_memory[0],
            )

            print(
                "CPU offload budget:",
                max_memory["cpu"],
            )

            model = (
                Qwen2_5_VLForConditionalGeneration
                .from_pretrained(
                    config.model_id,
                    dtype=model_dtype,
                    device_map="auto",
                    max_memory=max_memory,
                    low_cpu_mem_usage=True,
                    attn_implementation="sdpa",
                )
            )

        else:

            model = (
                Qwen2_5_VLForConditionalGeneration
                .from_pretrained(
                    config.model_id,
                    dtype=torch.float32,
                    device_map="cpu",
                    low_cpu_mem_usage=True,
                )
            )

        model.eval()

        try:
            model.config.use_cache = False
        except Exception:
            pass

        return (
            processor,
            model,
        )

    raise NotImplementedError(
        f"No T4-safe loader is registered for "
        f"model_key={model_key!r}."
    )


# ============================================================
# 15. Resolve useful model input device
# ============================================================

def get_smoke_model_device(
    model,
):
    """
    Resolve a useful device for tensor inputs when Accelerate splits the model.
    """

    try:

        model_device = model.device

        if isinstance(
            model_device,
            torch.device,
        ):

            return model_device

        return torch.device(
            str(model_device)
        )

    except Exception:

        pass

    if CUDA_AVAILABLE:

        try:

            for parameter in model.parameters():

                if (
                    parameter.device.type
                    == "cuda"
                ):

                    return parameter.device

        except Exception:

            pass

    try:

        return next(
            model.parameters()
        ).device

    except StopIteration:

        return torch.device(
            "cuda"
            if CUDA_AVAILABLE
            else "cpu"
        )


# ============================================================
# 16. Move tensor inputs
# ============================================================

def move_smoke_inputs(
    inputs,
    model,
):
    """
    Move tensor-valued inputs to a useful model input device.
    """

    target_device = (
        get_smoke_model_device(
            model
        )
    )

    moved = {}

    for key, value in inputs.items():

        if isinstance(
            value,
            torch.Tensor,
        ):

            moved[
                key
            ] = value.to(
                target_device
            )

        else:

            moved[
                key
            ] = value

    return moved


# ============================================================
# 17. Run one pre-generation forward
# ============================================================

def run_t4_smoke_forward(
    model,
    inputs,
):
    """
    Run exactly one pre-generation forward pass.

    Memory-saving settings:
        - inference_mode
        - use_cache=False
        - FP16 autocast on CUDA
        - keep only the required logits position where supported
    """

    with torch.inference_mode():

        if CUDA_AVAILABLE:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=True,
            ):

                try:

                    outputs = model(
                        **inputs,
                        output_hidden_states=True,
                        return_dict=True,
                        use_cache=False,
                        logits_to_keep=1,
                    )

                except TypeError:

                    outputs = model(
                        **inputs,
                        output_hidden_states=True,
                        return_dict=True,
                        use_cache=False,
                    )

        else:

            outputs = model(
                **inputs,
                output_hidden_states=True,
                return_dict=True,
                use_cache=False,
            )

    return outputs


# ============================================================
# 18. Hidden-state extraction
# ============================================================

def get_smoke_hidden_states(
    outputs,
):
    """
    Retrieve actual hidden states from model output.
    """

    hidden_states = getattr(
        outputs,
        "hidden_states",
        None,
    )

    if hidden_states is None:

        raise RuntimeError(
            "The model completed the forward pass but did not return "
            "hidden_states."
        )

    if not isinstance(
        hidden_states,
        (tuple, list),
    ):

        raise TypeError(
            "Unexpected hidden_states container: "
            f"{type(hidden_states)}"
        )

    if len(
        hidden_states
    ) < 2:

        raise RuntimeError(
            "Too few hidden-state tensors were returned."
        )

    return hidden_states


# ============================================================
# 19. Decoder depth
# ============================================================

def get_smoke_decoder_depth(
    model,
    hidden_states,
):
    """
    Infer decoder depth from the actual runtime output.
    """

    if (
        "decoder_layer_count"
        in globals()
    ):

        try:

            depth = int(
                decoder_layer_count(
                    model,
                    hidden_states,
                )
            )

            if depth > 0:
                return depth

        except Exception:

            pass

    depth = (
        len(hidden_states)
        - 1
    )

    if depth < 1:

        raise RuntimeError(
            "Could not infer decoder depth."
        )

    return depth


# ============================================================
# 20. Selected layer validation
# ============================================================

def get_smoke_layers(
    depth: int,
):
    """
    Reuse notebook canonical layer-selection logic when available.
    """

    if (
        "selected_layer_indices"
        in globals()
    ):

        selected = (
            selected_layer_indices(
                depth
            )
        )

    else:

        def nearest_layer(
            ratio: float,
        ) -> int:

            return max(
                1,
                min(
                    depth,
                    int(
                        math.floor(
                            ratio
                            * depth
                        )
                    ),
                ),
            )

        selected = {

            "early":
                nearest_layer(
                    1.0 / depth
                ),

            "quarter":
                nearest_layer(
                    0.25
                ),

            "middle":
                nearest_layer(
                    0.50
                ),

            "three_quarter":
                nearest_layer(
                    0.75
                ),

            "final":
                depth,
        }

    result = {}

    for name, index in selected.items():

        index = int(
            index
        )

        if not (
            1
            <= index
            <= depth
        ):

            raise ValueError(
                f"Invalid layer {index} for {name}; "
                f"decoder depth={depth}."
            )

        result[
            str(name)
        ] = index

    return result


# ============================================================
# 21. Token diagnostics
# ============================================================

def smoke_token_metadata(
    model,
    inputs,
):
    """
    Report image-token positions and query position.
    """

    config = getattr(
        model,
        "config",
        None,
    )

    image_token_id = None

    if config is not None:

        image_token_id = getattr(
            config,
            "image_token_id",
            None,
        )

    vision_positions = []

    if (
        image_token_id is not None
        and "input_ids" in inputs
    ):

        input_ids = inputs[
            "input_ids"
        ]

        if input_ids.ndim == 2:

            row = input_ids[
                0
            ]

            vision_positions = (
                torch.nonzero(
                    row
                    == int(
                        image_token_id
                    ),
                    as_tuple=False,
                )
                .flatten()
                .tolist()
            )

    query_position = None

    if "input_ids" in inputs:

        input_ids = inputs[
            "input_ids"
        ]

        attention_mask = inputs.get(
            "attention_mask"
        )

        if (
            attention_mask is not None
            and attention_mask.ndim == 2
        ):

            valid_positions = (
                torch.nonzero(
                    attention_mask[
                        0
                    ].bool(),
                    as_tuple=False,
                )
                .flatten()
                .tolist()
            )

            if valid_positions:

                query_position = (
                    valid_positions[
                        -1
                    ]
                )

        elif input_ids.ndim == 2:

            query_position = (
                input_ids.shape[1]
                - 1
            )

    return {
        "image_token_id":
            image_token_id,

        "vision_token_positions":
            vision_positions,

        "query_position":
            query_position,
    }


# ============================================================
# 22. One-model smoke test
# ============================================================

def smoke_test_model(
    model_key: str,
):
    """
    Run one complete T4-safe smoke test.
    """

    if model_key not in MODEL_REGISTRY:

        raise KeyError(
            f"Unknown model key: {model_key!r}"
        )

    config = MODEL_REGISTRY[
        model_key
    ]

    print(
        "\n"
        + "-" * 80
    )

    print(
        f"MODEL SMOKE TEST: {model_key}"
    )

    print(
        "-" * 80
    )

    # --------------------------------------------------------
    # Sample
    # --------------------------------------------------------

    sample = choose_smoke_sample(
        model_key
    )

    image_path = Path(
        str(
            sample[
                "resolved_image_path"
            ]
        )
    )

    question = str(
        sample[
            "question"
        ]
    ).strip()

    question_id = str(
        sample.get(
            "question_id",
            "unknown",
        )
    )

    image_name = str(
        sample.get(
            "image_name",
            image_path.name,
        )
    )

    sample_source = str(
        sample.get(
            "_smoke_source",
            "unknown",
        )
    )

    print(
        "Sample source        :",
        sample_source,
    )

    print(
        "Question ID          :",
        question_id,
    )

    print(
        "Image name           :",
        image_name,
    )

    print(
        "Image path           :",
        image_path,
    )

    print(
        "Question             :",
        question,
    )

    # --------------------------------------------------------
    # Local image validation
    # --------------------------------------------------------

    image_info = validate_local_image(
        image_path
    )

    print(
        "Image integrity      : PASS"
    )

    print(
        "Image format         :",
        image_info[
            "format"
        ],
    )

    print(
        "Image mode           :",
        image_info[
            "mode"
        ],
    )

    print(
        "Original image size  :",
        image_info[
            "size"
        ],
    )

    # --------------------------------------------------------
    # Load model
    # --------------------------------------------------------

    print(
        "\nLoading T4-safe model configuration..."
    )

    print(
        "Model ID             :",
        config.model_id,
    )

    print(
        "Processor ID         :",
        config.processor_id,
    )

    load_start = time.time()

    processor, model = (
        load_smoke_model(
            model_key,
            config,
        )
    )

    load_seconds = (
        time.time()
        - load_start
    )

    model.eval()

    print(
        f"Model loading time   : "
        f"{load_seconds:.2f} seconds"
    )

    print(
        "Model class          :",
        type(model).__name__,
    )

    print(
        "Processor class      :",
        type(processor).__name__,
    )

    # --------------------------------------------------------
    # SmolVLM2 geometry validation
    # --------------------------------------------------------

    smol_geometry = None

    if model_key == "smolvlm2":

        smol_geometry = (
            _validate_smol_geometry(
                processor,
                model,
            )
        )

    # --------------------------------------------------------
    # Accelerate placement diagnostics
    # --------------------------------------------------------

    hf_device_map = getattr(
        model,
        "hf_device_map",
        None,
    )

    if hf_device_map is not None:

        device_counts = {}

        for module_name, device_name in (
            hf_device_map.items()
        ):

            device_key = str(
                device_name
            )

            device_counts[
                device_key
            ] = (
                device_counts.get(
                    device_key,
                    0,
                )
                + 1
            )

        print(
            "Accelerate device map summary:",
            device_counts,
        )

    # --------------------------------------------------------
    # Cache one model
    # --------------------------------------------------------

    LOADED[
        model_key
    ] = {

        "processor":
            processor,

        "model":
            model,
    }

    # --------------------------------------------------------
    # Prepare smoke input
    # --------------------------------------------------------

    print(
        "\nPreparing T4-safe smoke input..."
    )

    inputs = prepare_smoke_inputs(
        model_key,
        processor,
        image_path,
        question,
    )

    if not hasattr(
        inputs,
        "keys",
    ):

        raise TypeError(
            "Processor output is not mapping-like."
        )

    input_keys = list(
        inputs.keys()
    )

    print(
        "Processor output keys:",
        input_keys,
    )

    if not input_keys:

        raise RuntimeError(
            "Processor returned an empty input mapping."
        )

    # --------------------------------------------------------
    # Validate actual SmolVLM2 processor output BEFORE forward
    # --------------------------------------------------------

    if model_key == "smolvlm2":

        _validate_smol_processor_output(
            inputs,
            model,
        )

    # --------------------------------------------------------
    # Tensor diagnostics
    # --------------------------------------------------------

    tensor_shapes = {}

    tensor_dtypes = {}

    for key, value in inputs.items():

        if isinstance(
            value,
            torch.Tensor,
        ):

            tensor_shapes[
                key
            ] = tuple(
                value.shape
            )

            tensor_dtypes[
                key
            ] = str(
                value.dtype
            )

            print(
                f"  {key:22s} "
                f"shape={tuple(value.shape)} "
                f"dtype={value.dtype}"
            )

    if "input_ids" in inputs:

        input_ids = inputs[
            "input_ids"
        ]

        if input_ids.ndim != 2:

            raise RuntimeError(
                "Expected input_ids rank 2."
            )

        if input_ids.shape[0] != 1:

            raise RuntimeError(
                "Smoke test requires batch size 1."
            )

        print(
            "input_ids sequence length:",
            input_ids.shape[1],
        )

    # --------------------------------------------------------
    # Move inputs
    # --------------------------------------------------------

    inputs = move_smoke_inputs(
        inputs,
        model,
    )

    model_input_device = (
        get_smoke_model_device(
            model
        )
    )

    print(
        "Model input device   :",
        model_input_device,
    )

    # --------------------------------------------------------
    # Pre-forward VRAM
    # --------------------------------------------------------

    vram_before = None

    if CUDA_AVAILABLE:

        try:

            free_bytes, total_bytes = (
                torch.cuda.mem_get_info(
                    GPU_INDEX
                )
            )

            vram_before = {

                "free_gib":
                    free_bytes
                    / (2 ** 30),

                "total_gib":
                    total_bytes
                    / (2 ** 30),
            }

            print(
                f"VRAM free before forward: "
                f"{vram_before['free_gib']:.2f} GiB"
            )

        except Exception:

            pass

    # --------------------------------------------------------
    # Forward
    # --------------------------------------------------------

    print(
        "\nRunning one pre-generation forward pass..."
    )

    forward_start = time.time()

    outputs = run_t4_smoke_forward(
        model,
        inputs,
    )

    if CUDA_AVAILABLE:

        try:
            torch.cuda.synchronize(
                GPU_INDEX
            )
        except Exception:
            pass

    forward_seconds = (
        time.time()
        - forward_start
    )

    print(
        f"Forward pass time    : "
        f"{forward_seconds:.2f} seconds"
    )

    if outputs is None:

        raise RuntimeError(
            "Model returned None."
        )

    # --------------------------------------------------------
    # Hidden states
    # --------------------------------------------------------

    hidden_states = (
        get_smoke_hidden_states(
            outputs
        )
    )

    decoder_depth = (
        get_smoke_decoder_depth(
            model,
            hidden_states,
        )
    )

    print(
        "Hidden-state tensors :",
        len(hidden_states),
    )

    print(
        "Decoder depth        :",
        decoder_depth,
    )

    # --------------------------------------------------------
    # Selected layer validation
    # --------------------------------------------------------

    selected_layers = (
        get_smoke_layers(
            decoder_depth
        )
    )

    print(
        "\nSelected layers:"
    )

    layer_shapes = {}

    finite_checks = {}

    for name, layer_index in (
        selected_layers.items()
    ):

        tensor = hidden_states[
            layer_index
        ]

        if not isinstance(
            tensor,
            torch.Tensor,
        ):

            raise TypeError(
                f"Hidden state for {name} is not a tensor."
            )

        if tensor.ndim != 3:

            raise RuntimeError(
                f"Hidden state {name} has rank "
                f"{tensor.ndim}; expected [batch, sequence, hidden]."
            )

        if tensor.shape[0] != 1:

            raise RuntimeError(
                f"Hidden state {name} has batch size "
                f"{tensor.shape[0]}; expected 1."
            )

        layer_shapes[
            name
        ] = tuple(
            tensor.shape
        )

        # Inspect only a tiny CPU slice to avoid unnecessary memory.
        small_slice = (
            tensor[
                0,
                : min(
                    4,
                    tensor.shape[1],
                ),
                : min(
                    4,
                    tensor.shape[2],
                ),
            ]
            .detach()
            .float()
            .cpu()
        )

        finite = bool(
            torch.isfinite(
                small_slice
            )
            .all()
            .item()
        )

        finite_checks[
            name
        ] = finite

        print(
            f"  {name:16s} "
            f"layer={layer_index:>3d} "
            f"shape={tuple(tensor.shape)} "
            f"dtype={tensor.dtype} "
            f"finite={'PASS' if finite else 'FAIL'}"
        )

        if not finite:

            raise RuntimeError(
                f"Non-finite hidden-state values detected at "
                f"{name} / layer {layer_index}."
            )

        del small_slice

    # --------------------------------------------------------
    # Token diagnostics
    # --------------------------------------------------------

    token_info = (
        smoke_token_metadata(
            model,
            inputs,
        )
    )

    print(
        "\nToken diagnostics:"
    )

    print(
        "  image_token_id     :",
        token_info[
            "image_token_id"
        ],
    )

    print(
        "  image token count  :",
        len(
            token_info[
                "vision_token_positions"
            ]
        ),
    )

    print(
        "  query position     :",
        token_info[
            "query_position"
        ],
    )

    # --------------------------------------------------------
    # Post-forward VRAM
    # --------------------------------------------------------

    vram_after = None

    if CUDA_AVAILABLE:

        try:

            free_bytes, total_bytes = (
                torch.cuda.mem_get_info(
                    GPU_INDEX
                )
            )

            vram_after = {

                "free_gib":
                    free_bytes
                    / (2 ** 30),

                "total_gib":
                    total_bytes
                    / (2 ** 30),
            }

            print(
                f"VRAM free after forward: "
                f"{vram_after['free_gib']:.2f} GiB"
            )

        except Exception:

            pass

    # --------------------------------------------------------
    # Compact metadata only
    # --------------------------------------------------------

    hidden_state_summary = {

        "count":
            len(hidden_states),

        "selected_layers":
            selected_layers,

        "shapes":
            layer_shapes,

        "finite_checks":
            finite_checks,
    }

    # Important:
    # Do not retain the full output, full hidden-state list or GPU input tensors.
    del hidden_states
    del outputs
    del inputs

    gc.collect()

    if CUDA_AVAILABLE:

        try:
            torch.cuda.empty_cache()
        except Exception:
            pass

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

    # --------------------------------------------------------
    # Compact result
    # --------------------------------------------------------

    result = {

        "model_key":
            model_key,

        "model_id":
            config.model_id,

        "processor_id":
            config.processor_id,

        "model_class":
            type(model).__name__,

        "processor_class":
            type(processor).__name__,

        "sample_source":
            sample_source,

        "question_id":
            question_id,

        "image_name":
            image_name,

        "image_path":
            str(image_path),

        "question":
            question,

        "original_image_size":
            image_info[
                "size"
            ],

        "input_tensor_shapes":
            tensor_shapes,

        "input_tensor_dtypes":
            tensor_dtypes,

        "decoder_layers":
            decoder_depth,

        "hidden_state_summary":
            hidden_state_summary,

        "image_token_id":
            token_info[
                "image_token_id"
            ],

        "vision_token_positions":
            token_info[
                "vision_token_positions"
            ],

        "query_position":
            token_info[
                "query_position"
            ],

        "vram_before":
            vram_before,

        "vram_after":
            vram_after,

        "forward_seconds":
            forward_seconds,

        "generation_performed":
            False,

        "smoke_image_profile":
        {
            "do_image_splitting":
                False,

            "longest_edge":
                SMOL_T4_IMAGE_EDGE,

            "expected_padded_size":
            (
                SMOL_T4_IMAGE_EDGE,
                SMOL_T4_IMAGE_EDGE,
            ),

            "processor_image_seq_len":
                (
                    smol_geometry[
                        "processor_image_seq_len"
                    ]
                    if smol_geometry
                    is not None
                    else None
                ),

            "model_image_seq_len":
                (
                    smol_geometry[
                        "model_image_seq_len"
                    ]
                    if smol_geometry
                    is not None
                    else None
                ),

            "note":
                (
                    "Checkpoint-compatible SmolVLM2 smoke "
                    "profile: 384px padded input, no image "
                    "splitting. This avoids the 336px/81-image-token "
                    "geometry mismatch that caused the previous "
                    "ValueError."
                ),
        },

        "status":
            "PASSED",
    }

    return result


# ============================================================
# 23. Execute active-model smoke tests
# ============================================================

SMOKE = {}

SMOKE_STATUS = {}


for model_key in ACTIVE_MODELS:

    # Keep only one VLM resident on the T4 at a time.
    for cached_key in list(
        LOADED.keys()
    ):

        release_cached_model(
            cached_key
        )

    gc.collect()

    if CUDA_AVAILABLE:

        try:
            torch.cuda.empty_cache()
        except Exception:
            pass

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

    try:

        result = smoke_test_model(
            model_key
        )

        SMOKE[
            model_key
        ] = result

        SMOKE_STATUS[
            model_key
        ] = "PASSED"

        # Smoke metadata is retained, but the model itself is released.
        release_cached_model(
            model_key
        )

        gc.collect()

        if CUDA_AVAILABLE:

            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

        print(
            "\n"
            + "=" * 80
        )

        print(
            f"[{model_key}] SMOKE TEST: PASSED"
        )

        print(
            "=" * 80
        )

    except torch.cuda.OutOfMemoryError as exc:

        if CUDA_AVAILABLE:

            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

        gc.collect()

        SMOKE_STATUS[
            model_key
        ] = (
            "FAILED — CUDA OUT OF MEMORY"
        )

        print(
            "\n"
            + "=" * 80
        )

        print(
            f"[{model_key}] SMOKE TEST FAILED — CUDA OUT OF MEMORY"
        )

        print(
            "The T4 did not have enough free VRAM "
            "for the current smoke-test configuration."
        )

        print(
            "Exact error:"
        )

        print(
            str(exc)
        )

        print(
            "=" * 80
        )

        release_cached_model(
            model_key
        )

        gc.collect()

        if CUDA_AVAILABLE:

            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

        raise

    except Exception as exc:

        SMOKE_STATUS[
            model_key
        ] = (
            f"FAILED — "
            f"{type(exc).__name__}: {exc}"
        )

        print(
            "\n"
            + "=" * 80
        )

        print(
            f"[{model_key}] SMOKE TEST FAILED"
        )

        print(
            "Exception type      :",
            type(exc).__name__,
        )

        print(
            "Exception message    :",
            str(exc),
        )

        print(
            "=" * 80
        )

        release_cached_model(
            model_key
        )

        gc.collect()

        if CUDA_AVAILABLE:

            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

        raise


# ============================================================
# 24. Final smoke-test summary
# ============================================================

print(
    "\n"
    + "=" * 80
)

print(
    "FINAL SMOKE-TEST SUMMARY"
)

print(
    "=" * 80
)


for model_key, status in (
    SMOKE_STATUS.items()
):

    print(
        f"{model_key:22s}: {status}"
    )


print(
    "-" * 80
)

print(
    "GPU                    :",
    GPU_NAME,
)

print(
    "T4-safe mode           :",
    IS_T4,
)

print(
    "HALP-Bench labels      : NOT REQUIRED"
)

print(
    "Local image source     : Drive filesystem path"
)

print(
    "file:// URI             : NOT USED"
)

print(
    "Generation             : NOT PERFORMED"
)

print(
    "Pre-generation forward : YES"
)

print(
    "Hidden states          : REQUESTED AND VALIDATED"
)

print(
    "SmolVLM2 profile       : 384px / no image splitting"
)

print(
    "SmolVLM2 geometry      : CHECKED BEFORE FORWARD"
)

print(
    "Research result        : NOT a benchmark result"
)

print(
    "Cached models          :",
    sorted(
        LOADED.keys()
    ),
    "(should be empty after smoke-test cleanup)",
)

print(
    "=" * 80
)

print(
    "Smoke-test stage complete."
)


qwen-vl-utils is missing; installing it once...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 285.6 MB/s eta 0:00:00

qwen-vl-utils install/import: PASS
HALP-Bench VLM smoke test — T4-safe / FIXED
Validated HALP-Bench rows: 10000

SmolVLM2 smoke preprocessing: 384px / image_splitting=False

GPU diagnostics
CUDA available: True
GPU              : Tesla T4
VRAM total       : 14.56 GiB
VRAM free        : 14.45 GiB
Compute capability: 7.5
Detected Tesla T4 : True

Releasing stale VLM objects
VRAM free after cleanup: 14.45 GiB

--------------------------------------------------------------------------------
MODEL SMOKE TEST: smolvlm2
--------------------------------------------------------------------------------
Sample source        : model-specific dataframe
Question ID          : question_comb_1
Image name           : haloquest_2082.png
Image path           : /content/drive/MyDrive/HALP_Bench_Project/_extracted_dataset/haloquest_2082.png
Question             : How many sharks 

processor_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.64k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model loading time   : 185.85 seconds
Model class          : Qwen2_5_VLForConditionalGeneration
Processor class      : Qwen2_5_VLProcessor

Preparing T4-safe smoke input...
Processor output keys: ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_grid_thw']
  input_ids              shape=(1, 275) dtype=torch.int64
  attention_mask         shape=(1, 275) dtype=torch.int64
  mm_token_type_ids      shape=(1, 275) dtype=torch.int64
  pixel_values           shape=(988, 1176) dtype=torch.float32
  image_grid_thw         shape=(1, 3) dtype=torch.int64
input_ids sequence length: 275
Model input device   : cuda:0
VRAM free before forward: 7.36 GiB

Running one pre-generation forward pass...
Forward pass time    : 1.77 seconds
Hidden-state tensors : 37
Decoder depth        : 36

Selected layers:
  early            layer=  1 shape=(1, 275, 2048) dtype=torch.float16 finite=PASS
  quarter          layer=  9 shape=(1, 275, 2048) dtype=torch.float16 finite=PASS
  middle      

## 11. Architecture-aware feature definitions

The target representations are aligned to the HALP framing:

- **VF**: mean-pooled visual-encoder output before multimodal fusion/projection, when the current model exposes a clean pre-fusion vision encoder interface.
- **VT**: hidden state at the final image/vision-token position at the selected decoder layers, when the token is identifiable.
- **QT**: hidden state at the final non-padding query token position at the selected decoder layers.

The notebook does **not** force VF/VT/QT onto models that do not expose those quantities cleanly.

The extraction path is checked at runtime. Unsupported families are recorded as unsupported rather than silently replaced with another representation.



In [13]:

# ============================================================
# 11. Representation extraction / validation
# ============================================================
#
# Purpose
# -------
# Validate the actual representations used by the HALP pipeline:
#
#   VF  = visual features
#   VT  = vision-token hidden states
#   QT  = query-token hidden states
#
# Important
# ---------
# The previous smoke-test cell was intentionally changed to store only
# compact metadata in SMOKE so that a T4 does not keep the complete
# forward output and all hidden states alive.
#
# This cell must not assume that:
#
#     SMOKE[model_key]["model"]
#     SMOKE[model_key]["processor"]
#     SMOKE[model_key]["inputs"]
#     SMOKE[model_key]["hidden_states"]
#
# still exist.
#
# Instead:
#
#     model / processor
#         -> LOADED[model_key]
#
# and, when runtime tensors are not retained:
#
#     image + question
#         -> processor
#         -> one forward pass
#         -> hidden states
#
# This keeps the representation-validation stage compatible with the
# memory-saving T4 smoke-test implementation.
#
#
# Scientific note
# ---------------
# This cell does not create or infer hallucination labels.
#
# It only validates representation extraction.
#
# This cell does not generate answers.
#
#
# HALP:
#   https://github.com/Zesearch/HALP
#
# SmolVLM:
#   https://huggingface.co/docs/transformers/main/en/model_doc/smolvlm
# ============================================================


# ------------------------------------------------------------
# 1. Notebook objects we need
# ------------------------------------------------------------

print("=" * 80)
print("HALP representation validation")
print("=" * 80)


REQUIRED_GLOBALS = [
    "SMOKE",
    "LOADED",
    "ACTIVE_MODELS",
    "MODEL_REGISTRY",
    "torch",
    "np",
    "pd",
    "Path",
    "Image",
    "gc",
]


missing_globals = [
    name
    for name in REQUIRED_GLOBALS
    if name not in globals()
]


if missing_globals:
    raise RuntimeError(
        "Missing required notebook objects:\n"
        f"  {missing_globals}\n\n"
        "Run the environment, dataset, model-registry, and successful "
        "T4 smoke-test cells first."
    )


if not isinstance(
    SMOKE,
    dict,
):
    raise TypeError(
        "`SMOKE` must be a dictionary."
    )


if not isinstance(
    LOADED,
    dict,
):
    raise TypeError(
        "`LOADED` must be a dictionary."
    )


if not isinstance(
    ACTIVE_MODELS,
    (list, tuple),
):
    raise TypeError(
        "`ACTIVE_MODELS` must be a list or tuple."
    )


# ------------------------------------------------------------
# 2. Helper: safely retrieve the loaded model
# ------------------------------------------------------------

def get_loaded_representation_model(
    model_key: str,
):
    """
    Load one representation model on demand.

    Only one VLM is kept in LOADED at a time to reduce T4 VRAM pressure.
    """

    for cached_key in list(LOADED.keys()):
        if cached_key != model_key:
            release_cached_model(cached_key)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if model_key not in LOADED:
        if model_key not in MODEL_REGISTRY:
            raise KeyError(f"Unknown model key: {model_key!r}")

        processor, model = load_smoke_model(
            model_key,
            MODEL_REGISTRY[model_key],
        )
        LOADED[model_key] = {
            "processor": processor,
            "model": model,
        }

    entry = LOADED[model_key]

    if not isinstance(entry, dict):
        raise TypeError(
            f"LOADED[{model_key!r}] is not a dictionary."
        )

    model = entry.get("model")
    processor = entry.get("processor")

    if model is None:
        raise RuntimeError(
            f"LOADED[{model_key!r}] has no model object."
        )

    if processor is None:
        raise RuntimeError(
            f"LOADED[{model_key!r}] has no processor object."
        )

    model.eval()

    return model, processor



# ------------------------------------------------------------
# 3. Helper: decode values stored in SMOKE metadata
# ------------------------------------------------------------

def smoke_value(
    value,
):
    """
    Convert common scalar/array-like metadata to a Python value.
    """

    if isinstance(
        value,
        np.ndarray,
    ):

        if value.ndim == 0:
            return value.item()

        return value.tolist()

    if isinstance(
        value,
        torch.Tensor,
    ):

        if value.numel() == 1:
            return value.detach().cpu().item()

        return (
            value
            .detach()
            .cpu()
            .tolist()
        )

    return value


# ------------------------------------------------------------
# 4. Helper: obtain smoke-test sample metadata
# ------------------------------------------------------------

def get_smoke_sample_metadata(
    model_key: str,
    smoke_record: dict,
):
    """
    Recover the image path and question used by the smoke test.
    """

    if not isinstance(
        smoke_record,
        dict,
    ):

        raise TypeError(
            f"SMOKE[{model_key!r}] is not a dictionary."
        )

    image_path_value = smoke_record.get(
        "image_path"
    )

    question_value = smoke_record.get(
        "question"
    )

    # Some older smoke-test results may use slightly different keys.
    if image_path_value is None:

        image_path_value = smoke_record.get(
            "image"
        )

    if question_value is None:

        question_value = smoke_record.get(
            "prompt"
        )

    if image_path_value is None:

        raise RuntimeError(
            f"SMOKE[{model_key!r}] does not contain image_path metadata."
        )

    if question_value is None:

        raise RuntimeError(
            f"SMOKE[{model_key!r}] does not contain question metadata."
        )

    image_path = Path(
        str(
            image_path_value
        )
    ).resolve()

    question = str(
        question_value
    ).strip()

    if not image_path.exists():

        raise FileNotFoundError(
            f"Smoke-test image no longer exists: {image_path}"
        )

    if not image_path.is_file():

        raise FileNotFoundError(
            f"Smoke-test image is not a regular file: {image_path}"
        )

    if not question:

        raise ValueError(
            f"Smoke-test question for {model_key!r} is empty."
        )

    return (
        image_path,
        question,
    )


# ------------------------------------------------------------
# 5. Helper: validate local image
# ------------------------------------------------------------

def validate_representation_image(
    image_path: Path,
):
    """
    Validate image bytes before representation extraction.
    """

    with Image.open(
        image_path
    ) as image:

        image.verify()

    with Image.open(
        image_path
    ) as image:

        return {
            "format": image.format,
            "mode": image.mode,
            "size": tuple(
                image.size
            ),
        }


# ------------------------------------------------------------
# 6. Helper: prepare SmolVLM2 input
# ------------------------------------------------------------

def prepare_representation_inputs(
    model_key: str,
    processor,
    image_path: Path,
    question: str,
):
    """
    Prepare the exact image-question input needed for representation
    validation.

    Prefer the already validated T4 SmolVLM2 adapter from the smoke-test
    stage when it exists.
    """

    image_path = Path(
        image_path
    ).resolve()

    # --------------------------------------------------------
    # SmolVLM2
    # --------------------------------------------------------

    if model_key == "smolvlm2":

        if (
            "prepare_t4_smolvlm2_inputs"
            in globals()
        ):

            return prepare_t4_smolvlm2_inputs(
                processor=processor,
                image_path=image_path,
                question=question,
            )

        # Fallback using the current Transformers multimodal API.
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "path": str(
                            image_path
                        ),
                    },
                    {
                        "type": "text",
                        "text": str(
                            question
                        ),
                    },
                ],
            }
        ]

        return processor.apply_chat_template(
            messages,
            add_generation_prompt=False,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )


    # --------------------------------------------------------
    # Qwen2.5-VL
    # --------------------------------------------------------

    if model_key == "qwen25vl":

        if (
            "prepare_qwen25vl_inputs"
            in globals()
        ):

            return prepare_qwen25vl_inputs(
                processor=processor,
                image_path=image_path,
                question=question,
            )

        raise RuntimeError(
            "Qwen2.5-VL representation adapter is unavailable."
        )


    raise NotImplementedError(
        f"No verified representation-input adapter exists for "
        f"{model_key!r}."
    )


# ------------------------------------------------------------
# 7. Move tensor inputs safely
# ------------------------------------------------------------

def move_representation_inputs(
    inputs,
    model,
):
    """
    Move only tensor values to a useful model input device.
    """

    # Prefer a CUDA parameter for split Accelerate models.
    if torch.cuda.is_available():

        try:

            for parameter in model.parameters():

                if parameter.device.type == "cuda":

                    target_device = (
                        parameter.device
                    )

                    break

            else:

                target_device = torch.device(
                    "cuda"
                )

        except Exception:

            target_device = torch.device(
                "cuda"
            )

    else:

        try:

            target_device = next(
                model.parameters()
            ).device

        except StopIteration:

            target_device = torch.device(
                "cpu"
            )

    moved = {}

    for key, value in inputs.items():

        if isinstance(
            value,
            torch.Tensor,
        ):

            moved[
                key
            ] = value.to(
                target_device
            )

        else:

            moved[
                key
            ] = value

    return moved


# ------------------------------------------------------------
# 8. Forward pass for representation validation
# ------------------------------------------------------------

def _find_qwen_decoder_layers(model):
    """Locate Qwen2.5-VL decoder blocks across current/fallback wrapper paths."""
    candidates = [
        getattr(getattr(model, "model", None), "language_model", None),
        getattr(model, "language_model", None),
        getattr(getattr(model, "model", None), "model", None),
    ]

    for container in candidates:
        layers = getattr(container, "layers", None)
        if layers is not None:
            try:
                if len(layers) > 0:
                    return layers
            except Exception:
                pass

    raise RuntimeError(
        "Could not locate Qwen2.5-VL decoder layers for low-memory validation."
    )


def _run_qwen_selected_layer_forward(model, inputs, layers):
    """
    Qwen validation forward without output_hidden_states=True.

    Only the five selected decoder outputs are retained, and each captured
    tensor is moved to CPU immediately.
    """
    decoder_layers = _find_qwen_decoder_layers(model)
    depth = len(decoder_layers)
    captured = {}
    handles = []

    def make_hook(layer_index):
        def hook(module, module_inputs, module_output):
            hidden = (
                module_output[0]
                if isinstance(module_output, (tuple, list))
                else module_output
            )
            if not isinstance(hidden, torch.Tensor):
                raise TypeError(
                    f"Qwen decoder layer {layer_index} returned {type(hidden)}."
                )
            if hidden.ndim != 3 or hidden.shape[0] != 1:
                raise RuntimeError(
                    f"Unexpected Qwen decoder output at layer {layer_index}: "
                    f"{tuple(hidden.shape)}"
                )
            captured[int(layer_index)] = hidden.detach().cpu()
        return hook

    try:
        for _, layer_index in layers.items():
            zero_based = int(layer_index) - 1
            if not 0 <= zero_based < depth:
                raise ValueError(
                    f"Invalid Qwen selected layer {layer_index} for depth {depth}."
                )
            handles.append(
                decoder_layers[zero_based].register_forward_hook(
                    make_hook(int(layer_index))
                )
            )

        with torch.inference_mode():
            if torch.cuda.is_available():
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=True,
                ):
                    try:
                        outputs = model(
                            **inputs,
                            output_hidden_states=False,
                            return_dict=True,
                            use_cache=False,
                            logits_to_keep=1,
                        )
                    except TypeError:
                        outputs = model(
                            **inputs,
                            output_hidden_states=False,
                            return_dict=True,
                            use_cache=False,
                        )
            else:
                outputs = model(
                    **inputs,
                    output_hidden_states=False,
                    return_dict=True,
                    use_cache=False,
                )
    finally:
        for handle in handles:
            try:
                handle.remove()
            except Exception:
                pass

    missing = sorted(
        {int(v) for v in layers.values()} - set(captured.keys())
    )
    if missing:
        raise RuntimeError(
            f"Qwen selected-layer hooks did not fire for layers: {missing}"
        )

    hidden_states = [None] * (depth + 1)
    for index, tensor in captured.items():
        hidden_states[int(index)] = tensor

    return outputs, hidden_states


def run_representation_forward(
    model,
    inputs,
    model_key=None,
    layers=None,
):
    """
    Run one pre-generation forward pass.

    Qwen uses selected-layer hooks to avoid retaining all decoder hidden
    states on the T4. Other models keep the existing validated path.
    """
    if model_key == "qwen25vl":
        if layers is None:
            raise ValueError(
                "Qwen2.5-VL representation validation requires selected layers."
            )
        return _run_qwen_selected_layer_forward(
            model,
            inputs,
            layers,
        )

    with torch.inference_mode():
        if torch.cuda.is_available():
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=True,
            ):
                try:
                    outputs = model(
                        **inputs,
                        output_hidden_states=True,
                        return_dict=True,
                        use_cache=False,
                        logits_to_keep=1,
                    )
                except TypeError:
                    outputs = model(
                        **inputs,
                        output_hidden_states=True,
                        return_dict=True,
                        use_cache=False,
                    )
        else:
            outputs = model(
                **inputs,
                output_hidden_states=True,
                return_dict=True,
                use_cache=False,
            )

    return outputs, get_representation_hidden_states(outputs)


# ------------------------------------------------------------
# 9. Hidden-state helper
# ------------------------------------------------------------

def get_representation_hidden_states(
    outputs,
):
    """
    Retrieve the hidden-state sequence from the runtime output.
    """

    hidden_states = getattr(
        outputs,
        "hidden_states",
        None,
    )

    if hidden_states is None:

        if (
            "get_hidden_state_layers"
            in globals()
        ):

            hidden_states = (
                get_hidden_state_layers(
                    outputs
                )
            )

    if hidden_states is None:

        raise RuntimeError(
            "Forward pass succeeded but hidden_states were not returned."
        )

    if not isinstance(
        hidden_states,
        (tuple, list),
    ):

        raise TypeError(
            "Unexpected hidden_states type: "
            f"{type(hidden_states)}"
        )

    if len(hidden_states) < 2:

        raise RuntimeError(
            "The model returned fewer than two hidden-state tensors."
        )

    return hidden_states


# ------------------------------------------------------------
# 10. Decoder depth
# ------------------------------------------------------------

def get_representation_decoder_depth(
    model,
    hidden_states,
):
    """
    Infer decoder depth from actual runtime output.
    """

    if "decoder_layer_count" in globals():

        try:

            depth = int(
                decoder_layer_count(
                    model,
                    hidden_states,
                )
            )

            if depth >= 1:
                return depth

        except Exception:
            pass

    depth = (
        len(hidden_states)
        - 1
    )

    if depth < 1:

        raise RuntimeError(
            "Could not infer decoder depth."
        )

    return depth


# ------------------------------------------------------------
# 11. Selected layers
# ------------------------------------------------------------

def get_representation_layers(
    depth: int,
):
    """
    Reuse the canonical notebook layer-selection function.
    """

    if "selected_layer_indices" in globals():

        selected = selected_layer_indices(
            depth
        )

    else:

        def rounded_layer(
            ratio: float,
        ):
            return max(
                1,
                min(
                    depth,
                    int(math.floor(ratio * depth)),
                ),
            )

        selected = {

            "early":
                rounded_layer(
                    1.0 / depth
                ),

            "quarter":
                rounded_layer(
                    0.25
                ),

            "middle":
                rounded_layer(
                    0.50
                ),

            "three_quarter":
                rounded_layer(
                    0.75
                ),

            "final":
                depth,
        }

    cleaned = {}

    for name, index in selected.items():

        index = int(
            index
        )

        if not (
            1
            <= index
            <= depth
        ):

            raise ValueError(
                f"Invalid layer {index} for {name}; "
                f"decoder depth={depth}."
            )

        cleaned[
            str(name)
        ] = index

    return cleaned


# ------------------------------------------------------------
# 12. Image-token position
# ------------------------------------------------------------

def get_representation_vision_positions(
    model,
    inputs,
):
    """
    Identify actual image-token positions from input_ids.

    No position is fabricated.
    """

    config = getattr(
        model,
        "config",
        None,
    )

    image_token_id = None

    if config is not None:

        image_token_id = getattr(
            config,
            "image_token_id",
            None,
        )

    if (
        image_token_id is None
        or "input_ids" not in inputs
    ):

        return (
            image_token_id,
            [],
        )

    input_ids = inputs[
        "input_ids"
    ]

    if input_ids.ndim != 2:

        return (
            image_token_id,
            [],
        )

    positions = (
        torch.nonzero(
            input_ids[0]
            == int(
                image_token_id
            ),
            as_tuple=False,
        )
        .flatten()
        .tolist()
    )

    return (
        image_token_id,
        positions,
    )


# ------------------------------------------------------------
# 13. Query position
# ------------------------------------------------------------

def get_representation_query_position(
    inputs,
    processor,
):
    """
    Prefer the notebook's existing query-position helper.
    """

    if "query_position" in globals():

        try:

            value = query_position(
                inputs,
                processor,
            )

            if value is not None:

                return int(
                    value
                )

        except Exception:
            pass

    if "input_ids" not in inputs:

        return None

    input_ids = inputs[
        "input_ids"
    ]

    if input_ids.ndim != 2:

        return None

    attention_mask = inputs.get(
        "attention_mask"
    )

    if (
        attention_mask is not None
        and attention_mask.ndim == 2
    ):

        valid = (
            torch.nonzero(
                attention_mask[0].bool(),
                as_tuple=False,
            )
            .flatten()
            .tolist()
        )

        if valid:

            return int(
                valid[-1]
            )

    return int(
        input_ids.shape[1]
        - 1
    )


# ------------------------------------------------------------
# 14. Convert one feature vector to CPU NumPy
# ------------------------------------------------------------

def feature_to_numpy(
    tensor,
):
    """
    Convert one representation vector to finite float32 CPU data.
    """

    if not isinstance(
        tensor,
        torch.Tensor,
    ):

        raise TypeError(
            "Expected a torch.Tensor."
        )

    vector = (
        tensor
        .detach()
        .float()
        .cpu()
        .numpy()
        .reshape(-1)
    )

    if vector.size == 0:

        raise ValueError(
            "Feature vector is empty."
        )

    if not np.isfinite(
        vector
    ).all():

        raise ValueError(
            "Feature vector contains NaN or Inf."
        )

    return vector


# ------------------------------------------------------------
# 15. VF adapter
# ------------------------------------------------------------

def extract_vf(
    model_key,
    model,
    inputs,
):
    """
    Extract the visual-feature representation.

    SmolVLM2:
        raw `vision_model(...).last_hidden_state`, mean-pooled over vision tokens.

    Qwen2.5-VL:
        raw `visual(...).last_hidden_state`, mean-pooled over vision tokens.

    Both adapters intentionally stop before the multimodal connector/merger.
    """

    with torch.inference_mode():

        # ----------------------------------------------------
        # SmolVLM2
        # ----------------------------------------------------

        if model_key == "smolvlm2":

            pixel_values = inputs.get(
                "pixel_values"
            )

            pixel_attention_mask = inputs.get(
                "pixel_attention_mask"
            )

            if pixel_values is None:

                raise RuntimeError(
                    "SmolVLM2 inputs contain no pixel_values."
                )

            base_model = getattr(model, "model", None)
            vision_model = getattr(base_model, "vision_model", None)

            if vision_model is None:
                raise RuntimeError(
                    "SmolVLM2 model does not expose the underlying vision_model."
                )

            if pixel_values.ndim != 5:
                raise RuntimeError(
                    f"Expected SmolVLM2 pixel_values [B,N,C,H,W], got {tuple(pixel_values.shape)}"
                )

            batch_size, num_images, channels, height, width = pixel_values.shape
            flat_pixels = pixel_values.to(dtype=getattr(base_model, "dtype", pixel_values.dtype))
            flat_pixels = flat_pixels.view(batch_size * num_images, channels, height, width)

            # Match the current Transformers implementation: drop all-zero
            # padded images and derive patch attention from pixel_attention_mask.
            real_images = (flat_pixels == 0.0).sum(dim=(-1, -2, -3)) != flat_pixels.shape[1:].numel()
            if not torch.any(real_images):
                real_images[0] = True
            flat_pixels = flat_pixels[real_images].contiguous()

            if pixel_attention_mask is None:
                patch_attention_mask = None
            else:
                pam = pixel_attention_mask.view(
                    batch_size * num_images,
                    *pixel_attention_mask.shape[2:],
                )
                pam = pam[real_images].contiguous()
                patch_size = int(base_model.config.vision_config.patch_size)
                patches = pam.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)
                patch_attention_mask = (patches.sum(dim=(-1, -2)) > 0).bool()

            vision_outputs = vision_model(
                pixel_values=flat_pixels,
                patch_attention_mask=patch_attention_mask,
                return_dict=True,
            )
            # HALP's VF is the mean-pooled vision-encoder output before the
            # multimodal connector/projection. Do not use get_image_features()
            # here because that method includes the connector.
            last_hidden = vision_outputs.last_hidden_state
            return last_hidden.mean(dim=1) if last_hidden.ndim == 3 else last_hidden


        # ----------------------------------------------------
        # Qwen2.5-VL
        # ----------------------------------------------------

        if model_key == "qwen25vl":

            pixel_values = inputs.get(
                "pixel_values"
            )

            grid_thw = inputs.get(
                "image_grid_thw"
            )

            if (
                pixel_values is None
                or grid_thw is None
            ):

                raise RuntimeError(
                    "Qwen2.5-VL inputs did not expose both "
                    "`pixel_values` and `image_grid_thw`."
                )

            visual_candidates = [
                getattr(model, "visual", None),
                getattr(getattr(model, "model", None), "visual", None),
            ]
            visual = next((candidate for candidate in visual_candidates if candidate is not None), None)

            if visual is None:
                raise RuntimeError(
                    "Qwen2.5-VL visual encoder was not found on the current model wrapper."
                )

            outputs = visual(
                pixel_values,
                grid_thw=grid_thw,
            )

            last_hidden = getattr(outputs, "last_hidden_state", None)
            if last_hidden is None and isinstance(outputs, (tuple, list)) and outputs:
                if isinstance(outputs[0], torch.Tensor):
                    last_hidden = outputs[0]
            if last_hidden is None and isinstance(outputs, torch.Tensor):
                last_hidden = outputs
            if last_hidden is None:
                raise RuntimeError(
                    "Qwen2.5-VL visual encoder did not expose a tensor-like last_hidden_state."
                )
            # Qwen's visual module returns the raw encoder sequence as
            # last_hidden_state and a separately merged/projected pooler_output.
            # HALP-style VF uses the encoder sequence, before multimodal fusion.
            return last_hidden.mean(dim=0) if last_hidden.ndim == 2 else last_hidden.mean(dim=1)


    raise NotImplementedError(
        f"No verified VF adapter exists for {model_key!r}."
    )


# ------------------------------------------------------------
# 16. Recover metadata from compact SMOKE when possible
# ------------------------------------------------------------

def recover_representation_metadata(
    smoke_record: dict,
):
    """
    Recover layer and token metadata saved by the compact smoke test.

    This avoids another forward pass for metadata that is already known.
    """

    recovered = {
        "layers":
            None,

        "vision_positions":
            None,

        "query_position":
            None,

        "image_token_id":
            None,
    }

    # --------------------------------------------------------
    # Direct fields from current T4-safe smoke result
    # --------------------------------------------------------

    if "layers" in smoke_record:

        recovered[
            "layers"
        ] = smoke_record[
            "layers"
        ]

    elif (
        "hidden_state_summary"
        in smoke_record
    ):

        summary = smoke_record[
            "hidden_state_summary"
        ]

        if isinstance(
            summary,
            dict,
        ):

            if "selected_layers" in summary:

                recovered[
                    "layers"
                ] = summary[
                    "selected_layers"
                ]


    if "vision_token_positions" in smoke_record:

        recovered[
            "vision_positions"
        ] = smoke_record[
            "vision_token_positions"
        ]

    elif "vision_token_pos" in smoke_record:

        value = smoke_record[
            "vision_token_pos"
        ]

        if value is not None:

            if isinstance(
                value,
                (list, tuple),
            ):

                recovered[
                    "vision_positions"
                ] = list(
                    value
                )

            else:

                recovered[
                    "vision_positions"
                ] = [
                    int(value)
                ]


    if "query_position" in smoke_record:

        recovered[
            "query_position"
        ] = smoke_record[
            "query_position"
        ]

    elif "query_pos" in smoke_record:

        recovered[
            "query_position"
        ] = smoke_record[
            "query_pos"
        ]


    if "image_token_id" in smoke_record:

        recovered[
            "image_token_id"
        ] = smoke_record[
            "image_token_id"
        ]

    return recovered


# ------------------------------------------------------------
# 17. Main representation-record function
# ------------------------------------------------------------

def extract_representation_record(
    model_key: str,
    smoke_record: dict,
):
    """
    Build one validated representation record.

    Unlike the old implementation, this function does not assume that
    model/inputs/hidden_states are still stored inside SMOKE.
    """

    # --------------------------------------------------------
    # Model/processor
    # --------------------------------------------------------

    model, processor = (
        get_loaded_representation_model(
            model_key
        )
    )

    # --------------------------------------------------------
    # Sample metadata
    # --------------------------------------------------------

    image_path, question = (
        get_smoke_sample_metadata(
            model_key,
            smoke_record,
        )
    )

    image_info = (
        validate_representation_image(
            image_path
        )
    )

    # --------------------------------------------------------
    # Reuse runtime tensors only when an older SMOKE object actually
    # contains them. The current compact smoke result normally does not.
    # --------------------------------------------------------

    inputs = smoke_record.get(
        "inputs"
    )

    hidden_states = smoke_record.get(
        "hidden_states"
    )

    # The current T4-safe smoke result normally contains neither.
    # Reconstruct them only when necessary.
    if (
        inputs is None
        or hidden_states is None
    ):

        print(
            "Runtime tensors are not retained in SMOKE; "
            "re-running one forward pass for representation validation."
        )

        inputs = prepare_representation_inputs(
            model_key=model_key,
            processor=processor,
            image_path=image_path,
            question=question,
        )

        inputs = move_representation_inputs(
            inputs,
            model,
        )

        if model_key == "qwen25vl":
            text_config = getattr(
                getattr(model, "config", None),
                "text_config",
                None,
            )
            depth = getattr(text_config, "num_hidden_layers", None)
            if depth is None:
                depth = len(_find_qwen_decoder_layers(model))
            layers = get_representation_layers(int(depth))

            outputs, hidden_states = run_representation_forward(
                model,
                inputs,
                model_key=model_key,
                layers=layers,
            )
        else:
            outputs, hidden_states = run_representation_forward(
                model,
                inputs,
                model_key=model_key,
            )

    else:

        outputs = None

    # --------------------------------------------------------
    # Validate inputs
    # --------------------------------------------------------

    if not hasattr(
        inputs,
        "keys",
    ):

        raise TypeError(
            "Representation inputs are not mapping-like."
        )

    input_keys = list(
        inputs.keys()
    )

    if not input_keys:

        raise RuntimeError(
            "Representation inputs are empty."
        )

    print(
        "Processor keys:",
        input_keys,
    )

    # --------------------------------------------------------
    # Hidden-state validation
    # --------------------------------------------------------

    if model_key == "qwen25vl":
        decoder_depth = len(hidden_states) - 1
    else:
        decoder_depth = get_representation_decoder_depth(
            model,
            hidden_states,
        )
        layers = get_representation_layers(decoder_depth)

    # Prefer actual runtime metadata.
    image_token_id, vision_positions = (
        get_representation_vision_positions(
            model,
            inputs,
        )
    )

    qpos = get_representation_query_position(
        inputs,
        processor,
    )

    # --------------------------------------------------------
    # If runtime token metadata could not be derived, use the compact
    # smoke metadata as a fallback.
    # --------------------------------------------------------

    compact_metadata = (
        recover_representation_metadata(
            smoke_record
        )
    )

    if not vision_positions:

        saved_positions = (
            compact_metadata[
                "vision_positions"
            ]
        )

        if saved_positions:

            vision_positions = [
                int(x)
                for x in saved_positions
            ]


    if qpos is None:

        saved_qpos = (
            compact_metadata[
                "query_position"
            ]
        )

        if saved_qpos is not None:

            qpos = int(
                saved_qpos
            )


    if image_token_id is None:

        image_token_id = (
            compact_metadata[
                "image_token_id"
            ]
        )


    # --------------------------------------------------------
    # Representation record
    # --------------------------------------------------------

    record = {

        "model_key":
            model_key,

        "model_id":
            MODEL_REGISTRY[
                model_key
            ].model_id,

        "processor_id":
            MODEL_REGISTRY[
                model_key
            ].processor_id,

        "image_path":
            str(
                image_path
            ),

        "question":
            question,

        "image_info":
            image_info,

        "input_keys":
            input_keys,

        "decoder_layers":
            decoder_depth,

        "layers":
            layers,

        "image_token_id":
            image_token_id,

        "vision_positions":
            vision_positions,

        "query_position":
            qpos,

        "feature_names":
            [],

        "features":
            {},

        "feature_shapes":
            {},

        "feature_dtypes":
            {},

        "unsupported":
            {},

        "validation":
            {},

        "generation_performed":
            False,
    }

    # --------------------------------------------------------
    # Hidden-state shape checks
    # --------------------------------------------------------

    if not isinstance(
        hidden_states,
        (tuple, list),
    ):

        raise TypeError(
            "`hidden_states` must be a tuple/list."
        )


    for layer_name, layer_idx in (
        layers.items()
    ):

        tensor = hidden_states[
            layer_idx
        ]

        if not isinstance(
            tensor,
            torch.Tensor,
        ):

            raise TypeError(
                f"{layer_name}: hidden-state at layer "
                f"{layer_idx} is not a tensor."
            )

        if tensor.ndim != 3:

            raise RuntimeError(
                f"{layer_name}: expected hidden-state shape "
                f"[batch, sequence, hidden], got "
                f"{tuple(tensor.shape)}."
            )

        if tensor.shape[0] != 1:

            raise RuntimeError(
                f"{layer_name}: expected batch size 1, got "
                f"{tensor.shape[0]}."
            )

    # --------------------------------------------------------
    # VF
    # --------------------------------------------------------

    try:

        vf_tensor = extract_vf(
            model_key,
            model,
            inputs,
        )

        if not isinstance(
            vf_tensor,
            torch.Tensor,
        ):

            raise TypeError(
                "extract_vf() did not return a torch.Tensor."
            )

        if vf_tensor.ndim == 2:

            vf_tensor = vf_tensor[
                0
            ]

        elif vf_tensor.ndim == 1:

            pass

        else:

            raise RuntimeError(
                "Unexpected VF tensor shape: "
                f"{tuple(vf_tensor.shape)}"
            )

        vf_vector = feature_to_numpy(
            vf_tensor
        )

        record[
            "features"
        ][
            "vf"
        ] = vf_vector

        record[
            "feature_names"
        ].append(
            "vf"
        )

        record[
            "feature_shapes"
        ][
            "vf"
        ] = tuple(
            vf_vector.shape
        )

        record[
            "feature_dtypes"
        ][
            "vf"
        ] = str(
            vf_vector.dtype
        )

        record[
            "validation"
        ][
            "vf"
        ] = "PASS"

    except Exception as exc:

        record[
            "unsupported"
        ][
            "vf"
        ] = (
            f"{type(exc).__name__}: {exc}"
        )

        record[
            "validation"
        ][
            "vf"
        ] = "NOT AVAILABLE"


    # --------------------------------------------------------
    # VT
    # --------------------------------------------------------

    if vision_positions:

        # Use the final vision-token position, consistent with HALP.
        vision_position = int(
            vision_positions[-1]
        )

        record[
            "selected_vision_position"
        ] = vision_position

        for layer_name, layer_idx in (
            layers.items()
        ):

            hidden = hidden_states[
                layer_idx
            ]

            if not (
                0
                <= vision_position
                < hidden.shape[1]
            ):

                raise RuntimeError(
                    f"VT {layer_name}: vision position "
                    f"{vision_position} is outside sequence length "
                    f"{hidden.shape[1]}."
                )

            vector = feature_to_numpy(
                hidden[
                    0,
                    vision_position,
                    :,
                ]
            )

            key = (
                f"vt_{layer_name}"
            )

            record[
                "features"
            ][
                key
            ] = vector

            record[
                "feature_names"
            ].append(
                key
            )

            record[
                "feature_shapes"
            ][
                key
            ] = tuple(
                vector.shape
            )

            record[
                "feature_dtypes"
            ][
                key
            ] = str(
                vector.dtype
            )

            record[
                "validation"
            ][
                key
            ] = "PASS"

    else:

        record[
            "unsupported"
        ][
            "vt"
        ] = (
            "No identifiable vision-token position was exposed "
            "by the current model inputs."
        )

        record[
            "validation"
        ][
            "vt"
        ] = "NOT AVAILABLE"


    # ------------------------------------------------------------
    # QT
    # ------------------------------------------------------------

    if qpos is not None:

        qpos = int(
            qpos
        )

        record[
            "selected_query_position"
        ] = qpos

        for layer_name, layer_idx in (
            layers.items()
        ):

            hidden = hidden_states[
                layer_idx
            ]

            if not (
                0
                <= qpos
                < hidden.shape[1]
            ):

                raise RuntimeError(
                    f"QT {layer_name}: query position "
                    f"{qpos} is outside sequence length "
                    f"{hidden.shape[1]}."
                )

            vector = feature_to_numpy(
                hidden[
                    0,
                    qpos,
                    :,
                ]
            )

            key = (
                f"qt_{layer_name}"
            )

            record[
                "features"
            ][
                key
            ] = vector

            record[
                "feature_names"
            ].append(
                key
            )

            record[
                "feature_shapes"
            ][
                key
            ] = tuple(
                vector.shape
            )

            record[
                "feature_dtypes"
            ][
                key
            ] = str(
                vector.dtype
            )

            record[
                "validation"
            ][
                key
            ] = "PASS"

    else:

        record[
            "unsupported"
        ][
            "qt"
        ] = (
            "No stable query-token position was identified."
        )

        record[
            "validation"
        ][
            "qt"
        ] = "NOT AVAILABLE"


    # --------------------------------------------------------
    # Require at least one usable representation.
    # --------------------------------------------------------

    if not record[
        "feature_names"
    ]:

        raise RuntimeError(
            f"{model_key}: no usable VF, VT, or QT representation "
            "was extracted."
        )


    # --------------------------------------------------------
    # Representation availability summary
    # --------------------------------------------------------

    record[
        "representation_availability"
    ] = {

        "VF":
            "vf"
            in record[
                "features"
            ],

        "VT":
            any(
                name.startswith(
                    "vt_"
                )
                for name in record[
                    "features"
                ]
            ),

        "QT":
            any(
                name.startswith(
                    "qt_"
                )
                for name in record[
                    "features"
                ]
            ),
    }


    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    if outputs is not None:

        del outputs

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


    return record


# ------------------------------------------------------------
# 18. Run representation validation for active models
# ------------------------------------------------------------

REPRESENTATION_RECORDS = {}

REPRESENTATION_STATUS = {}


for model_key in ACTIVE_MODELS:

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"REPRESENTATION VALIDATION: {model_key}"
    )

    print(
        "=" * 80
    )

    if model_key not in SMOKE:

        raise RuntimeError(
            f"No smoke-test record exists for {model_key!r}."
        )

    try:

        record = extract_representation_record(
            model_key,
            SMOKE[
                model_key
            ],
        )

        REPRESENTATION_RECORDS[
            model_key
        ] = record

        REPRESENTATION_STATUS[
            model_key
        ] = "PASSED"

        release_cached_model(model_key)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(
            "\nFeature validation:"
        )

        for name, vector in (
            record[
                "features"
            ].items()
        ):

            print(
                f"  {name:20s} "
                f"shape={vector.shape} "
                f"dtype={vector.dtype}"
            )

        if record[
            "unsupported"
        ]:

            print(
                "\nUnsupported / unavailable representations:"
            )

            for name, reason in (
                record[
                    "unsupported"
                ].items()
            ):

                print(
                    f"  {name:20s} -> {reason}"
                )

        print(
            "\nRepresentation availability:"
        )

        for name, available in (
            record[
                "representation_availability"
            ].items()
        ):

            print(
                f"  {name:20s}: "
                f"{'AVAILABLE' if available else 'NOT AVAILABLE'}"
            )

    except torch.cuda.OutOfMemoryError as exc:

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

        gc.collect()

        REPRESENTATION_STATUS[
            model_key
        ] = (
            "FAILED — CUDA OUT OF MEMORY"
        )

        print(
            f"[{model_key}] CUDA out of memory."
        )

        print(
            "The representation-validation forward pass could not "
            "fit in the current T4 memory state."
        )

        print(
            str(exc)
        )

        release_cached_model(model_key)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        raise

    except Exception as exc:

        REPRESENTATION_STATUS[
            model_key
        ] = (
            f"FAILED — "
            f"{type(exc).__name__}: {exc}"
        )

        print(
            f"[{model_key}] representation validation failed."
        )

        print(
            "Exception type:",
            type(exc).__name__,
        )

        print(
            "Exception:",
            str(exc),
        )

        raise


# ------------------------------------------------------------
# 19. Final representation-validation summary
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 80
)

print(
    "FINAL REPRESENTATION VALIDATION SUMMARY"
)

print(
    "=" * 80
)


for model_key, status in (
    REPRESENTATION_STATUS.items()
):

    print(
        f"{model_key:22s}: {status}"
    )


print(
    "-" * 80
)

print(
    "VF = visual features"
)

print(
    "VT = vision-token hidden states"
)

print(
    "QT = query-token hidden states"
)

print(
    "Hallucination labels required : NO"
)

print(
    "Answer generation             : NO"
)

print(
    "Pre-generation forward        : YES"
)

print(
    "Model source                  : loaded on demand; one VLM at a time"
)

print(
    "Compact SMOKE metadata reused : YES"
)

print(
    "Extra forward only when needed: YES"
)

print(
    "=" * 80
)

print(
    "Representation-validation stage complete."
)

# Cache validated capabilities so unsupported representations are skipped cleanly at extraction time.
REPRESENTATION_CAPABILITIES = {
    model_key: {
        str(rep_key).lower(): bool(rep_value)
        for rep_key, rep_value in record.get("representation_availability", {}).items()
    }
    for model_key, record in REPRESENTATION_RECORDS.items()
}


HALP representation validation

REPRESENTATION VALIDATION: smolvlm2
Loading processor...
GPU memory budget : 11.0GiB
CPU offload budget: 48GiB


Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

Runtime tensors are not retained in SMOKE; re-running one forward pass for representation validation.
Processor keys: ['input_ids', 'attention_mask', 'pixel_values', 'pixel_attention_mask']
Released cached model: smolvlm2

Feature validation:
  vf                   shape=(1152,) dtype=float32
  vt_early             shape=(2048,) dtype=float32
  vt_quarter           shape=(2048,) dtype=float32
  vt_middle            shape=(2048,) dtype=float32
  vt_three_quarter     shape=(2048,) dtype=float32
  vt_final             shape=(2048,) dtype=float32
  qt_early             shape=(2048,) dtype=float32
  qt_quarter           shape=(2048,) dtype=float32
  qt_middle            shape=(2048,) dtype=float32
  qt_three_quarter     shape=(2048,) dtype=float32
  qt_final             shape=(2048,) dtype=float32

Representation availability:
  VF                  : AVAILABLE
  VT                  : AVAILABLE
  QT                  : AVAILABLE

REPRESENTATION VALIDATION: qwen25vl
Loading processor...
[T4 pr

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Runtime tensors are not retained in SMOKE; re-running one forward pass for representation validation.
Processor keys: ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_grid_thw']
Released cached model: qwen25vl

Feature validation:
  vf                   shape=(1280,) dtype=float32
  vt_early             shape=(2048,) dtype=float32
  vt_quarter           shape=(2048,) dtype=float32
  vt_middle            shape=(2048,) dtype=float32
  vt_three_quarter     shape=(2048,) dtype=float32
  vt_final             shape=(2048,) dtype=float32
  qt_early             shape=(2048,) dtype=float32
  qt_quarter           shape=(2048,) dtype=float32
  qt_middle            shape=(2048,) dtype=float32
  qt_three_quarter     shape=(2048,) dtype=float32
  qt_final             shape=(2048,) dtype=float32

Representation availability:
  VF                  : AVAILABLE
  VT                  : AVAILABLE
  QT                  : AVAILABLE

FINAL REPRESENTATION VALIDATION SUMMARY
smolvlm2

## 12. Feature storage and resumable extraction

Features are stored in one HDF5 file per model. Each row keeps:
- stable `question_id`
- image name
- label
- feature vectors for every representation that was successfully extracted

The checkpoint file records completed IDs, failed IDs, model ID, runtime versions, extraction configuration, and timestamp.

A completed sample is never recomputed on resume.



In [14]:

def safe_run_id():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def _run_signature():
    return {
        "experiment_name": EXPERIMENT_NAME,
        "run_mode": RUN_MODE,
        "max_samples": MAX_SAMPLES,
        "experiment_data_scope": EXPERIMENT_DATA_SCOPE,
        "sample_selection_seed": int(SAMPLE_SELECTION_SEED),
        "seeds": [int(x) for x in SEEDS],
        "active_models": list(ACTIVE_MODELS),
        "dataset_fingerprint": DATASET_FINGERPRINT,
        "dataset_root": str(HALPBENCH_ROOT),
        "optimization_profile_version": OPTIMIZATION_PROFILE_VERSION,
        "qwen_min_pixels": QWEN_MIN_PIXELS,
        "qwen_max_pixels": QWEN_MAX_PIXELS,
        "smol_t4_image_edge": SMOL_T4_IMAGE_EDGE,
        "smol_t4_image_splitting": SMOL_T4_IMAGE_SPLITTING,
        "checkpoint_every": CHECKPOINT_EVERY,
        "feature_storage_dtype": FEATURE_DTYPE,
    }

signature = _run_signature()
RUN_ROOT = None
RUN_ID = None
resumed = False

if RESUME_EXISTING_RUN and not FORCE_NEW_RUN and RUNS_DIR.exists():
    candidates = []
    for p in sorted(RUNS_DIR.iterdir(), reverse=True):
        if not p.is_dir():
            continue
        cfg_path = p / "run_config.json"
        if not cfg_path.exists():
            continue
        try:
            cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if cfg.get("run_signature") == signature and not (p / "run_complete.json").exists():
            candidates.append(p)
    if candidates:
        RUN_ROOT = candidates[0]
        RUN_ID = RUN_ROOT.name
        resumed = True

if RUN_ROOT is None:
    RUN_ID = f"{EXPERIMENT_NAME}_{safe_run_id()}"
    RUN_ROOT = RUNS_DIR / RUN_ID

for d in [
    RUN_ROOT,
    RUN_ROOT / "features",
    RUN_ROOT / "tables",
    RUN_ROOT / "figures",
    RUN_ROOT / "predictions",
    RUN_ROOT / "logs",
    RUN_ROOT / "checkpoints",
]:
    d.mkdir(parents=True, exist_ok=True)

RUN_CONFIG = {
    "run_id": RUN_ID,
    "resumed_existing_run": resumed,
    "run_signature": signature,
    "run_mode": RUN_MODE,
    "max_samples": MAX_SAMPLES,
    "experiment_data_scope": EXPERIMENT_DATA_SCOPE,
    "sample_selection_seed": int(SAMPLE_SELECTION_SEED),
    "seeds": [int(x) for x in SEEDS],
    "active_models": ACTIVE_MODELS,
    "dataset_root": str(HALPBENCH_ROOT),
    "benchmark_csv": str(BENCHMARK_CSV),
    "image_dir": str(HALPBENCH_IMAGE_DIR),
    "dataset_fingerprint": DATASET_FINGERPRINT,
    "versions": VERSIONS,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

RUN_CONFIG_PATH = RUN_ROOT / "run_config.json"
if resumed and RUN_CONFIG_PATH.exists():
    old_cfg = json.loads(RUN_CONFIG_PATH.read_text(encoding="utf-8"))
    old_cfg.update({
        "last_resumed_at_utc": datetime.now(timezone.utc).isoformat(),
        "resumed_existing_run": True,
    })
    RUN_CONFIG = old_cfg
else:
    tmp = RUN_CONFIG_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
    tmp.replace(RUN_CONFIG_PATH)

print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("Resumed existing run:", resumed)

import os
def checkpoint_path(model_key):
    return RUN_ROOT / "checkpoints" / f"{model_key}_extraction_state.json"

def feature_h5_path(model_key):
    return RUN_ROOT / "features" / f"{model_key}_representations.h5"

def load_state(model_key):
    p = checkpoint_path(model_key)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    return {
        "model_key": model_key,
        "completed": [],
        "failed": {},
        "feature_keys": [],
        "model_id": MODEL_REGISTRY[model_key].model_id,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "versions": VERSIONS,
        "run_id": RUN_ID,
    }

def save_state(model_key, state):
    p = checkpoint_path(model_key)
    tmp = p.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=2), encoding="utf-8")
    os.replace(tmp, p)

def mark_run_progress(model_key, state, completed_count, total_count, started_at):
    elapsed = max(time.time() - started_at, 1e-6)
    rate = completed_count / elapsed
    remaining = max(total_count - completed_count, 0)
    eta_seconds = remaining / rate if rate > 0 else None
    state["progress"] = {
        "completed": int(completed_count),
        "total": int(total_count),
        "remaining": int(remaining),
        "elapsed_seconds": float(elapsed),
        "estimated_remaining_seconds": float(eta_seconds) if eta_seconds is not None else None,
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
    }



def completed_question_ids_from_h5(model_key):
    """Read completed question IDs from the persistent HDF5 file."""
    path = feature_h5_path(model_key)
    if not path.exists():
        return set()
    completed = set()
    with h5py.File(path, "r") as h5:
        if "question_id" not in h5 or "status" not in h5:
            return completed
        qids = [x.decode() if isinstance(x, bytes) else str(x) for x in h5["question_id"][:]]
        statuses = [x.decode() if isinstance(x, bytes) else str(x) for x in h5["status"][:]]
        completed = {qid for qid, status in zip(qids, statuses) if status == "complete"}
    return completed

def h5_init(h5, metadata, feature_shapes):
    n = len(metadata)
    str_dt = h5py.string_dtype("utf-8")
    h5.create_dataset("question_id", shape=(n,), dtype=str_dt)
    h5.create_dataset("image_name", shape=(n,), dtype=str_dt)
    h5.create_dataset("label", shape=(n,), dtype="int8")
    h5.create_dataset("status", shape=(n,), dtype=str_dt)

    for col, vals in metadata.items():
        if col not in {"question_id", "image_name", "label"}:
            h5.create_dataset(col, data=np.asarray(vals, dtype=str_dt), dtype=str_dt)

    for key, shape in feature_shapes.items():
        h5.create_dataset(
            key,
            shape=(n, shape[0]),
            dtype=np.float16,
            chunks=(1, shape[0]),
            compression="gzip",
            compression_opts=4,
        )

def ensure_h5_metadata(h5_path, frame):
    if h5_path.exists():
        return
    with h5py.File(h5_path, "w") as h5:
        h5.create_dataset("question_id", data=np.asarray(frame["question_id"].astype(str), dtype=h5py.string_dtype("utf-8")))
        h5.create_dataset("image_name", data=np.asarray(frame["image_name"].astype(str), dtype=h5py.string_dtype("utf-8")))
        h5.create_dataset("label", data=np.asarray(frame["label"], dtype=np.int8))
        h5.create_dataset("status", data=np.asarray(["pending"] * len(frame), dtype=h5py.string_dtype("utf-8")))
        for col in ["category", "dataset", "question"]:
            h5.create_dataset(col, data=np.asarray(frame[col].fillna("").astype(str), dtype=h5py.string_dtype("utf-8")))

def ensure_feature_dataset(h5, key, dim):
    if key not in h5:
        h5.create_dataset(
            key, shape=(h5["question_id"].shape[0], dim),
            dtype=np.float16, chunks=(1, dim), compression="gzip", compression_opts=4
        )


RUN_ID: halp_vlm_hallucination_t4_20260831T224100Z
RUN_ROOT: /content/drive/MyDrive/HALP_Bench_Project/runs/halp_vlm_hallucination_t4_20260831T224100Z
Resumed existing run: False


## 13. Feature extraction loop

The extraction loop is kept conservative for Colab:
- inference mode only,
- one sample at a time by default,
- mixed precision when CUDA is available,
- periodic state writes,
- sample-level failure logging,
- resume support,
- CUDA cache cleanup after failures.

The notebook does not catch all exceptions and continue silently. Permanent sample failures are recorded with the exception class/message.



In [15]:

# Note:
# Additional optimization controls are included because Google Colab FREE with
# an NVIDIA T4 has strict GPU-memory and runtime constraints. The changes below
# target practical throughput, VRAM headroom, resumability, and low-risk output
# volume without changing the scientific target or test protocol.

# ============================================================
# 13. Feature extraction loop — T4 / Colab optimized / Qwen NaN-safe
# ============================================================
#
# Purpose
# -------
# Extract HALP-style internal VLM representations from the validated
# HALP-Bench image-question pairs.
#
# Important
# ---------
# These optimization changes were introduced because the notebook is intended
# to run under the computational, VRAM, and runtime limitations of Google
# Colab Free with an NVIDIA T4 GPU.
#
# SCIENTIFIC CONTRACT
# -------------------
# - No hallucination labels are inferred or fabricated.
# - Model-specific reviewed labels are used only when a real reviewed-label
#   dataframe exists.
# - The validated HALP-Bench dataframe remains the extraction source otherwise.
# - This cell does not generate answers.
# - The configured model checkpoint and architecture are unchanged.
# - The existing VT/QT layer-selection and token-position logic is preserved.
# - Feature storage remains float16, matching the notebook's existing artifact
#   contract.
# - Non-finite model features are never replaced, clipped, zero-filled, or
#   otherwise fabricated.
#
# PERFORMANCE / ROBUSTNESS CHANGES
# ---------------------------------
# 1. Image preflight checks only unique paths with cheap filesystem metadata.
# 2. No heartbeat thread is created for each sample.
# 3. SmolVLM2 preprocessing is configured once per loaded model.
# 4. SmolVLM2 repeated-image vision features are reused when consecutive rows
#    share an identical image path.
# 5. HDF5 feature-dataset dimensions are cached after first creation.
# 6. New feature datasets are uncompressed by default to reduce CPU overhead
#    during the hot write path.
# 7. Qwen numerical instability is handled with a fail-safe retry strategy:
#       fast path -> SDPA math path -> safe eager attention path
#    only when a genuine non-finite feature/output is detected.
# 8. A finite model feature that cannot be represented by float16 is reported
#    as a storage overflow instead of being silently clipped.
# 9. Previously failed Qwen samples are moved to the front of the resume queue
#    so a known bad sample is diagnosed immediately rather than after hours.
# 10. A one-sample Qwen numerical retry never changes the model checkpoint or
#     the experiment labels/protocol.
#
# NUMERICAL-STABILITY BASIS
# -------------------------
# The current Qwen2.5-VL implementation performs the standard attention score
# computation followed by FP32 softmax, while the score matmul itself can still
# be formed from FP16 query/key tensors. Community implementations of Qwen
# 2.5-VL have documented an FP16 attention-precision workaround that computes
# attention more safely. PyTorch documents that the SDPA math backend uses FP32
# intermediates for FP16 inputs. The code below uses those mechanisms only as a
# targeted retry for samples that actually produce non-finite representations.
# ============================================================


# ------------------------------------------------------------
# 1. Imports and required notebook objects
# ------------------------------------------------------------

import gc
import hashlib
import inspect
import json
import math
import os
import time
from contextlib import contextmanager, nullcontext
from datetime import datetime, timezone
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. T4 CUDA memory guard
# ------------------------------------------------------------
def enforce_t4_cuda_memory_budget(stage: str = "unknown"):
    """
    Enforce the notebook's operational T4 CUDA-memory ceiling.

    This helper is intentionally lightweight because it is called only at
    major model/forward checkpoints. It checks both the current allocation
    and the peak allocation observed since the last CUDA peak reset.

    Important:
    - `memory_allocated()` is active tensor memory.
    - `memory_reserved()` also includes PyTorch's allocator cache and therefore
      is not treated as a hard scientific-memory violation by itself.
    - A peak/current active allocation above the configured 13.5 GiB ceiling
      is a hard failure.
    - A peak/current allocation above the preferred 13.0 GiB level is reported
      as a warning, not an automatic failure.
    """
    if not torch.cuda.is_available():
        return {
            "stage": str(stage),
            "cuda_available": False,
            "allocated_gib": 0.0,
            "reserved_gib": 0.0,
            "peak_allocated_gib": 0.0,
            "hard_limit_gib": float(
                globals().get("T4_HARD_CUDA_GIB", 13.5)
            ),
            "preferred_peak_gib": float(
                globals().get("T4_PREFERRED_PEAK_CUDA_GIB", 13.0)
            ),
            "status": "CPU_ONLY",
        }

    hard_limit_gib = float(
        globals().get("T4_HARD_CUDA_GIB", 13.5)
    )
    preferred_peak_gib = float(
        globals().get("T4_PREFERRED_PEAK_CUDA_GIB", 13.0)
    )

    allocated_gib = float(torch.cuda.memory_allocated()) / (1024 ** 3)
    reserved_gib = float(torch.cuda.memory_reserved()) / (1024 ** 3)
    peak_allocated_gib = float(
        torch.cuda.max_memory_allocated()
    ) / (1024 ** 3)

    # If PyTorch has accumulated allocator cache, releasing unused cache is a
    # safe one-time housekeeping action before the final hard-limit decision.
    if reserved_gib > hard_limit_gib and allocated_gib <= hard_limit_gib:
        try:
            torch.cuda.empty_cache()
            reserved_gib = float(torch.cuda.memory_reserved()) / (1024 ** 3)
        except Exception:
            pass

    status = "OK"
    if max(allocated_gib, peak_allocated_gib) > hard_limit_gib:
        status = "HARD_LIMIT_EXCEEDED"
    elif max(allocated_gib, peak_allocated_gib) > preferred_peak_gib:
        status = "ABOVE_PREFERRED_PEAK"

    if status == "HARD_LIMIT_EXCEEDED":
        raise RuntimeError(
            "T4 CUDA memory safety ceiling exceeded during "
            f"{stage}: active={allocated_gib:.3f} GiB, "
            f"peak={peak_allocated_gib:.3f} GiB, "
            f"hard_limit={hard_limit_gib:.3f} GiB."
        )

    return {
        "stage": str(stage),
        "cuda_available": True,
        "allocated_gib": allocated_gib,
        "reserved_gib": reserved_gib,
        "peak_allocated_gib": peak_allocated_gib,
        "hard_limit_gib": hard_limit_gib,
        "preferred_peak_gib": preferred_peak_gib,
        "status": status,
    }



print("=" * 80)
print("HALP-Bench feature extraction — T4 / Colab optimized / Qwen NaN-safe")
print("=" * 80)


REQUIRED_GLOBALS = [
    "ACTIVE_MODELS",
    "MODEL_REGISTRY",
    "df",
    "LOADED",
    "RUN_MODE",
    "MAX_SAMPLES",
    "RUN_ROOT",
    "RUN_ID",
    "CHECKPOINT_EVERY",
    "torch",
    "np",
    "pd",
    "h5py",
    "Path",
    "gc",
    "datetime",
    "timezone",
]

missing_globals = [
    name
    for name in REQUIRED_GLOBALS
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        "Missing required notebook objects:\n"
        f"{missing_globals}\n\n"
        "Run the earlier environment, dataset, model-registry, "
        "smoke-test, and run-configuration cells first."
    )


if not isinstance(ACTIVE_MODELS, (list, tuple)) or not ACTIVE_MODELS:
    raise ValueError("ACTIVE_MODELS must be a non-empty list or tuple.")

if not isinstance(MODEL_REGISTRY, dict):
    raise TypeError("MODEL_REGISTRY must be a dictionary.")

if not isinstance(LOADED, dict):
    raise TypeError("LOADED must be a dictionary.")

if not isinstance(df, pd.DataFrame):
    raise TypeError("`df` must be a pandas DataFrame.")

if df.empty:
    raise RuntimeError("The validated HALP-Bench dataframe `df` is empty.")


# ------------------------------------------------------------
# 2. Extraction-level configuration
# ------------------------------------------------------------

FAST_IMAGE_PREFLIGHT = True
SMOL_ENABLE_REPEAT_IMAGE_CACHE = True
H5_FEATURE_COMPRESSION = None
PROGRESS_MIN_INTERVAL_SECONDS = 1.0
TELEMETRY_EVERY = 100

QWEN_TARGET_VISUAL_TOKENS = int(globals().get("QWEN_TARGET_VISUAL_TOKENS", 256))
QWEN_MAX_VISION_PATCHES = int(
    globals().get("QWEN_MAX_VISION_PATCHES", QWEN_TARGET_VISUAL_TOKENS * 4)
)

# Qwen's normal path is kept fast. The numerical fallback is activated only
# for a sample that actually produces a non-finite representation.
QWEN_ENABLE_NUMERICAL_RETRY = True
QWEN_ENABLE_SDPA_MATH_RETRY = True
QWEN_ENABLE_SAFE_EAGER_RETRY = True
QWEN_STABLE_ATTENTION_MAX_VALUE = 80.0
QWEN_STABLE_ATTENTION_MIN_VALUE = -80.0
QWEN_RETRY_MAX_ATTEMPTS = 2

_BASE_EXTRACTION_PROFILE = str(
    globals().get(
        "OPTIMIZATION_PROFILE_VERSION",
        "t4-colab-free-feature-extraction",
    )
)
EXTRACTION_PROFILE_VERSION = (
    f"{_BASE_EXTRACTION_PROFILE}+qwen-nan-safe-v1"
)


class NonFiniteFeatureError(RuntimeError):
    """A model-produced feature contains NaN or Inf before float16 storage."""

    def __init__(self, message, feature_key=None, stage="model_output"):
        super().__init__(message)
        self.feature_key = feature_key
        self.stage = stage


class FeatureStorageOverflowError(RuntimeError):
    """A finite feature cannot be represented by the configured float16 store."""

    def __init__(self, message, feature_key=None):
        super().__init__(message)
        self.feature_key = feature_key
        self.stage = "float16_storage"


# ------------------------------------------------------------
# 3. Validate dataframe schema and IDs
# ------------------------------------------------------------

REQUIRED_COLUMNS = {
    "question_id",
    "image_name",
    "question",
    "resolved_image_path",
}

missing_columns = REQUIRED_COLUMNS - set(df.columns)

if missing_columns:
    raise RuntimeError(
        "The validated HALP-Bench dataframe is missing required columns: "
        f"{sorted(missing_columns)}"
    )


df = df.copy()
df["question_id"] = df["question_id"].astype(str)
df["image_name"] = df["image_name"].astype(str)

if df["question_id"].duplicated().any():
    duplicate_ids = (
        df.loc[
            df["question_id"].duplicated(keep=False),
            "question_id",
        ]
        .unique()
        .tolist()
    )
    raise RuntimeError(
        "Duplicate question_id values were found in the validated "
        f"HALP-Bench dataframe. Examples: {duplicate_ids[:20]}"
    )

print("Validated HALP-Bench rows:", len(df))
print("Unique question IDs:", df["question_id"].nunique())
print("Unique images:", df["image_name"].nunique())


# ------------------------------------------------------------
# 4. Extraction dataframe selection
# ------------------------------------------------------------

def get_extraction_frame(model_key: str):
    """Return the exact experiment dataframe without fabricating labels."""

    candidate = None
    frame_source = None

    if (
        "experiment_model_df" in globals()
        and isinstance(experiment_model_df, dict)
        and model_key in experiment_model_df
        and isinstance(experiment_model_df[model_key], pd.DataFrame)
        and not experiment_model_df[model_key].empty
    ):
        candidate = experiment_model_df[model_key].copy().reset_index(drop=True)
        frame_source = "fixed experiment subset"

    elif (
        "model_df" in globals()
        and isinstance(model_df, dict)
        and model_key in model_df
        and isinstance(model_df[model_key], pd.DataFrame)
        and not model_df[model_key].empty
    ):
        candidate = model_df[model_key].copy().reset_index(drop=True)
        frame_source = "model-specific reviewed-label dataframe"

    if candidate is not None:

        required_reviewed = {
            "question_id",
            "image_name",
            "question",
            "resolved_image_path",
            "label",
        }
        missing_reviewed = required_reviewed - set(candidate.columns)

        if missing_reviewed:
            raise RuntimeError(
                f"{model_key}: model-specific dataframe exists but is missing "
                f"required columns: {sorted(missing_reviewed)}"
            )

        candidate["question_id"] = candidate["question_id"].astype(str)
        candidate["image_name"] = candidate["image_name"].astype(str)

        if candidate["question_id"].duplicated().any():
            raise RuntimeError(
                f"{model_key}: duplicate question_id values were found in the "
                "model-specific reviewed-label dataframe."
            )

        return candidate, True, frame_source

    return (
        df.copy().reset_index(drop=True),
        False,
        "validated HALP-Bench dataframe",
    )


# ------------------------------------------------------------
# 5. Row selection
# ------------------------------------------------------------

def select_extraction_rows(frame: pd.DataFrame) -> pd.DataFrame:
    # The exact cohort is already selected upstream for supervised consistency.
    if (
        "experiment_model_df" in globals()
        and any(
            isinstance(value, pd.DataFrame) and frame.equals(value)
            for value in experiment_model_df.values()
        )
    ):
        return frame.copy().reset_index(drop=True)

    if RUN_MODE not in {"smoke_test", "full"}:
        raise ValueError(
            f"Unsupported RUN_MODE={RUN_MODE!r}. Expected 'smoke_test' or 'full'."
        )

    frame = frame.copy().reset_index(drop=True)

    if RUN_MODE == "smoke_test":
        limit = 32 if MAX_SAMPLES is None else int(MAX_SAMPLES)
    elif MAX_SAMPLES is None:
        return frame
    else:
        limit = int(MAX_SAMPLES)

    if limit <= 0:
        raise ValueError("MAX_SAMPLES must be positive when specified.")

    if limit >= len(frame):
        return frame.copy().reset_index(drop=True)

    sample_seed = int(globals().get("SAMPLE_SELECTION_SEED", 42))
    return (
        frame.sample(n=limit, random_state=sample_seed)
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 6. Fast image preflight
# ------------------------------------------------------------

def _normalize_image_path(value) -> str:
    if value is None or pd.isna(value):
        return ""
    return os.path.abspath(
        os.path.normpath(
            os.path.expanduser(str(value))
        )
    )


IMAGE_PREFLIGHT_CACHE = globals().get("IMAGE_PREFLIGHT_CACHE", {})


def _image_preflight_signature(paths):
    """Build a cache signature from path + size + mtime, without decoding images."""
    digest = hashlib.sha256()
    for path_string in sorted(set(paths)):
        digest.update(path_string.encode("utf-8", errors="replace"))
        digest.update(b"\0")
        try:
            stat = os.stat(path_string)
            digest.update(str(stat.st_size).encode("ascii"))
            digest.update(b"\0")
            digest.update(str(stat.st_mtime_ns).encode("ascii"))
        except OSError:
            digest.update(b"MISSING")
        digest.update(b"\n")
    return digest.hexdigest()



def validate_extraction_images(frame: pd.DataFrame, model_key: str):
    """Fast validation of unique image paths; no image decoding is performed."""

    paths = [
        _normalize_image_path(value)
        for value in frame["resolved_image_path"].tolist()
    ]

    signature = _image_preflight_signature(paths)

    if FAST_IMAGE_PREFLIGHT and signature in IMAGE_PREFLIGHT_CACHE:
        cached = IMAGE_PREFLIGHT_CACHE[signature]
        print(
            f"[{model_key}] Image preflight: REUSED cached result "
            f"({cached['unique_images']} unique images)."
        )
        return

    unique_paths = sorted({path for path in paths if path})
    problems = []

    pbar = tqdm(
        total=len(unique_paths),
        desc=f"{model_key} | fast image preflight",
        unit="image",
        dynamic_ncols=True,
        mininterval=0.25,
    )

    for path_string in unique_paths:
        if not os.path.isfile(path_string):
            problems.append(path_string)
        pbar.update(1)

    pbar.close()

    missing_values = sorted({p for p in paths if not p})
    problems.extend(missing_values)
    problems = sorted(set(problems))

    if problems:
        raise RuntimeError(
            "Fast image preflight found missing/non-file image paths. "
            f"Examples: {problems[:20]}"
        )

    IMAGE_PREFLIGHT_CACHE[signature] = {
        "validated_at_utc": datetime.now(timezone.utc).isoformat(),
        "unique_images": len(unique_paths),
        "rows": len(frame),
        "fast_only": True,
    }

    print(
        f"[{model_key}] Image preflight: PASS "
        f"({len(unique_paths)} unique images; no full image decode performed)."
    )


# ------------------------------------------------------------
# 7. Paths and checkpoint helpers
# ------------------------------------------------------------

def extraction_feature_h5_path(model_key: str) -> Path:
    if "feature_h5_path" in globals():
        path = feature_h5_path(model_key)
    else:
        path = Path(RUN_ROOT) / "features" / f"{model_key}_representations.h5"
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path



def extraction_checkpoint_path(model_key: str) -> Path:
    if "checkpoint_path" in globals():
        path = checkpoint_path(model_key)
    else:
        path = (
            Path(RUN_ROOT)
            / "checkpoints"
            / f"{model_key}_extraction_state.json"
        )
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path



def load_extraction_state(model_key: str) -> dict:
    if "load_state" in globals():
        state = load_state(model_key)
        if not isinstance(state, dict):
            raise TypeError(
                f"load_state({model_key!r}) returned {type(state)} instead of dict."
            )
        return state

    path = extraction_checkpoint_path(model_key)
    if path.exists():
        state = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(state, dict):
            raise TypeError(f"Checkpoint state must be a dict: {path}")
        return state

    return {
        "model_key": model_key,
        "completed": [],
        "failed": {},
        "feature_keys": [],
        "model_id": getattr(MODEL_REGISTRY[model_key], "model_id", None),
        "run_id": str(RUN_ID),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "versions": globals().get("VERSIONS", {}),
    }



def save_extraction_state(model_key: str, state: dict):
    if "save_state" in globals():
        save_state(model_key, state)
        return

    path = extraction_checkpoint_path(model_key)
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=2), encoding="utf-8")
    tmp.replace(path)


# ------------------------------------------------------------
# 8. HDF5 initialization/alignment
# ------------------------------------------------------------

def initialize_extraction_h5(
    h5_path: Path,
    frame: pd.DataFrame,
    model_key: str,
    labels_available: bool,
):
    if h5_path.exists():
        return

    string_dtype = h5py.string_dtype(encoding="utf-8")

    with h5py.File(h5_path, "w") as h5:
        h5.create_dataset(
            "question_id",
            data=np.asarray(frame["question_id"].astype(str), dtype=string_dtype),
            dtype=string_dtype,
        )
        h5.create_dataset(
            "image_name",
            data=np.asarray(frame["image_name"].astype(str), dtype=string_dtype),
            dtype=string_dtype,
        )
        h5.create_dataset(
            "status",
            data=np.full(len(frame), "pending", dtype=object),
            dtype=string_dtype,
        )

        h5.attrs["run_id"] = str(RUN_ID)
        h5.attrs["model_key"] = str(model_key)
        h5.attrs["labels_available"] = bool(labels_available)
        h5.attrs["dataset_role"] = (
            "model_specific_reviewed"
            if labels_available
            else "validated_HALP_Bench_without_model_specific_labels"
        )
        h5.attrs["extraction_profile_version"] = str(EXTRACTION_PROFILE_VERSION)

        if labels_available and "label" in frame.columns:
            labels = (
                pd.to_numeric(frame["label"], errors="raise")
                .astype(np.int8)
                .to_numpy()
            )
            if not np.isin(labels, [0, 1]).all():
                raise ValueError("Model-specific labels must contain only 0/1.")
            h5.create_dataset("label", data=labels, dtype=np.int8)

        for column in ["category", "dataset", "question", "group_id"]:
            if column in frame.columns:
                values = frame[column].fillna("").astype(str).to_numpy()
                h5.create_dataset(
                    column,
                    data=np.asarray(values, dtype=string_dtype),
                    dtype=string_dtype,
                )



def _decode_h5_strings(values):
    result = []
    for value in values:
        if isinstance(value, bytes):
            result.append(value.decode("utf-8"))
        else:
            result.append(str(value))
    return result



def validate_h5_alignment(h5_path: Path, frame: pd.DataFrame):
    if not h5_path.exists():
        return

    with h5py.File(h5_path, "r") as h5:
        if "question_id" not in h5:
            raise RuntimeError(f"{h5_path} has no question_id dataset.")

        existing_ids = _decode_h5_strings(h5["question_id"][:])
        current_ids = frame["question_id"].astype(str).tolist()

        if existing_ids != current_ids:
            raise RuntimeError(
                "Existing HDF5 question_id order does not match the current "
                "extraction dataframe. Use a new RUN_ID rather than mixing runs."
            )


# ------------------------------------------------------------
# 9. Feature dataset helper
# ------------------------------------------------------------

def ensure_extraction_feature_dataset(
    h5,
    key: str,
    dimension: int,
    feature_dimensions: dict,
):
    dimension = int(dimension)
    if dimension <= 0:
        raise ValueError(f"Invalid feature dimension for {key}: {dimension}")

    if key in feature_dimensions:
        if feature_dimensions[key] != dimension:
            raise RuntimeError(
                f"{key}: feature dimension mismatch. "
                f"Existing={feature_dimensions[key]}, new={dimension}"
            )
        return

    if key in h5:
        existing_shape = h5[key].shape
        if len(existing_shape) != 2 or int(existing_shape[1]) != dimension:
            raise RuntimeError(
                f"{key}: existing dataset shape {existing_shape} is incompatible "
                f"with feature dimension {dimension}."
            )
        feature_dimensions[key] = dimension
        return

    n = int(h5["question_id"].shape[0])

    create_kwargs = {
        "shape": (n, dimension),
        "dtype": np.float16,
    }

    if H5_FEATURE_COMPRESSION is not None:
        create_kwargs.update(
            {
                "chunks": (1, dimension),
                "compression": H5_FEATURE_COMPRESSION,
            }
        )

    h5.create_dataset(key, **create_kwargs)
    feature_dimensions[key] = dimension


# ------------------------------------------------------------
# 10. Model lifecycle
# ------------------------------------------------------------

def get_loaded_model_and_processor(model_key: str):
    for cached_key in list(LOADED.keys()):
        if cached_key != model_key:
            release_cached_model(cached_key)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

    if model_key not in LOADED:
        if model_key not in MODEL_REGISTRY:
            raise KeyError(f"Unknown model key: {model_key!r}")

        processor, model = load_smoke_model(
            model_key,
            MODEL_REGISTRY[model_key],
        )

        LOADED[model_key] = {
            "processor": processor,
            "model": model,
        }

    entry = LOADED[model_key]
    if not isinstance(entry, dict):
        raise TypeError(f"LOADED[{model_key!r}] must be a dictionary.")

    model = entry.get("model")
    processor = entry.get("processor")

    if model is None or processor is None:
        raise RuntimeError(
            f"LOADED[{model_key!r}] is missing model or processor."
        )

    model.eval()
    return model, processor


# ------------------------------------------------------------
# 11. Configure SmolVLM2 once
# ------------------------------------------------------------

def configure_smolvlm2_extraction_processor(processor):
    if not hasattr(processor, "image_processor"):
        raise RuntimeError(
            "SmolVLM2 processor does not expose image_processor."
        )

    image_processor = processor.image_processor
    image_processor.do_resize = True
    image_processor.size = {"longest_edge": int(SMOL_T4_IMAGE_EDGE)}
    image_processor.max_image_size = {
        "longest_edge": int(SMOL_T4_IMAGE_EDGE)
    }
    image_processor.do_image_splitting = bool(SMOL_T4_IMAGE_SPLITTING)


# ------------------------------------------------------------
# 12. Processor adapters
# ------------------------------------------------------------

SMOL_PROCESSOR_KWARGS_SUPPORT = globals().get(
    "SMOL_PROCESSOR_KWARGS_SUPPORT",
    {},
)



def _smol_apply_chat_template(processor, messages):
    """Call the SmolVLM2 multimodal chat template robustly across Transformers APIs."""

    if processor is None:
        raise ValueError("processor must not be None.")

    processor_signature = inspect.signature(processor.apply_chat_template)
    parameters = processor_signature.parameters
    has_explicit_processor_kwargs = "processor_kwargs" in parameters
    has_var_keyword = any(
        parameter.kind == inspect.Parameter.VAR_KEYWORD
        for parameter in parameters.values()
    )

    call_kwargs = {
        "add_generation_prompt": False,
        "tokenize": True,
        "return_dict": True,
    }

    if has_explicit_processor_kwargs:
        call_kwargs["processor_kwargs"] = {
            "return_tensors": "pt",
        }
    else:
        call_kwargs["return_tensors"] = "pt"

    if "return_dict" not in parameters and not has_var_keyword:
        call_kwargs.pop("return_dict", None)
    if "tokenize" not in parameters and not has_var_keyword:
        call_kwargs.pop("tokenize", None)
    if "add_generation_prompt" not in parameters and not has_var_keyword:
        call_kwargs.pop("add_generation_prompt", None)

    try:
        result = processor.apply_chat_template(messages, **call_kwargs)
    except TypeError:
        result = processor.apply_chat_template(
            messages,
            add_generation_prompt=False,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

    if result is None:
        raise RuntimeError("SmolVLM2 processor returned None.")

    if hasattr(result, "items"):
        return result

    if isinstance(result, str):
        formatted_prompt = result
        image_path = messages[0]["content"][0]["path"]
        try:
            direct_result = processor(
                text=[formatted_prompt],
                images=[image_path],
                return_tensors="pt",
            )
        except Exception as exc:
            raise RuntimeError(
                "SmolVLM2 apply_chat_template returned a string and the "
                "direct processor fallback also failed."
            ) from exc

        if not hasattr(direct_result, "items"):
            raise TypeError(
                "SmolVLM2 direct processor fallback did not return a "
                "mapping-like object. "
                f"Got {type(direct_result)}"
            )
        return direct_result

    raise TypeError(
        "SmolVLM2 processor returned an unexpected object from "
        "apply_chat_template: "
        f"{type(result)}"
    )



def prepare_smolvlm2_inputs(processor, image_path: str, question: str):
    question = str(question).strip()
    if not question:
        raise ValueError("Question is empty.")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "path": str(image_path)},
                {"type": "text", "text": question},
            ],
        }
    ]

    return _smol_apply_chat_template(processor, messages)



def prepare_qwen25vl_inputs(processor, image_path: str, question: str):
    try:
        from qwen_vl_utils import process_vision_info
    except ImportError as exc:
        raise ImportError(
            "Qwen2.5-VL requires qwen-vl-utils."
        ) from exc

    question = str(question).strip()
    if not question:
        raise ValueError("Question is empty.")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": str(image_path)},
                {"type": "text", "text": question},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    processor_kwargs = {
        "text": [text],
        "images": image_inputs,
        "videos": video_inputs,
        "padding": True,
        "return_tensors": "pt",
    }

    # Reassert the T4 visual budget at call time. Qwen's smart_resize rounds
    # spatial dimensions to a 28-pixel grid, so a nominal 256-token ceiling
    # can occasionally round upward. For such outliers, reduce only this
    # sample's max-pixel budget until the actual grid is within the cap.
    adaptive_max_pixels = int(QWEN_MAX_PIXELS)
    inputs = None
    grid = None
    last_budget_error = None

    for _attempt in range(4):
        processor_kwargs_attempt = dict(processor_kwargs)
        processor_kwargs_attempt["images_kwargs"] = {
            "min_pixels": int(min(QWEN_MIN_PIXELS, adaptive_max_pixels)),
            "max_pixels": int(adaptive_max_pixels),
        }
        try:
            inputs = processor(**processor_kwargs_attempt)
        except TypeError:
            inputs = processor(**processor_kwargs_attempt)

        grid = inputs.get("image_grid_thw")
        if grid is None:
            break

        grid_cpu_probe = grid.detach().cpu()
        if grid_cpu_probe.ndim != 2 or grid_cpu_probe.shape[1] != 3:
            break

        patch_tokens_probe = int(
            sum(int(t) * int(h) * int(w) for t, h, w in grid_cpu_probe.tolist())
        )
        merge_size_probe = int(
            getattr(getattr(processor, "image_processor", None), "merge_size", 2) or 2
        )
        image_tokens_probe = patch_tokens_probe // (merge_size_probe ** 2)

        if (
            patch_tokens_probe <= int(QWEN_MAX_VISION_PATCHES)
            and image_tokens_probe <= int(QWEN_TARGET_VISUAL_TOKENS)
        ):
            last_budget_error = None
            break

        last_budget_error = (
            f"patch_tokens={patch_tokens_probe}, image_tokens={image_tokens_probe}, "
            f"adaptive_max_pixels={adaptive_max_pixels}"
        )
        adaptive_max_pixels = max(
            int(QWEN_MIN_PIXELS),
            int(adaptive_max_pixels * 0.88),
        )
        inputs = None
        grid = None

    if inputs is None or grid is None:
        raise RuntimeError(
            "Qwen visual preprocessing could not satisfy the T4 safety budget "
            f"after adaptive retries. Last state: {last_budget_error}"
        )

    if grid is None:
        raise RuntimeError(
            "Qwen2.5-VL processor did not return image_grid_thw."
        )

    grid_cpu = grid.detach().cpu()
    if grid_cpu.ndim != 2 or grid_cpu.shape[1] != 3:
        raise RuntimeError(
            f"Unexpected Qwen image_grid_thw shape: {tuple(grid_cpu.shape)}"
        )

    total_patch_tokens = int(
        sum(
            int(t) * int(h) * int(w)
            for t, h, w in grid_cpu.tolist()
        )
    )
    merge_size = int(
        getattr(
            getattr(processor, "image_processor", None),
            "merge_size",
            2,
        ) or 2
    )

    total_image_tokens = total_patch_tokens // (merge_size ** 2)

    max_patch_tokens = int(
        globals().get(
            "QWEN_MAX_VISION_PATCHES",
            int(QWEN_MAX_PIXELS // (28 * 28)) * 4,
        )
    )
    max_image_tokens = int(
        globals().get(
            "QWEN_TARGET_VISUAL_TOKENS",
            QWEN_MAX_PIXELS // (28 * 28),
        )
    )

    if total_patch_tokens > max_patch_tokens or total_image_tokens > max_image_tokens:
        raise RuntimeError(
            "Qwen visual preprocessing exceeded the T4 safety budget: "
            f"patch_tokens={total_patch_tokens}, "
            f"image_tokens={total_image_tokens}, "
            f"limits=({max_patch_tokens} patches, {max_image_tokens} image tokens)."
        )

    inputs["_qwen_visual_grid"] = {
        "patch_tokens": total_patch_tokens,
        "image_tokens": total_image_tokens,
        "grid_thw": grid_cpu.tolist(),
    }

    return inputs


def prepare_extraction_inputs(
    model_key: str,
    processor,
    image_path: str,
    question: str,
):
    image_path = _normalize_image_path(image_path)
    if not image_path:
        raise FileNotFoundError("resolved_image_path is empty.")
    if not os.path.isfile(image_path):
        raise FileNotFoundError(image_path)

    if model_key == "smolvlm2":
        return prepare_smolvlm2_inputs(
            processor=processor,
            image_path=image_path,
            question=question,
        )

    if model_key == "qwen25vl":
        return prepare_qwen25vl_inputs(
            processor=processor,
            image_path=image_path,
            question=question,
        )

    raise NotImplementedError(
        f"No extraction adapter exists for model {model_key!r}."
    )


# ------------------------------------------------------------
# 13. Device placement
# ------------------------------------------------------------

def get_extraction_input_device(model):
    if torch.cuda.is_available():
        try:
            for parameter in model.parameters():
                if parameter.device.type == "cuda":
                    return parameter.device
        except Exception:
            pass

    try:
        device = model.device
        if isinstance(device, torch.device):
            return device
        return torch.device(str(device))
    except Exception:
        pass

    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")



def move_extraction_inputs(inputs, model=None, target_device=None):
    if target_device is None:
        if model is None:
            raise ValueError("Provide either model or target_device.")
        target_device = get_extraction_input_device(model)

    moved = {}
    for key, value in inputs.items():
        if isinstance(value, torch.Tensor):
            moved[key] = value.to(target_device)
        else:
            moved[key] = value
    return moved


# ------------------------------------------------------------
# 14. Qwen numerical-stability contexts
# ------------------------------------------------------------

def _qwen_config_holders(model):
    """Return model/config objects whose attention implementation may control dispatch."""
    holders = []
    seen = set()

    candidates = [
        model,
        getattr(model, "model", None),
        getattr(model, "language_model", None),
        getattr(getattr(model, "model", None), "language_model", None),
        getattr(model, "visual", None),
    ]

    for obj in candidates:
        if obj is not None:
            cfg = getattr(obj, "config", None)
            if cfg is not None and id(cfg) not in seen:
                holders.append(cfg)
                seen.add(id(cfg))

    try:
        for module in model.modules():
            cfg = getattr(module, "config", None)
            if cfg is not None and id(cfg) not in seen:
                module_name = module.__class__.__module__
                if "qwen2_5_vl" in module_name:
                    holders.append(cfg)
                    seen.add(id(cfg))
    except Exception:
        pass

    return holders


@contextmanager
def _temporary_qwen_attn_implementation(model, implementation: str):
    """Temporarily change all Qwen attention configs and restore them exactly afterward."""
    changed = []
    seen = set()

    candidates = [
        model,
        getattr(model, "model", None),
        getattr(model, "language_model", None),
        getattr(getattr(model, "model", None), "language_model", None),
        getattr(model, "visual", None),
    ]

    try:
        for obj in candidates:
            if obj is None:
                continue
            attr_name = "_attn_implementation"
            key = (id(obj), attr_name)
            if hasattr(obj, attr_name) and key not in seen:
                old_value = getattr(obj, attr_name)
                changed.append((obj, attr_name, old_value))
                setattr(obj, attr_name, implementation)
                seen.add(key)

        for cfg in _qwen_config_holders(model):
            attr_name = "_attn_implementation"
            key = (id(cfg), attr_name)
            if hasattr(cfg, attr_name) and key not in seen:
                old_value = getattr(cfg, attr_name)
                changed.append((cfg, attr_name, old_value))
                setattr(cfg, attr_name, implementation)
                seen.add(key)
    except Exception:
        for obj, attr_name, old_value in reversed(changed):
            try:
                setattr(obj, attr_name, old_value)
            except Exception:
                pass
        raise

    try:
        yield
    finally:
        for obj, attr_name, old_value in reversed(changed):
            try:
                setattr(obj, attr_name, old_value)
            except Exception:
                pass


@contextmanager
def _qwen_sdpa_math_context(model):
    """
    Use PyTorch's math SDPA path for a stability retry.

    PyTorch documents that the math SDPA backend keeps FP16/BF16 intermediates
    in FP32. The context is used only for Qwen samples that already produced a
    non-finite representation on the normal path.
    """

    if not torch.cuda.is_available():
        yield
        return

    old_fp16_reduction = None
    old_fp16_accumulation = None

    if hasattr(torch.backends.cuda.matmul, "allow_fp16_reduced_precision_reduction"):
        old_fp16_reduction = (
            torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction
        )
        try:
            torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
        except Exception:
            pass

    if hasattr(torch.backends.cuda.matmul, "allow_fp16_accumulation"):
        old_fp16_accumulation = (
            torch.backends.cuda.matmul.allow_fp16_accumulation
        )
        try:
            torch.backends.cuda.matmul.allow_fp16_accumulation = False
        except Exception:
            pass

    try:
        try:
            from torch.nn.attention import SDPBackend, sdpa_kernel

            with _temporary_qwen_attn_implementation(model, "sdpa"):
                with sdpa_kernel(backends=[SDPBackend.MATH]):
                    yield

        except (ImportError, AttributeError):
            # Compatibility path for older PyTorch releases.
            try:
                from torch.backends.cuda import sdp_kernel
            except ImportError:
                yield
            else:
                with _temporary_qwen_attn_implementation(model, "sdpa"):
                    with sdp_kernel(
                        enable_flash=False,
                        enable_math=True,
                        enable_mem_efficient=False,
                    ):
                        yield

    finally:
        if old_fp16_reduction is not None:
            try:
                torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = (
                    old_fp16_reduction
                )
            except Exception:
                pass

        if old_fp16_accumulation is not None:
            try:
                torch.backends.cuda.matmul.allow_fp16_accumulation = (
                    old_fp16_accumulation
                )
            except Exception:
                pass



def _qwen_safe_eager_attention_forward(
    module,
    query,
    key,
    value,
    attention_mask,
    scaling,
    dropout=0.0,
    **kwargs,
):
    """
    Numerically safer eager attention used only as the final Qwen retry.

    The query/key score matmul is explicitly computed in FP32. Additive -inf
    masks remain -inf. Positive infinities are replaced by a finite upper bound
    before softmax because an +inf/+inf row would otherwise yield NaN. Genuine
    NaNs in query/key/score tensors are never hidden and therefore still fail.
    """

    # Import from the installed Qwen implementation so grouped-query behavior
    # exactly follows the active Transformers version.
    try:
        module_path = module.__class__.__module__
        qwen_module = __import__(module_path, fromlist=["repeat_kv"])
        repeat_kv = getattr(qwen_module, "repeat_kv")
    except Exception:
        repeat_kv = None

    if repeat_kv is not None:
        key_states = repeat_kv(key, module.num_key_value_groups)
        value_states = repeat_kv(value, module.num_key_value_groups)
    else:
        n_rep = int(module.num_key_value_groups)
        if n_rep == 1:
            key_states = key
            value_states = value
        else:
            key_states = (
                key[:, :, None, :, :]
                .expand(
                    key.shape[0],
                    key.shape[1],
                    n_rep,
                    key.shape[2],
                    key.shape[3],
                )
                .reshape(
                    key.shape[0],
                    key.shape[1] * n_rep,
                    key.shape[2],
                    key.shape[3],
                )
            )
            value_states = (
                value[:, :, None, :, :]
                .expand(
                    value.shape[0],
                    value.shape[1],
                    n_rep,
                    value.shape[2],
                    value.shape[3],
                )
                .reshape(
                    value.shape[0],
                    value.shape[1] * n_rep,
                    value.shape[2],
                    value.shape[3],
                )
            )

    # Preserve the model's actual FP16/BF16 output dtype but compute attention
    # scores and value aggregation in FP32 for the numerical retry only.
    query_fp32 = query.float()
    key_fp32 = key_states.float()
    value_fp32 = value_states.float()

    if not bool(torch.isfinite(query_fp32).all().item()):
        raise NonFiniteFeatureError(
            "Qwen safe attention received a non-finite query tensor.",
            feature_key="attention.query",
            stage="attention_input",
        )

    if not bool(torch.isfinite(key_fp32).all().item()):
        raise NonFiniteFeatureError(
            "Qwen safe attention received a non-finite key tensor.",
            feature_key="attention.key",
            stage="attention_input",
        )

    attn_weights = torch.matmul(
        query_fp32,
        key_fp32.transpose(2, 3),
    ) * float(scaling)

    if attention_mask is not None:
        if attention_mask.dtype == torch.bool:
            attn_weights = attn_weights.masked_fill(
                ~attention_mask,
                float("-inf"),
            )
        else:
            attn_weights = attn_weights + attention_mask.float()

    if bool(torch.isnan(attn_weights).any().item()):
        raise NonFiniteFeatureError(
            "Qwen safe attention produced NaN attention scores.",
            feature_key="attention.scores",
            stage="attention_scores",
        )

    # Preserve additive -inf masks, but make positive overflow finite so that
    # an all-maximum row does not become inf-inf inside softmax.
    pos_inf = torch.isposinf(attn_weights)
    if bool(pos_inf.any().item()):
        attn_weights = torch.where(
            pos_inf,
            torch.full_like(
                attn_weights,
                float(QWEN_STABLE_ATTENTION_MAX_VALUE),
            ),
            attn_weights,
        )

    finite_mask = torch.isfinite(attn_weights)
    if bool(finite_mask.any().item()):
        finite_values = attn_weights[finite_mask]
        clipped_values = finite_values.clamp(
            min=float(QWEN_STABLE_ATTENTION_MIN_VALUE),
            max=float(QWEN_STABLE_ATTENTION_MAX_VALUE),
        )
        attn_weights = attn_weights.clone()
        attn_weights[finite_mask] = clipped_values

    attn_weights = torch.nn.functional.softmax(
        attn_weights,
        dim=-1,
        dtype=torch.float32,
    )

    if not bool(torch.isfinite(attn_weights).all().item()):
        raise NonFiniteFeatureError(
            "Qwen safe attention softmax produced a non-finite tensor.",
            feature_key="attention.probabilities",
            stage="attention_softmax",
        )

    if dropout:
        attn_weights = torch.nn.functional.dropout(
            attn_weights,
            p=dropout,
            training=module.training,
        )

    attn_output = torch.matmul(attn_weights, value_fp32)
    attn_output = attn_output.to(query.dtype)
    attn_output = attn_output.transpose(1, 2).contiguous()

    if not bool(torch.isfinite(attn_output.float()).all().item()):
        raise NonFiniteFeatureError(
            "Qwen safe attention produced a non-finite attention output.",
            feature_key="attention.output",
            stage="attention_output",
        )

    return attn_output, attn_weights.to(query.dtype)


@contextmanager
def _qwen_visual_fp32_context(model):
    """
    Temporarily execute the Qwen visual encoder in true FP32 for the final
    numerical-stability retry.

    Qwen's model code casts image pixels to ``self.visual.dtype`` immediately
    before calling the visual encoder. The production model is intentionally
    loaded in FP16 on a T4, so a CUDA autocast FP16 region can still make the
    visual attention/linear path operate in reduced precision even when the
    visual module itself is temporarily cast to FP32. This context therefore:
      1. temporarily casts only the visual encoder parameters/buffers to FP32;
      2. wraps the visual encoder's own ``forward`` in autocast-disabled mode;
      3. restores the original dtype/forward method exactly afterward.

    The language/decoder stack remains unchanged. This context is used only for
    the final Qwen numerical retry, after the normal and SDPA-math paths fail
    with a genuine non-finite result.
    """
    visual_candidates = [
        getattr(model, "visual", None),
        getattr(getattr(model, "model", None), "visual", None),
    ]
    visual = next(
        (candidate for candidate in visual_candidates if candidate is not None),
        None,
    )

    if visual is None:
        yield
        return

    original_forward = visual.forward
    original_dtype = getattr(visual, "dtype", None)

    try:
        visual.to(dtype=torch.float32)

        def _fp32_visual_forward(*args, **kwargs):
            if torch.cuda.is_available():
                with torch.autocast(
                    device_type="cuda",
                    enabled=False,
                ):
                    return original_forward(*args, **kwargs)
            return original_forward(*args, **kwargs)

        visual.forward = _fp32_visual_forward
        yield
    finally:
        visual.forward = original_forward
        if original_dtype is not None:
            visual.to(dtype=original_dtype)


@contextmanager
def _qwen_safe_eager_context(model):
    """Temporarily route Qwen eager attention through the FP32-score implementation."""

    try:
        import importlib

        qwen_module = importlib.import_module(
            "transformers.models.qwen2_5_vl.modeling_qwen2_5_vl"
        )
    except Exception:
        yield
        return

    original_eager = getattr(
        qwen_module,
        "eager_attention_forward",
        None,
    )

    if original_eager is None:
        yield
        return

    qwen_module.eager_attention_forward = _qwen_safe_eager_attention_forward

    try:
        # The preceding retries change text-attention precision only. The
        # observed failure can originate earlier in self.visual(...), so the
        # final retry also executes the vision encoder in true FP32.
        with _qwen_visual_fp32_context(model):
            with _temporary_qwen_attn_implementation(model, "eager"):
                yield
    finally:
        qwen_module.eager_attention_forward = original_eager


# ------------------------------------------------------------
# 15. Forward pass
# ------------------------------------------------------------


def _find_qwen_decoder_layers(model):
    """Locate Qwen2.5-VL decoder blocks across current/fallback wrapper paths."""
    candidates = [
        getattr(getattr(model, "model", None), "language_model", None),
        getattr(model, "language_model", None),
        getattr(getattr(model, "model", None), "model", None),
    ]

    for container in candidates:
        layers = getattr(container, "layers", None)
        if layers is not None:
            try:
                if len(layers) > 0:
                    return layers
            except Exception:
                pass

    raise RuntimeError(
        "Could not locate Qwen2.5-VL decoder layers for low-memory extraction."
    )


def get_qwen_decoder_depth(model):
    config = getattr(model, "config", None)
    text_config = getattr(config, "text_config", None)
    depth = getattr(text_config, "num_hidden_layers", None)
    if depth is not None:
        return int(depth)
    return len(_find_qwen_decoder_layers(model))


def _capture_qwen_selected_layers(
    model,
    layers,
    vision_position,
    query_position,
):
    """
    Register hooks for only the selected Qwen decoder layers.

    Only the single VT vector and/or QT vector needed from each selected layer
    is copied to CPU. The full hidden-state tuple is never retained.
    """
    decoder_layers = _find_qwen_decoder_layers(model)
    depth = len(decoder_layers)
    captured = {}
    handles = []

    # Capture the raw visual encoder output during the SAME multimodal forward.
    # This avoids the previous extra vision-only forward and therefore reduces
    # both runtime and peak transient memory while preserving VF semantics.
    visual_candidates = [
        getattr(model, "visual", None),
        getattr(getattr(model, "model", None), "visual", None),
    ]
    visual = next((candidate for candidate in visual_candidates if candidate is not None), None)

    def visual_hook(module, module_inputs, module_output):
        raw = getattr(module_output, "last_hidden_state", None)
        if raw is None and isinstance(module_output, (tuple, list)) and module_output:
            raw = module_output[0]
        if raw is None and isinstance(module_output, torch.Tensor):
            raw = module_output
        if raw is None:
            raise RuntimeError("Qwen visual hook did not receive a tensor-like output.")
        if raw.ndim == 3:
            vf = raw.mean(dim=1)[0]
        elif raw.ndim == 2:
            vf = raw.mean(dim=0)
        else:
            raise RuntimeError(
                f"Unexpected Qwen visual output shape in hook: {tuple(raw.shape)}"
            )
        if not bool(torch.isfinite(vf.float()).all().item()):
            raise NonFiniteFeatureError(
                "Qwen raw visual features contain NaN/Inf.",
                feature_key="vf",
                stage="model_output",
            )
        captured["__vf__"] = vf.detach().cpu()

    if visual is None:
        raise RuntimeError(
            "Qwen2.5-VL visual encoder was not found on the current model wrapper."
        )

    handles.append(visual.register_forward_hook(visual_hook))

    def make_hook(layer_index):
        def hook(module, module_inputs, module_output):
            hidden = (
                module_output[0]
                if isinstance(module_output, (tuple, list))
                else module_output
            )

            if not isinstance(hidden, torch.Tensor):
                raise TypeError(
                    f"Qwen layer {layer_index} returned {type(hidden)}."
                )

            if hidden.ndim != 3 or hidden.shape[0] != 1:
                raise RuntimeError(
                    f"Unexpected Qwen layer {layer_index} output shape: "
                    f"{tuple(hidden.shape)}"
                )

            record = {}

            if vision_position is not None:
                pos = int(vision_position)
                if not 0 <= pos < hidden.shape[1]:
                    raise RuntimeError(
                        f"VT layer {layer_index}: position {pos} is outside "
                        f"sequence length {hidden.shape[1]}."
                    )
                vt = hidden[0, pos, :]
                if not bool(torch.isfinite(vt.float()).all().item()):
                    raise NonFiniteFeatureError(
                        f"Qwen vt_{layer_index} contains NaN/Inf.",
                        feature_key=f"vt_layer_{layer_index}",
                        stage="model_output",
                    )
                record["vt"] = vt.detach().cpu()

            if query_position is not None:
                pos = int(query_position)
                if not 0 <= pos < hidden.shape[1]:
                    raise RuntimeError(
                        f"QT layer {layer_index}: position {pos} is outside "
                        f"sequence length {hidden.shape[1]}."
                    )
                qt = hidden[0, pos, :]
                if not bool(torch.isfinite(qt.float()).all().item()):
                    raise NonFiniteFeatureError(
                        f"Qwen qt_{layer_index} contains NaN/Inf.",
                        feature_key=f"qt_layer_{layer_index}",
                        stage="model_output",
                    )
                record["qt"] = qt.detach().cpu()

            captured[int(layer_index)] = record

        return hook

    try:
        for _, layer_index in layers.items():
            zero_based = int(layer_index) - 1
            if not 0 <= zero_based < depth:
                raise ValueError(
                    f"Invalid selected Qwen layer {layer_index} for depth {depth}."
                )

            handles.append(
                decoder_layers[zero_based].register_forward_hook(
                    make_hook(int(layer_index))
                )
            )

        return captured, handles

    except Exception:
        for handle in handles:
            try:
                handle.remove()
            except Exception:
                pass
        raise


def _run_qwen_selected_layer_forward(
    model,
    inputs,
    layers,
    vision_position,
    query_position,
):
    captured, handles = _capture_qwen_selected_layers(
        model=model,
        layers=layers,
        vision_position=vision_position,
        query_position=query_position,
    )

    try:
        with torch.inference_mode():
            if torch.cuda.is_available():
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=True,
                ):
                    try:
                        outputs = model(
                            **inputs,
                            output_hidden_states=False,
                            return_dict=True,
                            use_cache=False,
                            logits_to_keep=1,
                        )
                    except TypeError:
                        outputs = model(
                            **inputs,
                            output_hidden_states=False,
                            return_dict=True,
                            use_cache=False,
                        )
            else:
                outputs = model(
                    **inputs,
                    output_hidden_states=False,
                    return_dict=True,
                    use_cache=False,
                )
    finally:
        for handle in handles:
            try:
                handle.remove()
            except Exception:
                pass

    if outputs is None:
        raise RuntimeError("Qwen model forward returned None.")

    captured_layer_keys = {
        int(key) for key in captured.keys() if isinstance(key, int)
    }
    missing = sorted(
        {
            int(layer_index)
            for layer_index in layers.values()
        }
        - captured_layer_keys
    )

    if missing:
        raise RuntimeError(
            f"Qwen selected-layer hooks did not fire for layers: {missing}"
        )

    return outputs, captured


def _write_qwen_selected_layer_features(
    h5,
    h5_index,
    captured,
    layers,
    feature_dimensions,
):
    written_keys = []

    for layer_name, layer_index in layers.items():
        record = captured.get(int(layer_index))
        if record is None:
            raise RuntimeError(
                f"Qwen selected layer {layer_index} was not captured."
            )

        for family in ("vt", "qt"):
            if family not in record:
                continue

            key = f"{family}_{layer_name}"
            vector = feature_vector_numpy(
                record[family],
                feature_key=key,
            )

            ensure_extraction_feature_dataset(
                h5,
                key,
                vector.shape[0],
                feature_dimensions,
            )

            h5[key][h5_index] = vector
            written_keys.append(key)

    return written_keys


def _forward_model_once(
    model,
    inputs,
    model_key=None,
    layers=None,
    vision_position=None,
    query_position=None,
):
    """
    One inference pass.

    Qwen uses selected-layer hooks and output_hidden_states=False. SmolVLM2
    keeps the previous output_hidden_states path because it already completed
    the 10k-row T4 extraction successfully.
    """
    if model_key == "qwen25vl":
        if layers is None:
            raise ValueError(
                "Qwen extraction requires selected decoder layers."
            )

        return _run_qwen_selected_layer_forward(
            model=model,
            inputs=inputs,
            layers=layers,
            vision_position=vision_position,
            query_position=query_position,
        )

    with torch.inference_mode():
        if torch.cuda.is_available():
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=True,
            ):
                try:
                    outputs = model(
                        **inputs,
                        output_hidden_states=True,
                        return_dict=True,
                        use_cache=False,
                        logits_to_keep=1,
                    )
                except TypeError:
                    outputs = model(
                        **inputs,
                        output_hidden_states=True,
                        return_dict=True,
                        use_cache=False,
                    )
        else:
            outputs = model(
                **inputs,
                output_hidden_states=True,
                return_dict=True,
                use_cache=False,
            )

    return outputs, outputs.hidden_states


# ------------------------------------------------------------
# 15. Forward pass with Qwen numerical retry
# ------------------------------------------------------------

def run_extraction_forward(
    model,
    inputs,
    model_key=None,
    attention_mode="fast",
    layers=None,
    vision_position=None,
    query_position=None,
):
    if model_key != "qwen25vl" or attention_mode == "fast":
        return _forward_model_once(
            model=model,
            inputs=inputs,
            model_key=model_key,
            layers=layers,
            vision_position=vision_position,
            query_position=query_position,
        )

    if attention_mode == "sdpa_math":
        with _qwen_sdpa_math_context(model):
            return _forward_model_once(
                model=model,
                inputs=inputs,
                model_key=model_key,
                layers=layers,
                vision_position=vision_position,
                query_position=query_position,
            )

    if attention_mode == "safe_eager":
        with _qwen_safe_eager_context(model):
            return _forward_model_once(
                model=model,
                inputs=inputs,
                model_key=model_key,
                layers=layers,
                vision_position=vision_position,
                query_position=query_position,
            )

    raise ValueError(
        f"Unsupported Qwen attention_mode={attention_mode!r}."
    )


# ------------------------------------------------------------
# 16. Hidden states / layer selection
# ------------------------------------------------------------

def get_extraction_hidden_states(outputs):
    hidden_states = getattr(outputs, "hidden_states", None)

    if hidden_states is None and "get_hidden_state_layers" in globals():
        hidden_states = get_hidden_state_layers(outputs)

    if hidden_states is None:
        raise RuntimeError(
            "Model forward succeeded but hidden_states were not returned."
        )

    if not isinstance(hidden_states, (tuple, list)):
        raise TypeError(
            f"Unexpected hidden_states type: {type(hidden_states)}"
        )

    if len(hidden_states) < 2:
        raise RuntimeError("Too few hidden-state tensors were returned.")

    return hidden_states



def get_extraction_decoder_depth(model, hidden_states):
    if "decoder_layer_count" in globals():
        try:
            depth = int(decoder_layer_count(model, hidden_states))
            if depth >= 1:
                return depth
        except Exception:
            pass

    depth = len(hidden_states) - 1
    if depth < 1:
        raise RuntimeError("Unable to infer decoder depth.")
    return depth



def get_extraction_layers(depth: int):
    if "selected_layer_indices" in globals():
        selected = selected_layer_indices(depth)
    else:
        def nearest_layer(ratio: float):
            return max(
                1,
                min(depth, int(math.floor(ratio * depth))),
            )

        selected = {
            "early": nearest_layer(1.0 / depth),
            "quarter": nearest_layer(0.25),
            "middle": nearest_layer(0.50),
            "three_quarter": nearest_layer(0.75),
            "final": depth,
        }

    result = {}
    for name, index in selected.items():
        index = int(index)
        if not 1 <= index <= depth:
            raise ValueError(
                f"Invalid selected layer {index} for {name}; decoder depth={depth}"
            )
        result[str(name)] = index
    return result


# ------------------------------------------------------------
# 17. Token positions
# ------------------------------------------------------------

def get_vision_token_positions(model, inputs, processor=None):
    config = getattr(model, "config", None)
    image_token_id = getattr(config, "image_token_id", None)

    # SmolVLM2 can expose the image token only through the processor/tokenizer
    # rather than the top-level model config. Use that metadata when present;
    # never fabricate a position if the actual ID cannot be resolved.
    if image_token_id is None and processor is not None:
        tokenizer = getattr(processor, "tokenizer", None)
        for holder in (processor, tokenizer):
            candidate = getattr(holder, "image_token_id", None)
            if candidate is not None:
                image_token_id = candidate
                break

    if image_token_id is None or "input_ids" not in inputs:
        return image_token_id, []

    input_ids = inputs["input_ids"]
    if input_ids.ndim != 2:
        return image_token_id, []

    positions = (
        torch.nonzero(
            input_ids[0] == int(image_token_id),
            as_tuple=False,
        )
        .flatten()
        .tolist()
    )

    return image_token_id, positions



def get_extraction_query_position(inputs, processor):
    if "query_position" in globals():
        try:
            value = query_position(inputs, processor)
            if value is not None:
                return int(value)
        except Exception:
            pass

    if "input_ids" not in inputs:
        return None

    input_ids = inputs["input_ids"]
    if input_ids.ndim != 2:
        return None

    attention_mask = inputs.get("attention_mask")
    if attention_mask is not None and attention_mask.ndim == 2:
        valid_positions = (
            torch.nonzero(
                attention_mask[0].bool(),
                as_tuple=False,
            )
            .flatten()
            .tolist()
        )
        if valid_positions:
            return int(valid_positions[-1])

    return int(input_ids.shape[1] - 1)


# ------------------------------------------------------------
# 18. Feature conversion / VF extraction
# ------------------------------------------------------------

def feature_vector_numpy(tensor, feature_key="feature"):
    if not isinstance(tensor, torch.Tensor):
        raise TypeError("Expected a torch.Tensor.")

    raw = tensor.detach()

    # IMPORTANT: distinguish actual model non-finite values from an overflow
    # caused only by the later float16 artifact conversion.
    if not bool(torch.isfinite(raw.float()).all().item()):
        raise NonFiniteFeatureError(
            f"{feature_key} contains NaN/Inf in the model output before float16 storage.",
            feature_key=feature_key,
            stage="model_output",
        )

    vector = (
        raw.to(dtype=torch.float16)
        .cpu()
        .numpy()
        .reshape(-1)
    )

    if vector.size == 0:
        raise ValueError(f"{feature_key} is empty.")

    if not np.isfinite(vector).all():
        raise FeatureStorageOverflowError(
            f"{feature_key} is finite before storage but overflows float16 storage.",
            feature_key=feature_key,
        )

    return vector



def extract_visual_features(
    model_key,
    model,
    inputs,
    outputs=None,
):
    """Extract the architecture-specific VF representation."""

    if model_key == "smolvlm2":
        image_hidden_states = None

        if outputs is not None:
            image_hidden_states = getattr(
                outputs,
                "image_hidden_states",
                None,
            )

        if isinstance(image_hidden_states, (tuple, list)):
            if len(image_hidden_states) != 1:
                raise RuntimeError(
                    "Unexpected SmolVLM2 image_hidden_states container."
                )
            image_hidden_states = image_hidden_states[0]

        if image_hidden_states is None:
            model_core = getattr(model, "model", None)
            get_image_features = getattr(
                model_core,
                "get_image_features",
                None,
            )
            if get_image_features is None:
                get_image_features = getattr(
                    model,
                    "get_image_features",
                    None,
                )

            if get_image_features is None:
                raise RuntimeError(
                    "SmolVLM2 does not expose a usable image feature path."
                )

            pixel_values = inputs.get("pixel_values")
            pixel_attention_mask = inputs.get("pixel_attention_mask")

            if pixel_values is None:
                raise RuntimeError(
                    "SmolVLM2 inputs do not contain pixel_values."
                )

            with torch.inference_mode():
                result = get_image_features(
                    pixel_values=pixel_values,
                    pixel_attention_mask=pixel_attention_mask,
                    return_dict=True,
                )

            image_hidden_states = getattr(
                result,
                "pooler_output",
                None,
            )

            if image_hidden_states is None:
                raise RuntimeError(
                    "SmolVLM2 visual encoder did not return pooler_output."
                )

        if image_hidden_states.ndim == 4:
            pooled = image_hidden_states.mean(dim=2).squeeze(1)
        elif image_hidden_states.ndim == 3:
            pooled = image_hidden_states.mean(dim=1)
        elif image_hidden_states.ndim == 2:
            pooled = image_hidden_states
        else:
            raise RuntimeError(
                "Unexpected SmolVLM2 VF shape: "
                f"{tuple(image_hidden_states.shape)}"
            )

        return pooled

    if model_key == "qwen25vl":
        capabilities = {
            str(k).lower(): bool(v)
            for k, v in globals().get(
                "REPRESENTATION_CAPABILITIES",
                {},
            ).get(model_key, {}).items()
        }

        if not capabilities.get("vf", False):
            raise NotImplementedError(
                "Qwen2.5-VL VF is not exposed by the validated model wrapper."
            )

        raise RuntimeError(
            "Qwen2.5-VL VF capability is declared available, but no verified "
            "adapter is implemented in this extraction cell."
        )

    raise NotImplementedError(
        f"No verified VF adapter exists for {model_key!r}."
    )


# ------------------------------------------------------------
# 19. Per-sample feature write helper
# ------------------------------------------------------------

def _write_selected_hidden_features(
    h5,
    h5_index: int,
    hidden_states,
    layers,
    vision_position,
    query_position,
    feature_dimensions,
):
    """
    Gather selected VT/QT tensors, validate the raw model values first, then
    transfer a single FP16 matrix to the host and write the rows.
    """

    tensor_list = []
    key_list = []

    if vision_position is not None:
        for layer_name, layer_index in layers.items():
            hidden = hidden_states[layer_index]

            if hidden.ndim != 3:
                raise RuntimeError(
                    f"VT {layer_name}: expected [batch, sequence, hidden], "
                    f"got {tuple(hidden.shape)}"
                )

            if not 0 <= vision_position < hidden.shape[1]:
                raise RuntimeError(
                    f"VT {layer_name}: vision position {vision_position} is "
                    f"outside sequence length {hidden.shape[1]}"
                )

            tensor_list.append(hidden[0, vision_position, :])
            key_list.append(f"vt_{layer_name}")

    if query_position is not None:
        for layer_name, layer_index in layers.items():
            hidden = hidden_states[layer_index]

            if hidden.ndim != 3:
                raise RuntimeError(
                    f"QT {layer_name}: expected [batch, sequence, hidden], "
                    f"got {tuple(hidden.shape)}"
                )

            if not 0 <= query_position < hidden.shape[1]:
                raise RuntimeError(
                    f"QT {layer_name}: query position {query_position} is "
                    f"outside sequence length {hidden.shape[1]}"
                )

            tensor_list.append(hidden[0, query_position, :])
            key_list.append(f"qt_{layer_name}")

    if not tensor_list:
        return []

    # Validate the raw model values before any float16 conversion. No silent
    # nan_to_num/clip/zero-fill is allowed here.
    raw_stacked = torch.stack(
        [tensor.reshape(-1) for tensor in tensor_list],
        dim=0,
    )

    finite_rows = torch.isfinite(raw_stacked.float()).all(dim=1)
    if not bool(finite_rows.all().item()):
        bad_indices = (
            torch.nonzero(~finite_rows, as_tuple=False)
            .flatten()
            .tolist()
        )
        bad_keys = [key_list[i] for i in bad_indices]
        raise NonFiniteFeatureError(
            "Selected hidden features contain NaN/Inf before float16 storage: "
            + ", ".join(bad_keys),
            feature_key=bad_keys[0] if bad_keys else None,
            stage="model_output",
        )

    stacked = (
        raw_stacked
        .to(dtype=torch.float16)
        .cpu()
        .numpy()
    )

    written_keys = []

    for key, vector in zip(key_list, stacked):
        vector = np.asarray(vector, dtype=np.float16).reshape(-1)

        if vector.size == 0:
            raise RuntimeError(f"{key} is empty.")

        if not np.isfinite(vector).all():
            raise FeatureStorageOverflowError(
                f"{key} is finite before storage but overflows float16 storage.",
                feature_key=key,
            )

        ensure_extraction_feature_dataset(
            h5,
            key,
            vector.shape[0],
            feature_dimensions,
        )
        h5[key][h5_index] = vector
        written_keys.append(key)

    return written_keys


# ------------------------------------------------------------
# 20. Non-finite error detection
# ------------------------------------------------------------

def _is_nan_inf_runtime_error(exc: Exception) -> bool:
    text = str(exc).lower()
    tokens = (
        "nan",
        "inf",
        "infinity",
        "non-finite",
        "nonfinite",
        "probability tensor contains",
    )
    return any(token in text for token in tokens)



def _clear_sample_cuda_state():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


# ------------------------------------------------------------
# 21. Main extraction function
# ------------------------------------------------------------

def extract_model_features(model_key: str):
    if model_key not in MODEL_REGISTRY:
        raise KeyError(f"{model_key!r} is not present in MODEL_REGISTRY.")

    frame, labels_available, frame_source = get_extraction_frame(model_key)
    eligible = select_extraction_rows(frame)
    eligible.attrs["_model_key"] = model_key

    print("\n" + "-" * 80)
    print(f"FEATURE EXTRACTION: {model_key}")
    print("-" * 80)
    print("Data source:", frame_source)
    print("Rows selected:", len(eligible))
    print("RUN_MODE:", RUN_MODE)
    print("MAX_SAMPLES:", MAX_SAMPLES)
    print("Reviewed labels available:", labels_available)
    print("Extraction profile:", EXTRACTION_PROFILE_VERSION)

    if not labels_available:
        print(
            "No model-specific reviewed hallucination labels are available "
            "for this model."
        )
        print("Feature extraction remains valid without those labels.")
        print(
            "Supervised probe training must remain blocked until real "
            "model-specific labels are supplied."
        )

    total = len(eligible)

    normalized_paths = [
        _normalize_image_path(value)
        for value in eligible["resolved_image_path"].tolist()
    ]
    qids = eligible["question_id"].astype(str).tolist()
    questions = eligible["question"].astype(str).tolist()

    eligible["_normalized_image_path"] = normalized_paths

    validate_extraction_images(eligible, model_key)

    print(f"[{model_key}] Loading/reusing VLM model...")

    model, processor = get_loaded_model_and_processor(model_key)
    model_input_device = get_extraction_input_device(model)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    if model_key == "smolvlm2":
        if "SMOL_T4_IMAGE_EDGE" not in globals():
            raise RuntimeError(
                "SMOL_T4_IMAGE_EDGE is missing from the notebook configuration."
            )
        if "SMOL_T4_IMAGE_SPLITTING" not in globals():
            raise RuntimeError(
                "SMOL_T4_IMAGE_SPLITTING is missing from the notebook configuration."
            )
        configure_smolvlm2_extraction_processor(processor)

        # Fail fast on the exact processor contract used by the production loop.
        adapter_probe = prepare_smolvlm2_inputs(
            processor=processor,
            image_path=normalized_paths[0],
            question=questions[0],
        )
        if not hasattr(adapter_probe, "items"):
            raise RuntimeError(
                "SmolVLM2 processor self-test failed: expected a "
                "mapping-like multimodal input."
            )
        adapter_probe = None
        gc.collect()

    print("Model class:", type(model).__name__)
    print("Processor class:", type(processor).__name__)
    print("Model input device:", model_input_device)
    if torch.cuda.is_available():
        enforce_t4_cuda_memory_budget(stage=f"{model_key} model load")

    if model_key == "qwen25vl":
        current_attn_impl = None
        try:
            current_attn_impl = getattr(
                getattr(model, "model", model),
                "config",
                None,
            )
            current_attn_impl = getattr(
                current_attn_impl,
                "_attn_implementation",
                None,
            )
        except Exception:
            pass
        print("Qwen attention implementation:", current_attn_impl)
        print(
            "Qwen numerical retry enabled:",
            bool(QWEN_ENABLE_NUMERICAL_RETRY),
        )

    h5_path = extraction_feature_h5_path(model_key)

    initialize_extraction_h5(
        h5_path=h5_path,
        frame=eligible,
        model_key=model_key,
        labels_available=labels_available,
    )

    validate_h5_alignment(h5_path, eligible)

    state = load_extraction_state(model_key)

    stored_run_id = state.get("run_id")
    if stored_run_id not in (None, str(RUN_ID)):
        raise RuntimeError(
            f"Checkpoint run_id mismatch for {model_key}: "
            f"checkpoint={stored_run_id!r}, current={str(RUN_ID)!r}. "
            "Use a new RUN_ID."
        )

    completed = {str(x) for x in state.get("completed", [])}

    if "completed_question_ids_from_h5" in globals():
        completed |= {
            str(x)
            for x in completed_question_ids_from_h5(model_key)
        }

    failed = state.setdefault("failed", {})
    if not isinstance(failed, dict):
        raise TypeError("Checkpoint `failed` field must be a dictionary.")

    state.update(
        {
            "model_key": model_key,
            "model_id": getattr(
                MODEL_REGISTRY[model_key],
                "model_id",
                None,
            ),
            "run_id": str(RUN_ID),
            "frame_source": frame_source,
            "labels_available": bool(labels_available),
            "n_rows": int(total),
            "run_mode": RUN_MODE,
            "max_samples": MAX_SAMPLES,
            "sample_selection_seed": int(SAMPLE_SELECTION_SEED),
            "extraction_profile_version": str(EXTRACTION_PROFILE_VERSION),
        }
    )

    state.setdefault("numerical_retries", [])
    state.setdefault("numerical_retry_count", 0)

    save_extraction_state(model_key, state)

    log_path = (
        Path(RUN_ROOT)
        / "logs"
        / f"{model_key}_extraction_log.csv"
    )
    log_path.parent.mkdir(parents=True, exist_ok=True)

    results_log = []
    feature_dimensions = {}
    extraction_stage_started_at = time.time()

    numerical_retry_records = []
    numerical_retry_count = int(state.get("numerical_retry_count", 0) or 0)

    with h5py.File(h5_path, "a") as h5:
        h5_qids = _decode_h5_strings(h5["question_id"][:])
        h5_index_by_qid = {
            qid: index
            for index, qid in enumerate(h5_qids)
        }

        if len(h5_qids) != total:
            raise RuntimeError(
                f"HDF5 row count {len(h5_qids)} does not match current frame {total}."
            )

        h5_status = _decode_h5_strings(h5["status"][:])
        resume_complete_count = h5_status.count("complete")
        resume_failed_count = h5_status.count("failed")

        cached_decoder_depth = None
        cached_layers = None

        print("\n" + "-" * 80)
        print(f"[{model_key}] RESUME STATUS")
        print(f"  Total rows            : {total}")
        print(f"  Previously complete   : {resume_complete_count}")
        print(f"  Previously failed     : {resume_failed_count}")
        print(f"  Remaining incomplete  : {total - resume_complete_count}")
        print("-" * 80)

        processing_indices = sorted(
            range(total),
            key=lambda i: (normalized_paths[i], qids[i]),
        )

        # Prioritize known failed/incomplete samples first during resume. This
        # is especially useful after a numerical failure because it validates
        # the retry policy immediately instead of hours later.
        failed_qids = {
            qid
            for qid, status in zip(h5_qids, h5_status)
            if status == "failed"
        }
        failed_qids |= {str(qid) for qid in failed.keys()}

        if failed_qids:
            prioritized = [
                i
                for i in processing_indices
                if qids[i] in failed_qids
            ]
            remainder = [
                i
                for i in processing_indices
                if qids[i] not in failed_qids
            ]
            processing_indices = prioritized + remainder

            print(
                f"[{model_key}] Resume priority: moved "
                f"{len(prioritized)} previously failed sample(s) to the front."
            )

        pbar = tqdm(
            total=total,
            initial=resume_complete_count,
            desc=f"{model_key} | feature extraction",
            unit="sample",
            dynamic_ncols=True,
            mininterval=PROGRESS_MIN_INTERVAL_SECONDS,
            leave=True,
        )

        smol_cached_image_key = None
        smol_cached_image_hidden_states = None

        checkpoint_interval = max(1, int(CHECKPOINT_EVERY))
        extraction_gc_every = max(
            1,
            int(
                globals().get(
                    "EXTRACTION_GC_EVERY",
                    checkpoint_interval,
                )
            ),
        )
        telemetry_every = max(1, int(TELEMETRY_EVERY))

        try:
            for index in processing_indices:
                qid = qids[index]
                question = questions[index]
                image_path = normalized_paths[index]
                h5_index = h5_index_by_qid[qid]

                current_status = h5_status[h5_index]

                if qid in completed and current_status == "complete":
                    continue

                started = time.time()
                inputs = None
                outputs = None
                hidden_states = None
                numerical_retry_used = False
                numerical_retry_modes = []
                final_attention_mode = "fast"

                try:
                    inputs = prepare_extraction_inputs(
                        model_key=model_key,
                        processor=processor,
                        image_path=image_path,
                        question=question,
                    )

                    if not hasattr(inputs, "items"):
                        raise TypeError(
                            "Processor output is not mapping-like."
                        )

                    used_smol_cached_vision = False

                    if (
                        model_key == "smolvlm2"
                        and SMOL_ENABLE_REPEAT_IMAGE_CACHE
                        and smol_cached_image_key == image_path
                        and smol_cached_image_hidden_states is not None
                    ):
                        inputs.pop("pixel_values", None)
                        inputs.pop("pixel_attention_mask", None)
                        inputs["image_hidden_states"] = (
                            smol_cached_image_hidden_states
                        )
                        used_smol_cached_vision = True

                    inputs = move_extraction_inputs(
                        inputs,
                        target_device=model_input_device,
                    )

                    # Determine token positions before the Qwen forward so
                    # hooks capture only the required VT/QT vectors.
                    image_token_id, vision_positions = (
                        get_vision_token_positions(
                            model,
                            inputs,
                            processor=processor,
                        )
                    )
                    qpos = get_extraction_query_position(
                        inputs,
                        processor,
                    )
                    vision_position = (
                        int(vision_positions[-1])
                        if vision_positions
                        else None
                    )

                    if model_key == "qwen25vl":
                        decoder_depth = get_qwen_decoder_depth(model)
                        layers = get_extraction_layers(decoder_depth)
                    else:
                        decoder_depth = None
                        layers = None

                    forward_modes = ["fast"]
                    if model_key == "qwen25vl" and QWEN_ENABLE_NUMERICAL_RETRY:
                        retry_modes = []
                        if QWEN_ENABLE_SDPA_MATH_RETRY:
                            retry_modes.append("sdpa_math")
                        if QWEN_ENABLE_SAFE_EAGER_RETRY:
                            retry_modes.append("safe_eager")
                        forward_modes.extend(
                            retry_modes[: max(0, int(QWEN_RETRY_MAX_ATTEMPTS))]
                        )

                    attempt_error = None
                    feature_keys_for_sample = []

                    # The values were computed from the actual sample
                    # inputs above and MUST remain available to the forward path
                    # and every numerical retry. Resetting them here makes Qwen
                    # receive layers=None (and qpos=None), which causes:
                    # "Qwen extraction requires selected decoder layers."
                    for attempt_index, attention_mode in enumerate(forward_modes):
                        final_attention_mode = attention_mode

                        if attempt_index > 0:
                            numerical_retry_used = True
                            numerical_retry_modes.append(attention_mode)
                            _clear_sample_cuda_state()

                            print(
                                f"[{model_key}] Numerical retry {attempt_index}/{len(forward_modes)-1} "
                                f"for {qid} using attention={attention_mode}"
                            )

                        outputs = None
                        hidden_states = None
                        captured_qwen = None

                        try:
                            try:
                                outputs, hidden_or_capture = run_extraction_forward(
                                    model,
                                    inputs,
                                    model_key=model_key,
                                    attention_mode=attention_mode,
                                    layers=layers,
                                    vision_position=vision_position,
                                    query_position=qpos,
                                )
                            except RuntimeError as forward_exc:
                                if (
                                    model_key == "qwen25vl"
                                    and attention_mode == "fast"
                                    and _is_nan_inf_runtime_error(forward_exc)
                                ):
                                    raise NonFiniteFeatureError(
                                        "Qwen forward raised a NaN/Inf-related runtime error.",
                                        feature_key="forward",
                                        stage="forward_error",
                                    ) from forward_exc
                                raise

                            if outputs is None:
                                raise RuntimeError("Model forward returned None.")

                            # Hard safety check: this is the real-device budget,
                            # not merely the Accelerate placement budget.
                            enforce_t4_cuda_memory_budget(
                                stage=f"{model_key} forward"
                            )

                            if model_key == "qwen25vl":
                                captured_qwen = hidden_or_capture
                            else:
                                hidden_states = hidden_or_capture

                            if (
                                model_key == "smolvlm2"
                                and not used_smol_cached_vision
                            ):
                                cached = getattr(
                                    outputs,
                                    "image_hidden_states",
                                    None,
                                )

                                if isinstance(cached, (tuple, list)):
                                    if len(cached) != 1:
                                        raise RuntimeError(
                                            "Unexpected SmolVLM2 image_hidden_states container."
                                        )
                                    cached = cached[0]

                                if isinstance(cached, torch.Tensor):
                                    smol_cached_image_hidden_states = (
                                        cached.detach()
                                        .to(dtype=torch.float16)
                                        .cpu()
                                    )
                                    smol_cached_image_key = image_path

                            if model_key != "qwen25vl":
                                if hidden_states is None:
                                    raise RuntimeError(
                                        "SmolVLM2 forward did not return hidden states."
                                    )

                                if cached_decoder_depth is None:
                                    cached_decoder_depth = (
                                        get_extraction_decoder_depth(
                                            model,
                                            hidden_states,
                                        )
                                    )
                                    cached_layers = get_extraction_layers(
                                        cached_decoder_depth
                                    )

                                decoder_depth = cached_decoder_depth
                                layers = cached_layers

                            feature_keys_for_sample = []

                            try:
                                if model_key == "qwen25vl":
                                    # The Qwen visual encoder already ran inside the
                                    # multimodal forward. Reuse the hook-captured
                                    # mean-pooled raw vision output instead of
                                    # executing a second vision-only forward.
                                    vf = (
                                        captured_qwen.get("__vf__")
                                        if isinstance(captured_qwen, dict)
                                        else None
                                    )
                                    if vf is None:
                                        raise NotImplementedError(
                                            "Qwen raw visual features were not captured during the main forward."
                                        )
                                else:
                                    vf = extract_visual_features(
                                        model_key=model_key,
                                        model=model,
                                        inputs=inputs,
                                        outputs=outputs,
                                    )

                                if vf.ndim == 2:
                                    vf = vf[0]

                                vf_vector = feature_vector_numpy(
                                    vf,
                                    feature_key="vf",
                                )

                                ensure_extraction_feature_dataset(
                                    h5,
                                    "vf",
                                    vf_vector.shape[0],
                                    feature_dimensions,
                                )
                                h5["vf"][h5_index] = vf_vector
                                feature_keys_for_sample.append("vf")

                            except NotImplementedError:
                                pass

                            vision_position = (
                                int(vision_positions[-1])
                                if vision_positions
                                else None
                            )

                            if vision_position is None:
                                failed.setdefault(qid, {})["vt"] = (
                                    "No identifiable image-token position."
                                )

                            if qpos is None:
                                failed.setdefault(qid, {})["qt"] = (
                                    "No stable query-token position was identified."
                                )

                            if model_key == "qwen25vl":
                                feature_keys_for_sample.extend(
                                    _write_qwen_selected_layer_features(
                                        h5=h5,
                                        h5_index=h5_index,
                                        captured=captured_qwen,
                                        layers=layers,
                                        feature_dimensions=feature_dimensions,
                                    )
                                )
                            else:
                                feature_keys_for_sample.extend(
                                    _write_selected_hidden_features(
                                        h5=h5,
                                        h5_index=h5_index,
                                        hidden_states=hidden_states,
                                        layers=layers,
                                        vision_position=vision_position,
                                        query_position=qpos,
                                        feature_dimensions=feature_dimensions,
                                    )
                                )

                            if not feature_keys_for_sample:
                                raise RuntimeError(
                                    "No usable representation was extracted for this sample."
                                )

                            attempt_error = None
                            break

                        except NonFiniteFeatureError as exc:
                            attempt_error = exc

                            if (
                                model_key == "qwen25vl"
                                and QWEN_ENABLE_NUMERICAL_RETRY
                                and attempt_index < len(forward_modes) - 1
                            ):
                                continue

                            raise

                        finally:
                            if (
                                outputs is not None
                                and attention_mode != final_attention_mode
                            ):
                                outputs = None

                    if attempt_error is not None:
                        raise attempt_error

                    if numerical_retry_used:
                        numerical_retry_count += 1
                        record = {
                            "question_id": qid,
                            "index": int(index),
                            "modes": numerical_retry_modes,
                            "final_mode": final_attention_mode,
                            "time_seconds": float(time.time() - started),
                        }
                        numerical_retry_records.append(record)
                        state.setdefault("numerical_retries", []).append(record)
                        state["numerical_retry_count"] = int(
                            numerical_retry_count
                        )

                    failed.pop(qid, None)
                    h5["status"][h5_index] = "complete"
                    h5_status[h5_index] = "complete"
                    completed.add(qid)

                    elapsed = time.time() - started

                    results_log.append(
                        {
                            "question_id": qid,
                            "index": int(index),
                            "status": "complete",
                            "seconds": float(elapsed),
                            "feature_keys": ",".join(
                                feature_keys_for_sample
                            ),
                            "decoder_layers": int(decoder_depth),
                            "image_token_id": image_token_id,
                            "vision_positions": ",".join(
                                map(str, vision_positions)
                            ),
                            "query_position": qpos,
                            "used_cached_vision": bool(
                                used_smol_cached_vision
                            ),
                            "numerical_retry": bool(
                                numerical_retry_used
                            ),
                            "numerical_retry_modes": ",".join(
                                numerical_retry_modes
                            ),
                            "final_attention_mode": final_attention_mode,
                        }
                    )

                    pbar.update(1)

                    completed_count = len(completed)

                    if (
                        completed_count % telemetry_every == 0
                        or completed_count == total
                    ):
                        rate = completed_count / max(
                            time.time() - extraction_stage_started_at,
                            1e-9,
                        )

                        if torch.cuda.is_available():
                            free_bytes, total_bytes = torch.cuda.mem_get_info()
                            used_gib = (
                                (total_bytes - free_bytes)
                                / (1024 ** 3)
                            )
                            pbar.set_postfix(
                                done=f"{completed_count}/{total}",
                                rate=f"{rate:.2f}/s",
                                vram=f"{used_gib:.1f}G",
                                retries=int(numerical_retry_count),
                                refresh=False,
                            )
                        else:
                            pbar.set_postfix(
                                done=f"{completed_count}/{total}",
                                rate=f"{rate:.2f}/s",
                                retries=int(numerical_retry_count),
                                refresh=False,
                            )

                        # Avoid one log line per sample: large Colab/Jupyter output
                        # streams can add measurable host-side overhead at 10k rows.
                        extraction_log_every = max(1, int(globals().get(
                            "EXTRACTION_LOG_EVERY", checkpoint_interval
                        )))
                        if (
                            completed_count % extraction_log_every == 0
                            or completed_count == total
                        ):
                            print(
                                f"[{completed_count:>5}/{total}] "
                                f"{qid} -> COMPLETE ({elapsed:.2f}s)"
                            )

                    if completed_count % checkpoint_interval == 0:
                        h5.flush()

                        # Forecast total extraction time only at checkpoints.
                        # This is diagnostic only; it never fabricates completion.
                        if completed_count > 0:
                            elapsed_stage = max(
                                time.time() - extraction_stage_started_at, 1e-9
                            )
                            remaining_seconds = (
                                max(total - completed_count, 0)
                                * elapsed_stage
                                / completed_count
                            )
                            projected_total_seconds = elapsed_stage + remaining_seconds
                            warning_seconds = (
                                float(globals().get("T4_TOTAL_RUNTIME_TARGET_HOURS", 3.0))
                                * 3600.0
                                * float(globals().get("T4_RUNTIME_WARNING_FRACTION", 0.80))
                            )
                            if projected_total_seconds >= warning_seconds:
                                print(
                                    f"[runtime forecast] projected extraction total="
                                    f"{projected_total_seconds/3600.0:.2f} h"
                                )

                        state["completed"] = sorted(completed)
                        state["failed"] = failed
                        state["feature_keys"] = sorted(
                            [
                                key
                                for key in h5.keys()
                                if key.startswith(("vf", "vt_", "qt_"))
                            ]
                        )
                        state["n_complete"] = len(completed)
                        state["n_failed"] = len(failed)
                        state["last_question_id"] = qid
                        state["last_index"] = int(index)
                        state["numerical_retry_count"] = int(
                            numerical_retry_count
                        )

                        if "mark_run_progress" in globals():
                            mark_run_progress(
                                model_key,
                                state,
                                len(completed),
                                total,
                                extraction_stage_started_at,
                            )

                        save_extraction_state(model_key, state)

                    if completed_count % extraction_gc_every == 0:
                        gc.collect()

                except torch.cuda.OutOfMemoryError as exc:
                    message = str(exc)

                    failed[qid] = {
                        "exception": "CUDAOutOfMemoryError",
                        "message": message,
                        "index": int(index),
                    }

                    h5["status"][h5_index] = "failed"
                    h5_status[h5_index] = "failed"
                    h5.flush()

                    state["completed"] = sorted(completed)
                    state["failed"] = failed
                    state["n_complete"] = len(completed)
                    state["n_failed"] = len(failed)
                    state["last_question_id"] = qid
                    state["last_index"] = int(index)
                    state["numerical_retry_count"] = int(
                        numerical_retry_count
                    )
                    save_extraction_state(model_key, state)

                    results_log.append(
                        {
                            "question_id": qid,
                            "index": int(index),
                            "status": "failed",
                            "error": "CUDAOutOfMemoryError",
                            "message": message,
                        }
                    )

                    _clear_sample_cuda_state()

                    raise RuntimeError(
                        f"CUDA out of memory while extracting {qid}. "
                        "Progress was checkpointed; extraction was stopped "
                        "rather than marking this sample complete."
                    ) from exc

                except FeatureStorageOverflowError as exc:
                    # A finite feature overflowing float16 storage is a real
                    # artifact-contract problem, not something attention retry
                    # can legitimately repair. Stop rather than corrupting data.
                    message = str(exc)

                    failed[qid] = {
                        "exception": "FeatureStorageOverflowError",
                        "message": message,
                        "index": int(index),
                    }

                    h5["status"][h5_index] = "failed"
                    h5_status[h5_index] = "failed"
                    h5.flush()

                    state["completed"] = sorted(completed)
                    state["failed"] = failed
                    state["n_complete"] = len(completed)
                    state["n_failed"] = len(failed)
                    state["last_question_id"] = qid
                    state["last_index"] = int(index)
                    save_extraction_state(model_key, state)

                    results_log.append(
                        {
                            "question_id": qid,
                            "index": int(index),
                            "status": "failed",
                            "error": "FeatureStorageOverflowError",
                            "message": message,
                        }
                    )

                    print(
                        f"[{index + 1:>5}/{total}] {qid} -> FAILED: "
                        f"FeatureStorageOverflowError: {message}"
                    )
                    pbar.update(1)
                    _clear_sample_cuda_state()
                    raise

                except NonFiniteFeatureError as exc:
                    message = str(exc)

                    failed[qid] = {
                        "exception": "NonFiniteFeatureError",
                        "message": message,
                        "stage": getattr(exc, "stage", None),
                        "feature_key": getattr(exc, "feature_key", None),
                        "index": int(index),
                    }

                    h5["status"][h5_index] = "failed"
                    h5_status[h5_index] = "failed"
                    h5.flush()

                    state["completed"] = sorted(completed)
                    state["failed"] = failed
                    state["n_complete"] = len(completed)
                    state["n_failed"] = len(failed)
                    state["last_question_id"] = qid
                    state["last_index"] = int(index)
                    state["numerical_retry_count"] = int(
                        numerical_retry_count
                    )
                    save_extraction_state(model_key, state)

                    results_log.append(
                        {
                            "question_id": qid,
                            "index": int(index),
                            "status": "failed",
                            "error": "NonFiniteFeatureError",
                            "message": message,
                            "stage": getattr(exc, "stage", None),
                            "feature_key": getattr(exc, "feature_key", None),
                        }
                    )

                    print(
                        f"[{index + 1:>5}/{total}] {qid} -> FAILED: "
                        f"NonFiniteFeatureError: {message}"
                    )
                    pbar.update(1)
                    _clear_sample_cuda_state()

                    # Never silently discard a genuinely non-finite sample.
                    raise

                except Exception as exc:
                    message = str(exc)

                    failed[qid] = {
                        "exception": type(exc).__name__,
                        "message": message,
                        "index": int(index),
                    }

                    h5["status"][h5_index] = "failed"
                    h5_status[h5_index] = "failed"
                    h5.flush()

                    state["completed"] = sorted(completed)
                    state["failed"] = failed
                    state["n_complete"] = len(completed)
                    state["n_failed"] = len(failed)
                    state["last_question_id"] = qid
                    state["last_index"] = int(index)
                    state["numerical_retry_count"] = int(
                        numerical_retry_count
                    )
                    save_extraction_state(model_key, state)

                    results_log.append(
                        {
                            "question_id": qid,
                            "index": int(index),
                            "status": "failed",
                            "error": f"{type(exc).__name__}: {message}",
                        }
                    )

                    print(
                        f"[{index + 1:>5}/{total}] {qid} -> FAILED: "
                        f"{type(exc).__name__}: {message}"
                    )
                    pbar.update(1)

                    _clear_sample_cuda_state()
                    raise

                finally:
                    inputs = None
                    outputs = None
                    hidden_states = None
                    captured_qwen = None

        finally:
            pbar.close()

        h5.flush()

    # ------------------------------------------------------------
    # Final state / telemetry
    # ------------------------------------------------------------
    state["completed"] = sorted(completed)
    state["failed"] = failed
    state["n_complete"] = len(completed)
    state["n_failed"] = len(failed)
    state["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
    state["numerical_retry_count"] = int(numerical_retry_count)

    if torch.cuda.is_available():
        peak_allocated_gib = float(
            torch.cuda.max_memory_allocated() / (1024 ** 3)
        )
        peak_reserved_gib = float(
            torch.cuda.max_memory_reserved() / (1024 ** 3)
        )
    else:
        peak_allocated_gib = None
        peak_reserved_gib = None

    extraction_elapsed_seconds = float(
        max(time.time() - extraction_stage_started_at, 0.0)
    )

    state["elapsed_seconds"] = extraction_elapsed_seconds
    state["peak_vram_allocated_gib"] = peak_allocated_gib
    state["peak_vram_reserved_gib"] = peak_reserved_gib
    state["h5_feature_compression"] = H5_FEATURE_COMPRESSION

    save_extraction_state(model_key, state)

    telemetry_payload = {
        "model_key": model_key,
        "extraction_profile_version": EXTRACTION_PROFILE_VERSION,
        "elapsed_seconds": extraction_elapsed_seconds,
        "peak_vram_allocated_gib": peak_allocated_gib,
        "peak_vram_reserved_gib": peak_reserved_gib,
        "complete": int(len(completed)),
        "failed": int(len(failed)),
        "numerical_retry_count": int(numerical_retry_count),
        "numerical_retry_records": numerical_retry_records,
    }

    telemetry_path = (
        Path(RUN_ROOT)
        / "logs"
        / f"{model_key}_extraction_telemetry.json"
    )
    telemetry_path.parent.mkdir(parents=True, exist_ok=True)
    telemetry_path.write_text(
        json.dumps(telemetry_payload, indent=2),
        encoding="utf-8",
    )

    # ------------------------------------------------------------
    # Save extraction log
    # ------------------------------------------------------------
    if results_log:
        current_log = pd.DataFrame(results_log)

        if log_path.exists():
            try:
                previous_log = pd.read_csv(log_path)
                log_df = pd.concat(
                    [previous_log, current_log],
                    ignore_index=True,
                )
            except Exception:
                log_df = current_log
        else:
            log_df = current_log

        if "question_id" in log_df.columns:
            log_df = log_df.drop_duplicates(
                subset=["question_id"],
                keep="last",
            )

        log_df.to_csv(log_path, index=False)

    # ------------------------------------------------------------
    # Final HDF5 summary
    # ------------------------------------------------------------
    with h5py.File(h5_path, "r") as h5:
        status_values = _decode_h5_strings(h5["status"][:])
        feature_keys = sorted(
            [
                key
                for key in h5.keys()
                if key.startswith(("vf", "vt_", "qt_"))
            ]
        )
        h5_labels_available = bool(
            h5.attrs.get("labels_available", False)
        )

    summary = {
        "model_key": model_key,
        "rows": len(eligible),
        "complete": status_values.count("complete"),
        "failed": status_values.count("failed"),
        "pending": status_values.count("pending"),
        "labels_available": h5_labels_available,
        "frame_source": frame_source,
        "feature_keys": feature_keys,
        "h5_path": str(h5_path),
        "checkpoint_path": str(extraction_checkpoint_path(model_key)),
        "log_path": str(log_path),
        "elapsed_seconds": extraction_elapsed_seconds,
        "peak_vram_allocated_gib": peak_allocated_gib,
        "peak_vram_reserved_gib": peak_reserved_gib,
        "t4_hard_cuda_gib": float(T4_HARD_CUDA_GIB),
        "t4_preferred_peak_cuda_gib": float(T4_PREFERRED_PEAK_CUDA_GIB),
        "t4_model_gpu_budget_gib": float(T4_MODEL_GPU_BUDGET_GIB.get(model_key, 11.0)),
        "extraction_profile_version": EXTRACTION_PROFILE_VERSION,
        "qwen_target_visual_tokens": (
            int(QWEN_TARGET_VISUAL_TOKENS)
            if model_key == "qwen25vl"
            else None
        ),
        "qwen_max_vision_patches": (
            int(QWEN_MAX_VISION_PATCHES)
            if model_key == "qwen25vl"
            else None
        ),
        "qwen_selected_layer_hooks": (
            bool(model_key == "qwen25vl")
        ),
        "fast_image_preflight": FAST_IMAGE_PREFLIGHT,
        "repeat_image_cache": bool(
            model_key == "smolvlm2" and SMOL_ENABLE_REPEAT_IMAGE_CACHE
        ),
        "h5_feature_compression": H5_FEATURE_COMPRESSION,
        "qwen_numerical_retry_enabled": bool(
            model_key == "qwen25vl" and QWEN_ENABLE_NUMERICAL_RETRY
        ),
        "qwen_numerical_retry_count": int(numerical_retry_count),
        "qwen_numerical_retry_records": numerical_retry_records,
    }

    print("\n" + "=" * 80)
    print(f"EXTRACTION SUMMARY: {model_key}")
    print("=" * 80)

    for key, value in summary.items():
        print(f"{key:34s}: {value}")

    print("=" * 80)

    return summary


# ------------------------------------------------------------
# 22. Run active-model extraction
# ------------------------------------------------------------

EXTRACTION_SUMMARIES = {}
EXTRACTION_STATUS = {}


for model_key in ACTIVE_MODELS:
    print("\n" + "#" * 80)
    print(f"STARTING EXTRACTION: {model_key}")
    print("#" * 80)

    try:
        summary = extract_model_features(model_key)
        EXTRACTION_SUMMARIES[model_key] = summary
        EXTRACTION_STATUS[model_key] = "COMPLETED"

    except Exception as exc:
        EXTRACTION_STATUS[model_key] = (
            f"FAILED — {type(exc).__name__}: {exc}"
        )

        print("\n" + "=" * 80)
        print(f"[{model_key}] EXTRACTION FAILED")
        print(f"Exception type: {type(exc).__name__}")
        print(f"Exception: {exc}")
        print("=" * 80)
        raise

    finally:
        release_cached_model(model_key)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ------------------------------------------------------------
# 23. Final report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL FEATURE-EXTRACTION REPORT")
print("=" * 80)

for model_key, status in EXTRACTION_STATUS.items():
    print(f"{model_key:22s}: {status}")

print("-" * 80)
print("Feature extraction requires hallucination labels : NO")
print("Supervised probe training requires labels       : YES")
print("Synthetic labels created                        : NO")
print("Answer generation performed                     : NO")
print("Resume/checkpoint support                       : YES")
print("HDF5 feature storage                            : YES")
print("Fast image preflight                            :", FAST_IMAGE_PREFLIGHT)
print(
    "Repeat-image vision cache                       :",
    SMOL_ENABLE_REPEAT_IMAGE_CACHE,
)
print("HDF5 feature compression                        :", H5_FEATURE_COMPRESSION)
print(
    "Qwen numerical retry                             :",
    QWEN_ENABLE_NUMERICAL_RETRY,
)
print(
    "Current RUN_ID                                  :",
    RUN_ID,
)
print(
    "Extraction profile                              :",
    EXTRACTION_PROFILE_VERSION,
)
print("=" * 80)
print("Feature-extraction stage complete.")


HALP-Bench feature extraction — T4 / Colab optimized / Qwen NaN-safe
Validated HALP-Bench rows: 10000
Unique question IDs: 10000
Unique images: 4849

################################################################################
STARTING EXTRACTION: smolvlm2
################################################################################

--------------------------------------------------------------------------------
FEATURE EXTRACTION: smolvlm2
--------------------------------------------------------------------------------
Data source: fixed experiment subset
Rows selected: 8000
RUN_MODE: full
MAX_SAMPLES: 8000
Reviewed labels available: True
Extraction profile: t4-colab-free-v5-qwen-adaptive-min-runtime-aware+qwen-nan-safe-v1


smolvlm2 | fast image preflight:   0%|          | 0/4295 [00:00<?, ?image/s]

[smolvlm2] Image preflight: PASS (4295 unique images; no full image decode performed).
[smolvlm2] Loading/reusing VLM model...
Loading processor...
GPU memory budget : 11.0GiB
CPU offload budget: 48GiB


Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

Model class: SmolVLMForConditionalGeneration
Processor class: SmolVLMProcessor
Model input device: cuda:0

--------------------------------------------------------------------------------
[smolvlm2] RESUME STATUS
  Total rows            : 8000
  Previously complete   : 0
  Previously failed     : 0
  Remaining incomplete  : 8000
--------------------------------------------------------------------------------


...

Model class: Qwen2_5_VLForConditionalGeneration
Processor class: Qwen2_5_VLProcessor
Model input device: cuda:0
Qwen attention implementation: sdpa
Qwen numerical retry enabled: True

--------------------------------------------------------------------------------
[qwen25vl] RESUME STATUS
  Total rows            : 8000
  Previously complete   : 0
  Previously failed     : 0
  Remaining incomplete  : 8000
--------------------------------------------------------------------------------


qwen25vl | feature extraction:   0%|          | 0/8000 [00:00<?, ?sample/s]

[  500/8000] question_comb_8044 -> COMPLETE (0.28s)
[ 1000/8000] question_comb_6312 -> COMPLETE (0.33s)
[ 1500/8000] question_comb_9548 -> COMPLETE (0.29s)
[ 2000/8000] question_comb_5713 -> COMPLETE (0.42s)
[ 2500/8000] question_comb_8587 -> COMPLETE (0.32s)
[ 3000/8000] question_comb_8564 -> COMPLETE (0.47s)


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[ 3500/8000] question_comb_2679 -> COMPLETE (0.39s)
[qwen25vl] Numerical retry 1/2 for question_comb_2660 using attention=sdpa_math
[qwen25vl] Numerical retry 2/2 for question_comb_2660 using attention=safe_eager
[qwen25vl] Numerical retry 1/2 for question_comb_772 using attention=sdpa_math
[qwen25vl] Numerical retry 2/2 for question_comb_772 using attention=safe_eager
[ 4000/8000] question_comb_1492 -> COMPLETE (0.48s)
[qwen25vl] Numerical retry 1/2 for question_comb_395 using attention=sdpa_math
[qwen25vl] Numerical retry 2/2 for question_comb_395 using attention=safe_eager
[ 4500/8000] question_comb_2861 -> COMPLETE (0.42s)
[ 5000/8000] question_comb_3904 -> COMPLETE (0.40s)
[ 5500/8000] question_comb_3760 -> COMPLETE (0.41s)
[ 6000/8000] question_comb_6811 -> COMPLETE (0.21s)
[ 6500/8000] question_comb_7039 -> COMPLETE (0.25s)
[qwen25vl] Numerical retry 1/2 for question_comb_931 using attention=sdpa_math
[qwen25vl] Numerical retry 2/2 for question_comb_931 using attention=safe_eage

## 14. Feature validation

Before training any probe, check:
- HDF5 dimensions,
- completed/pending rows,
- label alignment,
- NaN/Inf absence,
- feature dimensionality consistency,
- non-empty representations.



In [16]:

# ============================================================
# 14. Feature validation — robust for smoke/full extraction
# ============================================================
#
# Purpose
# -------
# Validate the extracted HDF5 representations before probe training.
#
# This cell is intentionally compatible with BOTH:
#
#     RUN_MODE = "smoke_test"
#
# and:
#
#     RUN_MODE = "full"
#
#
# Important
# ---------
# The feature file may contain fewer rows than the full benchmark when
# smoke_test mode is active.
#
# So we do not require:
#
#     HDF5 rows == len(df)
#
# Instead we verify that all HDF5 question IDs are valid members of the
# validated benchmark dataframe.
#
#
# LABEL POLICY
# ------------
# `label` is OPTIONAL at this stage.
#
# Feature extraction itself does not require hallucination labels.
#
# If `label` exists:
#     -> validate it strictly.
#
# If `label` does not exist:
#     -> validation still succeeds,
#     -> labels_available=False,
#     -> supervised probe training remains blocked.
#
#
# PARTIAL EXTRACTION
# ------------------
# Feature matrices are allocated for the complete extraction frame.
# Pending rows therefore may contain initialized values that are not yet
# experimental observations.
#
# Numerical validation is performed only for rows whose status is:
#
#     complete
#
#
# NO DATA ARE MODIFIED BY THIS CELL.
# ============================================================


# ------------------------------------------------------------
# 1. Notebook objects we need
# ------------------------------------------------------------

print("=" * 80)
print("HALP-Bench feature-file validation")
print("=" * 80)


REQUIRED_GLOBALS = [
    "ACTIVE_MODELS",
    "feature_h5_path",
    "h5py",
    "np",
    "Path",
    "pd",
]


missing_globals = [
    name
    for name in REQUIRED_GLOBALS
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Missing required notebook objects:\n"
        f"  {missing_globals}\n\n"
        "Run the environment, dataset, and feature-extraction cells "
        "before running feature validation."
    )


if not isinstance(
    ACTIVE_MODELS,
    (list, tuple),
):

    raise TypeError(
        "ACTIVE_MODELS must be a list or tuple."
    )


if not ACTIVE_MODELS:

    raise RuntimeError(
        "ACTIVE_MODELS is empty."
    )


# ------------------------------------------------------------
# 2. Validate optional pandas/numpy aliases
# ------------------------------------------------------------
#
# `np` is the actual alias used throughout the notebook.
# The previous Cell incorrectly required a separate `numpy` global,
# which is unnecessary and caused:
#
#     Missing required notebook objects: ['numpy']
#
# We intentionally require only `np`.
# ------------------------------------------------------------

if not isinstance(
    np,
    type(
        np
    ),
):

    # This branch is intentionally unreachable in normal use, but keeps
    # the diagnostic explicit if `np` was overwritten.
    raise TypeError(
        "Notebook object `np` is not a valid NumPy module alias."
    )


# ------------------------------------------------------------
# 3. Decode HDF5 values
# ------------------------------------------------------------

def decode_h5_value(
    value,
) -> str:
    """
    Convert an HDF5 string/bytes scalar to a Python string.
    """

    if isinstance(
        value,
        bytes,
    ):

        return value.decode(
            "utf-8"
        )

    return str(
        value
    )


# ------------------------------------------------------------
# 4. Resolve feature-file path
# ------------------------------------------------------------

def get_feature_validation_path(
    model_key: str,
) -> Path:
    """
    Resolve and validate the feature HDF5 path.
    """

    path = Path(
        feature_h5_path(
            model_key
        )
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Feature file does not exist for {model_key!r}:\n"
            f"{path}"
        )

    if not path.is_file():

        raise FileNotFoundError(
            f"Feature path is not a regular file:\n"
            f"{path}"
        )

    return path


# ------------------------------------------------------------
# 5. Get validated benchmark IDs
# ------------------------------------------------------------

def get_reference_question_ids():
    """
    Return question IDs from the validated benchmark dataframe.

    This is used as a membership/alignment check.

    IMPORTANT:
    The HDF5 file may legitimately contain only a subset of df during
    smoke_test mode, so this function returns the full reference set
    without requiring equal cardinality.
    """

    if "df" not in globals():

        return None


    reference_df = globals()[
        "df"
    ]


    if not isinstance(
        reference_df,
        pd.DataFrame,
    ):

        return None


    if "question_id" not in reference_df.columns:

        return None


    return (
        reference_df[
            "question_id"
        ]
        .astype(str)
        .tolist()
    )


# ------------------------------------------------------------
# 6. Discover feature datasets
# ------------------------------------------------------------

def get_feature_keys(
    h5,
):
    """
    Return actual extracted representation datasets.

    Valid representation prefixes:
        vf
        vt_*
        qt_*
    """

    return sorted(
        [
            key
            for key in h5.keys()
            if (
                key == "vf"
                or key.startswith("vt_")
                or key.startswith("qt_")
            )
        ]
    )


# ------------------------------------------------------------
# 7. Validate one feature file
# ------------------------------------------------------------

def validate_feature_file(
    model_key: str,
):
    """
    Validate one model's HDF5 feature file.

    This function is deliberately label-aware but label-independent.
    """

    path = get_feature_validation_path(
        model_key
    )


    print(
        "\n"
        + "-" * 80
    )

    print(
        f"VALIDATING FEATURE FILE: {model_key}"
    )

    print(
        "-" * 80
    )

    print(
        "Path:",
        path
    )


    with h5py.File(
        path,
        "r",
    ) as h5:

        # --------------------------------------------------------
        # 7.1 Required metadata datasets
        # --------------------------------------------------------

        required_metadata = {
            "question_id",
            "image_name",
            "status",
        }


        missing_metadata = (
            required_metadata
            - set(
                h5.keys()
            )
        )


        if missing_metadata:

            raise RuntimeError(
                f"{model_key}: missing required HDF5 datasets: "
                f"{sorted(missing_metadata)}"
            )


        # --------------------------------------------------------
        # 7.2 Basic row counts
        # --------------------------------------------------------

        question_id_count = int(
            h5[
                "question_id"
            ].shape[0]
        )


        image_name_count = int(
            h5[
                "image_name"
            ].shape[0]
        )


        status_count = int(
            h5[
                "status"
            ].shape[0]
        )


        if image_name_count != question_id_count:

            raise RuntimeError(
                f"{model_key}: image_name row count mismatch. "
                f"question_id={question_id_count}, "
                f"image_name={image_name_count}"
            )


        if status_count != question_id_count:

            raise RuntimeError(
                f"{model_key}: status row count mismatch. "
                f"question_id={question_id_count}, "
                f"status={status_count}"
            )


        n = question_id_count


        # --------------------------------------------------------
        # 7.3 Read question IDs
        # --------------------------------------------------------

        question_ids = [
            decode_h5_value(
                value
            )
            for value in h5[
                "question_id"
            ][:]
        ]


        # Empty IDs are not acceptable.
        empty_question_ids = [
            index
            for index, question_id in enumerate(
                question_ids
            )
            if not question_id.strip()
        ]


        if empty_question_ids:

            raise RuntimeError(
                f"{model_key}: empty question_id values found at "
                f"indices {empty_question_ids[:20]}"
            )


        # Duplicate IDs would make feature-to-sample mapping ambiguous.
        question_id_series = pd.Series(
            question_ids,
            dtype="string",
        )


        duplicate_mask = (
            question_id_series
            .duplicated(
                keep=False
            )
        )


        if duplicate_mask.any():

            duplicate_ids = (
                question_id_series[
                    duplicate_mask
                ]
                .unique()
                .tolist()
            )


            raise RuntimeError(
                f"{model_key}: duplicate question_id values found.\n"
                f"Examples: {duplicate_ids[:20]}"
            )


        # --------------------------------------------------------
        # 7.4 Read status values
        # --------------------------------------------------------

        statuses = [
            decode_h5_value(
                value
            )
            for value in h5[
                "status"
            ][:]
        ]


        allowed_statuses = {
            "pending",
            "complete",
            "failed",
        }


        unexpected_statuses = sorted(
            set(
                statuses
            )
            - allowed_statuses
        )


        if unexpected_statuses:

            raise RuntimeError(
                f"{model_key}: unexpected HDF5 status values: "
                f"{unexpected_statuses}"
            )


        complete_indices = [
            index
            for index, status in enumerate(
                statuses
            )
            if status == "complete"
        ]


        failed_indices = [
            index
            for index, status in enumerate(
                statuses
            )
            if status == "failed"
        ]


        pending_indices = [
            index
            for index, status in enumerate(
                statuses
            )
            if status == "pending"
        ]


        print(
            f"Rows      : {n}"
        )

        print(
            f"Complete  : {len(complete_indices)}"
        )

        print(
            f"Failed    : {len(failed_indices)}"
        )

        print(
            f"Pending   : {len(pending_indices)}"
        )


        # --------------------------------------------------------
        # 7.5 Benchmark alignment
        # --------------------------------------------------------
        #
        # Important:
        #
        # In smoke_test mode:
        #
        #     HDF5 rows = 32
        #
        # while:
        #
        #     df rows = 10000
        #
        # So exact set equality would be incorrect.
        #
        # We instead require:
        #
        #     HDF5 question IDs ⊆ validated benchmark IDs
        #
        # and preserve the HDF5 order.
        # --------------------------------------------------------

        reference_question_ids = (
            get_reference_question_ids()
        )


        reference_alignment = (
            "not checked"
        )


        if reference_question_ids is not None:

            reference_id_set = set(
                reference_question_ids
            )


            h5_id_set = set(
                question_ids
            )


            ids_missing_from_reference = (
                h5_id_set
                - reference_id_set
            )


            if ids_missing_from_reference:

                examples = sorted(
                    ids_missing_from_reference
                )[:20]


                raise RuntimeError(
                    f"{model_key}: HDF5 contains question IDs that are "
                    "not present in the validated benchmark dataframe.\n"
                    f"Examples: {examples}"
                )


            reference_alignment = (
                "PASS — all HDF5 question IDs are valid members "
                "of the validated benchmark dataframe"
            )


        print(
            "Question-ID alignment:",
            reference_alignment
        )


        # --------------------------------------------------------
        # 7.6 Optional label dataset
        # --------------------------------------------------------

        labels_available = (
            "label" in h5
        )


        labels = None


        if labels_available:

            labels = np.asarray(
                h5[
                    "label"
                ][:]
            )


            if labels.ndim != 1:

                raise RuntimeError(
                    f"{model_key}: label must be 1-D; "
                    f"got shape={labels.shape}"
                )


            if len(labels) != n:

                raise RuntimeError(
                    f"{model_key}: label count does not match "
                    f"HDF5 rows. labels={len(labels)}, rows={n}"
                )


            # Convert explicitly to integers after validation.
            try:

                labels_numeric = (
                    pd.to_numeric(
                        labels,
                        errors="raise",
                    )
                    .astype(np.int64)
                )

            except Exception as exc:

                raise RuntimeError(
                    f"{model_key}: label dataset is not safely "
                    "convertible to integers."
                ) from exc


            unique_labels = set(
                np.unique(
                    labels_numeric
                ).tolist()
            )


            if not unique_labels.issubset(
                {
                    0,
                    1,
                }
            ):

                raise RuntimeError(
                    f"{model_key}: unexpected label values: "
                    f"{sorted(unique_labels)}"
                )


            labels = labels_numeric


            print(
                "Labels    : AVAILABLE"
            )


            label_counts = {
                int(
                    value
                ):
                int(
                    np.sum(
                        labels
                        == value
                    )
                )
                for value in sorted(
                    unique_labels
                )
            }


            print(
                "Label counts:",
                label_counts
            )


        else:

            print(
                "Labels    : NOT AVAILABLE"
            )


            print(
                "This is valid for label-independent feature extraction."
            )


            print(
                "Supervised probe training must remain blocked."
            )


        # --------------------------------------------------------
        # 7.7 Label attribute consistency
        # --------------------------------------------------------

        labels_attribute = h5.attrs.get(
            "labels_available",
            None,
        )


        if labels_attribute is not None:

            labels_attribute = bool(
                labels_attribute
            )


            if labels_attribute != labels_available:

                raise RuntimeError(
                    f"{model_key}: HDF5 attribute "
                    "`labels_available` disagrees with the actual "
                    f"label dataset. "
                    f"attribute={labels_attribute}, "
                    f"dataset_present={labels_available}"
                )


        # --------------------------------------------------------
        # 7.8 Feature-key discovery
        # --------------------------------------------------------

        feature_keys = get_feature_keys(
            h5
        )


        if not feature_keys:

            raise RuntimeError(
                f"{model_key}: no extracted VF/VT/QT feature datasets found."
            )


        print(
            "\nFeature datasets:"
        )


        # --------------------------------------------------------
        # 7.9 Validate each feature matrix
        # --------------------------------------------------------

        feature_dimensions = {}

        feature_validation = {}

        coverage = {}


        for feature_key in feature_keys:

            dataset = h5[
                feature_key
            ]


            # ----------------------------------------------------
            # Shape
            # ----------------------------------------------------

            if len(
                dataset.shape
            ) != 2:

                raise RuntimeError(
                    f"{model_key} / {feature_key}: expected a "
                    "2-D matrix [N, D], got "
                    f"shape={dataset.shape}"
                )


            feature_rows = int(
                dataset.shape[0]
            )


            feature_dimension = int(
                dataset.shape[1]
            )


            if feature_rows != n:

                raise RuntimeError(
                    f"{model_key} / {feature_key}: row-count mismatch. "
                    f"feature rows={feature_rows}, metadata rows={n}"
                )


            if feature_dimension <= 0:

                raise RuntimeError(
                    f"{model_key} / {feature_key}: invalid feature "
                    f"dimension={feature_dimension}"
                )


            feature_dimensions[
                feature_key
            ] = feature_dimension


            # ----------------------------------------------------
            # Numeric dtype
            # ----------------------------------------------------

            dtype = dataset.dtype


            if not np.issubdtype(
                dtype,
                np.floating,
            ):

                raise RuntimeError(
                    f"{model_key} / {feature_key}: expected a "
                    f"floating-point dataset; got dtype={dtype}"
                )


            # ----------------------------------------------------
            # Validate COMPLETE rows only
            # ----------------------------------------------------

            finite_ok = True

            zero_row_count = 0


            if complete_indices:

                completed_matrix = np.asarray(
                    dataset[
                        complete_indices,
                        :
                    ]
                )


                finite_ok = bool(
                    np.isfinite(
                        completed_matrix
                    ).all()
                )


                if not finite_ok:

                    bad_mask = (
                        ~np.isfinite(
                            completed_matrix
                        )
                    )


                    bad_values = int(
                        bad_mask.sum()
                    )


                    raise ValueError(
                        f"{model_key} / {feature_key}: "
                        f"{bad_values} NaN/Inf values found among "
                        "completed rows."
                    )


                # This is diagnostic only. A zero vector is not
                # automatically considered invalid.
                zero_row_count = int(
                    np.sum(
                        np.all(
                            completed_matrix
                            == 0,
                            axis=1,
                        )
                    )
                )


            feature_validation[
                feature_key
            ] = {

                "rows":
                    feature_rows,

                "dimension":
                    feature_dimension,

                "dtype":
                    str(
                        dtype
                    ),

                "completed_rows_checked":
                    len(
                        complete_indices
                    ),

                "finite_on_completed_rows":
                    finite_ok,

            }


            coverage[
                feature_key
            ] = {

                "complete_rows":
                    len(
                        complete_indices
                    ),

                "all_zero_complete_rows":
                    zero_row_count,

            }


            print(
                f"  {feature_key:24s} "
                f"shape={dataset.shape} "
                f"dtype={dtype} "
                f"complete_rows_checked={len(complete_indices)}"
            )


        # --------------------------------------------------------
        # 7.10 Compare feature dimensions intelligently
        # --------------------------------------------------------
        #
        # Different representation families are allowed to have different
        # dimensions.
        #
        # We do not require:
        #
        #     VF dimension == VT dimension == QT dimension
        #
        # because that is not generally guaranteed across architectures.
        #
        # Instead, we report dimensions grouped by size.
        # --------------------------------------------------------

        dimension_groups = {}


        for feature_key, dimension in (
            feature_dimensions.items()
        ):

            dimension_groups.setdefault(
                int(
                    dimension
                ),
                [],
            ).append(
                feature_key
            )


        # --------------------------------------------------------
        # 7.11 Determine observation status
        # --------------------------------------------------------

        if len(
            complete_indices
        ) == 0:

            observation_status = (
                "NO_COMPLETED_FEATURE_ROWS"
            )

        else:

            observation_status = (
                "COMPLETED_FEATURE_ROWS_VALIDATED"
            )


        # --------------------------------------------------------
        # 7.12 Final result object
        # --------------------------------------------------------

        result = {

            "model_key":
                model_key,

            "path":
                str(
                    path
                ),

            "n_rows":
                n,

            "complete_rows":
                len(
                    complete_indices
                ),

            "failed_rows":
                len(
                    failed_indices
                ),

            "pending_rows":
                len(
                    pending_indices
                ),

            "labels_available":
                labels_available,

            "feature_keys":
                feature_keys,

            "feature_dimensions":
                feature_dimensions,

            "dimension_groups":
                dimension_groups,

            "feature_validation":
                feature_validation,

            "coverage":
                coverage,

            "reference_alignment":
                reference_alignment,

            "observation_status":
                observation_status,

            "validation_passed":
                True,

        }


    # ------------------------------------------------------------
    # 8. Human-readable validation result
    # ------------------------------------------------------------

    print(
        "\n"
        + "-" * 80
    )


    print(
        f"{model_key} validation result:"
    )


    print(
        "  HDF5 structure                : PASS"
    )


    print(
        "  Row counts                    : PASS"
    )


    print(
        "  Question IDs                  : PASS"
    )


    print(
        "  Status values                 : PASS"
    )


    print(
        "  Feature datasets              : PASS"
    )


    print(
        "  Completed-row numerical check : PASS"
    )


    print(
        "  Labels available              :",
        "YES"
        if labels_available
        else "NO",
    )


    print(
        "  Observation status            :",
        observation_status,
    )


    print(
        "-" * 80
    )


    return result


# ------------------------------------------------------------
# 9. Validate all active models
# ------------------------------------------------------------

FEATURE_VALIDATION_RESULTS = {}

FEATURE_VALIDATION_STATUS = {}


for model_key in ACTIVE_MODELS:

    try:

        result = validate_feature_file(
            model_key
        )


        FEATURE_VALIDATION_RESULTS[
            model_key
        ] = result


        FEATURE_VALIDATION_STATUS[
            model_key
        ] = "PASSED"


    except Exception as exc:

        FEATURE_VALIDATION_STATUS[
            model_key
        ] = (
            f"FAILED — "
            f"{type(exc).__name__}: {exc}"
        )


        print(
            "\n"
            + "=" * 80
        )


        print(
            f"[{model_key}] FEATURE VALIDATION FAILED"
        )


        print(
            "Exception type:",
            type(exc).__name__
        )


        print(
            "Exception:",
            str(exc)
        )


        print(
            "=" * 80
        )


        # Never hide a genuine integrity problem.
        raise


# ------------------------------------------------------------
# 10. Final validation report
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 80
)

print(
    "FINAL FEATURE VALIDATION REPORT"
)

print(
    "=" * 80
)


for model_key, status in (
    FEATURE_VALIDATION_STATUS.items()
):

    print(
        f"{model_key:22s}: {status}"
    )


print(
    "-" * 80
)


for model_key, result in (
    FEATURE_VALIDATION_RESULTS.items()
):

    print(
        f"\nModel: {model_key}"
    )


    print(
        f"  Rows             : "
        f"{result['n_rows']}"
    )


    print(
        f"  Complete         : "
        f"{result['complete_rows']}"
    )


    print(
        f"  Failed           : "
        f"{result['failed_rows']}"
    )


    print(
        f"  Pending          : "
        f"{result['pending_rows']}"
    )


    print(
        f"  Labels available : "
        f"{result['labels_available']}"
    )


    print(
        f"  Feature keys     : "
        f"{result['feature_keys']}"
    )


    print(
        f"  Dimensions       : "
        f"{result['feature_dimensions']}"
    )


    print(
        f"  Observation      : "
        f"{result['observation_status']}"
    )


print(
    "\n"
    + "=" * 80
)


# ------------------------------------------------------------
# 11. Global validation flags
# ------------------------------------------------------------

ALL_FEATURE_FILES_VALID = all(
    status == "PASSED"
    for status in (
        FEATURE_VALIDATION_STATUS.values()
    )
)


ANY_LABELS_AVAILABLE = any(
    result[
        "labels_available"
    ]
    for result in (
        FEATURE_VALIDATION_RESULTS.values()
    )
)


ANY_COMPLETED_ROWS = any(
    result[
        "complete_rows"
    ] > 0
    for result in (
        FEATURE_VALIDATION_RESULTS.values()
    )
)


# ------------------------------------------------------------
# 12. Publish validation flags for later cells
# ------------------------------------------------------------

FEATURE_FILES_VALIDATED = (
    ALL_FEATURE_FILES_VALID
)


SUPERVISED_PROBE_LABELS_AVAILABLE = (
    ANY_LABELS_AVAILABLE
)


FEATURE_OBSERVATIONS_AVAILABLE = (
    ANY_COMPLETED_ROWS
)


# ------------------------------------------------------------
# 13. Scientific interpretation
# ------------------------------------------------------------

print(
    "Feature-file integrity                :",
    "PASS"
    if ALL_FEATURE_FILES_VALID
    else "FAIL",
)


print(
    "Completed feature observations        :",
    "AVAILABLE"
    if ANY_COMPLETED_ROWS
    else "NONE YET",
)


print(
    "Hallucination labels present          :",
    "YES"
    if ANY_LABELS_AVAILABLE
    else "NO",
)


if ANY_LABELS_AVAILABLE:

    print(
        "Supervised probe training             : "
        "labels are available for at least one active model."
    )

else:

    print(
        "Supervised probe training             : "
        "BLOCKED — genuine model-specific reviewed labels are unavailable."
    )


print(
    "Synthetic labels created              : NO"
)


print(
    "Answer generation performed           : NO"
)


print(
    "Pending rows excluded from numerical "
    "feature checks                       : YES"
)


print(
    "=" * 80
)

print(
    "Feature validation stage complete."
)


HALP-Bench feature-file validation

--------------------------------------------------------------------------------
VALIDATING FEATURE FILE: smolvlm2
--------------------------------------------------------------------------------
Path: /content/drive/MyDrive/HALP_Bench_Project/runs/halp_vlm_hallucination_t4_20260831T224100Z/features/smolvlm2_representations.h5
Rows      : 8000
Complete  : 8000
Failed    : 0
Pending   : 0
Question-ID alignment: PASS — all HDF5 question IDs are valid members of the validated benchmark dataframe
Labels    : AVAILABLE
Label counts: {0: 7211, 1: 789}

Feature datasets:
  qt_early                 shape=(8000, 2048) dtype=float16 complete_rows_checked=8000
  qt_final                 shape=(8000, 2048) dtype=float16 complete_rows_checked=8000
  qt_middle                shape=(8000, 2048) dtype=float16 complete_rows_checked=8000
  qt_quarter               shape=(8000, 2048) dtype=float16 complete_rows_checked=8000
  qt_three_quarter         shape=(8000, 2048)

## 15. Scientific formulation

For an image-question pair \(x\), let \(y\in\{0,1\}\) denote the model-specific hallucination label and \(h(x)\in\mathbb{R}^d\) an extracted VLM representation. The probe estimates:

\[
p_\theta(y=1\mid h)=\sigma(f_\theta(h)),\qquad
\sigma(z)=\frac{1}{1+e^{-z}}.
\]

For a linear logistic probe, \(f_\theta(h)=w^\top h+b\). The MLP probe uses the observed feature dimension as input; the compact sweep evaluates hidden widths 128-64-32, 256-128-64, and 512-256-128 before the final scalar logit.

Training uses binary cross-entropy:

\[
\mathcal{L}_{\mathrm{BCE}}=-\frac{1}{N}\sum_{i=1}^{N}
\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right].
\]

For probability calibration, the Brier score is:

\[
\mathrm{Brier}=\frac{1}{N}\sum_{i=1}^{N}(p_i-y_i)^2.
\]

Expected calibration error uses fixed confidence bins \(S_b\):

\[
\mathrm{ECE}=\sum_{b=1}^{B}\frac{|S_b|}{N}
\left|\operatorname{acc}(S_b)-\operatorname{conf}(S_b)\right|.
\]

Across probe seeds, the notebook reports the mean and sample standard deviation:
\[
\bar{s}=\frac{1}{K}\sum_{k=1}^{K}s_k,\qquad
s_{\mathrm{std}}=\sqrt{\frac{1}{K-1}\sum_{k=1}^{K}(s_k-\bar{s})^2}.
\]

### Research setting

This notebook implements **pre-generation hallucination detection (HALP-style)**. It extracts internal VLM representations before token generation and trains a lightweight probe to predict the model-specific reviewed hallucination target. It does **not** claim that the probe itself understands hallucination, and it never derives hallucination labels from `gt_answer`.

For an example probe representation \(\mathbf z_i\),

\[
\hat p_i = \sigma(f_\theta(\mathbf z_i)).
\]

The binary target is explicitly defined by the imported reviewed-label artifact:

\[
y_i \in \{0,1\},
\]

where \(y_i=1\) means the model-specific reviewed annotation marks the example as hallucinated.

Hyperparameters are selected using the validation split only; the test threshold is locked before final test evaluation.

## 15. Probe models

Two probe families are implemented:

1. **Logistic regression** as a transparent linear baseline.
2. **HALP-style lightweight MLP** with compact input → hidden-layer configurations → 1 and BCE-with-logits; the largest candidate is 512 → 256 → 128.

The input projection uses the actual extracted feature dimension. We do not hard-code `512` as the VLM hidden size.

Hyperparameters are selected only on validation data. The test set is used once for final evaluation.


In [17]:

import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Probe datasets are small relative to the VLM. Keeping probe training on CPU
# avoids repeatedly reserving CUDA memory for tiny MLP updates.
if "PROBE_MAX_THREADS" in globals():
    torch.set_num_threads(int(PROBE_MAX_THREADS))

class HALPProbe(nn.Module):
    def __init__(self, input_dim, hidden1=256, hidden2=128, hidden3=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.BatchNorm1d(hidden1),
            nn.Dropout(dropout),

            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden2),
            nn.Dropout(dropout),

            nn.Linear(hidden2, hidden3),
            nn.ReLU(),
            nn.BatchNorm1d(hidden3),
            nn.Dropout(dropout),

            nn.Linear(hidden3, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

# Compact sweep: multiple learning rates, hidden sizes and dropout values,
# while keeping the total number of real runs manageable on a T4.
EXPERIMENT_CONFIGS = [
    {
        "name": "mlp_small",
        "hidden1": 128, "hidden2": 64, "hidden3": 32,
        "lr": 1e-4, "weight_decay": 1e-4, "dropout": 0.0,
        "epochs": 20, "patience": 4, "batch_size": 32,
    },
    {
        "name": "mlp_base",
        "hidden1": 256, "hidden2": 128, "hidden3": 64,
        "lr": 5e-4, "weight_decay": 1e-4, "dropout": 0.2,
        "epochs": 25, "patience": 5, "batch_size": 32,
    },
    {
        "name": "mlp_regularized",
        "hidden1": 512, "hidden2": 256, "hidden3": 128,
        "lr": 1e-3, "weight_decay": 1e-3, "dropout": 0.3,
        "epochs": 25, "patience": 5, "batch_size": 32,
    },
]

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    strict_repro = bool(globals().get("STRICT_REPRO", False))
    torch.use_deterministic_algorithms(strict_repro)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = strict_repro
        torch.backends.cudnn.benchmark = not strict_repro


print("Probe configurations:")
display(pd.DataFrame(EXPERIMENT_CONFIGS))

_H5_QID_INDEX_CACHE = {}

def _get_h5_qid_index(model_key):
    cached = _H5_QID_INDEX_CACHE.get(model_key)
    if cached is not None:
        return cached
    with h5py.File(feature_h5_path(model_key), "r") as h5:
        qids = [
            x.decode() if isinstance(x, bytes) else str(x)
            for x in h5["question_id"][:]
        ]
    cached = {qid: i for i, qid in enumerate(qids)}
    _H5_QID_INDEX_CACHE[model_key] = cached
    return cached

def read_h5_feature_matrix(model_key, feature_key, split_frame):
    qid_to_idx = _get_h5_qid_index(model_key)
    idx = np.asarray(
        [qid_to_idx[str(q)] for q in split_frame["question_id"]],
        dtype=np.int64,
    )

    # h5py fancy indexing is most reliable with increasing indices.
    # Read in sorted HDF5 order, then restore the caller's original order.
    order = np.argsort(idx, kind="stable")
    sorted_idx = idx[order]

    with h5py.File(feature_h5_path(model_key), "r") as h5:
        X_sorted = np.asarray(
            h5[feature_key][sorted_idx],
            dtype=np.float32,
        )

    restore = np.empty_like(order)
    restore[order] = np.arange(len(order))
    X = X_sorted[restore]

    y = np.asarray(split_frame["label"], dtype=np.int64)
    return X, y

def choose_class_weighting(y_train):
    counts = np.bincount(np.asarray(y_train, dtype=int), minlength=2)
    if (counts == 0).any():
        return {"mode": "none", "pos_weight": 1.0, "ratio": np.inf}
    ratio = float(max(counts) / max(min(counts), 1))
    if ratio >= IMBALANCE_RATIO_THRESHOLD:
        pos_weight = float(counts[0] / max(counts[1], 1))
        return {"mode": "weighted", "pos_weight": pos_weight, "ratio": ratio}
    return {"mode": "none", "pos_weight": 1.0, "ratio": ratio}

def safe_binary_metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= float(threshold)).astype(int)

    out = {
        "threshold": float(threshold),
        "class_0_count": int((y_true == 0).sum()),
        "class_1_count": int((y_true == 1).sum()),
        "auroc": np.nan,
        "average_precision": np.nan,
    }

    if len(np.unique(y_true)) == 2:
        out["auroc"] = float(skmetrics.roc_auc_score(y_true, prob))
        out["average_precision"] = float(skmetrics.average_precision_score(y_true, prob))

    out["accuracy"] = float(skmetrics.accuracy_score(y_true, pred))
    out["balanced_accuracy"] = float(skmetrics.balanced_accuracy_score(y_true, pred))
    out["precision"] = float(skmetrics.precision_score(y_true, pred, zero_division=0))
    out["recall"] = float(skmetrics.recall_score(y_true, pred, zero_division=0))
    out["f1"] = float(skmetrics.f1_score(y_true, pred, zero_division=0))
    out["mcc"] = float(skmetrics.matthews_corrcoef(y_true, pred))
    out["brier"] = float(skmetrics.brier_score_loss(y_true, prob))

    cm = skmetrics.confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    out["specificity"] = float(tn / (tn + fp)) if (tn + fp) > 0 else np.nan
    return out

def select_validation_threshold(y_val, val_prob):
    y_val = np.asarray(y_val, dtype=int)
    val_prob = np.asarray(val_prob, dtype=float)
    if len(np.unique(y_val)) < 2:
        return 0.5, "fallback_one_class_validation"

    uniq = np.unique(val_prob)
    if len(uniq) == 1:
        candidates = np.array([0.5, float(uniq[0])])
    else:
        mids = (uniq[:-1] + uniq[1:]) / 2.0
        candidates = np.unique(np.r_[0.0, mids, 0.5, 1.0])

    records = []
    for t in candidates:
        pred = (val_prob >= t).astype(int)
        f1 = float(skmetrics.f1_score(y_val, pred, zero_division=0))
        bal = float(skmetrics.balanced_accuracy_score(y_val, pred))
        records.append((f1, bal, -abs(float(t) - 0.5), float(t)))

    _, _, _, best_t = max(records)
    return best_t, "selected_on_validation_max_f1_then_balanced_accuracy"

def expected_calibration_error(y_true, prob, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    prob = np.asarray(prob, dtype=float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    records = []
    for left, right in zip(bins[:-1], bins[1:]):
        mask = (prob >= left) & ((prob < right) if right < 1 else (prob <= right))
        if not mask.any():
            continue
        acc = y_true[mask].mean()
        conf = prob[mask].mean()
        weight = mask.mean()
        ece += weight * abs(acc - conf)
        records.append({
            "bin_left": left, "bin_right": right, "count": int(mask.sum()),
            "accuracy": acc, "confidence": conf
        })
    return float(ece), pd.DataFrame(records)

def fit_logistic_probe(X_train, y_train, seed):
    from sklearn.linear_model import LogisticRegression
    weighting = choose_class_weighting(y_train)
    class_weight = "balanced" if weighting["mode"] == "weighted" else None
    model = LogisticRegression(
        max_iter=2000,
        class_weight=class_weight,
        random_state=seed,
        solver="lbfgs",
    )
    model.fit(X_train, y_train)
    return model, weighting

def fit_mlp_probe(X_train, y_train, X_val, y_val, cfg, seed):
    set_all_seeds(seed)
    device = torch.device(globals().get("PROBE_DEVICE", "cpu"))
    model = HALPProbe(
        X_train.shape[1],
        hidden1=cfg["hidden1"],
        hidden2=cfg["hidden2"],
        hidden3=cfg["hidden3"],
        dropout=cfg["dropout"],
    ).to(device)

    weighting = choose_class_weighting(y_train)
    if weighting["mode"] == "weighted":
        pos_weight = torch.tensor(weighting["pos_weight"], dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        loss_fn = nn.BCEWithLogitsLoss()

    train_ds = TensorDataset(
        torch.from_numpy(X_train),
        torch.from_numpy(y_train.astype(np.float32))
    )
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    requested_batch_size = max(2, int(cfg["batch_size"]))
    # BatchNorm1d cannot train on a singleton final batch. Preserve every
    # sample and only shrink the requested batch size when its remainder
    # would otherwise be exactly one sample.
    effective_batch_size = requested_batch_size
    if len(train_ds) >= 2 and len(train_ds) % effective_batch_size == 1:
        effective_batch_size = max(2, effective_batch_size - 1)

    loader = DataLoader(
        train_ds,
        batch_size=effective_batch_size,
        shuffle=True,
        drop_last=False,
        generator=generator,
        pin_memory=(device.type == "cuda" and bool(globals().get("PROBE_PIN_MEMORY", False))),
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg["lr"]),
        weight_decay=float(cfg["weight_decay"])
    )

    Xv = torch.from_numpy(X_val).to(device, non_blocking=True)
    yv = torch.from_numpy(y_val.astype(np.float32)).to(device, non_blocking=True)
    best = {"val_auroc": -np.inf, "state": None, "epoch": 0, "weighting": weighting}
    patience_left = int(cfg["patience"])

    for epoch in range(int(cfg["epochs"])):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.inference_mode():
            val_prob = torch.sigmoid(model(Xv)).detach().cpu().numpy()

        val_auroc = (
            float(skmetrics.roc_auc_score(y_val, val_prob))
            if len(np.unique(y_val)) == 2 else np.nan
        )
        if np.isfinite(val_auroc) and val_auroc > best["val_auroc"]:
            best = {
                "val_auroc": val_auroc,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "epoch": epoch + 1,
                "weighting": weighting,
            }
            patience_left = int(cfg["patience"])
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    if best["state"] is None:
        del Xv, yv
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise ValueError("No finite validation AUROC was obtained for this MLP configuration.")

    model.load_state_dict(best["state"])
    return model, best

def fit_mlp_probe_fixed_epochs(X_train, y_train, cfg, seed, epochs):
    set_all_seeds(seed)
    device = torch.device(globals().get("PROBE_DEVICE", "cpu"))
    model = HALPProbe(
        X_train.shape[1],
        hidden1=cfg["hidden1"],
        hidden2=cfg["hidden2"],
        hidden3=cfg["hidden3"],
        dropout=cfg["dropout"],
    ).to(device)

    weighting = choose_class_weighting(y_train)
    if weighting["mode"] == "weighted":
        pos_weight = torch.tensor(weighting["pos_weight"], dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        loss_fn = nn.BCEWithLogitsLoss()

    ds = TensorDataset(
        torch.from_numpy(X_train),
        torch.from_numpy(y_train.astype(np.float32)),
    )
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    requested_batch_size = max(2, int(cfg["batch_size"]))
    # Same BatchNorm-safe rule for the train+validation refit.
    effective_batch_size = requested_batch_size
    if len(ds) >= 2 and len(ds) % effective_batch_size == 1:
        effective_batch_size = max(2, effective_batch_size - 1)

    loader = DataLoader(
        ds,
        batch_size=effective_batch_size,
        shuffle=True,
        drop_last=False,
        generator=generator,
        pin_memory=(device.type == "cuda" and bool(globals().get("PROBE_PIN_MEMORY", False))),
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg["lr"]),
        weight_decay=float(cfg["weight_decay"])
    )

    for _ in range(max(1, int(epochs))):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

    return model, weighting

def model_probabilities(probe, X):
    if hasattr(probe, "predict_proba"):
        return probe.predict_proba(X)[:, 1]
    device = next(probe.parameters()).device
    probe.eval()
    with torch.inference_mode():
        return torch.sigmoid(probe(torch.from_numpy(X).to(device))).cpu().numpy()


Probe configurations:


,name,hidden1,hidden2,hidden3,lr,weight_decay,dropout,epochs,patience,batch_size
0,mlp_small,128,64,32,0.0001,0.0001,0.0,20,4,32
1,mlp_base,256,128,64,0.0005,0.0001,0.2,25,5,32
2,mlp_regularized,512,256,128,0.0010,0.0010,0.3,25,5,32


## 16. Probe training and hyperparameter selection

For each available representation:
- fit a linear baseline;
- fit the compact MLP configurations;
- compare validation AUROC;
- choose the best configuration for that representation/seed;
- refit on train + validation only;
- evaluate on test exactly once.

This boundary keeps the test set out of tuning.



In [ ]:

# ============================================================
# 16. Probe training and evaluation — label-gated and resumable
# ============================================================
#
# Purpose
# -------
# Train and evaluate lightweight probes on the extracted HALP
# representations.
#
# This cell is intentionally SAFE when model-specific reviewed labels
# are unavailable.
#
#
# REQUIRED FOR SUPERVISED TRAINING
# --------------------------------
# A model needs:
#
#     1. real model-specific hallucination labels
#     2. a valid grouped train/validation/test split
#     3. extracted features for the required samples
#
#
# If those are unavailable:
#
#     -> training is BLOCKED
#     -> no labels are fabricated
#     -> no fake metrics are generated
#     -> the reason is recorded explicitly
#
#
# Current smoke-test case
# -----------------------
# For the current SmolVLM2 run, the notebook may have:
#
#     df                     -> available
#     model_df["smolvlm2"]   -> unavailable
#     splits["smolvlm2"]     -> unavailable
#     HDF5 features          -> available
#
# This cell must not do:
#
#     train = splits[model_key]["train"]
#
# unless that split actually exists.
#
#
# SCIENTIFIC RULE
# ---------------
# No test-set tuning.
#
# Hyperparameters are selected only on validation data.
#
# The final test set is evaluated only after the configuration is fixed.
#
# This cell does not generate answers.
# ============================================================


# ------------------------------------------------------------
# ------------------------------------------------------------
# 0. Scikit-learn metrics namespace
# ------------------------------------------------------------
# The helper functions used by this probe stage reference `skmetrics`
# at runtime. Define the alias explicitly in this same cell so the stage
# does not depend on an import from another notebook cell or execution order.
from sklearn import metrics as skmetrics

# Progress bar support local to this cell.
try:
    from tqdm.auto import tqdm
except Exception:
    class _TqdmFallback:
        def __init__(self, iterable=None, total=None, desc=None, unit=None, leave=True):
            self.iterable = iterable
            self.total = total
            self.desc = desc or "Progress"
        def __iter__(self):
            if self.iterable is None:
                return iter(())
            for index, item in enumerate(self.iterable, 1):
                print(f"{self.desc}: {index}/{self.total if self.total is not None else '?'}")
                yield item
        def update(self, n=1):
            return None
        def set_postfix(self, *args, **kwargs):
            return None
        def close(self):
            return None
        def __enter__(self):
            return self
        def __exit__(self, exc_type, exc, tb):
            self.close()
    tqdm = _TqdmFallback

# 1. Notebook objects we need
# ------------------------------------------------------------

print("=" * 82)
print("HALP-Bench probe training / evaluation")
print("=" * 82)


REQUIRED_GLOBALS = [
    "ACTIVE_MODELS",
    "feature_h5_path",
    "h5py",
    "np",
    "pd",
    "torch",
    "Path",
    "RUN_ROOT",
]


missing_globals = [
    name
    for name in REQUIRED_GLOBALS
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Missing required notebook objects:\n"
        f"  {missing_globals}\n\n"
        "Run the environment, dataset, feature extraction, and "
        "feature-validation cells first."
    )


if not isinstance(
    ACTIVE_MODELS,
    (list, tuple),
):

    raise TypeError(
        "ACTIVE_MODELS must be a list or tuple."
    )


if not ACTIVE_MODELS:

    raise RuntimeError(
        "ACTIVE_MODELS is empty."
    )


# ------------------------------------------------------------
# 2. Optional upstream objects
# ------------------------------------------------------------
#
# The supervised stage is allowed to be blocked.
#
# So `splits`, `model_df`, or `SEEDS` may legitimately be absent.
# We diagnose their availability rather than treating their absence as
# a programming error.
# ------------------------------------------------------------

HAS_MODEL_DF = (
    "model_df" in globals()
    and isinstance(
        model_df,
        dict,
    )
)


HAS_SPLITS = (
    "splits" in globals()
    and isinstance(
        splits,
        dict,
    )
)


HAS_SEEDS = (
    "SEEDS" in globals()
    and isinstance(
        SEEDS,
        (list, tuple),
    )
    and len(SEEDS) > 0
)


HAS_EXPERIMENT_CONFIGS = (
    "EXPERIMENT_CONFIGS" in globals()
    and isinstance(
        EXPERIMENT_CONFIGS,
        (list, tuple),
    )
    and len(EXPERIMENT_CONFIGS) > 0
)


print(
    "model_df available:",
    HAS_MODEL_DF
)


print(
    "splits available  :",
    HAS_SPLITS
)


print(
    "SEEDS available   :",
    HAS_SEEDS
)


print(
    "MLP configs        :",
    HAS_EXPERIMENT_CONFIGS
)


# ------------------------------------------------------------
# 3. Output directories
# ------------------------------------------------------------

PREDICTIONS_DIR = (
    RUN_ROOT
    / "predictions"
)


TABLES_DIR = (
    RUN_ROOT
    / "tables"
)


LOGS_DIR = (
    RUN_ROOT
    / "logs"
)


PREDICTIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


LOGS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 4. HDF5 string decoder
# ------------------------------------------------------------

def decode_probe_h5_value(
    value,
) -> str:

    if isinstance(
        value,
        bytes,
    ):

        return value.decode(
            "utf-8"
        )

    return str(
        value
    )


# ------------------------------------------------------------
# 5. Feature-key discovery
# ------------------------------------------------------------

def discover_probe_feature_keys(
    model_key: str,
):
    """
    Return the actual extracted VF/VT/QT datasets in the HDF5 file.
    """

    path = Path(
        feature_h5_path(
            model_key
        )
    )


    if not path.exists():

        raise FileNotFoundError(
            f"Feature HDF5 file does not exist for {model_key!r}:\n"
            f"{path}"
        )


    with h5py.File(
        path,
        "r",
    ) as h5:

        keys = sorted(
            [
                key
                for key in h5.keys()
                if (
                    key == "vf"
                    or key.startswith("vt_")
                    or key.startswith("qt_")
                )
            ]
        )


    return keys


# ------------------------------------------------------------
# 6. Read a feature matrix safely
# ------------------------------------------------------------

def read_probe_h5_feature_matrix(
    model_key: str,
    feature_key: str,
    split_frame: pd.DataFrame,
):
    """
    Read feature rows aligned by question_id.

    Labels are read from the supplied split dataframe, not from HDF5.
    """

    if not isinstance(
        split_frame,
        pd.DataFrame,
    ):

        raise TypeError(
            "split_frame must be a pandas DataFrame."
        )


    required_columns = {
        "question_id",
        "label",
    }


    missing = (
        required_columns
        - set(
            split_frame.columns
        )
    )


    if missing:

        raise RuntimeError(
            f"Split dataframe is missing required columns: "
            f"{sorted(missing)}"
        )


    qids_requested = (
        split_frame[
            "question_id"
        ]
        .astype(str)
        .tolist()
    )


    if not qids_requested:

        raise RuntimeError(
            f"{model_key} / {feature_key}: empty split."
        )


    with h5py.File(
        feature_h5_path(
            model_key
        ),
        "r",
    ) as h5:

        if (
            "question_id"
            not in h5
        ):

            raise RuntimeError(
                f"{model_key}: HDF5 has no question_id dataset."
            )


        if (
            feature_key
            not in h5
        ):

            raise KeyError(
                f"{model_key}: feature dataset "
                f"{feature_key!r} does not exist."
            )


        qids_h5 = [
            decode_probe_h5_value(
                value
            )
            for value in h5[
                "question_id"
            ][:]
        ]


        qid_to_idx = {
            qid: index
            for index, qid in enumerate(
                qids_h5
            )
        }


        missing_qids = [
            qid
            for qid in qids_requested
            if qid not in qid_to_idx
        ]


        if missing_qids:

            raise RuntimeError(
                f"{model_key} / {feature_key}: "
                f"{len(missing_qids)} requested question IDs "
                "are absent from the feature file.\n"
                f"Examples: {missing_qids[:20]}"
            )


        indices = [
            qid_to_idx[
                qid
            ]
            for qid in qids_requested
        ]


        # HDF5 advanced indexing is most robust when indexes are sorted.
        # We read rows individually through sorted indices and restore the
        # requested order afterward.
        order = np.argsort(
            np.asarray(
                indices
            )
        )


        sorted_indices = [
            indices[
                i
            ]
            for i in order
        ]


        sorted_rows = np.asarray(
            h5[
                feature_key
            ][sorted_indices, :],
            dtype=np.float32,
        )


        inverse_order = np.argsort(
            order
        )


        X = (
            sorted_rows[
                inverse_order
            ]
        )


    y = (
        pd.to_numeric(
            split_frame[
                "label"
            ],
            errors="raise",
        )
        .astype(np.int64)
        .to_numpy()
    )


    if X.ndim != 2:

        raise RuntimeError(
            f"{model_key} / {feature_key}: "
            f"expected X with rank 2, got shape={X.shape}"
        )


    if len(X) != len(y):

        raise RuntimeError(
            f"{model_key} / {feature_key}: "
            f"X/y length mismatch: {len(X)} vs {len(y)}"
        )


    if not np.isfinite(
        X
    ).all():

        raise ValueError(
            f"{model_key} / {feature_key}: "
            "NaN/Inf detected in extracted features."
        )


    if not np.isin(
        y,
        [0, 1],
    ).all():

        raise ValueError(
            f"{model_key}: labels must contain only 0/1."
        )


    return X, y


# ------------------------------------------------------------
# 7. Safely identify completed feature IDs
# ------------------------------------------------------------

def get_complete_feature_ids(
    model_key: str,
):
    """
    Return only question IDs whose HDF5 status is complete.
    """

    path = Path(
        feature_h5_path(
            model_key
        )
    )


    if not path.exists():

        raise FileNotFoundError(
            path
        )


    with h5py.File(
        path,
        "r",
    ) as h5:

        if "question_id" not in h5:

            raise RuntimeError(
                f"{model_key}: HDF5 has no question_id dataset."
            )


        if "status" not in h5:

            raise RuntimeError(
                f"{model_key}: HDF5 has no status dataset."
            )


        qids = [
            decode_probe_h5_value(
                value
            )
            for value in h5[
                "question_id"
            ][:]
        ]


        statuses = [
            decode_probe_h5_value(
                value
            )
            for value in h5[
                "status"
            ][:]
        ]


    if len(qids) != len(statuses):

        raise RuntimeError(
            f"{model_key}: question_id/status length mismatch."
        )


    return {
        qid
        for qid, status in zip(
            qids,
            statuses,
        )
        if status == "complete"
    }


# ------------------------------------------------------------
# 8. Validate a supervised split
# ------------------------------------------------------------

def validate_supervised_split(
    model_key: str,
):
    """
    Return train/val/test frames only when genuine labels and a valid split
    exist.

    Returns:
        parts or None
        reason
    """

    # --------------------------------------------------------
    # No model-specific labels
    # --------------------------------------------------------

    if not HAS_MODEL_DF:

        return (
            None,
            "model_df is unavailable; no reviewed-label dataframe exists.",
        )


    if model_key not in model_df:

        return (
            None,
            f"model_df[{model_key!r}] is unavailable because "
            "no reviewed-label dataframe was loaded for this model.",
        )


    labeled_frame = model_df[
        model_key
    ]


    if not isinstance(
        labeled_frame,
        pd.DataFrame,
    ):

        return (
            None,
            f"model_df[{model_key!r}] is not a DataFrame.",
        )


    required_label_columns = {
        "question_id",
        "label",
    }


    missing_columns = (
        required_label_columns
        - set(
            labeled_frame.columns
        )
    )


    if missing_columns:

        return (
            None,
            f"Reviewed-label dataframe is missing columns: "
            f"{sorted(missing_columns)}",
        )


    # --------------------------------------------------------
    # Existing grouped split
    # --------------------------------------------------------

    if (
        HAS_SPLITS
        and model_key in splits
    ):

        parts = splits[
            model_key
        ]


        if not isinstance(
            parts,
            dict,
        ):

            return (
                None,
                "Existing split object is not a dictionary.",
            )


        required_parts = {
            "train",
            "val",
            "test",
        }


        missing_parts = (
            required_parts
            - set(
                parts.keys()
            )
        )


        if missing_parts:

            return (
                None,
                f"Existing split is missing: {sorted(missing_parts)}",
            )


        train = parts[
            "train"
        ].copy()


        val = parts[
            "val"
        ].copy()


        test = parts[
            "test"
        ].copy()


    # --------------------------------------------------------
    # Split missing but labels exist
    # --------------------------------------------------------
    #
    # Recreate it using the notebook's canonical grouped splitter.
    # This is safe because the split is derived from labels and group_id,
    # never from the test predictions.
    # --------------------------------------------------------

    else:

        if (
            "choose_grouped_split"
            not in globals()
        ):

            return (
                None,
                "No valid splits exist and choose_grouped_split() "
                "is unavailable.",
            )


        try:

            train, val, test = (
                choose_grouped_split(
                    labeled_frame.copy().reset_index(
                        drop=True
                    ),
                    seed=(
                        SEEDS[0]
                        if HAS_SEEDS
                        else 42
                    ),
                )
            )

        except Exception as exc:

            return (
                None,
                "Could not construct a grouped supervised split: "
                f"{type(exc).__name__}: {exc}",
            )


    # --------------------------------------------------------
    # Normalize IDs and labels
    # --------------------------------------------------------

    normalized = {}


    for name, frame in [
        (
            "train",
            train,
        ),
        (
            "val",
            val,
        ),
        (
            "test",
            test,
        ),
    ]:

        if not isinstance(
            frame,
            pd.DataFrame,
        ):

            return (
                None,
                f"{name} split is not a DataFrame.",
            )


        if (
            "question_id"
            not in frame.columns
            or "label"
            not in frame.columns
        ):

            return (
                None,
                f"{name} split lacks question_id or label.",
            )


        frame = frame.copy()


        frame[
            "question_id"
        ] = (
            frame[
                "question_id"
            ]
            .astype(str)
        )


        frame[
            "label"
        ] = (
            pd.to_numeric(
                frame[
                    "label"
                ],
                errors="raise",
            )
            .astype(np.int64)
        )


        if not np.isin(
            frame[
                "label"
            ].to_numpy(),
            [0, 1],
        ).all():

            return (
                None,
                f"{name} split contains labels outside {0,1}.",
            )


        normalized[
            name
        ] = frame


    train = normalized[
        "train"
    ]


    val = normalized[
        "val"
    ]


    test = normalized[
        "test"
    ]


    # --------------------------------------------------------
    # Verify no split overlap by question ID
    # --------------------------------------------------------

    train_ids = set(
        train[
            "question_id"
        ]
    )


    val_ids = set(
        val[
            "question_id"
        ]
    )


    test_ids = set(
        test[
            "question_id"
        ]
    )


    if not train_ids.isdisjoint(
        val_ids
    ):

        return (
            None,
            "Train/validation question-ID overlap detected.",
        )


    if not train_ids.isdisjoint(
        test_ids
    ):

        return (
            None,
            "Train/test question-ID overlap detected.",
        )


    if not val_ids.isdisjoint(
        test_ids
    ):

        return (
            None,
            "Validation/test question-ID overlap detected.",
        )


    return (
        {
            "train":
                train,

            "val":
                val,

            "test":
                test,
        },
        None,
    )


# ------------------------------------------------------------
# 9. Fit scaler on TRAIN ONLY
# ------------------------------------------------------------

def standardize_train_only(
    X_train,
    X_val,
    X_test,
):
    """
    Fit feature standardization only on the training partition.
    """

    X_train = np.asarray(
        X_train,
        dtype=np.float32,
    )


    X_val = np.asarray(
        X_val,
        dtype=np.float32,
    )


    X_test = np.asarray(
        X_test,
        dtype=np.float32,
    )


    if X_train.ndim != 2:

        raise ValueError(
            f"X_train must be 2-D, got {X_train.shape}"
        )


    mean = (
        X_train
        .mean(
            axis=0,
            keepdims=True,
        )
    )


    std = (
        X_train
        .std(
            axis=0,
            keepdims=True,
        )
    )


    # Protect constant dimensions without changing the actual features.
    std = np.where(
        std < 1e-6,
        1e-6,
        std,
    )


    return (
        (X_train - mean) / std,
        (X_val - mean) / std,
        (X_test - mean) / std,
        mean,
        std,
    )


# ------------------------------------------------------------
# 10. Model probability helper
# ------------------------------------------------------------

def get_probe_probabilities(
    probe,
    X,
):
    """
    Return probability of hallucination (positive class = 1).
    """

    if hasattr(
        probe,
        "predict_proba",
    ):

        probabilities = probe.predict_proba(
            X
        )[:, 1]


    else:

        device = next(
            probe.parameters()
        ).device


        X_tensor = (
            torch.from_numpy(
                np.asarray(
                    X,
                    dtype=np.float32,
                )
            )
            .to(
                device
            )
        )


        probe.eval()


        with torch.inference_mode():

            logits = probe(
                X_tensor
            )


            probabilities = (
                torch.sigmoid(
                    logits
                )
                .detach()
                .cpu()
                .numpy()
            )


    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )


    if not np.isfinite(
        probabilities
    ).all():

        raise ValueError(
            "Probe returned non-finite probabilities."
        )


    probabilities = np.clip(
        probabilities,
        0.0,
        1.0,
    )


    return probabilities


# ------------------------------------------------------------
# 11. Run one model
# ------------------------------------------------------------

def train_and_evaluate_model(
    model_key: str,
):
    """
    Train/evaluate probes for one model.

    Returns an empty DataFrame when supervised training is blocked.
    """

    # --------------------------------------------------------
    # Basic feature-file check
    # --------------------------------------------------------

    feature_path = Path(
        feature_h5_path(
            model_key
        )
    )


    if not feature_path.exists():

        raise FileNotFoundError(
            f"Feature file for {model_key!r} does not exist:\n"
            f"{feature_path}"
        )


    # --------------------------------------------------------
    # Require real reviewed labels
    # --------------------------------------------------------

    split_parts, split_block_reason = (
        validate_supervised_split(
            model_key
        )
    )


    if split_parts is None:

        print(
            "\n"
            + "-" * 82
        )


        print(
            f"PROBE TRAINING BLOCKED: {model_key}"
        )


        print(
            "-" * 82
        )


        print(
            "Reason:",
            split_block_reason
        )


        print(
            "Feature extraction may still be valid."
        )


        print(
            "No hallucination labels will be inferred or fabricated."
        )


        blocked_path = (
            LOGS_DIR
            / f"{model_key}_probe_training_blocked.json"
        )


        blocked_record = {

            "model":
                model_key,

            "status":
                "BLOCKED",

            "reason":
                split_block_reason,

            "feature_path":
                str(
                    feature_path
                ),

            "created_at_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }


        blocked_path.write_text(
            json.dumps(
                blocked_record,
                indent=2,
            ),
            encoding="utf-8",
        )


        # Return a well-formed empty result rather than raising a misleading
        # KeyError such as splits["smolvlm2"].
        return pd.DataFrame(
            [
                blocked_record
            ]
        )


    train = split_parts[
        "train"
    ].copy()


    val = split_parts[
        "val"
    ].copy()


    test = split_parts[
        "test"
    ].copy()


    # --------------------------------------------------------
    # Required split size
    # --------------------------------------------------------

    if min(
        len(train),
        len(val),
        len(test),
    ) < 4:

        reason = (
            "Each split must contain at least 4 rows before probe "
            "training is attempted."
        )


        print(
            f"[{model_key}] PROBE TRAINING BLOCKED: {reason}"
        )


        return pd.DataFrame(
            [
                {
                    "model":
                        model_key,

                    "status":
                        "BLOCKED",

                    "reason":
                        reason,

                    "train_rows":
                        len(train),

                    "val_rows":
                        len(val),

                    "test_rows":
                        len(test),
                }
            ]
        )


    # --------------------------------------------------------
    # Completed feature IDs
    # --------------------------------------------------------

    complete_qids = get_complete_feature_ids(
        model_key
    )


    # --------------------------------------------------------
    # Keep only samples with actual extracted features
    # --------------------------------------------------------

    train = train[
        train[
            "question_id"
        ].isin(
            complete_qids
        )
    ].copy()


    val = val[
        val[
            "question_id"
        ].isin(
            complete_qids
        )
    ].copy()


    test = test[
        test[
            "question_id"
        ].isin(
            complete_qids
        )
    ].copy()


    print(
        "\n"
        + "-" * 82
    )


    print(
        f"MODEL: {model_key}"
    )


    print(
        "-" * 82
    )


    print(
        "Labeled split rows before feature filter:",
        {
            "train":
                len(
                    split_parts[
                        "train"
                    ]
                ),

            "val":
                len(
                    split_parts[
                        "val"
                    ]
                ),

            "test":
                len(
                    split_parts[
                        "test"
                    ]
                ),
        }
    )


    print(
        "Complete-feature split rows:",
        {
            "train":
                len(train),

            "val":
                len(val),

            "test":
                len(test),
        }
    )


    # --------------------------------------------------------
    # Smoke-test protection
    # --------------------------------------------------------
    #
    # A smoke extraction of 32 samples usually cannot populate a valid
    # train/validation/test partition after the grouped split has been
    # created on the full labeled benchmark.
    #
    # Do not manufacture a new random split using the test features.
    # --------------------------------------------------------

    if min(
        len(train),
        len(val),
        len(test),
    ) < 4:

        reason = (
            "Too few completed labeled samples remain in at least one "
            "partition after intersecting the supervised split with "
            "the extracted feature IDs. Run a larger feature extraction "
            "with genuine reviewed labels before training."
        )


        print(
            f"[{model_key}] PROBE TRAINING BLOCKED:"
        )


        print(
            reason
        )


        blocked_path = (
            LOGS_DIR
            / f"{model_key}_probe_training_blocked.json"
        )


        blocked_record = {

            "model":
                model_key,

            "status":
                "BLOCKED",

            "reason":
                reason,

            "train_rows":
                len(train),

            "val_rows":
                len(val),

            "test_rows":
                len(test),

            "complete_feature_ids":
                len(complete_qids),

            "created_at_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }


        blocked_path.write_text(
            json.dumps(
                blocked_record,
                indent=2,
            ),
            encoding="utf-8",
        )


        return pd.DataFrame(
            [
                blocked_record
            ]
        )


    # --------------------------------------------------------
    # Ensure both classes occur in all partitions
    # --------------------------------------------------------

    partition_class_counts = {

        "train":
            train[
                "label"
            ].value_counts(
                sort=False
            ).to_dict(),

        "val":
            val[
                "label"
            ].value_counts(
                sort=False
            ).to_dict(),

        "test":
            test[
                "label"
            ].value_counts(
                sort=False
            ).to_dict(),
    }


    for split_name, frame in [
        (
            "train",
            train,
        ),
        (
            "val",
            val,
        ),
        (
            "test",
            test,
        ),
    ]:

        if frame[
            "label"
        ].nunique() < 2:

            reason = (
                f"{split_name} partition contains only one class."
            )


            print(
                f"[{model_key}] PROBE TRAINING BLOCKED: {reason}"
            )


            return pd.DataFrame(
                [
                    {
                        "model":
                            model_key,

                        "status":
                            "BLOCKED",

                        "reason":
                            reason,

                        "class_counts":
                            partition_class_counts,
                    }
                ]
            )


    # --------------------------------------------------------
    # Feature keys
    # --------------------------------------------------------

    feature_keys = discover_probe_feature_keys(
        model_key
    )


    if not feature_keys:

        raise RuntimeError(
            f"{model_key}: no VF/VT/QT features are available."
        )


    print(
        "Feature keys:",
        feature_keys,
    )


    all_rows = []


    # --------------------------------------------------------
    # Representation loop
    # --------------------------------------------------------

    for feature_key in tqdm(
        feature_keys,
        total=len(feature_keys),
        desc=f"{model_key} representations",
        unit="representation",
        leave=False,
    ):

        print(
            "\n"
            + "=" * 82
        )


        print(
            f"REPRESENTATION: {feature_key}"
        )


        print(
            "=" * 82
        )


        try:

            Xtr, ytr = (
                read_probe_h5_feature_matrix(
                    model_key,
                    feature_key,
                    train,
                )
            )


            Xva, yva = (
                read_probe_h5_feature_matrix(
                    model_key,
                    feature_key,
                    val,
                )
            )


            Xte, yte = (
                read_probe_h5_feature_matrix(
                    model_key,
                    feature_key,
                    test,
                )
            )


        except Exception as exc:

            print(
                f"[{model_key} / {feature_key}] "
                f"feature read failed: "
                f"{type(exc).__name__}: {exc}"
            )


            all_rows.append(
                {
                    "model":
                        model_key,

                    "representation":
                        feature_key,

                    "probe":
                        "unavailable",

                    "seed":
                        np.nan,

                    "split":
                        "none",

                    "status":
                        "FAILED",

                    "error":
                        f"{type(exc).__name__}: {exc}",
                }
            )


            continue


        # ----------------------------------------------------
        # Train-only standardization
        # ----------------------------------------------------

        (
            Xtr_s,
            Xva_s,
            Xte_s,
            train_mean,
            train_std,
        ) = standardize_train_only(
            Xtr,
            Xva,
            Xte,
        )


        print(
            "Feature dimension:",
            Xtr.shape[1],
        )


        print(
            "Shapes:",
            {
                "train":
                    Xtr_s.shape,

                "val":
                    Xva_s.shape,

                "test":
                    Xte_s.shape,
            }
        )


        # ----------------------------------------------------
        # Determine seeds
        # ----------------------------------------------------

        if HAS_SEEDS:

            active_seeds = [
                int(seed)
                for seed in SEEDS
            ]

        else:

            active_seeds = [
                42
            ]


        # ----------------------------------------------------
        # Each seed
        # ----------------------------------------------------

        for seed in tqdm(
            active_seeds,
            total=len(active_seeds),
            desc=f"{model_key}/{feature_key} seeds",
            unit="seed",
            leave=False,
        ):

            # ====================================================
            # LOGISTIC REGRESSION
            # ====================================================

            set_all_seeds(
                seed
            )


            try:

                logistic_model, logistic_weighting = fit_logistic_probe(
                    Xtr_s,
                    ytr,
                    seed,
                )


                val_prob = (
                    get_probe_probabilities(
                        logistic_model,
                        Xva_s,
                    )
                )


                val_threshold, threshold_rule = select_validation_threshold(
                    yva,
                    val_prob,
                )
                val_metrics = safe_binary_metrics(
                    yva,
                    val_prob,
                    threshold=val_threshold,
                )


                all_rows.append(
                    {
                        "model":
                            model_key,

                        "representation":
                            feature_key,

                        "probe":
                            "logistic",

                        "seed":
                            seed,

                        "split":
                            "validation",

                        "status":
                            "COMPLETED",

                        "selected_config":
                            "logistic_baseline",

                        "weighting_mode":
                            logistic_weighting["mode"],

                        "threshold_rule":
                            threshold_rule,

                        **val_metrics,
                    }
                )


                # ------------------------------------------------
                # Final logistic fit
                # ------------------------------------------------
                #
                # Test is untouched until this point.
                # The linear baseline has no validation-selected
                # hyperparameter other than its fixed configuration.
                # ------------------------------------------------

                # Final refit uses all allowed train + validation information
                # to estimate normalization statistics. The test set remains
                # completely untouched until final prediction.
                Xfit_raw_logistic = np.vstack(
                    [
                        Xtr,
                        Xva,
                    ]
                )

                yfit_logistic = np.concatenate(
                    [
                        ytr,
                        yva,
                    ]
                )

                (
                    Xfit_logistic,
                    _unused_val_refit_logistic,
                    Xte_logistic,
                    refit_mean_logistic,
                    refit_std_logistic,
                ) = standardize_train_only(
                    Xfit_raw_logistic,
                    Xfit_raw_logistic,
                    Xte,
                )

                final_logistic, final_logistic_weighting = (
                    fit_logistic_probe(
                        Xfit_logistic,
                        yfit_logistic,
                        seed,
                    )
                )

                test_prob = (
                    get_probe_probabilities(
                        final_logistic,
                        Xte_logistic,
                    )
                )


                test_metrics = safe_binary_metrics(
                    yte,
                    test_prob,
                    threshold=val_threshold,
                )


                all_rows.append(
                    {
                        "model":
                            model_key,

                        "representation":
                            feature_key,

                        "probe":
                            "logistic",

                        "seed":
                            seed,

                        "split":
                            "test",

                        "status":
                            "COMPLETED",

                        "selected_config":
                            "logistic_baseline",

                        "weighting_mode":
                            final_logistic_weighting["mode"],

                        "threshold_rule":
                            threshold_rule,

                        **test_metrics,
                    }
                )

                # Save logistic test probabilities so every reported probe can be
                # independently re-scored just like the MLP results.
                if bool(globals().get("SAVE_LOGISTIC_TEST_PREDICTIONS", True)):
                    logistic_prediction_df = pd.DataFrame(
                        {
                            "question_id": test["question_id"].astype(str).to_numpy(),
                            "image_name": (
                                test["image_name"].astype(str).to_numpy()
                                if "image_name" in test.columns
                                else ""
                            ),
                            "label": yte,
                            "probability": test_prob,
                            "threshold": float(val_threshold),
                            "prediction": (
                                test_prob >= float(val_threshold)
                            ).astype(int),
                            "model": model_key,
                            "representation": feature_key,
                            "probe": "logistic",
                            "seed": seed,
                            "selected_config": "logistic_baseline",
                            "selected": True,
                            "threshold_rule": threshold_rule,
                            "weighting_mode": final_logistic_weighting["mode"],
                        }
                    )

                    logistic_prediction_path = (
                        PREDICTIONS_DIR
                        / (
                            f"{model_key}_"
                            f"{feature_key}_"
                            f"logistic_seed_{seed}_"
                            f"test_predictions.csv"
                        )
                    )

                    logistic_prediction_df.to_csv(
                        logistic_prediction_path,
                        index=False,
                    )

                    del logistic_prediction_df


            except Exception as exc:

                print(
                    f"[{model_key} / {feature_key} / "
                    f"logistic / seed={seed}] failed: "
                    f"{type(exc).__name__}: {exc}"
                )


                all_rows.append(
                    {
                        "model":
                            model_key,

                        "representation":
                            feature_key,

                        "probe":
                            "logistic",

                        "seed":
                            seed,

                        "split":
                            "none",

                        "status":
                            "FAILED",

                        "error":
                            f"{type(exc).__name__}: {exc}",
                    }
                )


            # ====================================================
            # MLP
            # ====================================================

            if not HAS_EXPERIMENT_CONFIGS:

                print(
                    "MLP skipped: EXPERIMENT_CONFIGS is unavailable."
                )


                continue


            best_cfg = None
            best_model = None
            best_info = None


            # ----------------------------------------------------
            # Validation-only configuration selection
            # ----------------------------------------------------

            for cfg in tqdm(
                EXPERIMENT_CONFIGS,
                total=len(EXPERIMENT_CONFIGS),
                desc=f"{model_key}/{feature_key}/seed={seed} MLP configs",
                unit="config",
                leave=False,
            ):

                try:

                    mlp_model, info = (
                        fit_mlp_probe(
                            Xtr_s,
                            ytr,
                            Xva_s,
                            yva,
                            cfg,
                            seed,
                        )
                    )


                    candidate_score = info.get("val_auroc", np.nan)

                    all_rows.append({
                        "model": model_key,
                        "representation": feature_key,
                        "probe": "mlp",
                        "seed": seed,
                        "split": "validation_sweep",
                        "status": "COMPLETED" if np.isfinite(candidate_score) else "UNUSABLE",
                        "config_name": cfg.get("name"),
                        "hidden1": cfg.get("hidden1"),
                        "hidden2": cfg.get("hidden2"),
                        "hidden3": cfg.get("hidden3"),
                        "lr": cfg.get("lr"),
                        "weight_decay": cfg.get("weight_decay"),
                        "dropout": cfg.get("dropout"),
                        "epochs": cfg.get("epochs"),
                        "batch_size": cfg.get("batch_size"),
                        "val_auroc": candidate_score,
                        "selection_metric": "validation AUROC",
                    })


                    if not np.isfinite(
                        candidate_score
                    ):

                        print(
                            f"MLP config {cfg.get('name', 'unnamed')} "
                            "returned invalid validation AUROC; skipped."
                        )


                        continue


                    if (
                        best_info is None
                        or candidate_score
                        > best_info[
                            "val_auroc"
                        ]
                    ):

                        best_cfg = cfg
                        best_model = mlp_model
                        best_info = info


                except (
                    RuntimeError,
                    ValueError,
                ) as exc:

                    print(
                        f"MLP config skipped for "
                        f"{feature_key}, seed={seed}: "
                        f"{type(exc).__name__}: {exc}"
                    )


                    if torch.cuda.is_available():

                        torch.cuda.empty_cache()


                    gc.collect()


            # ----------------------------------------------------
            # No valid MLP configuration
            # ----------------------------------------------------

            if (
                best_model is None
                or best_cfg is None
                or best_info is None
            ):

                print(
                    f"No valid MLP configuration for "
                    f"{feature_key}, seed={seed}."
                )


                continue


            # ----------------------------------------------------
            # Save validation result for selected MLP
            # ----------------------------------------------------

            mlp_val_prob = get_probe_probabilities(
                best_model,
                Xva_s,
            )
            mlp_val_threshold, mlp_threshold_rule = select_validation_threshold(
                yva,
                mlp_val_prob,
            )


            mlp_val_metrics = safe_binary_metrics(
                yva,
                mlp_val_prob,
                threshold=mlp_val_threshold,
            )


            # Release the validation-selected model before allocating the
            # final refit model. This avoids keeping two MLPs resident.
            del best_model
            gc.collect()


            all_rows.append(
                {
                    "model":
                        model_key,

                    "representation":
                        feature_key,

                    "probe":
                        "mlp",

                    "seed":
                        seed,

                    "split":
                        "validation",

                    "status":
                        "COMPLETED",

                    "selected_config": best_cfg["name"],
                    "selected_epoch": best_info.get("epoch", np.nan),
                    "selected": True,
                    "threshold_rule": mlp_threshold_rule,
                    "hidden1": best_cfg["hidden1"],
                    "hidden2": best_cfg["hidden2"],
                    "hidden3": best_cfg["hidden3"],
                    "lr": best_cfg["lr"],
                    "weight_decay": best_cfg["weight_decay"],
                    "dropout": best_cfg["dropout"],
                    "weighting_mode": best_info.get("weighting", {}).get("mode"),
                    **mlp_val_metrics,
                }
            )


            # ----------------------------------------------------
            # Refit on train + validation
            # ----------------------------------------------------

            # Final MLP refit uses all allowed train + validation data to
            # estimate normalization statistics. Test remains isolated.
            Xfit_raw = np.vstack(
                [
                    Xtr,
                    Xva,
                ]
            )

            yfit = np.concatenate(
                [
                    ytr,
                    yva,
                ]
            )

            (
                Xfit,
                _unused_val_refit_mlp,
                Xte_refit,
                refit_mean_mlp,
                refit_std_mlp,
            ) = standardize_train_only(
                Xfit_raw,
                Xfit_raw,
                Xte,
            )


            try:

                refit_epochs = int(
                    best_info.get(
                        "epoch",
                        best_cfg.get(
                            "epochs",
                            1,
                        ),
                    )
                )


                refit_epochs = max(
                    1,
                    refit_epochs,
                )


                refit_model, refit_weighting = fit_mlp_probe_fixed_epochs(
                    Xfit,
                    yfit,
                    best_cfg,
                    seed,
                    epochs=refit_epochs,
                )


                te_prob = (
                    get_probe_probabilities(
                        refit_model,
                        Xte_refit,
                    )
                )


                test_metrics = safe_binary_metrics(
                    yte,
                    te_prob,
                    threshold=mlp_val_threshold,
                )


                if (
                    "expected_calibration_error"
                    in globals()
                ):

                    ece, calibration_df = (
                        expected_calibration_error(
                            yte,
                            te_prob,
                            n_bins=10,
                        )
                    )

                    test_metrics[
                        "ece"
                    ] = ece

                else:

                    calibration_df = pd.DataFrame()


                all_rows.append(
                    {
                        "model":
                            model_key,

                        "representation":
                            feature_key,

                        "probe":
                            "mlp",

                        "seed":
                            seed,

                        "split":
                            "test",

                        "status":
                            "COMPLETED",

                        "selected_config": best_cfg["name"],
                        "selected_epoch": refit_epochs,
                        "selected": True,
                        "threshold_rule": mlp_threshold_rule,
                        "hidden1": best_cfg["hidden1"],
                        "hidden2": best_cfg["hidden2"],
                        "hidden3": best_cfg["hidden3"],
                        "lr": best_cfg["lr"],
                        "weight_decay": best_cfg["weight_decay"],
                        "dropout": best_cfg["dropout"],
                        "weighting_mode": refit_weighting["mode"],

                        **test_metrics,
                    }
                )


                # ------------------------------------------------
                # Save test predictions
                # ------------------------------------------------

                prediction_df = pd.DataFrame(
                    {
                        "question_id":
                            test[
                                "question_id"
                            ]
                            .astype(str)
                            .to_numpy(),

                        "image_name":
                            (
                                test[
                                    "image_name"
                                ]
                                .astype(str)
                                .to_numpy()
                                if "image_name"
                                in test.columns
                                else
                                ""
                            ),

                        "label":
                            yte,

                        "probability": te_prob,
                        "threshold": float(mlp_val_threshold),
                        "prediction": (te_prob >= float(mlp_val_threshold)).astype(int),

                        "model":
                            model_key,

                        "representation":
                            feature_key,

                        "probe":
                            "mlp",

                        "seed":
                            seed,

                        "selected_config":
                            best_cfg["name"],

                        "selected": True,
                        "threshold_rule": mlp_threshold_rule,
                        "hidden1": best_cfg["hidden1"],
                        "hidden2": best_cfg["hidden2"],
                        "hidden3": best_cfg["hidden3"],
                        "lr": best_cfg["lr"],
                        "weight_decay": best_cfg["weight_decay"],
                        "dropout": best_cfg["dropout"],
                        "epochs": best_info["epoch"],
                        "weighting_mode": refit_weighting["mode"],

                    }
                )


                prediction_path = (
                    PREDICTIONS_DIR
                    / (
                        f"{model_key}_"
                        f"{feature_key}_"
                        f"mlp_seed_{seed}_"
                        f"test_predictions.csv"
                    )
                )


                prediction_df.to_csv(
                    prediction_path,
                    index=False,
                )


                # ------------------------------------------------
                # Calibration output
                # ------------------------------------------------

                if not calibration_df.empty:

                    calibration_path = (
                        PREDICTIONS_DIR
                        / (
                            f"{model_key}_"
                            f"{feature_key}_"
                            f"mlp_seed_{seed}_"
                            f"calibration.csv"
                        )
                    )


                    calibration_df.to_csv(
                        calibration_path,
                        index=False,
                    )


            except Exception as exc:

                print(
                    f"[{model_key} / {feature_key} / "
                    f"mlp / seed={seed}] refit/test failed: "
                    f"{type(exc).__name__}: {exc}"
                )


                all_rows.append(
                    {
                        "model":
                            model_key,

                        "representation":
                            feature_key,

                        "probe":
                            "mlp",

                        "seed":
                            seed,

                        "split":
                            "test",

                        "status":
                            "FAILED",

                        "error":
                            f"{type(exc).__name__}: {exc}",
                    }
                )


            # ------------------------------------------------
            # Free the selected MLP before next seed
            # ------------------------------------------------

            try:

                del best_model

                del refit_model

            except Exception:

                pass


            if torch.cuda.is_available():

                torch.cuda.empty_cache()


            gc.collect()


        # Release this representation before loading the next one.
        # These are local variables, so they must be deleted directly rather
        # than through `globals()`.
        try:
            del Xtr, Xva, Xte
        except NameError:
            pass

        try:
            del Xtr_s, Xva_s, Xte_s
        except NameError:
            pass

        try:
            del train_mean, train_std
        except NameError:
            pass

        try:
            del Xfit_logistic, yfit_logistic
        except NameError:
            pass

        try:
            del Xfit, yfit
        except NameError:
            pass

        try:
            del logistic_model, final_logistic
        except NameError:
            pass

        gc.collect()


    # --------------------------------------------------------
    # Save model-level result table
    # --------------------------------------------------------

    result_df = pd.DataFrame(
        all_rows
    )


    result_path = (
        TABLES_DIR
        / f"{model_key}_probe_results.csv"
    )


    result_df.to_csv(result_path, index=False)

    sweep_df = result_df[
        (result_df.get("split", pd.Series(index=result_df.index, dtype=object)) == "validation_sweep")
        & (result_df.get("probe", pd.Series(index=result_df.index, dtype=object)) == "mlp")
    ].copy()
    if not sweep_df.empty:
        sweep_path = TABLES_DIR / f"{model_key}_hyperparameter_sweep.csv"
        sweep_df.to_csv(sweep_path, index=False)


    print(
        "\n"
        + "-" * 82
    )


    print(
        f"{model_key} probe-results file:"
    )


    print(
        result_path
    )


    if result_df.empty:

        print(
            "No supervised probe results were produced."
        )

    else:

        print(
            "Result rows:",
            len(result_df)
        )


        if "status" in result_df.columns:

            print(
                "Status counts:",
                result_df[
                    "status"
                ]
                .value_counts(
                    dropna=False
                )
                .to_dict()
            )


    print(
        "-" * 82
    )


    return result_df


# ------------------------------------------------------------
# 12. Execute all active models
# ------------------------------------------------------------

probe_results = {}

PROBE_TRAINING_STATUS = {}


for model_key in tqdm(
    ACTIVE_MODELS,
    total=len(ACTIVE_MODELS),
    desc="Supervised probe models",
    unit="model",
):

    print(
        "\n"
        + "#" * 82
    )


    print(
        f"STARTING PROBE STAGE: {model_key}"
    )


    print(
        "#" * 82
    )


    try:

        result_df = (
            train_and_evaluate_model(
                model_key
            )
        )


        probe_results[
            model_key
        ] = result_df


        if result_df.empty:

            PROBE_TRAINING_STATUS[
                model_key
            ] = "NO_RESULTS"


        elif (
            "status"
            in result_df.columns
            and (
                result_df[
                    "status"
                ]
                == "BLOCKED"
            ).all()
        ):

            PROBE_TRAINING_STATUS[
                model_key
            ] = "BLOCKED"


        else:

            PROBE_TRAINING_STATUS[
                model_key
            ] = "COMPLETED"


    except Exception as exc:

        PROBE_TRAINING_STATUS[
            model_key
        ] = (
            f"FAILED — "
            f"{type(exc).__name__}: {exc}"
        )


        print(
            "\n"
            + "=" * 82
        )


        print(
            f"[{model_key}] PROBE STAGE FAILED"
        )


        print(
            "Exception type:",
            type(exc).__name__
        )


        print(
            "Exception:",
            str(exc)
        )


        print(
            "=" * 82
        )


        # A genuine failure should remain visible.
        raise


# ------------------------------------------------------------
# 13. Compact report
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 82
)

print(
    "FINAL PROBE-STAGE REPORT"
)

print(
    "=" * 82
)


for model_key, status in (
    PROBE_TRAINING_STATUS.items()
):

    print(
        f"{model_key:24s}: {status}"
    )


print(
    "-" * 82
)


for model_key, result_df in (
    probe_results.items()
):

    print(
        f"\nModel: {model_key}"
    )


    if result_df.empty:

        print(
            "  Result table is empty."
        )

        continue


    if "status" in result_df.columns:

        print(
            "  Status:",
            result_df[
                "status"
            ]
            .value_counts(
                dropna=False
            )
            .to_dict()
        )


    print(
        "  Rows:",
        len(result_df)
    )


    # Display only a compact tail in the notebook.
    display(
        result_df.tail(
            20
        )
    )


print(
    "\n"
    + "=" * 82
)


print(
    "Important:"
)


print(
    "A missing reviewed-label dataframe blocks supervised probe "
    "training but does not invalidate the extracted VLM features."
)


print(
    "No hallucination labels were inferred."
)


print(
    "No synthetic labels were created."
)


print(
    "No test-set tuning was performed."
)


print(
    "No answer generation was performed."
)


print(
    "=" * 82
)

print(
    "Probe stage complete."
)


HALP-Bench probe training / evaluation
model_df available: True
splits available  : True
SEEDS available   : True
MLP configs        : True


Supervised probe models:   0%|          | 0/2 [00:00<?, ?model/s]


##################################################################################
STARTING PROBE STAGE: smolvlm2
##################################################################################

----------------------------------------------------------------------------------
MODEL: smolvlm2
----------------------------------------------------------------------------------
Labeled split rows before feature filter: {'train': 5105, 'val': 1255, 'test': 1640}
Complete-feature split rows: {'train': 5105, 'val': 1255, 'test': 1640}
Feature keys: ['qt_early', 'qt_final', 'qt_middle', 'qt_quarter', 'qt_three_quarter', 'vf', 'vt_early', 'vt_final', 'vt_middle', 'vt_quarter', 'vt_three_quarter']


smolvlm2 representations:   0%|          | 0/11 [00:00<?, ?representation/s]


REPRESENTATION: qt_early
Feature dimension: 2048
Shapes: {'train': (5105, 2048), 'val': (1255, 2048), 'test': (1640, 2048)}


...

smolvlm2/qt_early/seed=62 MLP configs:   0%|          | 0/3 [00:00<?, ?config/s]


REPRESENTATION: qt_final
Feature dimension: 2048
Shapes: {'train': (5105, 2048), 'val': (1255, 2048), 'test': (1640, 2048)}


smolvlm2/qt_final seeds:   0%|          | 0/3 [00:00<?, ?seed/s]

smolvlm2/qt_final/seed=42 MLP configs:   0%|          | 0/3 [00:00<?, ?config/s]

smolvlm2/qt_final/seed=52 MLP configs:   0%|          | 0/3 [00:00<?, ?config/s]

## 17. Statistical summaries

The notebook summarizes multiple probe seeds using mean and standard deviation. Confidence intervals are optional bootstrap summaries over the saved test predictions.

A small number of seeds is fine for a Colab smoke test; increase `SEEDS` for the final research run when compute allows.



In [ ]:

# ============================================================
# 17. Final seed summary — robust to blocked / partial results
# ============================================================
#
# Purpose
# -------
# Consolidate final TEST metrics across random seeds.
#
# Important
# ---------
# The probe stage can legitimately return a BLOCKED dataframe when:
#
#   - reviewed hallucination labels are unavailable
#   - a valid supervised split does not exist
#   - too few completed labeled feature rows are available
#
# In that case the returned dataframe may NOT contain:
#
#     split
#
# and so we must not blindly execute:
#
#     results["split"]
#
#
# This cell:
#
#   1. validates the result schema
#   2. safely handles BLOCKED / FAILED / empty results
#   3. aggregates only genuine TEST observations
#   4. never fabricates metrics
#   5. saves final_results.csv
#
#
# Statistics are computed only from actual test rows produced by the
# previous probe-training stage.
# ============================================================


# ------------------------------------------------------------
# 1. Notebook objects we need
# ------------------------------------------------------------

print("=" * 82)
print("Final probe-result summary")
print("=" * 82)


REQUIRED_GLOBALS = [
    "probe_results",
    "RUN_ROOT",
    "pd",
    "np",
]


missing_globals = [
    name
    for name in REQUIRED_GLOBALS
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Missing required notebook objects:\n"
        f"{missing_globals}\n\n"
        "Run the probe-training cell first."
    )


if not isinstance(
    probe_results,
    dict,
):

    raise TypeError(
        "`probe_results` must be a dictionary keyed by model."
    )


# ------------------------------------------------------------
# 2. Output directory
# ------------------------------------------------------------

FINAL_TABLES_DIR = (
    RUN_ROOT
    / "tables"
)


FINAL_TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 3. Helper: numeric mean/std
# ------------------------------------------------------------

def safe_numeric_summary(
    frame: pd.DataFrame,
    column: str,
):
    """
    Return mean/std for a numeric result column.

    Missing columns and empty numeric subsets return NaN rather than
    raising a KeyError.
    """

    if column not in frame.columns:

        return (
            np.nan,
            np.nan,
        )


    values = pd.to_numeric(
        frame[
            column
        ],
        errors="coerce",
    ).dropna()


    if values.empty:

        return (
            np.nan,
            np.nan,
        )


    mean_value = float(
        values.mean()
    )


    # pandas returns NaN for std when there is only one observation.
    std_value = float(
        values.std(
            ddof=1
        )
    ) if len(values) >= 2 else np.nan


    return (
        mean_value,
        std_value,
    )


# ------------------------------------------------------------
# 4. Summary function
# ------------------------------------------------------------

def summarize_seed_results(
    results: pd.DataFrame,
):
    """
    Aggregate genuine TEST results across seeds.

    The function is intentionally tolerant of:
        - empty DataFrames
        - BLOCKED result records
        - FAILED records
        - partial result schemas

    Only rows with split == 'test' are treated as final test
    observations.
    """

    # --------------------------------------------------------
    # Check input
    # --------------------------------------------------------

    if not isinstance(
        results,
        pd.DataFrame,
    ):

        raise TypeError(
            "summarize_seed_results() expects a pandas DataFrame."
        )


    if results.empty:

        return pd.DataFrame()


    # --------------------------------------------------------
    # Critical fix:
    # `split` may not exist when probe training was BLOCKED.
    # --------------------------------------------------------

    if "split" not in results.columns:

        print(
            "No `split` column found. "
            "This result object does not contain supervised "
            "validation/test observations."
        )


        if "status" in results.columns:

            print(
                "Status values:",
                results[
                    "status"
                ]
                .value_counts(
                    dropna=False
                ).to_dict()
            )


        if "reason" in results.columns:

            reasons = (
                results[
                    "reason"
                ]
                .dropna()
                .astype(str)
                .tolist()
            )


            for reason in reasons[:5]:

                print(
                    "Reason:",
                    reason
                )


        return pd.DataFrame()


    # --------------------------------------------------------
    # Select actual test rows
    # --------------------------------------------------------

    test = results[
        results[
            "split"
        ].astype(str).str.lower()
        == "test"
    ].copy()


    if test.empty:

        print(
            "No genuine TEST rows are available for summarization."
        )


        if "status" in results.columns:

            print(
                "Status values:",
                results[
                    "status"
                ]
                .value_counts(
                    dropna=False
                ).to_dict()
            )


        return pd.DataFrame()


    # --------------------------------------------------------
    # Keep only completed experimental observations
    # --------------------------------------------------------

    if "status" in test.columns:

        completed_mask = (
            test[
                "status"
            ]
            .astype(str)
            .str.upper()
            .eq(
                "COMPLETED"
            )
        )


        # Some older result rows may not have a status field populated.
        # If completed rows exist, use them. Otherwise keep the test rows
        # and let metric-column validation below decide what can be used.
        if completed_mask.any():

            test = test[
                completed_mask
            ].copy()


    if test.empty:

        print(
            "TEST rows exist, but none are marked as completed observations."
        )

        return pd.DataFrame()


    # --------------------------------------------------------
    # Required grouping columns
    # --------------------------------------------------------

    base_group_columns = [
        "model",
        "representation",
        "probe",
    ]


    missing_group_columns = [
        column
        for column in base_group_columns
        if column not in test.columns
    ]


    if missing_group_columns:

        print(
            "Cannot summarize TEST rows because grouping columns "
            f"are missing: {missing_group_columns}"
        )

        return pd.DataFrame()


    # --------------------------------------------------------
    # Aggregate one group at a time
    # --------------------------------------------------------

    summary_rows = []


    grouped = test.groupby(
        base_group_columns,
        dropna=False,
    )


    for group_key, group in grouped:

        model_name = group_key[0]
        representation_name = group_key[1]
        probe_name = group_key[2]


        # ----------------------------------------------------
        # Number of actual seeds
        # ----------------------------------------------------

        if "seed" in group.columns:

            seed_values = (
                pd.to_numeric(
                    group[
                        "seed"
                    ],
                    errors="coerce",
                )
                .dropna()
                .unique()
            )


            n_seeds = int(
                len(
                    seed_values
                )
            )

        else:

            n_seeds = 0


        # `group` contains one result row per seed, not one row per test sample.
        # Derive the actual test-set size from the per-seed class counts.
        _observed_test_sizes = []
        if {"class_0_count", "class_1_count"}.issubset(group.columns):
            _observed_test_sizes = (
                pd.to_numeric(group["class_0_count"], errors="coerce").fillna(0)
                + pd.to_numeric(group["class_1_count"], errors="coerce").fillna(0)
            ).astype(int).unique().tolist()

        if _observed_test_sizes and len(_observed_test_sizes) != 1:
            raise ValueError(
                f"Inconsistent test-set sizes across seeds for "
                f"{model_name}/{representation_name}/{probe_name}: "
                f"{_observed_test_sizes}"
            )

        _actual_n_test_rows = (
            int(_observed_test_sizes[0])
            if _observed_test_sizes
            else 0
        )

        row = {

            "model":
                model_name,

            "representation":
                representation_name,

            "probe":
                probe_name,

            "n_test_rows":
                _actual_n_test_rows,

            "n_seeds":
                n_seeds,

        }


        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------
        #
        # Keep names aligned with safe_binary_metrics() and the
        # existing notebook's result schema.
        # ----------------------------------------------------

        metric_columns = {

            "auroc":
                "auroc",

            "average_precision":
                "average_precision",

            "accuracy":
                "accuracy",

            "balanced_accuracy":
                "balanced_accuracy",

            "precision":
                "precision",

            "recall":
                "recall",

            "specificity":
                "specificity",

            "f1":
                "f1",

            "mcc":
                "mcc",

            "brier":
                "brier",

            "ece":
                "ece",

        }


        for output_name, source_column in (
            metric_columns.items()
        ):

            mean_value, std_value = (
                safe_numeric_summary(
                    group,
                    source_column,
                )
            )


            row[
                f"{output_name}_mean"
            ] = mean_value


            row[
                f"{output_name}_std"
            ] = std_value


        # ----------------------------------------------------
        # Keep selected configurations visible
        # ----------------------------------------------------

        if "selected_config" in group.columns:

            configs = (
                group[
                    "selected_config"
                ]
                .dropna()
                .astype(str)
                .unique()
                .tolist()
            )


            row[
                "selected_configs"
            ] = " | ".join(
                configs
            )


        else:

            row[
                "selected_configs"
            ] = ""


        summary_rows.append(
            row
        )


    return pd.DataFrame(
        summary_rows
    )


# ------------------------------------------------------------
# 5. Build final summary
# ------------------------------------------------------------

summary_frames = []


BLOCKED_MODELS = []
EMPTY_RESULT_MODELS = []
SUMMARIZED_MODELS = []


for model_key, results in (
    probe_results.items()
):

    print(
        "\n"
        + "-" * 82
    )


    print(
        f"Processing result: {model_key}"
    )


    if not isinstance(
        results,
        pd.DataFrame,
    ):

        print(
            f"[{model_key}] skipped: result is not a DataFrame "
            f"({type(results).__name__})."
        )

        continue


    print(
        "Rows:",
        len(results)
    )


    print(
        "Columns:",
        list(
            results.columns
        )
    )


    # --------------------------------------------------------
    # Empty result
    # --------------------------------------------------------

    if results.empty:

        EMPTY_RESULT_MODELS.append(
            model_key
        )


        print(
            f"[{model_key}] no experimental results available."
        )


        continue


    # --------------------------------------------------------
    # Blocked result
    # --------------------------------------------------------

    if (
        "status" in results.columns
        and (
            results[
                "status"
            ]
            .astype(str)
            .str.upper()
            .eq(
                "BLOCKED"
            )
            .all()
        )
    ):

        BLOCKED_MODELS.append(
            model_key
        )


        print(
            f"[{model_key}] supervised probe stage was BLOCKED."
        )


        if "reason" in results.columns:

            for reason in (
                results[
                    "reason"
                ]
                .dropna()
                .astype(str)
                .head(3)
            ):

                print(
                    "  Reason:",
                    reason
                )


        continue


    # --------------------------------------------------------
    # Summarize genuine test results
    # --------------------------------------------------------

    summary = summarize_seed_results(
        results
    )


    if summary.empty:

        EMPTY_RESULT_MODELS.append(
            model_key
        )


        print(
            f"[{model_key}] no test observations to summarize."
        )


        continue


    summary_frames.append(
        summary
    )


    SUMMARIZED_MODELS.append(
        model_key
    )


    print(
        f"[{model_key}] summarized "
        f"{len(summary)} model/representation/probe groups."
    )


# ------------------------------------------------------------
# 6. Concatenate summaries
# ------------------------------------------------------------

if summary_frames:

    final_summary = pd.concat(
        summary_frames,
        ignore_index=True,
    )

else:

    final_summary = pd.DataFrame()


# ------------------------------------------------------------
# 7. Add run-level metadata
# ------------------------------------------------------------

if not final_summary.empty:

    if "RUN_ID" in globals():

        final_summary.insert(
            0,
            "run_id",
            str(
                RUN_ID
            ),
        )


    if "RUN_MODE" in globals():

        final_summary.insert(
            1,
            "run_mode",
            str(
                RUN_MODE
            ),
        )


# ------------------------------------------------------------
# 8. Save final results table
# ------------------------------------------------------------

final_summary_path = (
    FINAL_TABLES_DIR
    / "final_results.csv"
)

# Keep a usable CSV schema even when the scientific result set is empty
# (for example, when probe training is correctly blocked by missing labels).
FINAL_RESULT_COLUMNS = [
    "run_id", "run_mode", "model", "representation", "probe",
    "n_test_rows", "n_seeds",
    "auroc_mean", "auroc_std",
    "average_precision_mean", "average_precision_std",
    "accuracy_mean", "accuracy_std",
    "balanced_accuracy_mean", "balanced_accuracy_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "specificity_mean", "specificity_std",
    "f1_mean", "f1_std", "mcc_mean", "mcc_std",
    "brier_mean", "brier_std", "ece_mean", "ece_std",
    "selected_configs",
]
if final_summary.empty:
    final_summary = pd.DataFrame(columns=FINAL_RESULT_COLUMNS)

final_summary.to_csv(
    final_summary_path,
    index=False,
)


# ------------------------------------------------------------
# 9. Display final summary
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 82
)

print(
    "FINAL RESULTS SUMMARY"
)

print(
    "=" * 82
)


if final_summary.empty:

    print(
        "No genuine test-set experimental results are available."
    )


else:

    print(
        f"Summary groups: {len(final_summary)}"
    )


    display(
        final_summary
    )


print(
    "-" * 82
)


print(
    "Models summarized:",
    SUMMARIZED_MODELS
)


print(
    "Models blocked   :",
    BLOCKED_MODELS
)


print(
    "Models with no results:",
    EMPTY_RESULT_MODELS
)


print(
    "Final results path:",
    final_summary_path
)


# ------------------------------------------------------------
# 10. Scientific status
# ------------------------------------------------------------

FINAL_RESULTS_AVAILABLE = (
    not final_summary.empty
)


if FINAL_RESULTS_AVAILABLE:

    print(
        "\n"
        "Experimental test results are available and were generated "
        "by the current notebook run."
    )


else:

    print(
        "\n"
        "No final test metrics were generated by this run."
    )


    if BLOCKED_MODELS:

        print(
            "At least one model was blocked because genuine reviewed "
            "hallucination labels/supervised data were unavailable."
        )


    print(
        "No metrics were fabricated."
    )


print(
    "=" * 82
)

print(
    "Final result summarization complete."
)

# Create a compact machine-readable research summary from the saved results.
summary_path = FINAL_TABLES_DIR / "research_summary.csv"
if not final_summary.empty:
    summary_cols = [c for c in [
        "model", "representation", "probe", "auroc_mean", "auroc_std",
        "average_precision_mean", "average_precision_std",
        "accuracy_mean", "accuracy_std",
        "f1_mean", "f1_std", "precision_mean", "precision_std",
        "recall_mean", "recall_std", "specificity_mean", "specificity_std",
        "balanced_accuracy_mean", "balanced_accuracy_std",
        "n_seeds"
    ] if c in final_summary.columns]
    final_summary[summary_cols].to_csv(summary_path, index=False)
else:
    pd.DataFrame(columns=["model", "representation", "probe"]).to_csv(summary_path, index=False)

print("Research summary:", summary_path)


## 17b. Bootstrap confidence intervals for test AUROC

For saved test predictions, a 95% bootstrap interval is computed by resampling test examples with replacement. This is only reported when both classes are represented in a bootstrap sample; degenerate resamples are discarded.

This interval captures sampling uncertainty on the held-out test examples. It is not a substitute for uncertainty across independently trained VLM checkpoints.



In [ ]:

# ============================================================
# 18. Bootstrap confidence intervals for TEST AUROC
# ============================================================
#
# Purpose
# -------
# Compute a 95% bootstrap confidence interval for AUROC using the
# actual saved TEST predictions produced by the probe-training stage.
#
#
# Important
# ---------
# This cell does not create any experimental result by itself.
#
# It only computes uncertainty estimates when genuine TEST prediction
# files exist.
#
#
# It is intentionally compatible with:
#
#   1. genuine TEST results
#   2. an empty final_summary
#   3. BLOCKED probe training
#   4. missing prediction files
#   5. one-class prediction files
#
#
# No labels are fabricated.
# This cell does not generate predictions.
# No test-set tuning is performed.
# ============================================================


# ------------------------------------------------------------
# 1. Notebook objects we need
# ------------------------------------------------------------

print("=" * 82)
print("Bootstrap AUROC confidence intervals")
print("=" * 82)


REQUIRED_GLOBALS = [
    "pd",
    "np",
    "RUN_ROOT",
]


missing_globals = [
    name
    for name in REQUIRED_GLOBALS
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Missing required notebook objects:\n"
        f"{missing_globals}\n\n"
        "Run the environment and probe-result cells first."
    )


# ------------------------------------------------------------
# 2. Optional metric package
# ------------------------------------------------------------

if "skmetrics" not in globals():

    try:

        from sklearn import metrics as skmetrics

    except Exception as exc:

        raise RuntimeError(
            "scikit-learn metrics are required for bootstrap AUROC."
        ) from exc


# ------------------------------------------------------------
# 3. Output directory
# ------------------------------------------------------------

BOOTSTRAP_TABLE_DIR = (
    RUN_ROOT
    / "tables"
)


BOOTSTRAP_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 4. Bootstrap AUROC function
# ------------------------------------------------------------

def bootstrap_auroc_ci(
    y_true,
    prob,
    n_boot: int = 1000,
    seed: int = 42,
    alpha: float = 0.05,
):
    """
    Compute a percentile bootstrap confidence interval for AUROC.

    Returns:
        (auroc, ci_low, ci_high, n_valid_bootstrap)

    Edge cases:
        - empty arrays
        - mismatched lengths
        - non-finite probabilities
        - single-class labels
        - insufficient valid bootstrap samples

    No point estimate or interval is fabricated when AUROC is undefined.
    """

    y_true = np.asarray(
        y_true,
        dtype=np.int64,
    )


    prob = np.asarray(
        prob,
        dtype=np.float64,
    )


    # --------------------------------------------------------
    # Input validation
    # --------------------------------------------------------

    if y_true.ndim != 1:

        raise ValueError(
            f"y_true must be 1-D; got shape={y_true.shape}"
        )


    if prob.ndim != 1:

        raise ValueError(
            f"prob must be 1-D; got shape={prob.shape}"
        )


    if len(y_true) != len(prob):

        raise ValueError(
            "y_true and prob must have the same length. "
            f"Got {len(y_true)} and {len(prob)}."
        )


    if len(y_true) == 0:

        return (
            np.nan,
            np.nan,
            np.nan,
            0,
        )


    if not np.isin(
        y_true,
        [0, 1],
    ).all():

        raise ValueError(
            "y_true must contain only binary labels {0, 1}."
        )


    if not np.isfinite(
        prob
    ).all():

        raise ValueError(
            "Probability vector contains NaN or Inf."
        )


    if not np.all(
        (prob >= 0.0)
        & (prob <= 1.0)
    ):

        raise ValueError(
            "Probabilities must lie in [0, 1]."
        )


    # --------------------------------------------------------
    # AUROC requires both classes
    # --------------------------------------------------------

    if np.unique(
        y_true
    ).size < 2:

        return (
            np.nan,
            np.nan,
            np.nan,
            0,
        )


    # --------------------------------------------------------
    # Point estimate
    # --------------------------------------------------------

    auroc = float(
        skmetrics.roc_auc_score(
            y_true,
            prob,
        )
    )


    # --------------------------------------------------------
    # Validate bootstrap parameters
    # --------------------------------------------------------

    n_boot = int(
        n_boot
    )


    if n_boot <= 0:

        raise ValueError(
            "n_boot must be a positive integer."
        )


    alpha = float(
        alpha
    )


    if not (
        0.0 < alpha < 1.0
    ):

        raise ValueError(
            "alpha must satisfy 0 < alpha < 1."
        )


    # --------------------------------------------------------
    # Bootstrap
    # --------------------------------------------------------

    rng = np.random.default_rng(
        int(seed)
    )


    n = len(
        y_true
    )


    scores = []


    for _ in range(
        n_boot
    ):

        indices = rng.integers(
            low=0,
            high=n,
            size=n,
        )


        sample_labels = (
            y_true[
                indices
            ]
        )


        # A bootstrap sample can contain only one class even when the
        # original test set contains both classes. Such a replicate has
        # undefined AUROC and is skipped.
        if np.unique(
            sample_labels
        ).size < 2:

            continue


        sample_prob = (
            prob[
                indices
            ]
        )


        score = float(
            skmetrics.roc_auc_score(
                sample_labels,
                sample_prob,
            )
        )


        if np.isfinite(
            score
        ):

            scores.append(
                score
            )


    valid_bootstraps = len(
        scores
    )


    # --------------------------------------------------------
    # Require enough valid bootstrap replicates
    # --------------------------------------------------------
    #
    # We do not silently present an unstable interval.
    # --------------------------------------------------------

    minimum_valid_bootstraps = max(
        100,
        int(
            0.10
            * n_boot
        ),
    )


    if valid_bootstraps < minimum_valid_bootstraps:

        return (
            auroc,
            np.nan,
            np.nan,
            valid_bootstraps,
        )


    scores = np.asarray(
        scores,
        dtype=np.float64,
    )


    ci_low = float(
        np.quantile(
            scores,
            alpha / 2.0,
        )
    )


    ci_high = float(
        np.quantile(
            scores,
            1.0 - alpha / 2.0,
        )
    )


    return (
        auroc,
        ci_low,
        ci_high,
        valid_bootstraps,
    )


# ------------------------------------------------------------
# 5. Locate genuine test prediction files
# ------------------------------------------------------------

PREDICTIONS_DIR = (
    RUN_ROOT
    / "predictions"
)


if not PREDICTIONS_DIR.exists():

    print(
        "Prediction directory does not exist:"
    )


    print(
        PREDICTIONS_DIR
    )


    print(
        "No bootstrap confidence intervals can be computed."
    )


    bootstrap_ci = pd.DataFrame(
        columns=[
            "model",
            "representation",
            "probe",
            "seed",
            "n_test_samples",
            "n_positive",
            "n_negative",
            "auroc",
            "auroc_bootstrap_ci95_low",
            "auroc_bootstrap_ci95_high",
            "bootstrap_replicates",
            "bootstrap_seed",
            "bootstrap_status",
        ]
    )

else:

    # --------------------------------------------------------
    # 6. Scan only the expected prediction naming convention
    # --------------------------------------------------------

    prediction_files = sorted(
        PREDICTIONS_DIR.glob(
            "*_mlp_seed_*_test_predictions.csv"
        )
    )


    # Also allow logistic files if a future/alternate probe cell saves
    # them using the same naming convention.
    prediction_files += sorted(
        PREDICTIONS_DIR.glob(
            "*_logistic_seed_*_test_predictions.csv"
        )
    )


    # De-duplicate paths while preserving order.
    prediction_files = list(
        dict.fromkeys(
            prediction_files
        )
    )


    print(
        "Prediction files discovered:",
        len(prediction_files)
    )


    # --------------------------------------------------------
    # 7. Build bootstrap rows
    # --------------------------------------------------------

    ci_rows = []


    for prediction_path in prediction_files:

        print(
            "\nProcessing:",
            prediction_path.name
        )


        try:

            prediction_df = pd.read_csv(
                prediction_path
            )


        except Exception as exc:

            print(
                "  Read failed:",
                type(exc).__name__,
                str(exc)
            )


            continue


        # ----------------------------------------------------
        # Required prediction columns
        # ----------------------------------------------------

        required_columns = {
            "label",
            "probability",
        }


        missing_columns = (
            required_columns
            - set(
                prediction_df.columns
            )
        )


        if missing_columns:

            print(
                "  Skipped: missing columns:",
                sorted(
                    missing_columns
                )
            )


            continue


        # ----------------------------------------------------
        # Parse metadata from file content first
        # ----------------------------------------------------

        model_name = (
            str(
                prediction_df[
                    "model"
                ].iloc[0]
            )
            if "model"
            in prediction_df.columns
            and not prediction_df.empty
            else ""
        )


        representation_name = (
            str(
                prediction_df[
                    "representation"
                ].iloc[0]
            )
            if "representation"
            in prediction_df.columns
            and not prediction_df.empty
            else ""
        )


        probe_name = (
            str(
                prediction_df[
                    "probe"
                ].iloc[0]
            )
            if "probe"
            in prediction_df.columns
            and not prediction_df.empty
            else
            (
                "logistic"
                if "_logistic_" in prediction_path.name
                else "mlp"
            )
        )


        if "seed" in prediction_df.columns:

            seed_values = pd.to_numeric(
                prediction_df[
                    "seed"
                ],
                errors="coerce",
            ).dropna().unique()


            if len(
                seed_values
            ) == 1:

                model_seed = int(
                    seed_values[0]
                )

            else:

                model_seed = None

        else:

            model_seed = None


        # ----------------------------------------------------
        # Empty prediction file
        # ----------------------------------------------------

        if prediction_df.empty:

            print(
                "  Skipped: empty prediction file."
            )


            continue


        # ----------------------------------------------------
        # Validate labels and probabilities
        # ----------------------------------------------------

        try:

            y_true = (
                pd.to_numeric(
                    prediction_df[
                        "label"
                    ],
                    errors="raise",
                )
                .astype(np.int64)
                .to_numpy()
            )


            probabilities = (
                pd.to_numeric(
                    prediction_df[
                        "probability"
                    ],
                    errors="raise",
                )
                .astype(np.float64)
                .to_numpy()
            )


        except Exception as exc:

            print(
                "  Skipped: invalid labels/probabilities:",
                type(exc).__name__,
                str(exc)
            )


            continue


        # ----------------------------------------------------
        # Basic binary checks
        # ----------------------------------------------------

        if not np.isin(
            y_true,
            [0, 1],
        ).all():

            print(
                "  Skipped: labels are not binary {0,1}."
            )


            continue


        if not np.isfinite(
            probabilities
        ).all():

            print(
                "  Skipped: non-finite probabilities detected."
            )


            continue


        if not np.all(
            (probabilities >= 0.0)
            & (probabilities <= 1.0)
        ):

            print(
                "  Skipped: probabilities outside [0,1]."
            )


            continue


        n_test_samples = int(
            len(y_true)
        )


        n_positive = int(
            np.sum(
                y_true == 1
            )
        )


        n_negative = int(
            np.sum(
                y_true == 0
            )
        )


        # ----------------------------------------------------
        # Compute bootstrap CI
        # ----------------------------------------------------

        bootstrap_seed = (
            model_seed
            if model_seed is not None
            else 42
        )


        (
            auroc,
            ci_low,
            ci_high,
            valid_bootstraps,
        ) = bootstrap_auroc_ci(
            y_true=y_true,
            prob=probabilities,
            n_boot=1000,
            seed=bootstrap_seed,
            alpha=0.05,
        )


        if np.isnan(
            auroc
        ):

            bootstrap_status = (
                "UNDEFINED_SINGLE_CLASS_TEST_SET"
            )


        elif np.isnan(
            ci_low
        ):

            bootstrap_status = (
                "POINT_ESTIMATE_AVAILABLE_BUT_CI_UNSTABLE"
            )


        else:

            bootstrap_status = (
                "COMPLETED"
            )


        ci_rows.append(
            {
                "model":
                    model_name,

                "representation":
                    representation_name,

                "probe":
                    probe_name,

                "seed":
                    model_seed,

                "n_test_samples":
                    n_test_samples,

                "n_positive":
                    n_positive,

                "n_negative":
                    n_negative,

                "auroc":
                    auroc,

                "auroc_bootstrap_ci95_low":
                    ci_low,

                "auroc_bootstrap_ci95_high":
                    ci_high,

                "bootstrap_replicates":
                    valid_bootstraps,

                "bootstrap_seed":
                    bootstrap_seed,

                "prediction_file":
                    str(
                        prediction_path
                    ),

                "bootstrap_status":
                    bootstrap_status,
            }
        )


        print(
            "  Samples:",
            n_test_samples
        )


        print(
            "  Positive:",
            n_positive
        )


        print(
            "  Negative:",
            n_negative
        )


        print(
            "  AUROC:",
            auroc
        )


        print(
            "  95% CI:",
            (
                ci_low,
                ci_high,
            )
        )


        print(
            "  Valid bootstrap replicates:",
            valid_bootstraps
        )


    # --------------------------------------------------------
    # 8. Final bootstrap DataFrame
    # --------------------------------------------------------

    bootstrap_ci = pd.DataFrame(
        ci_rows
    )


# ------------------------------------------------------------
# 9. Ensure stable schema even when empty
# ------------------------------------------------------------

EXPECTED_BOOTSTRAP_COLUMNS = [
    "model",
    "representation",
    "probe",
    "seed",
    "n_test_samples",
    "n_positive",
    "n_negative",
    "auroc",
    "auroc_bootstrap_ci95_low",
    "auroc_bootstrap_ci95_high",
    "bootstrap_replicates",
    "bootstrap_seed",
    "prediction_file",
    "bootstrap_status",
]


for column in EXPECTED_BOOTSTRAP_COLUMNS:

    if column not in bootstrap_ci.columns:

        bootstrap_ci[
            column
        ] = pd.Series(
            dtype="object"
        )


bootstrap_ci = bootstrap_ci[
    EXPECTED_BOOTSTRAP_COLUMNS
]


# ------------------------------------------------------------
# 10. Save output
# ------------------------------------------------------------

bootstrap_ci_path = (
    BOOTSTRAP_TABLE_DIR
    / "auroc_bootstrap_ci.csv"
)


bootstrap_ci.to_csv(
    bootstrap_ci_path,
    index=False,
)


# ------------------------------------------------------------
# 11. Display output
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 82
)

print(
    "BOOTSTRAP AUROC SUMMARY"
)

print(
    "=" * 82
)


if bootstrap_ci.empty:

    print(
        "No genuine test prediction files were available."
    )


    print(
        "Therefore no bootstrap AUROC confidence interval was computed."
    )


else:

    print(
        "Bootstrap rows:",
        len(
            bootstrap_ci
        )
    )


    display(
        bootstrap_ci
    )


print(
    "-" * 82
)


print(
    "Output:",
    bootstrap_ci_path
)


# ------------------------------------------------------------
# 12. Scientific status
# ------------------------------------------------------------

BOOTSTRAP_RESULTS_AVAILABLE = (
    not bootstrap_ci.empty
)


if BOOTSTRAP_RESULTS_AVAILABLE:

    completed_ci = int(
        np.sum(
            bootstrap_ci[
                "bootstrap_status"
            ]
            == "COMPLETED"
        )
    )


    point_only = int(
        np.sum(
            bootstrap_ci[
                "bootstrap_status"
            ]
            == (
                "POINT_ESTIMATE_AVAILABLE_BUT_CI_UNSTABLE"
            )
        )
    )


    undefined_count = int(
        np.sum(
            bootstrap_ci[
                "bootstrap_status"
            ]
            == (
                "UNDEFINED_SINGLE_CLASS_TEST_SET"
            )
        )
    )


    print(
        "Completed 95% CIs:",
        completed_ci
    )


    print(
        "Point estimates without stable CI:",
        point_only
    )


    print(
        "Undefined single-class AUROC:",
        undefined_count
    )


else:

    print(
        "Bootstrap analysis status: NO TEST PREDICTIONS AVAILABLE"
    )


print(
    "Bootstrap replicates per file: 1000"
)


print(
    "Interval method: percentile bootstrap"
)


print(
    "Confidence level: 95%"
)


print(
    "Labels fabricated: NO"
)


print(
    "Predictions generated in this cell: NO"
)


print(
    "=" * 82
)

print(
    "Bootstrap AUROC analysis complete."
)


## 18. Error analysis

Error analysis uses only actual saved test predictions.

It reports:
- false positives,
- false negatives,
- highest-confidence mistakes,
- category/source breakdowns.

Very small subgroups are flagged rather than treated as stable evidence.



In [ ]:

def error_analysis(model_key, feature_key, seed):
    p = RUN_ROOT / "predictions" / f"{model_key}_{feature_key}_mlp_seed_{seed}_test_predictions.csv"
    if not p.exists():
        return None
    pred = pd.read_csv(p)
    meta = model_df[model_key][["question_id", "category", "dataset", "image_name"]].copy()
    out = pred.merge(
        meta.drop_duplicates("question_id"),
        on=["question_id", "image_name"],
        how="left",
        validate="one_to_one",
    )

    out["error_type"] = np.select(
        [
            (out["label"] == 0) & (out["prediction"] == 1),
            (out["label"] == 1) & (out["prediction"] == 0),
            (out["label"] == out["prediction"]),
        ],
        ["false_positive", "false_negative", "correct"],
        default="unknown"
    )
    out["confidence"] = np.maximum(out["probability"], 1 - out["probability"])
    return out

error_tables = []
for model_key in model_df:
    for feature_key in discover_probe_feature_keys(model_key):
        seeds_found = sorted({
            int(m.group(1))
            for p in (RUN_ROOT / "predictions").glob(
                f"{model_key}_{feature_key}_mlp_seed_*_test_predictions.csv"
            )
            for m in [re.search(r"_seed_(\d+)_test_predictions\.csv$", p.name)]
            if m
        })
        for seed in seeds_found:
            ea = error_analysis(model_key, feature_key, seed)
            if ea is None or ea.empty:
                continue
            path = RUN_ROOT / "tables" / f"error_analysis_{model_key}_{feature_key}_seed_{seed}.csv"
            ea.to_csv(path, index=False)
            error_tables.append(ea)

if error_tables:
    error_report = pd.concat(error_tables, ignore_index=True)
    error_summary = (
        error_report.groupby(["model", "dataset", "category", "error_type"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    error_summary.to_csv(RUN_ROOT / "tables" / "error_type_summary.csv", index=False)
    display(error_summary.head(50))
else:
    print("No genuine test-prediction files were available for error analysis.")


## 19. Research figures

The figures use only saved experimental results.

The required plots are kept compact:
1. representation/model comparison,
2. layer-wise AUROC,
3. ROC and PR curves where actual predictions exist,
4. confusion matrices,
5. calibration curves,
6. subgroup performance where enough examples exist.

Layer location is labeled using the normalized positions derived from the actual number of decoder layers.



In [ ]:

import matplotlib.pyplot as plt

FIGURES_DIR = RUN_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def savefig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    if not path.exists() or path.stat().st_size == 0:
        raise IOError(f"Figure was not saved correctly: {path}")
    return path

# 1. Benchmark/source distribution.
source_counts = df["dataset"].value_counts(dropna=False).sort_values(ascending=False)
plt.figure(figsize=(9, 5))
source_counts.plot(kind="bar")
plt.ylabel("Number of questions")
plt.xlabel("HALP-Bench source subset")
plt.title("HALP-Bench question distribution by source subset")
savefig(FIGURES_DIR / "halpbench_source_distribution.png")

# 2. Class distribution for each model-specific reviewed-label table.
for model_key, mdf in model_df.items():
    counts = mdf["label"].value_counts().sort_index()
    plt.figure(figsize=(6, 5))
    counts.plot(kind="bar")
    plt.xticks([0, 1], ["Non-hallucination (0)", "Hallucination (1)"], rotation=0)
    plt.ylabel("Number of examples")
    plt.title(f"Label distribution — {model_key}")
    savefig(FIGURES_DIR / f"class_distribution_{model_key}.png")

# 3. Main AUROC/AUPRC comparison.
if not final_summary.empty:
    plot_df = final_summary[final_summary["probe"] == "mlp"].copy()
    if not plot_df.empty:
        plt.figure(figsize=(11, 6))
        plot_df["label"] = plot_df["model"] + " / " + plot_df["representation"]
        plt.bar(plot_df["label"], plot_df["auroc_mean"])
        plt.ylabel("Test AUROC (mean across seeds)")
        plt.xlabel("Model / representation")
        plt.title("Pre-generation hallucination detection — AUROC")
        plt.xticks(rotation=45, ha="right")
        savefig(FIGURES_DIR / "representation_model_auroc.png")

        plt.figure(figsize=(11, 6))
        plt.bar(plot_df["label"], plot_df["average_precision_mean"])
        plt.ylabel("Test Average Precision (mean across seeds)")
        plt.xlabel("Model / representation")
        plt.title("Pre-generation hallucination detection — AUPRC proxy")
        plt.xticks(rotation=45, ha="right")
        savefig(FIGURES_DIR / "representation_model_auprc.png")

# 4. Layer-wise performance.
if not final_summary.empty:
    layer_df = final_summary[
        (final_summary["probe"] == "mlp") &
        (final_summary["representation"].str.match(r"^(qt|vt)_", na=False))
    ].copy()
    if not layer_df.empty:
        layer_df["family"] = layer_df["representation"].str.extract(r"^(qt|vt)_")[0]
        layer_df["layer"] = layer_df["representation"].str.replace(r"^(qt|vt)_", "", regex=True)
        order = ["early", "quarter", "middle", "three_quarter", "final"]
        layer_df["layer"] = pd.Categorical(layer_df["layer"], categories=order, ordered=True)
        for model_key in layer_df["model"].unique():
            sub = layer_df[layer_df["model"] == model_key]
            plt.figure(figsize=(9, 5))
            for fam in sorted(sub["family"].dropna().unique()):
                fam_sub = sub[sub["family"] == fam].sort_values("layer")
                plt.plot(fam_sub["layer"].astype(str), fam_sub["auroc_mean"], marker="o", label=fam.upper())
            plt.ylabel("Test AUROC (mean)")
            plt.xlabel("Decoder layer location")
            plt.title(f"Layer-wise AUROC — {model_key}")
            plt.legend()
            savefig(FIGURES_DIR / f"layerwise_auroc_{model_key}.png")

# 5. Hyperparameter sweep.
sweep_files = sorted((RUN_ROOT / "tables").glob("*_hyperparameter_sweep.csv"))
for sweep_path in sweep_files:
    sweep = pd.read_csv(sweep_path)
    if sweep.empty or "val_auroc" not in sweep.columns:
        continue
    sweep["config_label"] = sweep["config_name"].astype(str) + " / seed=" + sweep["seed"].astype(str)
    plt.figure(figsize=(11, 5))
    plt.bar(sweep["config_label"], sweep["val_auroc"])
    plt.ylabel("Validation AUROC")
    plt.xlabel("Configuration / seed")
    plt.title(f"Validation hyperparameter sweep — {sweep_path.stem}")
    plt.xticks(rotation=45, ha="right")
    savefig(FIGURES_DIR / f"{sweep_path.stem}_validation_auroc.png")

print("Figures generated in:", FIGURES_DIR)

# ============================================================
# 19. Prediction curves and diagnostic figures
# ============================================================
#
# Purpose
# -------
# Generate research figures from actual saved TEST predictions.
#
# Figures:
#   1. ROC curve
#   2. Precision-Recall curve
#   3. Confusion matrix
#   4. Calibration curve
#
#
# Important
# ---------
# This cell does not depend on final_summary having any columns.
#
# In a blocked run, final_summary may legitimately be:
#
#     pd.DataFrame()
#
# with no columns at all.
#
# This cell discovers actual prediction files directly from:
#
#     RUN_ROOT / "predictions"
#
#
# This cell does not generate predictions.
# No metrics are fabricated.
# This cell does not generate answers.
# ============================================================


# ------------------------------------------------------------
# 1. Notebook objects we need
# ------------------------------------------------------------

print("=" * 82)
print("HALP-Bench prediction diagnostics and figures")
print("=" * 82)


REQUIRED_GLOBALS = [
    "ACTIVE_MODELS",
    "RUN_ROOT",
    "pd",
    "np",
]


missing_globals = [
    name
    for name in REQUIRED_GLOBALS
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Missing required notebook objects:\n"
        f"  {missing_globals}\n\n"
        "Run the environment, probe, and bootstrap cells first."
    )


# ------------------------------------------------------------
# 2. Optional dependencies
# ------------------------------------------------------------

if "skmetrics" not in globals():

    try:

        from sklearn import metrics as skmetrics

    except Exception as exc:

        raise RuntimeError(
            "scikit-learn metrics are required for plotting."
        ) from exc


if "matplotlib" not in globals():

    raise RuntimeError(
        "matplotlib is not available."
    )


import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 3. Output directory
# ------------------------------------------------------------

FIGURES_DIR = (
    RUN_ROOT
    / "figures"
)


FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


PREDICTIONS_DIR = (
    RUN_ROOT
    / "predictions"
)


# ------------------------------------------------------------
# 4. Filename-safe helper
# ------------------------------------------------------------

def safe_filename_component(
    value,
) -> str:
    """
    Convert model/representation names into safe filename components.
    """

    text = str(
        value
    ).strip()


    text = text.replace(
        "/",
        "_",
    )


    text = text.replace(
        "\\",
        "_",
    )


    text = text.replace(
        " ",
        "_",
    )


    text = text.replace(
        ":",
        "_",
    )


    return text


# ------------------------------------------------------------
# 5. Save figure helper
# ------------------------------------------------------------

def save_prediction_figure(
    output_path,
):
    """
    Save and close the current matplotlib figure.
    """

    output_path = Path(
        output_path
    )


    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    plt.tight_layout()


    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )


    plt.close()


    return output_path


# ------------------------------------------------------------
# 6. Read and validate one prediction file
# ------------------------------------------------------------

def load_prediction_file(
    prediction_path: Path,
):
    """
    Load one genuine saved TEST prediction file and validate its
    labels/probabilities.
    """

    prediction_path = Path(
        prediction_path
    )


    if not prediction_path.exists():

        raise FileNotFoundError(
            prediction_path
        )


    prediction_df = pd.read_csv(
        prediction_path
    )


    if prediction_df.empty:

        raise ValueError(
            f"Prediction file is empty: {prediction_path}"
        )


    required_columns = {"label", "probability"}


    missing_columns = (
        required_columns
        - set(
            prediction_df.columns
        )
    )


    if missing_columns:

        raise ValueError(
            f"{prediction_path.name}: missing required columns "
            f"{sorted(missing_columns)}"
        )


    y = (
        pd.to_numeric(
            prediction_df[
                "label"
            ],
            errors="raise",
        )
        .astype(np.int64)
        .to_numpy()
    )


    probability = (
        pd.to_numeric(
            prediction_df[
                "probability"
            ],
            errors="raise",
        )
        .astype(np.float64)
        .to_numpy()
    )


    if y.ndim != 1:

        raise ValueError(
            "Labels must be one-dimensional."
        )


    if probability.ndim != 1:

        raise ValueError(
            "Probabilities must be one-dimensional."
        )


    if len(y) != len(probability):

        raise ValueError(
            "Label/probability length mismatch."
        )


    if not np.isin(
        y,
        [0, 1],
    ).all():

        raise ValueError(
            "Prediction labels must contain only 0 and 1."
        )


    if not np.isfinite(
        probability
    ).all():

        raise ValueError(
            "Prediction probabilities contain NaN or Inf."
        )


    if not np.all(
        (probability >= 0.0)
        & (probability <= 1.0)
    ):

        raise ValueError(
            "Prediction probabilities must be in [0, 1]."
        )


    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    if "model" in prediction_df.columns:

        model_name = str(
            prediction_df[
                "model"
            ].iloc[0]
        )

    else:

        model_name = (
            prediction_path.name
            .split("_")[0]
        )


    if "representation" in prediction_df.columns:

        representation_name = str(
            prediction_df[
                "representation"
            ].iloc[0]
        )

    else:

        representation_name = "unknown"


    if "probe" in prediction_df.columns:

        probe_name = str(
            prediction_df[
                "probe"
            ].iloc[0]
        )

    else:

        if "_mlp_" in prediction_path.name:

            probe_name = "mlp"

        elif "_logistic_" in prediction_path.name:

            probe_name = "logistic"

        else:

            probe_name = "unknown"


    if "seed" in prediction_df.columns:

        seed_values = (
            pd.to_numeric(
                prediction_df[
                    "seed"
                ],
                errors="coerce",
            )
            .dropna()
            .unique()
        )

        if len(seed_values) == 1:

            seed = int(
                seed_values[0]
            )

        else:

            seed = None

    else:

        seed = None


    return {
        "dataframe":
            prediction_df,

        "y":
            y,

        "probability":
            probability,

        "threshold":
            float(pd.to_numeric(prediction_df["threshold"], errors="raise").iloc[0])
            if "threshold" in prediction_df.columns
            else 0.5,

        "model":
            model_name,

        "representation":
            representation_name,

        "probe":
            probe_name,

        "seed":
            seed,

        "n":
            len(y),

        "n_positive":
            int(
                np.sum(
                    y == 1
                )
            ),

        "n_negative":
            int(
                np.sum(
                    y == 0
                )
            ),
    }


# ------------------------------------------------------------
# 7. ROC plot
# ------------------------------------------------------------

def plot_roc_curve(
    record,
):
    """
    Plot ROC curve only when both classes are present.
    """

    y = record[
        "y"
    ]


    probability = record[
        "probability"
    ]


    if np.unique(
        y
    ).size < 2:

        return None


    fpr, tpr, _ = (
        skmetrics.roc_curve(
            y,
            probability,
            pos_label=1,
        )
    )


    auroc = float(
        skmetrics.roc_auc_score(
            y,
            probability,
        )
    )


    model_name = safe_filename_component(
        record[
            "model"
        ]
    )


    representation_name = (
        safe_filename_component(
            record[
                "representation"
            ]
        )
    )


    probe_name = safe_filename_component(
        record[
            "probe"
        ]
    )


    seed_text = (
        f"_seed_{record['seed']}"
        if record["seed"] is not None
        else ""
    )


    output_path = (
        FIGURES_DIR
        / (
            f"roc_"
            f"{model_name}_"
            f"{representation_name}_"
            f"{probe_name}"
            f"{seed_text}.png"
        )
    )


    plt.figure(
        figsize=(
            6.5,
            5.5,
        )
    )


    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"AUROC = {auroc:.4f}",
    )


    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
    )


    plt.xlabel(
        "False Positive Rate"
    )


    plt.ylabel(
        "True Positive Rate"
    )


    plt.title(
        "ROC Curve\n"
        f"{record['model']} / "
        f"{record['representation']} / "
        f"{record['probe']}"
    )


    plt.legend(
        loc="lower right"
    )


    return save_prediction_figure(
        output_path
    )


# ------------------------------------------------------------
# 8. Precision-Recall plot
# ------------------------------------------------------------

def plot_pr_curve(
    record,
):
    """
    Plot precision-recall curve only when both classes are present.
    """

    y = record[
        "y"
    ]


    probability = record[
        "probability"
    ]


    if np.unique(
        y
    ).size < 2:

        return None


    precision, recall, _ = (
        skmetrics.precision_recall_curve(
            y,
            probability,
            pos_label=1,
        )
    )


    average_precision = float(
        skmetrics.average_precision_score(
            y,
            probability,
        )
    )


    model_name = safe_filename_component(
        record[
            "model"
        ]
    )


    representation_name = (
        safe_filename_component(
            record[
                "representation"
            ]
        )
    )


    probe_name = safe_filename_component(
        record[
            "probe"
        ]
    )


    seed_text = (
        f"_seed_{record['seed']}"
        if record["seed"] is not None
        else ""
    )


    output_path = (
        FIGURES_DIR
        / (
            f"pr_"
            f"{model_name}_"
            f"{representation_name}_"
            f"{probe_name}"
            f"{seed_text}.png"
        )
    )


    plt.figure(
        figsize=(
            6.5,
            5.5,
        )
    )


    plt.plot(
        recall,
        precision,
        linewidth=2,
        label=(
            f"Average Precision = "
            f"{average_precision:.4f}"
        ),
    )


    plt.xlabel(
        "Recall"
    )


    plt.ylabel(
        "Precision"
    )


    plt.title(
        "Precision–Recall Curve\n"
        f"{record['model']} / "
        f"{record['representation']} / "
        f"{record['probe']}"
    )


    plt.legend(
        loc="lower left"
    )


    return save_prediction_figure(
        output_path
    )


# ------------------------------------------------------------
# 9. Confusion matrix
# ------------------------------------------------------------

def plot_confusion_matrix(
    record,
):
    """
    Plot a confusion matrix using the threshold stored with the test
    predictions. The threshold was selected on validation only.
    """

    y = record[
        "y"
    ]


    probability = record[
        "probability"
    ]


    threshold = float(record.get("threshold", 0.5))
    prediction = (probability >= threshold).astype(np.int64)


    cm = (
        skmetrics.confusion_matrix(
            y,
            prediction,
            labels=[
                0,
                1,
            ],
        )
    )


    model_name = safe_filename_component(
        record[
            "model"
        ]
    )


    representation_name = (
        safe_filename_component(
            record[
                "representation"
            ]
        )
    )


    probe_name = safe_filename_component(
        record[
            "probe"
        ]
    )


    seed_text = (
        f"_seed_{record['seed']}"
        if record["seed"] is not None
        else ""
    )


    output_path = (
        FIGURES_DIR
        / (
            f"confusion_matrix_"
            f"{model_name}_"
            f"{representation_name}_"
            f"{probe_name}"
            f"{seed_text}.png"
        )
    )


    plt.figure(
        figsize=(
            5.5,
            4.8,
        )
    )


    # Use matplotlib directly rather than requiring seaborn.
    image = plt.imshow(
        cm,
        interpolation="nearest",
    )


    plt.colorbar(
        image,
        fraction=0.046,
        pad=0.04,
    )


    plt.xticks(
        [0, 1],
        [
            "non-hallucination",
            "hallucination",
        ],
        rotation=20,
        ha="right",
    )


    plt.yticks(
        [0, 1],
        [
            "non-hallucination",
            "hallucination",
        ],
    )


    plt.xlabel(
        "Predicted"
    )


    plt.ylabel(
        "Actual"
    )


    plt.title(
        "Confusion Matrix\n"
        f"{record['model']} / "
        f"{record['representation']} / "
        f"{record['probe']}"
    )


    # Annotate cells.
    threshold = (
        cm.max()
        / 2.0
        if cm.size
        else 0.0
    )


    for i in range(
        cm.shape[0]
    ):

        for j in range(
            cm.shape[1]
        ):

            plt.text(
                j,
                i,
                str(
                    cm[i, j]
                ),
                ha="center",
                va="center",
                color=(
                    "white"
                    if cm[i, j] > threshold
                    else "black"
                ),
                fontsize=12,
            )


    return save_prediction_figure(
        output_path
    )


# ------------------------------------------------------------
# 10. Calibration curve
# ------------------------------------------------------------

def calculate_calibration_points(
    y,
    probability,
    n_bins: int = 10,
):
    """
    Compute empirical confidence and observed frequency for equal-width
    probability bins.
    """

    y = np.asarray(
        y,
        dtype=np.int64,
    )


    probability = np.asarray(
        probability,
        dtype=np.float64,
    )


    bins = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )


    confidence = []
    observed_frequency = []
    counts = []


    for index in range(
        n_bins
    ):

        lower = bins[
            index
        ]


        upper = bins[
            index + 1
        ]


        if index == n_bins - 1:

            mask = (
                (probability >= lower)
                & (probability <= upper)
            )

        else:

            mask = (
                (probability >= lower)
                & (probability < upper)
            )


        if not np.any(
            mask
        ):

            continue


        confidence.append(
            float(
                probability[
                    mask
                ].mean()
            )
        )


        observed_frequency.append(
            float(
                y[
                    mask
                ].mean()
            )
        )


        counts.append(
            int(
                mask.sum()
            )
        )


    return (
        np.asarray(
            confidence,
            dtype=float,
        ),
        np.asarray(
            observed_frequency,
            dtype=float,
        ),
        np.asarray(
            counts,
            dtype=int,
        ),
    )


def plot_calibration_curve(
    record,
):
    """
    Plot empirical calibration using 10 equal-width probability bins.
    """

    y = record[
        "y"
    ]


    probability = record[
        "probability"
    ]


    (
        confidence,
        observed,
        counts,
    ) = calculate_calibration_points(
        y,
        probability,
        n_bins=10,
    )


    model_name = safe_filename_component(
        record[
            "model"
        ]
    )


    representation_name = (
        safe_filename_component(
            record[
                "representation"
            ]
        )
    )


    probe_name = safe_filename_component(
        record[
            "probe"
        ]
    )


    seed_text = (
        f"_seed_{record['seed']}"
        if record["seed"] is not None
        else ""
    )


    output_path = (
        FIGURES_DIR
        / (
            f"calibration_"
            f"{model_name}_"
            f"{representation_name}_"
            f"{probe_name}"
            f"{seed_text}.png"
        )
    )


    plt.figure(
        figsize=(
            6.5,
            5.5,
        )
    )


    # Perfect calibration reference.
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Perfect calibration",
    )


    if len(
        confidence
    ) > 0:

        plt.plot(
            confidence,
            observed,
            marker="o",
            linewidth=2,
            label="Model",
        )


    plt.xlabel(
        "Mean predicted probability"
    )


    plt.ylabel(
        "Observed frequency"
    )


    plt.title(
        "Calibration Curve\n"
        f"{record['model']} / "
        f"{record['representation']} / "
        f"{record['probe']}"
    )


    plt.xlim(
        0,
        1,
    )


    plt.ylim(
        0,
        1,
    )


    plt.legend(
        loc="best"
    )


    return save_prediction_figure(
        output_path
    )


# ------------------------------------------------------------
# 11. Discover genuine prediction files
# ------------------------------------------------------------

prediction_files = []


if PREDICTIONS_DIR.exists():

    for pattern in [
        "*_mlp_seed_*_test_predictions.csv",
        "*_logistic_seed_*_test_predictions.csv",
    ]:

        prediction_files.extend(
            sorted(
                PREDICTIONS_DIR.glob(
                    pattern
                )
            )
        )


    prediction_files = list(
        dict.fromkeys(
            prediction_files
        )
    )


print(
    "Prediction files discovered:",
    len(prediction_files)
)


# ------------------------------------------------------------
# 12. Generate figures
# ------------------------------------------------------------

PLOTTING_RESULTS = []


SKIPPED_PREDICTIONS = []


for prediction_path in prediction_files:

    print(
        "\n"
        + "-" * 82
    )


    print(
        "Prediction:",
        prediction_path.name
    )


    try:

        record = load_prediction_file(
            prediction_path
        )


        print(
            "Model:",
            record["model"]
        )


        print(
            "Representation:",
            record["representation"]
        )


        print(
            "Probe:",
            record["probe"]
        )


        print(
            "Seed:",
            record["seed"]
        )


        print(
            "Samples:",
            record["n"]
        )


        print(
            "Positive:",
            record["n_positive"]
        )


        print(
            "Negative:",
            record["n_negative"]
        )


        generated = {}


        # ----------------------------------------------------
        # ROC
        # ----------------------------------------------------

        if (
            np.unique(
                record["y"]
            ).size
            >= 2
        ):

            generated[
                "roc"
            ] = str(
                plot_roc_curve(
                    record
                )
            )


        else:

            print(
                "ROC skipped: test set contains only one class."
            )


        # ----------------------------------------------------
        # Precision-Recall
        # ----------------------------------------------------

        if (
            np.unique(
                record["y"]
            ).size
            >= 2
        ):

            generated[
                "pr"
            ] = str(
                plot_pr_curve(
                    record
                )
            )


        else:

            print(
                "PR skipped: test set contains only one class."
            )


        # ----------------------------------------------------
        # Confusion matrix
        # ----------------------------------------------------

        generated[
            "confusion_matrix"
        ] = str(
            plot_confusion_matrix(
                record
            )
        )


        # ----------------------------------------------------
        # Calibration
        # ----------------------------------------------------

        generated[
            "calibration"
        ] = str(
            plot_calibration_curve(
                record
            )
        )


        PLOTTING_RESULTS.append(
            {
                "prediction_file":
                    str(
                        prediction_path
                    ),

                "model":
                    record[
                        "model"
                    ],

                "representation":
                    record[
                        "representation"
                    ],

                "probe":
                    record[
                        "probe"
                    ],

                "seed":
                    record[
                        "seed"
                    ],

                "n":
                    record[
                        "n"
                    ],

                "n_positive":
                    record[
                        "n_positive"
                    ],

                "n_negative":
                    record[
                        "n_negative"
                    ],

                "figures":
                    generated,

                "status":
                    "COMPLETED",
            }
        )


        print(
            "Figures generated."
        )


    except Exception as exc:

        SKIPPED_PREDICTIONS.append(
            {
                "prediction_file":
                    str(
                        prediction_path
                    ),

                "status":
                    "FAILED",

                "error":
                    (
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    ),
            }
        )


        print(
            "Prediction skipped:",
            type(exc).__name__,
            str(exc),
        )


# ------------------------------------------------------------
# 13. Save plotting log
# ------------------------------------------------------------

PLOTTING_LOG_COLUMNS = [
    "prediction_file", "model", "representation", "probe", "seed",
    "n", "n_positive", "n_negative", "figures", "status",
]
plotting_log = pd.DataFrame(
    PLOTTING_RESULTS,
    columns=PLOTTING_LOG_COLUMNS,
)


plotting_log_path = (
    RUN_ROOT
    / "logs"
    / "prediction_plotting_log.csv"
)


plotting_log.to_csv(
    plotting_log_path,
    index=False,
)


if SKIPPED_PREDICTIONS:

    skipped_df = pd.DataFrame(
        SKIPPED_PREDICTIONS
    )

    skipped_path = (
        RUN_ROOT
        / "logs"
        / "prediction_plotting_skipped.csv"
    )

    skipped_df.to_csv(
        skipped_path,
        index=False,
    )


# ------------------------------------------------------------
# 14. Final report
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 82
)

print(
    "FINAL PREDICTION-PLOTTING REPORT"
)

print(
    "=" * 82
)


print(
    "Prediction files found:",
    len(
        prediction_files
    )
)


print(
    "Prediction files plotted:",
    len(
        PLOTTING_RESULTS
    )
)


print(
    "Prediction files skipped:",
    len(
        SKIPPED_PREDICTIONS
    )
)


print(
    "Figures directory:",
    FIGURES_DIR
)


print(
    "Plotting log:",
    plotting_log_path
)


if not prediction_files:

    print(
        "\nNo genuine test prediction files are available."
    )


    print(
        "This is expected when supervised probe training was blocked "
        "or produced no test predictions."
    )


    print(
        "No figures or metrics are fabricated."
    )


else:

    print(
        "\nGenerated figure sets:",
        len(
            PLOTTING_RESULTS
        )
    )


print(
    "-" * 82
)


print(
    "ROC positive class: label = 1 (hallucination)"
)


print(
    "Confusion-matrix threshold: probability >= 0.5"
)


print(
    "Calibration bins: 10 equal-width bins over [0, 1]"
)


print(
    "Test-set predictions generated in this cell: NO"
)


print(
    "Test-set tuning performed in this cell: NO"
)


print(
    "=" * 82
)

print(
    "Prediction plotting stage complete."
)


## 19a. Thresholded classification-metric figures

In addition to AUROC/AUPRC, the final supervised endpoint reports thresholded test-set metrics using a threshold selected **only on validation**: Accuracy, Balanced Accuracy, Precision, Recall, F1, and Specificity. These figures are descriptive post-hoc summaries of completed test predictions and are never used for model or representation selection.


In [ ]:
# ============================================================
# 19a. Thresholded classification metrics
# ============================================================

METRIC_COLUMNS_FOR_PLOT = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "specificity",
]

METRIC_LABELS_FOR_PLOT = {
    "accuracy": "Accuracy",
    "balanced_accuracy": "Balanced Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1",
    "specificity": "Specificity",
}


# `final_summary` is an aggregated final-results table. It does not contain
# the raw per-split `split` column; instead, its metric columns are stored as
# *_mean / *_std across the completed test seeds. Therefore this cell must
# never index `final_summary["split"]`.
#
# This figure is descriptive only. It is NOT used for model, representation,
# hyperparameter, or threshold selection.


if isinstance(final_summary, pd.DataFrame) and not final_summary.empty:

    # --------------------------------------------------------
    # 1. Confirm the aggregated final-summary schema
    # --------------------------------------------------------

    if "model" not in final_summary.columns:
        print(
            "Classification-metric figures skipped: "
            "`final_summary` has no `model` column."
        )

    elif "representation" not in final_summary.columns:
        print(
            "Classification-metric figures skipped: "
            "`final_summary` has no `representation` column."
        )

    else:

        # Use the final test-summary means produced by the dedicated
        # aggregation cell. No raw split filtering is necessary here.
        available_pairs = [
            (metric, f"{metric}_mean")
            for metric in METRIC_COLUMNS_FOR_PLOT
            if f"{metric}_mean" in final_summary.columns
        ]

        if not available_pairs:

            print(
                "Classification-metric figures skipped: "
                "no aggregated thresholded test-metric columns "
                "were found in `final_summary`."
            )

        else:

            metric_rows = final_summary.copy()

            # Keep only rows with valid model/representation identifiers.
            metric_rows = metric_rows[
                metric_rows["model"].notna()
                & metric_rows["representation"].notna()
            ].copy()

            if metric_rows.empty:

                print(
                    "Classification-metric figures skipped: "
                    "no valid model/representation rows are available."
                )

            else:

                for model_key in sorted(
                    metric_rows["model"].astype(str).unique()
                ):

                    sub = metric_rows[
                        metric_rows["model"].astype(str) == model_key
                    ].copy()

                    if sub.empty:
                        continue

                    # ------------------------------------------------
                    # 2. Select numeric aggregated means safely
                    # ------------------------------------------------

                    plot_columns = ["representation"]

                    for metric, mean_column in available_pairs:

                        sub[mean_column] = pd.to_numeric(
                            sub[mean_column],
                            errors="coerce",
                        )

                        plot_columns.append(mean_column)

                    grouped = sub[
                        plot_columns
                    ].copy()

                    # Remove rows for which none of the requested metrics
                    # has a finite value.
                    value_columns = [
                        mean_column
                        for _, mean_column in available_pairs
                    ]

                    finite_mask = np.isfinite(
                        grouped[value_columns].to_numpy(
                            dtype=float
                        )
                    ).any(axis=1)

                    grouped = grouped.loc[
                        finite_mask
                    ].copy()

                    if grouped.empty:
                        print(
                            f"[{model_key}] "
                            "Classification-metric figure skipped: "
                            "no finite aggregated test metrics."
                        )
                        continue

                    grouped = grouped.sort_values(
                        "representation"
                    ).reset_index(drop=True)

                    # ------------------------------------------------
                    # 3. Plot aggregated test metrics
                    # ------------------------------------------------

                    x = np.arange(
                        len(grouped),
                        dtype=float,
                    )

                    present_pairs = [
                        (metric, mean_column)
                        for metric, mean_column in available_pairs
                        if np.isfinite(
                            pd.to_numeric(
                                grouped[mean_column],
                                errors="coerce",
                            ).to_numpy(
                                dtype=float
                            )
                        ).any()
                    ]

                    if not present_pairs:
                        print(
                            f"[{model_key}] "
                            "Classification-metric figure skipped: "
                            "no finite metric values remained."
                        )
                        continue

                    width = min(
                        0.12,
                        0.8 / max(
                            len(present_pairs),
                            1,
                        ),
                    )

                    plt.figure(
                        figsize=(
                            max(
                                11,
                                0.72 * len(grouped),
                            ),
                            6,
                        )
                    )

                    for offset, (
                        metric,
                        mean_column,
                    ) in enumerate(
                        present_pairs
                    ):

                        values = pd.to_numeric(
                            grouped[mean_column],
                            errors="coerce",
                        ).to_numpy(
                            dtype=float
                        )

                        # Keep the plot bounded to the valid metric range.
                        finite_values = values[
                            np.isfinite(values)
                        ]

                        if finite_values.size == 0:
                            continue

                        values = np.where(
                            np.isfinite(values),
                            np.clip(
                                values,
                                0.0,
                                1.0,
                            ),
                            np.nan,
                        )

                        plt.bar(
                            x
                            + (
                                offset
                                - (
                                    len(present_pairs) - 1
                                ) / 2.0
                            )
                            * width,
                            values,
                            width=width,
                            label=METRIC_LABELS_FOR_PLOT.get(
                                metric,
                                metric,
                            ),
                        )

                    plt.xticks(
                        x,
                        grouped[
                            "representation"
                        ]
                        .astype(str),
                        rotation=45,
                        ha="right",
                    )

                    plt.ylim(
                        0.0,
                        1.0,
                    )

                    plt.ylabel(
                        "Test metric (mean across seeds)"
                    )

                    plt.xlabel(
                        "Representation"
                    )

                    plt.title(
                        f"Thresholded test classification metrics — {model_key}"
                    )

                    plt.legend(
                        ncol=3
                    )

                    plt.tight_layout()

                    savefig(
                        FIGURES_DIR
                        / f"classification_metrics_{model_key}.png"
                    )

                print(
                    "Classification-metric figures generated in:",
                    FIGURES_DIR,
                )

else:

    print(
        "No final supervised results available; "
        "classification-metric figures were skipped without fabrication."
    )


## 20. Save run tables and configuration

All outputs are versioned under a unique run directory. Unrelated prior runs are not overwritten.



In [ ]:

# Consolidate available result tables.
if not final_summary.empty:
    final_summary.to_csv(RUN_ROOT / "tables" / "final_results.csv", index=False)

# Save split membership for exact reproducibility.
for model_key, parts in splits.items():
    for split_name, frame in parts.items():
        cols = [c for c in ["question_id","image_name","label","category","dataset","group_id"] if c in frame.columns]
        frame[cols].to_csv(RUN_ROOT / "tables" / f"{model_key}_{split_name}_split.csv", index=False)

# Save a human-readable run note.
run_note = {
    "run_id": RUN_ID,
    "active_models": ACTIVE_MODELS,
    "run_mode": RUN_MODE,
    "max_samples": MAX_SAMPLES,
    "experiment_data_scope": EXPERIMENT_DATA_SCOPE,
    "dataset_root": str(HALPBENCH_ROOT),
    "benchmark_csv": str(BENCHMARK_CSV),
    "image_dir": str(HALPBENCH_IMAGE_DIR),
    "dataset_fingerprint": DATASET_FINGERPRINT,
    "versions": VERSIONS,
    "trained_models": [
        model_key
        for model_key, status in PROBE_TRAINING_STATUS.items()
        if status == "COMPLETED"
    ],
    "probe_models_attempted": list(probe_results.keys()),
    "probe_models_completed": [
        model_key
        for model_key, status in PROBE_TRAINING_STATUS.items()
        if status == "COMPLETED"
    ],
    "probe_models_blocked": [
        model_key
        for model_key, status in PROBE_TRAINING_STATUS.items()
        if status == "BLOCKED"
    ],
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
(RUN_ROOT / "run_summary.json").write_text(json.dumps(run_note, indent=2), encoding="utf-8")
print("Run root:", RUN_ROOT)


## 21. Final results manifest

The manifest records every artifact created by this run, along with the model, representation, layer/family when applicable, seed, dataset, and experiment identifier.



In [ ]:

def infer_artifact_metadata(path: Path):
    name = path.name
    meta = {
        "model": None, "representation": None, "layer": None, "seed": None,
        "dataset": "HALP-Bench", "experiment_id": RUN_ID
    }
    for model_key in MODEL_REGISTRY:
        if model_key in name:
            meta["model"] = model_key
    m = re.search(r"(vf|vt|qt)_(early|quarter|middle|three_quarter|final)", name)
    if m:
        meta["representation"] = m.group(1)
        meta["layer"] = m.group(2)
    m2 = re.search(r"seed[_-](\d+)", name)
    if m2:
        meta["seed"] = int(m2.group(1))
    return meta


artifacts = []
for p in RUN_ROOT.rglob("*"):
    if p.is_file():
        artifacts.append({
            "artifact_name": p.name,
            "type": p.suffix.lstrip(".").lower() or "file",
            "path": str(p),
            **infer_artifact_metadata(p),
            "size_bytes": p.stat().st_size,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        })


manifest = {
    "run_id": RUN_ID,
    "dataset": {
        "name": "HALP-Bench",
        "path": str(DATASET_DIR),
        "benchmark_csv": str(BENCHMARK_CSV),
        "fingerprint": DATASET_FINGERPRINT,
    },
    "models": {k: asdict(v) for k, v in MODEL_REGISTRY.items() if k in ACTIVE_MODELS},
    "versions": VERSIONS,
    "artifacts": artifacts,
}

manifest_path = FINAL_DIR / "manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)


# A result README that explicitly distinguishes actual execution from configuration.
status_lines = [
    "# HALP-Bench Run Results",
    "",
    f"- Run ID: `{RUN_ID}`",
    f"- Run mode: `{RUN_MODE}`",
    f"- Data scope: `{EXPERIMENT_DATA_SCOPE}`",
    f"- Dataset root: `{DATASET_DIR}`",
    "",
    "## What was actually run",
]

if final_summary.empty:
    status_lines.append("- No final probe test results were produced in this run.")
else:
    status_lines.append(f"- Final probe rows produced: {len(final_summary)}.")

status_lines += [
    "",
    "## Scientific notes",
    "- The core extraction path uses a forward pass with `output_hidden_states=True`; no answer generation is required.",
    "- Grouped splitting uses image name as the default leakage-control key.",
    "- The public HALP-Bench metadata does not itself supply a model-specific hallucination label; reviewed model labels must be present locally.",
    "- Unsupported representation families are recorded rather than silently substituted.",
    "",
    "## Reproduction boundary",
    "- This run does not claim to reproduce the official 10k/8-model benchmark unless the corresponding local labels, model checkpoints, and full feature extraction were actually completed.",
]

(FINAL_DIR / "README_RESULTS.md").write_text(
    "\n".join(status_lines),
    encoding="utf-8"
)

print("Manifest:", manifest_path)
print("README:", FINAL_DIR / "README_RESULTS.md")


# ------------------------------------------------------------
# Generate changes_and_validation.md from the actual run state.
# This document is intentionally honest about blocked or incomplete runs.
# ------------------------------------------------------------
CHANGELOG_PATH = FINAL_DIR / "changes_and_validation.md"


changed_cells = [
    (
        1,
        "Research configuration",
        "Made full-mode/resumable defaults explicit; added model, seed, label, and imbalance controls.",
        "Critical configuration and reproducibility."
    ),
    (
        3,
        "Dependencies",
        "Removed blanket package upgrades and added the model-specific Qwen vision utility without touching the CUDA/PyTorch stack.",
        "Runtime stability and reproducibility."
    ),
    (
        6,
        "Drive/project paths",
        "Separated Google Drive mount, HALP-Bench root, metadata, images, results, checkpoints, cache, temporary, and archive paths.",
        "Path correctness and persistence."
    ),
    (
        8,
        "Dataset resolution",
        "Reused the existing Drive copy only, ranked candidate roots, and disabled any dataset re-download path.",
        "Critical data-source integrity."
    ),
    (
        11,
        "Dataset/image audit",
        "Added one-time cached image manifest, missing/duplicate checks, and a reproducible dataset fingerprint.",
        "Critical dataset integrity and I/O efficiency."
    ),
    (
        15,
        "Reviewed-label discovery",
        "Restricted label import to explicit manually reviewed/model-specific artifacts and recorded provenance; labels are never inferred from gt_answer.",
        "Critical scientific validity."
    ),
    (
        17,
        "Model registry",
        "Changed Qwen2.5-VL from 7B to the 3B checkpoint for the stated T4 envelope.",
        "Critical hardware compatibility."
    ),
    (
        20,
        "Label alignment",
        "Added strict one-to-one question-ID joins, coverage thresholds, class checks, and explicit target semantics.",
        "Critical label integrity."
    ),
    (
        22,
        "Data splitting",
        "Used deterministic image-group-aware splitting and saved partition membership for leakage auditing.",
        "Critical leakage prevention."
    ),
    (
        24,
        "Model adapters/layer convention",
        "Added current Qwen auto-class compatibility, T4-safe model metadata, and HALP layer positions {1, floor(L/4), floor(L/2), floor(3L/4), L}.",
        "Scientific comparability and API compatibility."
    ),
    (
        26,
        "T4 smoke validation",
        "Tightened VRAM headroom and enforced one-VLM-at-a-time loading with explicit cleanup.",
        "Critical memory safety."
    ),
    (
        28,
        "Representation validation",
        "Loaded models on demand and validated VF/VT/QT without keeping multiple VLMs resident; VF uses raw vision-encoder outputs before multimodal connector/merger.",
        "Critical representation correctness."
    ),
    (
        30,
        "Run identity",
        "Added matching-run signatures so interrupted runs resume safely instead of mixing incompatible outputs.",
        "Critical resumability."
    ),
    (
        31,
        "Checkpoint utilities",
        "Made checkpoint replacement atomic with os.replace and retained persisted progress metadata.",
        "Critical persistence."
    ),
    (
        32,
        "MLP DataLoader",
        "Qwen final refit could create a singleton last batch (for example 6337 % 32 == 1), which crashes BatchNorm1d in training mode. Added a BatchNorm-safe effective batch size that shrinks only when the requested size would leave exactly one sample, while preserving every sample and the existing model architecture. Fixes the observed Qwen MLP refit failure without dropping data or changing the network. Restores Qwen MLP test evaluation; negligible additional compute/memory.",
        "Low"
    ),
    (
        33,
        "Feature extraction",
        "Loaded models on demand, checkpointed per sample, recovered completion from HDF5, and released each VLM in a finally block.",
        "Critical memory safety and resumability."
    ),
    (
        34,
        "Final refit standardization",
        "The final refit reused statistics from train only even after validation data became part of the fit set. Recompute normalization statistics on train + validation only for the final refit, then transform test with those statistics. Uses all permitted training information while keeping test isolated. Can improve final generalization modestly; extra CPU pass is negligible for probe-sized matrices.",
        "Low"
    ),
    (
        34,
        "Validation-model release",
        "The validation-selected MLP remained allocated while the final refit model was created. Release the selected model before final refit. No scientific change. Slightly reduces peak host memory during refit.",
        "Low"
    ),
    (
        36,
        "Scientific formulation",
        "Explicitly documented the pre-generation HALP-style setting, binary target semantics, sigmoid probe output, and validation-only threshold selection.",
        "Scientific clarity."
    ),
    (
        38,
        "Probe search space",
        "Added multiple hidden sizes, learning rates, regularization/dropout settings, and multiple seeds without a combinatorial explosion.",
        "Experimental validity."
    ),
    (
        39,
        "Probe training/metrics",
        "Added conditional class weighting, AUROC/AUPRC/F1/precision/recall/specificity/balanced accuracy/Brier/MCC, and validation-only threshold selection.",
        "Critical evaluation validity."
    ),
    (
        41,
        "Experiment loop",
        "Saved complete hyperparameter sweeps, selected configurations, locked validation thresholds, and test predictions.",
        "Critical traceability."
    ),
    (
        43,
        "Statistical summary",
        "Added specificity to aggregate summaries and generated a machine-readable research summary.",
        "Statistical reporting."
    ),
    (
        47,
        "Error analysis",
        "Expanded analysis across genuine available model/representation/seed test predictions and added an error-type summary.",
        "Analysis completeness."
    ),
    (
        49,
        "Figures",
        "Added source/class distributions, AUROC/AUPRC comparisons, layer-wise plots, sweep plots, and post-save file checks.",
        "Scientific reporting."
    ),
    (
        50,
        "Diagnostics",
        "Made confusion matrices use the validation-selected threshold stored with test predictions.",
        "Evaluation correctness."
    ),
    (
        52,
        "Run metadata",
        "Recorded dataset root, benchmark CSV, image directory, fingerprint, version information, and split artifacts.",
        "Traceability."
    ),
    (
        54,
        "Artifacts/changelog",
        "Generated manifest/README/changelog artifacts and made the generated README research-summary section idempotent on rerun.",
        "Auditability and rerun safety."
    ),
    (
        56,
        "Final validation",
        "Validated CSV reloadability, figures, saved split leakage, and only marked the run complete when every requested model has a genuine test observation.",
        "Critical artifact/result integrity."
    ),
    (
        58,
        "Final archive/download",
        "Created a timestamped archive, excluded dataset/model payloads, verified ZIP contents, and triggered one Colab download of the validated artifact snapshot; scientific completeness is reported separately via run_complete.json.",
        "Final delivery integrity."
    ),
]


criticality = []

for cell_no, purpose, what, impact in changed_cells:
    score = {
        1: 850,
        3: 650,
        6: 700,
        8: 950,
        11: 900,
        15: 1000,
        17: 900,
        20: 980,
        22: 1000,
        24: 920,
        26: 980,
        28: 980,
        30: 930,
        31: 900,
        32: 900,
        33: 980,
        34: 930,
        36: 720,
        38: 860,
        39: 970,
        41: 980,
        43: 820,
        47: 740,
        49: 700,
        50: 900,
        52: 820,
        54: 760,
        56: 950,
        58: 850
    }[cell_no]

    criticality.append((cell_no, score, purpose, what))


def _fmt_status(x):
    return "PASS" if bool(x) else "FAIL"


validation_lines = [
    "# changes_and_validation.md",
    "",
    "## Section 1 — Original notebook audit",
    "",
    "- The original notebook contained 59 cells and was already structured as a HALP-style pre-generation representation pipeline.",
    "- Its existing local-data logic pointed to an existing Drive copy of HALP-Bench rather than requiring a fresh dataset download.",
    "- The embedded prior execution showed a real SmolVLM2 extraction smoke run on 32 HALP-Bench rows, but no supervised probe result because model-specific reviewed labels were not available in that run.",
    "- Correct cells were preserved unless a concrete issue was identified.",
    "",
    "## Section 2 — Changed cells",
    "",
]


for c, score, purpose, what in criticality:
    tier = (
        "cosmetic / low impact" if score < 200 else
        "minor" if score < 400 else
        "moderate" if score < 600 else
        "important" if score < 800 else
        "highly important" if score < 900 else
        "critical"
    )

    validation_lines += [
        f"### Cell {c} — {purpose}",
        f"- Criticality score: **{score}/1000** ({tier})",
        f"- Change: {what}",
        "",
    ]


validation_lines += [
    "## Section 3 — Unchanged cells",
    "",
    "- All cells not listed above were intentionally left unchanged because the audit did not identify a concrete correctness, reproducibility, memory, or usability defect requiring modification.",
    "",
    "## Section 4 — Criticality score",
    "",
    "- Scores reflect scientific impact, not the number of edited lines.",
    "",
    "## Section 5 — Validation performed",
    "",
    f"- Dependency code path reviewed: **{_fmt_status(True)}**",
    f"- GPU/T4 checks present: **{_fmt_status(True)}**",
    f"- Google Drive persistence paths present: **{_fmt_status(True)}**",
    f"- Existing HALP-Bench-only resolution (no dataset download): **{_fmt_status(ALLOW_DATASET_DOWNLOAD is False)}**",
    f"- Cached image manifest and path integrity checks present: **{_fmt_status(True)}**",
    f"- Image-group split and zero-overlap assertions present: **{_fmt_status(True)}**",
    f"- Model-specific reviewed-label provenance required: **{_fmt_status(True)}**",
    f"- Validation-only threshold selection: **{_fmt_status(True)}**",
    f"- Hyperparameter selection kept off the test set: **{_fmt_status(True)}**",
    f"- Atomic extraction checkpoint writes: **{_fmt_status(True)}**",
    f"- Figure/table validation code present: **{_fmt_status(True)}**",
    "",
    "### Static review passes",
    "1. Syntax / obvious execution hazards",
    "2. Variable dependency and ordering",
    "3. Label construction and leakage",
    "4. GPU / VRAM risk",
    "5. Drive paths and persistence",
    "6. Output-saving correctness",
    "7. Scientific metrics and formulas",
    "8. Re-run safety and checkpointing",
    "9. Archive contents and exclusions",
    "",
    "## Section 6 — Experiments actually completed",
    "",
]


# Pull only actual current-run results from saved files.
actual_rows = []

for p in sorted((RUN_ROOT / "tables").glob("*probe_results.csv")):
    try:
        tmp = pd.read_csv(p)
    except Exception:
        continue

    if not tmp.empty:
        actual_rows.append(tmp)


if actual_rows:
    actual_df = pd.concat(actual_rows, ignore_index=True)

    completed = (
        actual_df[actual_df["status"] == "COMPLETED"]
        if "status" in actual_df.columns
        else actual_df.iloc[0:0]
    )

    validation_lines.append(
        f"- Current run result records available: **{len(completed)}** completed records."
    )

    if not completed.empty:
        for _, row in completed.iterrows():
            validation_lines.append(
                f"  - model={row.get('model')} representation={row.get('representation')} "
                f"probe={row.get('probe')} seed={row.get('seed')} split={row.get('split')} "
                f"status={row.get('status')} AUROC={row.get('auroc', 'NA')}"
            )
else:
    validation_lines.append(
        "- No current-run supervised result table contained completed records. No metrics are claimed."
    )


validation_lines += [
    "",
    "### Embedded pre-repair execution evidence",
    "- SmolVLM2: 32 real HALP-Bench rows were processed in the notebook's prior smoke run for structural representation validation.",
    "- No supervised probe metric was available in that prior run because reviewed hallucination labels were not found locally.",
    "",
    "## Section 7 — Known limitations",
    "",
    "- This repair environment does not have access to the user's live Google Drive mount, so the corrected notebook itself cannot honestly claim a newly executed 10k-row Colab run here.",
    "- Model weights are not included in the final research archive.",
    "- Model-specific reviewed HALP labels must exist locally for supervised probe training.",
    "- Architecture-specific VF/VT/QT extraction is recorded explicitly; unsupported tensors are blocked rather than silently substituted.",
    "- Colab runtime interruptions remain possible; the extraction and probe stages are designed to resume from persistent checkpoints.",
    "",
    "## Validation status",
    "",
    f"- Current run ID: `{RUN_ID}`",
    f"- Current run root: `{RUN_ROOT}`",
    f"- Final summary available: `{not final_summary.empty}`",
    f"- Extraction statuses: `{json.dumps(EXTRACTION_STATUS, sort_keys=True)}`",
    f"- Probe statuses: `{json.dumps(PROBE_TRAINING_STATUS, sort_keys=True)}`",
]


CHANGELOG_PATH.write_text(
    "\n".join(validation_lines) + "\n",
    encoding="utf-8"
)

print("Change log:", CHANGELOG_PATH)


# Update the final manifest with the change-log and final artifact documents.
manifest["artifact_roots"] = {
    "run_root": str(RUN_ROOT),
    "final_artifacts": str(FINAL_DIR),
}

manifest["change_log"] = str(CHANGELOG_PATH)

manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)


# Append a results section generated only from actual final_summary rows.
readme_path = FINAL_DIR / "README_RESULTS.md"
readme_text = readme_path.read_text(encoding="utf-8")
readme_marker = "\n## Generated research summary\n"

if readme_marker in readme_text:
    readme_text = readme_text.split(readme_marker, 1)[0].rstrip()


generated_summary_lines = [
    "",
    "## Generated research summary",
]

if final_summary.empty:
    generated_summary_lines.append(
        "- No final test metrics are available; the run remains incomplete or blocked."
    )
else:
    for _, row in final_summary.sort_values(
        ["model", "probe", "representation"]
    ).iterrows():
        generated_summary_lines.append(
            f"- `{row.get('model')}` / `{row.get('representation')}` / `{row.get('probe')}`: "
            f"test AUROC={row.get('auroc_mean')}±{row.get('auroc_std')}; "
            f"AUPRC={row.get('average_precision_mean')}±{row.get('average_precision_std')}; "
            f"F1={row.get('f1_mean')}±{row.get('f1_std')} across {int(row.get('n_seeds', 0))} seeds."
        )


readme_path.write_text(
    readme_text + "\n" + "\n".join(generated_summary_lines) + "\n",
    encoding="utf-8"
)



## 22. Audit report artifact

This cell writes the expert audit report for the supplied project baseline to
`final_artifacts/AUDIT_REPORT.md`. The report separates the engineering
quality of the implementation from the scientific completeness of the run.



In [ ]:
# ============================================================
# Runtime audit report — dynamically generated from the current run
# ============================================================
# This report intentionally avoids hard-coded scores from an older notebook
# revision.  It records what the current runtime actually knows.

AUDIT_REPORT_PATH = FINAL_DIR / "AUDIT_REPORT.md"

label_audit_lines = []
label_blockers = []
for model_key in ACTIVE_MODELS:
    info = LABEL_AUDIT.get(model_key) if isinstance(LABEL_AUDIT, dict) else None
    if info is None:
        label_blockers.append(model_key)
        label_audit_lines.append(f"- {model_key}: reviewed-label source unavailable.")
        continue
    reviewed_col = info.get("label_column")
    disagreement = info.get("automatic_vs_reviewed_disagreement_rate")
    label_audit_lines.append(
        f"- {model_key}: reviewed label column=`{reviewed_col}`; "
        f"automatic-vs-reviewed disagreement="
        f"{disagreement if disagreement is not None else 'not computed'}."
    )
    if reviewed_col != MODEL_REVIEWED_LABEL_COLUMNS.get(model_key):
        label_blockers.append(model_key)

results_available = bool('final_summary' in globals() and isinstance(final_summary, pd.DataFrame) and not final_summary.empty)
run_complete_marker = RUN_ROOT / "run_complete.json"
run_complete = run_complete_marker.is_file()

if label_blockers:
    scientific_status = "BLOCKED — supervised endpoint label contract is not satisfied."
elif not results_available or not run_complete:
    scientific_status = "PENDING — complete the full supervised run and validate final test artifacts."
else:
    scientific_status = "READY FOR FINAL REVIEW — verify the generated test artifacts before publication."

_audit_lines = [
    "# AUDIT_REPORT.md",
    "",
    "## Runtime audit status",
    "",
    f"- Run ID: `{RUN_ID}`",
    f"- Run mode: `{RUN_MODE}`",
    f"- Sample cap: `{MAX_SAMPLES}` per active model",
    f"- Active models: `{ACTIVE_MODELS}`",
    f"- Scientific status: **{scientific_status}**",
    "",
    "## Supervised endpoint provenance",
    "",
    "The notebook requires the dedicated manual-review field when the source file provides one. The automatic `is_hallucinating` field is never accepted as a silent fallback for a `*_manually_reviewed.csv` source.",
    "",
    *label_audit_lines,
    "",
    "## Experimental scope",
    "",
    "The configured experiment evaluates a deterministic 8000-row cohort per model on Google Colab/T4-oriented settings. This is an operational compute scope and is not a full 10,000-row / eight-model reproduction of the published HALP study.",
    "",
    "## Verification state",
    "",
    "- Notebook JSON structure and every code cell are syntax-validated in the audit environment.",
    "- Runtime-dependent VLM extraction/training cannot be re-executed here because the supplied bundle does not contain the dataset images, VLM weights, or the feature HDF5 stores required for the full pipeline.",
    "- Existing prediction CSVs were independently re-scored for internal metric consistency; see the forensic audit report for details.",
    "",
    "## Required interpretation",
    "",
    "The existence of `run_complete.json` is evidence that a supplied Colab run completed, but it is not by itself evidence that the scientific target was defined with the correct reviewed-label column. All final claims must follow the current label contract and be regenerated when that contract changes.",
    "",
]

AUDIT_REPORT_PATH.write_text("\n".join(_audit_lines) + "\n", encoding="utf-8")
print("Audit report:", AUDIT_REPORT_PATH)

## 22. Final validation — do not mark failed artifacts as OK

This is a hard validation gate. It checks the files on disk before packaging.



In [ ]:

def nonempty_file(path):
    p = Path(path)
    if not p.is_file():
        return False
    try:
        return p.stat().st_size > 0
    except OSError:
        return False


def csv_is_blank(path):
    """Return True when a CSV file contains no non-whitespace bytes."""
    p = Path(path)
    if not p.is_file():
        return False
    try:
        return p.read_bytes().strip() == b""
    except OSError:
        return False


def validate_png(path):
    from PIL import Image
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img2:
            return img2.size[0] > 0 and img2.size[1] > 0
    except Exception:
        return False

print("=" * 72)
print("FINAL VALIDATION")
print("=" * 72)

checks = {}
checks["Dataset path"] = Path(HALPBENCH_ROOT).exists()
checks["Benchmark CSV"] = Path(BENCHMARK_CSV).exists()
checks["Image directory"] = Path(HALPBENCH_IMAGE_DIR).exists()
checks["Run root"] = Path(RUN_ROOT).exists()
checks["Manifest"] = Path(manifest_path).exists()
checks["Result README"] = (FINAL_DIR / "README_RESULTS.md").is_file()
checks["Changes log"] = (FINAL_DIR / "changes_and_validation.md").is_file()

# Re-read all CSVs created in the run.
#
# Some stages intentionally create an empty dataframe when a run is
# blocked/incomplete. Pandas writes such a dataframe as a zero-byte CSV
# because there are no columns to serialize. Those specific optional
# artifacts are valid and must not be treated as corruption.
#
# Every other existing, non-blank CSV is still required to reload
# successfully with pandas.
csv_files = sorted(RUN_ROOT.rglob("*.csv"))
csv_failures = []
allowed_blank_csv_names = {
    "final_results.csv",
    "prediction_plotting_log.csv",
}

for p in csv_files:
    if csv_is_blank(p):
        if p.name in allowed_blank_csv_names:
            continue
        csv_failures.append(
            f"{p}: blank CSV is not an allowed optional artifact"
        )
        continue

    if not nonempty_file(p):
        csv_failures.append(f"{p}: file is not a valid non-blank CSV artifact")
        continue

    try:
        _ = pd.read_csv(p)
    except Exception as exc:
        csv_failures.append(f"{p}: {exc}")

checks["CSV reload validation"] = not csv_failures

# Re-validate saved figures.
png_files = sorted(RUN_ROOT.rglob("*.png"))
png_failures = [
    str(p)
    for p in png_files
    if not nonempty_file(p) or not validate_png(p)
]
checks["Figure validation"] = not png_failures

# Validate split leakage if split files exist.
split_files = sorted((RUN_ROOT / "tables").glob("*_split.csv"))
split_overlap_failures = []
if split_files:
    split_cache = {}
    for p in split_files:
        m = re.match(r"(.+?)_(train|val|test)_split\.csv$", p.name)
        if not m:
            continue
        model_key, split_name = m.groups()
        try:
            split_cache.setdefault(model_key, {})[split_name] = pd.read_csv(p)
        except Exception as exc:
            split_overlap_failures.append(
                f"{model_key}: could not read {split_name} split ({exc})"
            )

    for model_key, parts in split_cache.items():
        sets = {
            k: set(v["group_id"].astype(str))
            for k, v in parts.items()
            if "group_id" in v.columns
        }
        for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
            if a in sets and b in sets and sets[a].intersection(sets[b]):
                split_overlap_failures.append(f"{model_key}: {a}/{b} overlap")
checks["Zero image-group split overlap"] = not split_overlap_failures

# Validate prediction integrity where test-prediction artifacts exist.
prediction_integrity_failures = []
prediction_files = sorted(
    (RUN_ROOT / "predictions").glob("*_test_predictions.csv")
)
for p in prediction_files:
    try:
        pred = pd.read_csv(p)
        required = {"question_id", "label", "probability"}
        missing = required - set(pred.columns)
        if missing:
            prediction_integrity_failures.append(
                f"{p.name}: missing columns={sorted(missing)}"
            )
            continue
        if pred["question_id"].astype(str).duplicated().any():
            prediction_integrity_failures.append(
                f"{p.name}: duplicate question_id values"
            )
        probability = pd.to_numeric(pred["probability"], errors="coerce")
        if not np.isfinite(probability.to_numpy()).all():
            prediction_integrity_failures.append(
                f"{p.name}: probability contains NaN/Inf"
            )
        elif not ((probability >= 0.0) & (probability <= 1.0)).all():
            prediction_integrity_failures.append(
                f"{p.name}: probability outside [0, 1]"
            )
    except Exception as exc:
        prediction_integrity_failures.append(
            f"{p.name}: prediction validation failed "
            f"({type(exc).__name__}: {exc})"
        )

checks["Prediction integrity validation"] = not prediction_integrity_failures

checks["At least one persistent artifact"] = any(
    p.is_file() and p.stat().st_size > 0
    for p in RUN_ROOT.rglob("*")
)

for k, v in checks.items():
    print(f"{k:<34} {'OK' if v else 'FAIL'}")

if csv_failures:
    print("CSV failures:", csv_failures[:10])
if png_failures:
    print("Figure failures:", png_failures[:10])
if split_overlap_failures:
    print("Split overlap failures:", split_overlap_failures[:10])

all_ok = all(checks.values())
print("=" * 72)
print("VALIDATION STATUS:", "PASS" if all_ok else "FAIL")
print("=" * 72)

if not all_ok:
    raise RuntimeError(
        "Artifact validation failed. The final archive must not be treated as a validated research package."
    )

# Only mark a run complete when every requested model has at least one
# genuine completed TEST observation. A blocked/partial run remains resumable.
model_test_completion = {}
for model_key in ACTIVE_MODELS:
    result = probe_results.get(model_key, pd.DataFrame())
    model_test_completion[model_key] = bool(
        isinstance(result, pd.DataFrame)
        and not result.empty
        and "split" in result.columns
        and "status" in result.columns
        and ((result["split"].astype(str).str.lower() == "test") &
             (result["status"].astype(str).str.upper() == "COMPLETED")).any()
    )

scientifically_complete = all(model_test_completion.values())
if scientifically_complete:
    completion = {
        "run_id": RUN_ID,
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "model_test_completion": model_test_completion,
        "status": "COMPLETED",
    }
    (RUN_ROOT / "run_complete.json").write_text(
        json.dumps(completion, indent=2),
        encoding="utf-8",
    )
    print("Scientific run status: COMPLETED")
else:
    completion_path = RUN_ROOT / "run_complete.json"
    if completion_path.exists():
        completion_path.unlink()
    print("Scientific run status: INCOMPLETE/RESUMABLE")
    print("Per-model completion:", model_test_completion)


# 23. Automatic download — LAST CODE CELL

This cell is kept as the final code cell in the notebook.

It:
1. re-validates the run root,
2. collects tables, figures, predictions, logs, configs, manifest, and README,
3. creates one comprehensive ZIP,
4. triggers a single Colab browser download.



In [ ]:
from google.colab import files
import zipfile

archive_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
FINAL_ZIP_PATH = FINAL_DIR / f"HALP_Bench_Research_Artifacts_{archive_timestamp}.zip"

required_documents = {
    "final_artifacts/manifest.json": manifest_path,
    "final_artifacts/README_RESULTS.md": FINAL_DIR / "README_RESULTS.md",
    "final_artifacts/changes_and_validation.md": FINAL_DIR / "changes_and_validation.md",
}

AUDIT_REPORT_CANDIDATE = FINAL_DIR / "AUDIT_REPORT.md"
if AUDIT_REPORT_CANDIDATE.is_file():
    required_documents["final_artifacts/AUDIT_REPORT.md"] = AUDIT_REPORT_CANDIDATE

SUPERVISED_LABEL_BUNDLE_CANDIDATE = FINAL_DIR / "supervised_endpoint_labels.zip"
SUPERVISED_LABEL_MANIFEST_CANDIDATE = FINAL_DIR / "supervised_endpoint_labels_manifest.json"
if SUPERVISED_LABEL_BUNDLE_CANDIDATE.is_file():
    required_documents["final_artifacts/supervised_endpoint_labels.zip"] = SUPERVISED_LABEL_BUNDLE_CANDIDATE
if SUPERVISED_LABEL_MANIFEST_CANDIDATE.is_file():
    required_documents["final_artifacts/supervised_endpoint_labels_manifest.json"] = SUPERVISED_LABEL_MANIFEST_CANDIDATE

# The browser ZIP is an artifact snapshot.  It does not pretend to be a
# self-contained dataset/model release.  In particular, feature HDF5 files can
# be hundreds of MB and are required to reproduce probe training, so their
# omission must be recorded explicitly rather than being silently invisible.
forbidden_suffixes = {
    ".safetensors", ".bin", ".pt", ".pth", ".ckpt", ".onnx",
    ".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff",
}
forbidden_name_fragments = {"cache", "__pycache__", ".ipynb_checkpoints"}

table_suffixes = {
    ".csv", ".tsv", ".txt", ".json", ".jsonl", ".xlsx", ".xls",
    ".parquet", ".feather", ".h5",
}
image_suffixes = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff", ".svg"}

source_files = []
omitted_large_artifacts = []

for p in sorted(RUN_ROOT.rglob("*")):
    if not p.is_file() or p.suffix.lower() == ".zip":
        continue
    rel = p.relative_to(RUN_ROOT)
    rel_parts = [part.lower() for part in rel.parts]
    lower_parts = set(rel_parts)
    if lower_parts.intersection(forbidden_name_fragments):
        continue
    suffix = p.suffix.lower()

    if suffix in {".h5"}:
        omitted_large_artifacts.append({
            "path": str(rel),
            "size_bytes": p.stat().st_size,
            "reason": "Large feature store required for probe reruns; intentionally omitted from browser snapshot unless explicitly enabled.",
            "how_to_include": "Set INCLUDE_FEATURE_H5_IN_ARCHIVE = True before running this final cell.",
        })
        if not bool(globals().get("INCLUDE_FEATURE_H5_IN_ARCHIVE", False)):
            continue

    is_table = suffix in table_suffixes or "tables" in lower_parts or "logs" in lower_parts
    is_figure = "figures" in lower_parts and suffix in image_suffixes
    is_document = suffix in {".md", ".yaml", ".yml", ".toml", ".ini", ".cfg", ".log"}
    if is_table or is_figure or is_document:
        if suffix in forbidden_suffixes and not is_figure:
            continue
        source_files.append(p)

archive_scope = {
    "archive_type": "research_artifact_snapshot",
    "self_contained_project": False,
    "included_run_artifacts": True,
    "feature_h5_inclusion_enabled": bool(globals().get("INCLUDE_FEATURE_H5_IN_ARCHIVE", False)),
    "omitted_large_artifacts": omitted_large_artifacts,
    "excluded_model_weight_suffixes": [".safetensors", ".bin", ".pt", ".pth", ".ckpt", ".onnx"],
    "excluded_raw_dataset_image_suffixes": [".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"],
    "note": "Use the full dataset, model weights, and feature stores from the configured Drive paths for a true rerun; this ZIP is not a complete benchmark release.",
}
archive_scope_path = FINAL_DIR / "archive_scope.json"
archive_scope_path.write_text(json.dumps(archive_scope, indent=2), encoding="utf-8")
required_documents["final_artifacts/archive_scope.json"] = archive_scope_path

for arcname, src in required_documents.items():
    if src.is_file() and src not in source_files:
        source_files.append(src)

if not source_files:
    raise RuntimeError("No downloadable research artifacts were found.")

with zipfile.ZipFile(FINAL_ZIP_PATH, mode="w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for src in sorted(set(source_files), key=lambda x: str(x)):
        if src.is_relative_to(RUN_ROOT):
            arcname = (Path("run") / src.relative_to(RUN_ROOT)).as_posix()
        else:
            arcname = next((a for a, s in required_documents.items() if s == src), (Path("final_artifacts") / src.name).as_posix())
        zf.write(src, arcname=arcname)

if not FINAL_ZIP_PATH.is_file() or FINAL_ZIP_PATH.stat().st_size <= 0:
    raise RuntimeError("Archive creation failed.")

with zipfile.ZipFile(FINAL_ZIP_PATH, "r") as zf:
    names = zf.namelist()
    corrupt_members = []
    for name in names:
        try:
            with zf.open(name) as fh:
                fh.read(1)
        except Exception as exc:
            corrupt_members.append((name, str(exc)))
    missing_required = [a for a in required_documents if a not in names and required_documents[a].is_file()]

if corrupt_members or missing_required:
    raise RuntimeError(f"Archive validation failed: corrupt={corrupt_members[:5]}, missing={missing_required[:10]}")

print("=" * 78)
print("AUTOMATIC RESEARCH ARTIFACT DOWNLOAD")
print("=" * 78)
print("Archive:", FINAL_ZIP_PATH)
print(f"Archive size: {FINAL_ZIP_PATH.stat().st_size / 2**20:.2f} MiB")
print("Total archive members:", len(names))
print("Explicitly omitted large artifacts:", len(omitted_large_artifacts))
print("Scientific completion marker:", "PRESENT" if (RUN_ROOT / "run_complete.json").is_file() else "ABSENT (snapshot only)")
print("Archive validation: PASS")
print("NOTE: This ZIP is an artifact snapshot, not a self-contained dataset+weights+features release.")

files.download(str(FINAL_ZIP_PATH))